# Tensor Shapes and Manual Backpropagation — 155 Progressive Exercises


> **Working copy:** This file may contain learner answers and scratch work. Use the paired `_virgin.ipynb` notebook whenever you want a clean retry.

This supplementary workbook prepares you for the manual-backpropagation portion of Andrej Karpathy's **Backprop Ninja** lecture. It begins with rank-zero tensors and one-operation graphs, then adds reductions, broadcasting, matrix multiplication, branches, embeddings, stable softmax, BatchNorm, and a complete next-character MLP.

The scaffold contains no official answer key. This working copy may contain learner solutions. Supplied fixtures build forward graphs and private autograd references for each manual derivative.

Exercises 001–015 show small forward inputs for mental arithmetic. Exercises 016–033 also show their forward tensors and upstream gradients. Most use an explicit per-entry pass before the general PyTorch expression; Exercise 022 deliberately uses only the scalable PyTorch pass because its supplied inverse-square-root derivative is too cumbersome for useful hand arithmetic. All exercise inputs, forward operations, named intermediates, outputs, and upstream gradients are visible in notebook cells. Only private expected results remain opaque.

## How to use this notebook

1. Run the test-helper cell once.
2. Work strictly from top to bottom.
3. In Exercises 001–033, write an explicit per-entry numeric expression before the general PyTorch tensor derivation, except in Exercise 022, where you translate a supplied derivative directly into PyTorch; afterward, use scalable tensor derivations.
4. Draw or narrate the local graph before writing a gradient expression.
5. Use only ordinary tensor operations in answer cells—never `.backward()`, `torch.autograd.grad`, or `.grad`.
6. Before computing values, predict every listed input, intermediate, output, upstream-gradient, and backward-gradient shape as a literal Python tuple; then run the supplied test.
7. Move on only after every required variable prints `PASS`.

The tests check shape, dtype, and values without displaying expected tensors. When stuck, ask for one hint about the current exercise rather than requesting the whole chain.


## The six-question backward checklist

For every forward line, ask:

1. What does each axis mean?
2. What are the input and output shapes?
3. Is the operation elementwise, a reduction, a broadcast, an index, or a matrix contraction?

For every backward line, ask:

1. What upstream gradient arrives, and what is its shape?
2. What is this operation's local derivative?
3. Did forward broadcasting create repeated paths that must be summed?
4. Did a forward reduction remove axes that backward must restore?
5. Does the variable appear along more than one path?
6. Have I written a literal Python-tuple shape for every input, intermediate, output, upstream gradient, and backward gradient in this exercise?

A name beginning with `d` means the derivative of the final scalar objective with respect to the named forward tensor. For example, `dprobs` has the same shape as `probs`.


## Course map

- Exercises 001–015: mental calculation plus PyTorch verification for ranks, axes, reductions, reshape, matmul, and lookup
- Exercises 016–033: local scalar and elementwise derivatives with mental and PyTorch passes
- Exercises 034–049: reduction backward
- Exercises 050–067: broadcasting and unbroadcasting
- Exercises 068–085: matrix multiplication and affine layers
- Exercises 086–101: chains, fan-out, and gradient accumulation
- Exercises 102–111: embedding lookups and indexed accumulation
- Exercises 112–127: stable softmax and mean negative log-likelihood
- Exercises 128–141: BatchNorm as atomic operations
- Exercises 142–155: complete next-character MLP capstone


In [1]:
# Supplied test infrastructure: run once after starting or restarting the kernel.
import torch

# Float64 keeps tiny hand-derived examples stable under different operation orders.
DTYPE = torch.float64
_REFS = {}
_MISSING = object()


def _store_forward(key, output):
    """Store a private forward reference without displaying the answer."""
    assert isinstance(output, torch.Tensor)
    _REFS[key] = {
        "out": output.detach().clone(),
        "out_shape": tuple(output.shape),
    }


def _capture(key, output, inputs, upstream=None, retain_graph=False):
    """Use autograd only to build private test references for manual derivations."""
    assert isinstance(output, torch.Tensor)
    if upstream is None:
        assert output.numel() == 1, "A non-scalar output needs an explicit upstream gradient."
        grad_outputs = None
    else:
        assert isinstance(upstream, torch.Tensor)
        assert upstream.shape == output.shape
        grad_outputs = upstream

    names = list(inputs)
    tensors = list(inputs.values())
    gradients = torch.autograd.grad(
        output,
        tensors,
        grad_outputs=grad_outputs,
        retain_graph=retain_graph,
        create_graph=False,
        allow_unused=False,
    )
    reference = {
        "out": output.detach().clone(),
        "out_shape": tuple(output.shape),
    }
    for name, gradient in zip(names, gradients):
        reference[name] = gradient.detach().clone()
    _REFS[key] = reference


def _capture_path(key, field, output, input_tensor, upstream, retain_graph=False):
    """Store one private branch contribution using autograd rather than an answer formula."""
    gradient = torch.autograd.grad(
        output,
        input_tensor,
        grad_outputs=upstream,
        retain_graph=retain_graph,
        create_graph=False,
    )[0]
    _REFS[key][field] = gradient.detach().clone()


def _capture_product_path(key, field, left, right, upstream):
    """Store one operand's local product path on detached test-only leaves."""
    left_local = left.detach().requires_grad_()
    right_local = right.detach().requires_grad_()
    product_local = left_local * right_local
    gradient = torch.autograd.grad(product_local, left_local, grad_outputs=upstream)[0]
    _REFS[key][field] = gradient.detach().clone()


def _store_shape(key, field, tensor):
    """Add an intermediate shape to an existing reference record."""
    assert key in _REFS
    _REFS[key][field] = tuple(tensor.shape)


def _check_tensor(variable_name, key, field):
    """Check type, shape, dtype, and values without revealing expected values."""
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, torch.Tensor), f"`{variable_name}` must be a torch.Tensor."
    expected = _REFS[key][field]
    assert actual.shape == expected.shape, (
        f"`{variable_name}` has shape {tuple(actual.shape)}; "
        f"the gradient/value must have shape {tuple(expected.shape)}."
    )
    assert actual.dtype == expected.dtype, (
        f"`{variable_name}` has dtype {actual.dtype}; expected {expected.dtype}."
    )
    torch.testing.assert_close(actual, expected, rtol=1e-7, atol=1e-9)
    print(f"PASS: {variable_name}")




def _assert_shape_prediction(variable_name, expected):
    """Compare one explicit shape tuple without exposing the expected tuple in tests."""
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, tuple), f"`{variable_name}` must be a Python tuple."
    assert actual == expected, f"`{variable_name}` is {actual}; reconsider the graph shapes."
    print(f"PASS: {variable_name}")


def _check_visible_tensor_shape(variable_name, tensor_name):
    """Check against a supplied or forward tensor already visible in the notebook."""
    tensor = globals().get(tensor_name, _MISSING)
    assert tensor is not _MISSING, f"Visible tensor `{tensor_name}` is not defined."
    assert isinstance(tensor, torch.Tensor), f"`{tensor_name}` must be a torch.Tensor."
    _assert_shape_prediction(variable_name, tuple(tensor.shape))


def _check_private_tensor_shape(variable_name, key, field):
    """Check against a private expected tensor without displaying its shape."""
    _assert_shape_prediction(variable_name, tuple(_REFS[key][field].shape))


def _check_shape(variable_name, key, field="out_shape"):
    """Require an explicit Python shape tuple before checking its value."""
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, tuple), f"`{variable_name}` must be a Python tuple."
    expected = _REFS[key][field]
    assert actual == expected, f"`{variable_name}` is {actual}; reconsider the axes."
    print(f"PASS: {variable_name}")


# Load private reference support used to check the notebook-visible exercises.
import importlib.util as _fixture_importlib
from pathlib import Path as _FixturePath

_fixture_filename = "_tensor_backprop_fixtures.py"
_fixture_relatives = (
    _FixturePath(_fixture_filename),
    _FixturePath("notebooks") / _fixture_filename,
    _FixturePath("coursework/05-backprop-ninja/notebooks") / _fixture_filename,
    _FixturePath("karpathy_ml_course/coursework/05-backprop-ninja/notebooks")
    / _fixture_filename,
)
_fixture_path = next(
    (
        base / relative
        for base in (_FixturePath.cwd(), *_FixturePath.cwd().parents)
        for relative in _fixture_relatives
        if (base / relative).is_file()
    ),
    None,
)
if _fixture_path is None:
    raise FileNotFoundError(
        f"Could not find {_fixture_filename}. Keep it beside the exercise notebook."
    )
_fixture_spec = _fixture_importlib.spec_from_file_location(
    "_tensor_backprop_fixtures", _fixture_path
)
assert _fixture_spec is not None and _fixture_spec.loader is not None
_fixture_module = _fixture_importlib.module_from_spec(_fixture_spec)
_fixture_spec.loader.exec_module(_fixture_module)
_run_fixture = _fixture_module.run_fixture
_run_forward_reference = _fixture_module.run_forward_reference
_run_gradient_reference = _fixture_module.run_gradient_reference


print("Test helpers ready. Start at Exercise 001 and do not use autograd in answer cells.")


Test helpers ready. Start at Exercise 001 and do not use autograd in answer cells.


## 1. Mental tensor calculations and PyTorch verification

Before differentiating, learn to narrate every axis and calculate small tensor operations by hand. Exercises 001–015 use two independent passes: first write the final tensor values literally from mental calculation, then compute the same result with the requested PyTorch operator. These first tasks contain no gradients.

Use this shape ledger:

1. Write each input shape.
2. Name what every axis represents.
3. Apply the operation's shape rule.
4. Check that the number of elements is preserved when reshaping.

A scalar tensor has shape `()`. A vector has one axis. A matrix has two axes.


### Two-pass rule for Exercises 001–015

Each exercise provides small input values visibly in Markdown.

#### Pass 1 — explicit per-entry calculation

Write the result as a literal tensor, but feel free to leave arithmetic expressions unsimplified when they communicate the operation more clearly:

```python
exNNN_manual = torch.tensor(
    [visible_number * scalar, visible_number * scalar],
    dtype=DTYPE,
)
```

Use only visible numeric values in this expression; do not refer to supplied tensor variables such as `x`, `y`, `left`, `right`, `table`, or `ids`. Python evaluates these scalar expressions before constructing the tensor. This pass exposes how each output entry is calculated and checks signs, factors, and index alignment without hiding the work behind a tensor operation.

Both forms are accepted:

```python
# Expressive, unsimplified arithmetic.
exNNN_manual = torch.tensor([2.0 * 3.0, -1.0 * 3.0], dtype=DTYPE)

# Fully simplified values.
exNNN_manual = torch.tensor([6.0, -3.0], dtype=DTYPE)
```

#### Pass 2 — scalable PyTorch expression

Independently perform the requested operation with the supplied tensor variables:

```python
exNNN_out = x * scale
```

This second form demonstrates PyTorch shape semantics. Same-shaped tensors combine elementwise; scalars and compatible size-one axes broadcast; reductions or matrix multiplications follow their own shape rules. The explicit first pass and scalable second pass are tested independently.


### Exercise 001 — Scalar plus constant

**Purpose:** Learn that adding a number to a scalar tensor still produces a scalar tensor.

**Inputs:** `x` is a scalar tensor with shape `()`.

**Visible values for the mental pass:**

```python
x = torch.tensor(2.0, dtype=DTYPE)
```

**Operation:** Add the scalar value `3.0` to the scalar tensor `x`.

**Two required passes:** First write `ex001_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex001_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** Tensor addition with the `+` operator.

**Required outputs:**

- `ex001_manual`: the scalar result written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex001_out`: the same scalar result computed with PyTorch addition


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex001_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex001_out_shape`: predict the shape of `ex001_out` as a literal Python tuple

**Next concept:** Vector times scalar.


In [2]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor(2.0, dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(1, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [3]:
# Exercise 001: complete both the mental and PyTorch passes.
# Explicit pass: `ex001_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex001_out` with the requested operator.
# Define `ex001_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex001_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex001_out_shape` — the shape of `ex001_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex001_x_shape = ()
ex001_out_shape = ()
print(tuple(x.shape))
ex001_manual = torch.tensor(2.0 + 3.0, dtype=DTYPE)
ex001_out = x + 3.0


()


In [4]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex001_x_shape", "x")
_check_private_tensor_shape("ex001_out_shape", "ex001", "out")
_check_tensor("ex001_manual", "ex001", "out")
_check_tensor("ex001_out", "ex001", "out")


PASS: ex001_x_shape
PASS: ex001_out_shape
PASS: ex001_manual
PASS: ex001_out


### Exercise 002 — Vector times scalar

**Purpose:** Learn that multiplying a vector by one number multiplies every element and keeps the vector's shape unchanged.

**Inputs:** `x` has shape `(4,)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([1.0, -2.0, 3.0, 4.0], dtype=DTYPE)
```

**Operation:** Multiply every element of `x` by the scalar value `2.5`.

**Two required passes:** First write `ex002_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex002_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** Elementwise multiplication with `*` and scalar broadcasting.

**Required outputs:**

- `ex002_manual`: the resulting vector values written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex002_out`: the same scaled vector computed with PyTorch multiplication


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex002_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex002_out_shape`: predict the shape of `ex002_out` as a literal Python tuple

**Next concept:** Elementwise vector addition.


In [5]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([1.0, -2.0, 3.0, 4.0], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(2, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [6]:
# Exercise 002: complete both the mental and PyTorch passes.
# Explicit pass: `ex002_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex002_out` with the requested operator.
# Define `ex002_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex002_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex002_out_shape` — the shape of `ex002_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex002_x_shape = (4,)
ex002_out_shape = (4,)
ex002_out = x * 2.5
ex002_manual = torch.tensor([2.5, -5.0, 7.5, 10.0], dtype=DTYPE)

In [7]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex002_x_shape", "x")
_check_private_tensor_shape("ex002_out_shape", "ex002", "out")
_check_tensor("ex002_manual", "ex002", "out")
_check_tensor("ex002_out", "ex002", "out")


PASS: ex002_x_shape
PASS: ex002_out_shape
PASS: ex002_manual
PASS: ex002_out


### Exercise 003 — Elementwise vector addition

**Purpose:** Learn that adding two vectors adds entries at matching positions: first with first, second with second, and so on.

**Inputs:** `x` and `y` both have shape `(4,)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
y = torch.tensor([-1.0, 0.5, 2.0, 3.0], dtype=DTYPE)
```

**Operation:** Add `x` and `y` position by position.

**Two required passes:** First write `ex003_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex003_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** Elementwise tensor addition with `+`.

**Required outputs:**

- `ex003_manual`: the resulting vector values written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex003_out`: the same elementwise sum computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex003_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex003_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex003_out_shape`: predict the shape of `ex003_out` as a literal Python tuple

**Next concept:** Elementwise vector multiplication.


In [8]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
y = torch.tensor([-1.0, 0.5, 2.0, 3.0], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(3, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [9]:
# Exercise 003: complete both the mental and PyTorch passes.
# Explicit pass: `ex003_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex003_out` with the requested operator.
# Define `ex003_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex003_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex003_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex003_out_shape` — the shape of `ex003_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex003_x_shape = (4,)
ex003_y_shape = (4,)
ex003_out_shape = (4,)
ex003_out = x + y
ex003_manual = torch.tensor([0.0, 2.5, 5, 7], dtype=DTYPE)

In [10]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex003_x_shape", "x")
_check_visible_tensor_shape("ex003_y_shape", "y")
_check_private_tensor_shape("ex003_out_shape", "ex003", "out")
_check_tensor("ex003_manual", "ex003", "out")
_check_tensor("ex003_out", "ex003", "out")


PASS: ex003_x_shape
PASS: ex003_y_shape
PASS: ex003_out_shape
PASS: ex003_manual
PASS: ex003_out


### Exercise 004 — Elementwise vector multiplication

**Purpose:** Learn that `*` multiplies matching vector entries and keeps them separate instead of combining them into one number.

**Inputs:** `x` and `y` both have shape `(3,)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([2.0, -1.0, 4.0], dtype=DTYPE)
y = torch.tensor([3.0, 5.0, -2.0], dtype=DTYPE)
```

**Operation:** Multiply `x` and `y` position by position without combining the products into a total.

**Two required passes:** First write `ex004_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex004_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** Elementwise multiplication with `*`; do not use a dot product.

**Required outputs:**

- `ex004_manual`: the resulting vector values written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex004_out`: the same elementwise product computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex004_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex004_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex004_out_shape`: predict the shape of `ex004_out` as a literal Python tuple

**Next concept:** Matrix plus scalar.


In [11]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([2.0, -1.0, 4.0], dtype=DTYPE)
y = torch.tensor([3.0, 5.0, -2.0], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(4, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [12]:
# Exercise 004: complete both the mental and PyTorch passes.
# Explicit pass: `ex004_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex004_out` with the requested operator.
# Define `ex004_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex004_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex004_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex004_out_shape` — the shape of `ex004_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex004_x_shape = (3,)
ex004_y_shape = (3,)
ex004_out_shape = (3,)
ex004_out = x * y
ex004_manual = torch.tensor([6.0, -5.0, -8.0], dtype=DTYPE)

In [13]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex004_x_shape", "x")
_check_visible_tensor_shape("ex004_y_shape", "y")
_check_private_tensor_shape("ex004_out_shape", "ex004", "out")
_check_tensor("ex004_manual", "ex004", "out")
_check_tensor("ex004_out", "ex004", "out")


PASS: ex004_x_shape
PASS: ex004_y_shape
PASS: ex004_out_shape
PASS: ex004_manual
PASS: ex004_out


### Exercise 005 — Matrix plus scalar

**Purpose:** Learn that adding one number to a matrix adds that number to every matrix entry.

**Inputs:** `x` has shape `(2, 3)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [0.0, 1.0, 2.0],
    [3.0, 4.0, 5.0],
], dtype=DTYPE)
```

**Operation:** Add `10.0` to every entry of `x`.

**Two required passes:** First write `ex005_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex005_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** Tensor addition with a broadcast scalar.

**Required outputs:**

- `ex005_manual`: the resulting matrix values written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex005_out`: the same shifted matrix computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex005_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex005_out_shape`: predict the shape of `ex005_out` as a literal Python tuple

**Next concept:** Elementwise matrix product.


In [14]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [0.0, 1.0, 2.0],
    [3.0, 4.0, 5.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(5, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [15]:
# Exercise 005: complete both the mental and PyTorch passes.
# Explicit pass: `ex005_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex005_out` with the requested operator.
# Define `ex005_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex005_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex005_out_shape` — the shape of `ex005_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex005_x_shape = (2, 3)
ex005_out_shape = (2, 3)
ex005_out = x + 10
ex005_manual = torch.tensor([
    [10.0, 11.0, 12.0],
    [13.0, 14.0, 15.0]
], dtype=DTYPE)

In [16]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex005_x_shape", "x")
_check_private_tensor_shape("ex005_out_shape", "ex005", "out")
_check_tensor("ex005_manual", "ex005", "out")
_check_tensor("ex005_out", "ex005", "out")


PASS: ex005_x_shape
PASS: ex005_out_shape
PASS: ex005_manual
PASS: ex005_out


### Exercise 006 — Elementwise matrix product

**Purpose:** Learn that `*` multiplies entries at matching row-column positions and keeps the matrix shape unchanged.

**Inputs:** `x` and `y` have shape `(2, 3)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
y = torch.tensor([
    [1.0, 0.0, -1.0],
    [2.0, 3.0, 4.0],
], dtype=DTYPE)
```

**Operation:** Multiply entries of `x` and `y` at matching row-column positions without reducing any axis.

**Two required passes:** First write `ex006_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex006_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** Elementwise multiplication with `*`; matrix multiplication is a different operation.

**Required outputs:**

- `ex006_manual`: the resulting matrix values written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex006_out`: the same elementwise matrix product computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex006_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex006_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex006_out_shape`: predict the shape of `ex006_out` as a literal Python tuple

**Next concept:** Reduce every axis.


In [17]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
y = torch.tensor([
    [1.0, 0.0, -1.0],
    [2.0, 3.0, 4.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(6, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [18]:
# Exercise 006: complete both the mental and PyTorch passes.
# Explicit pass: `ex006_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex006_out` with the requested operator.
# Define `ex006_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex006_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex006_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex006_out_shape` — the shape of `ex006_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex006_x_shape = (2, 3)
ex006_y_shape = (2, 3)
ex006_out_shape = (2, 3)
ex006_out = x * y
ex006_manual = torch.tensor([
    [1.0, 0.0, -3.0],
    [8.0, 15.0, 24.0]
], dtype=DTYPE)

In [19]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex006_x_shape", "x")
_check_visible_tensor_shape("ex006_y_shape", "y")
_check_private_tensor_shape("ex006_out_shape", "ex006", "out")
_check_tensor("ex006_manual", "ex006", "out")
_check_tensor("ex006_out", "ex006", "out")


PASS: ex006_x_shape
PASS: ex006_y_shape
PASS: ex006_out_shape
PASS: ex006_manual
PASS: ex006_out


### Exercise 007 — Reduce every axis

**Purpose:** Learn that summing every entry of a matrix produces one scalar tensor.

**Inputs:** `x` has shape `(2, 3)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
```

**Operation:** Add all six entries of `x` into one total.

**Two required passes:** First write `ex007_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex007_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** The `Tensor.sum` reduction. Decide whether any dimension argument is needed.

**Required outputs:**

- `ex007_manual`: the final total written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex007_out`: the same total computed with a PyTorch reduction


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex007_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex007_out_shape`: predict the shape of `ex007_out` as a literal Python tuple

**Next concept:** Reduce rows with dim 0.


In [20]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(7, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [21]:
# Exercise 007: complete both the mental and PyTorch passes.
# Explicit pass: `ex007_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex007_out` with the requested operator.
# Define `ex007_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex007_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex007_out_shape` — the shape of `ex007_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex007_x_shape = (2, 3)
ex007_out_shape = ()
ex007_manual = torch.tensor(21.0, dtype=DTYPE)
ex007_out = x.sum()

In [22]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex007_x_shape", "x")
_check_private_tensor_shape("ex007_out_shape", "ex007", "out")
_check_tensor("ex007_manual", "ex007", "out")
_check_tensor("ex007_out", "ex007", "out")


PASS: ex007_x_shape
PASS: ex007_out_shape
PASS: ex007_manual
PASS: ex007_out


### Exercise 008 — Reduce rows with dim 0

**Purpose:** Learn that `dim=0` combines the rows and leaves one result for each column.

**Inputs:** `x` has shape `(2, 3)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
```

**Operation:** Add downward through the rows so that each column gets its own total.

**Two required passes:** First write `ex008_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex008_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** The `Tensor.sum` reduction and the fact that rows are dimension 0.

**Required outputs:**

- `ex008_manual`: the column totals written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex008_out`: the same column totals computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex008_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex008_out_shape`: predict the shape of `ex008_out` as a literal Python tuple

**Next concept:** Reduce columns with dim 1.


In [23]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(8, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [24]:
# Exercise 008: complete both the mental and PyTorch passes.
# Explicit pass: `ex008_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex008_out` with the requested operator.
# Define `ex008_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex008_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex008_out_shape` — the shape of `ex008_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex008_x_shape = (2, 3)
ex008_out_shape = (3,)
ex008_manual = torch.tensor([5.0, 7.0, 9.0], dtype=DTYPE)
ex008_out = x.sum(dim=0)

In [25]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex008_x_shape", "x")
_check_private_tensor_shape("ex008_out_shape", "ex008", "out")
_check_tensor("ex008_manual", "ex008", "out")
_check_tensor("ex008_out", "ex008", "out")


PASS: ex008_x_shape
PASS: ex008_out_shape
PASS: ex008_manual
PASS: ex008_out


### Exercise 009 — Reduce columns with dim 1

**Purpose:** Learn that `dim=1` combines the columns and leaves one result for each row.

**Inputs:** `x` has shape `(2, 3)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
```

**Operation:** Add across the columns so that each row gets its own total.

**Two required passes:** First write `ex009_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex009_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** The `Tensor.sum` reduction and the fact that columns are dimension 1.

**Required outputs:**

- `ex009_manual`: the row totals written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex009_out`: the same row totals computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex009_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex009_out_shape`: predict the shape of `ex009_out` as a literal Python tuple

**Next concept:** Keep a reduced axis.


In [26]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(9, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [27]:
# Exercise 009: complete both the mental and PyTorch passes.
# Explicit pass: `ex009_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex009_out` with the requested operator.
# Define `ex009_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex009_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex009_out_shape` — the shape of `ex009_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex009_x_shape = (2, 3)
ex009_out_shape = (2,)
ex009_manual = torch.tensor([6.0, 15.0], dtype=DTYPE)
ex009_out = x.sum(dim=1)

In [28]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex009_x_shape", "x")
_check_private_tensor_shape("ex009_out_shape", "ex009", "out")
_check_tensor("ex009_manual", "ex009", "out")
_check_tensor("ex009_out", "ex009", "out")


PASS: ex009_x_shape
PASS: ex009_out_shape
PASS: ex009_manual
PASS: ex009_out


### Exercise 010 — Sum across columns and keep axis 1

**Purpose:** Learn that `keepdim=True` leaves a reduced axis in the result with length 1.

**Inputs:** `x` has shape `(2, 3)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
```

**Operation:** For each row, add its three column values. This reduces the column axis (axis 1), leaves one total per row, and keeps axis 1 present with length 1.

**Two required passes:** First write `ex010_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex010_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** `Tensor.sum`; reduce axis 1 (the column axis) and use the `keepdim` option. Assemble the PyTorch expression yourself.

**Required outputs:**

- `ex010_manual`: the row totals with the retained axis written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex010_out`: the same rank-preserving row reduction computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex010_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex010_out_shape`: predict the shape of `ex010_out` as a literal Python tuple

**Next concept:** Sum across rows and keep axis 0.


In [29]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(10, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [30]:
# Exercise 010: complete both the mental and PyTorch passes.
# Explicit pass: `ex010_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex010_out` with the requested operator.
# Define `ex010_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex010_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex010_out_shape` — the shape of `ex010_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex010_x_shape = (2, 3)
ex010_out_shape = (2, 1)
ex010_manual = torch.tensor([[6.0], [15.0]], dtype=DTYPE)
ex010_out = x.sum(dim=1, keepdim=True)

In [31]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex010_x_shape", "x")
_check_private_tensor_shape("ex010_out_shape", "ex010", "out")
_check_tensor("ex010_manual", "ex010", "out")
_check_tensor("ex010_out", "ex010", "out")


PASS: ex010_x_shape
PASS: ex010_out_shape
PASS: ex010_manual
PASS: ex010_out


### Exercise 011 — Sum across rows and keep axis 0

**Purpose:** Learn that `keepdim=True` can preserve axis 0 after reducing the rows.

**Inputs:** `x` has shape `(2, 3)`.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)
```

**Operation:** For each column, add its two row values. This reduces the row axis (axis 0), leaves one total per column, and keeps axis 0 present with length 1.

**Two required passes:** First write `ex011_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex011_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** `Tensor.sum`; reduce axis 0 (the row axis) and use the `keepdim` option. Assemble the PyTorch expression yourself.

**Required outputs:**

- `ex011_manual`: the column totals with the retained axis written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to the supplied input variable
- `ex011_out`: the same rank-preserving column-total reduction computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex011_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex011_out_shape`: predict the shape of `ex011_out` as a literal Python tuple

**Next concept:** Reshape without changing element count.


In [32]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(11, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [33]:
# Exercise 011: complete both the mental and PyTorch passes.
# Explicit pass: `ex011_manual` may use visible-number arithmetic or simplified values.
# Do not use the supplied input variable or operations in the manual expression.
# PyTorch pass: define `ex011_out` with the requested operator.
# Predict every visible tensor shape before computing values.
# Define `ex011_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex011_out_shape` — the shape of `ex011_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex011_x_shape = (2, 3)
ex011_out_shape = (1, 3)
ex011_manual = torch.tensor([[5.0, 7.0, 9.0]], dtype=DTYPE)
ex011_out = x.sum(dim=0, keepdim=True)

In [34]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex011_x_shape", "x")
_check_private_tensor_shape("ex011_out_shape", "ex011", "out")
_check_tensor("ex011_manual", "ex011", "out")
_check_tensor("ex011_out", "ex011", "out")


PASS: ex011_x_shape
PASS: ex011_out_shape
PASS: ex011_manual
PASS: ex011_out


### Exercise 012 — Reshape without changing element count

**Purpose:** Learn that reshaping changes how dimensions are grouped without changing the values or their total count.

**Inputs:** `x` has shape `(2, 3)` and six elements.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [0.0, 1.0, 2.0],
    [3.0, 4.0, 5.0],
], dtype=DTYPE)
```

**Operation:** Regroup the six values of `x` into three rows with two values per row without changing their order.

**Two required passes:** First write `ex012_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex012_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** `Tensor.reshape`; reshaping must preserve the total number of elements.

**Required outputs:**

- `ex012_manual`: the regrouped values written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex012_out`: the same regrouping computed with PyTorch reshape


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex012_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex012_out_shape`: predict the shape of `ex012_out` as a literal Python tuple

**Next concept:** Flatten non-batch axes.


In [35]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [0.0, 1.0, 2.0],
    [3.0, 4.0, 5.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(12, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [36]:
# Exercise 012: complete both the mental and PyTorch passes.
# Explicit pass: `ex012_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex012_out` with the requested operator.
# Define `ex012_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex012_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex012_out_shape` — the shape of `ex012_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex012_x_shape = (2, 3)
ex012_out_shape = (3, 2)
ex012_manual = torch.tensor([
    [0.0, 1.0],
    [2.0, 3.0],
    [4.0, 5.0],
], dtype=DTYPE)
ex012_out = x.reshape(3, 2)

In [37]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex012_x_shape", "x")
_check_private_tensor_shape("ex012_out_shape", "ex012", "out")
_check_tensor("ex012_manual", "ex012", "out")
_check_tensor("ex012_out", "ex012", "out")


PASS: ex012_x_shape
PASS: ex012_out_shape
PASS: ex012_manual
PASS: ex012_out


### Exercise 013 — Flatten non-batch axes

**Purpose:** Learn how to keep batch examples separate while joining each example's remaining dimensions into one vector.

**Inputs:** `x` has shape `(2, 3, 2)`: two examples, three positions, two features.

**Visible values for the mental pass:**

```python
x = torch.tensor([
    [[0.0, 1.0], [2.0, 3.0], [4.0, 5.0]],
    [[6.0, 7.0], [8.0, 9.0], [10.0, 11.0]],
], dtype=DTYPE)
```

**Operation:** Keep the first axis as the batch axis and combine each example's remaining two axes into one axis.

**Two required passes:** First write `ex013_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex013_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** `Tensor.reshape`, the batch size from `x.shape`, and one inferred dimension written as `-1`.

**Required outputs:**

- `ex013_manual`: the flattened values for both examples written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex013_out`: the same per-example flattening computed with PyTorch reshape


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex013_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex013_out_shape`: predict the shape of `ex013_out` as a literal Python tuple

**Next concept:** Matrix multiplication shape.


In [38]:
# Supplied inputs are visible so you can calculate the result mentally.
x = torch.tensor([
    [[0.0, 1.0], [2.0, 3.0], [4.0, 5.0]],
    [[6.0, 7.0], [8.0, 9.0], [10.0, 11.0]],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(13, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [39]:
# Exercise 013: complete both the mental and PyTorch passes.
# Explicit pass: `ex013_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex013_out` with the requested operator.
# Define `ex013_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex013_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex013_out_shape` — the shape of `ex013_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex013_x_shape = (2, 3, 2)
ex013_out_shape = (2, 6)
ex013_manual = torch.tensor([
    [0.0, 1.0, 2.0, 3.0, 4.0, 5.0],
    [6.0, 7.0, 8.0, 9.0, 10.0, 11.0]
], dtype=DTYPE)
ex013_out = x.reshape(x.shape[0], -1)

In [40]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex013_x_shape", "x")
_check_private_tensor_shape("ex013_out_shape", "ex013", "out")
_check_tensor("ex013_manual", "ex013", "out")
_check_tensor("ex013_out", "ex013", "out")


PASS: ex013_x_shape
PASS: ex013_out_shape
PASS: ex013_manual
PASS: ex013_out


### Exercise 014 — Matrix multiplication shape

**Purpose:** Learn how the two outside dimensions determine a matrix product's output shape after the shared inside dimension is combined.

**Inputs:** `left` has shape `(3, 4)` and `right` has shape `(4, 2)`.

**Visible values for the mental pass:**

```python
left = torch.tensor([
    [0.0, 1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0, 7.0],
    [8.0, 9.0, 10.0, 11.0],
], dtype=DTYPE)
right = torch.tensor([
    [0.0, 1.0],
    [2.0, 3.0],
    [4.0, 5.0],
    [6.0, 7.0],
], dtype=DTYPE)
```

**Operation:** Compute the matrix product of `left` and `right`.

**Two required passes:** First write `ex014_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex014_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** The matrix-multiplication operator `@`; pair each output position with one row and one column.

**Required outputs:**

- `ex014_manual`: the matrix-product values written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex014_out`: the same matrix product computed with PyTorch


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex014_left_shape`: predict the shape of `left` as a literal Python tuple
- `ex014_right_shape`: predict the shape of `right` as a literal Python tuple
- `ex014_out_shape`: predict the shape of `ex014_out` as a literal Python tuple

**Next concept:** Embedding-table lookup shape.


In [41]:
# Supplied inputs are visible so you can calculate the result mentally.
left = torch.tensor([
    [0.0, 1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0, 7.0],
    [8.0, 9.0, 10.0, 11.0],
], dtype=DTYPE)
right = torch.tensor([
    [0.0, 1.0],
    [2.0, 3.0],
    [4.0, 5.0],
    [6.0, 7.0],
], dtype=DTYPE)

# Register the private expected result without displaying its calculation.
_run_forward_reference(14, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [42]:
# Exercise 014: complete both the mental and PyTorch passes.
# Explicit pass: `ex014_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex014_out` with the requested operator.
# Define `ex014_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex014_left_shape` — the shape of `left` as a literal Python tuple.
# Define `ex014_right_shape` — the shape of `right` as a literal Python tuple.
# Define `ex014_out_shape` — the shape of `ex014_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex014_left_shape = (3, 4)
ex014_right_shape = (4, 2)
ex014_out_shape = (3, 2)
ex014_manual = torch.tensor([
    [0.0 + 2.0 + 8.0 + 18.0, 0.0 + 3.0 + 10.0 + 21.0],
    [0.0 + 10.0 + 24.0 + 42.0, 4.0 + 15.0 + 30.0 + 49.0],
    [0.0 + 18.0 + 40.0 + 66.0, 8.0 + 27.0 + 50.0 + 77.0],
], dtype=DTYPE)
ex014_out = left @ right

In [43]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex014_left_shape", "left")
_check_visible_tensor_shape("ex014_right_shape", "right")
_check_private_tensor_shape("ex014_out_shape", "ex014", "out")
_check_tensor("ex014_manual", "ex014", "out")
_check_tensor("ex014_out", "ex014", "out")


PASS: ex014_left_shape
PASS: ex014_right_shape
PASS: ex014_out_shape
PASS: ex014_manual
PASS: ex014_out


### Exercise 015 — Embedding-table lookup shape

**Purpose:** Learn that each integer ID is replaced by the corresponding row from an embedding table.

**Inputs:** `table` has shape `(5, 3)` and `ids` has shape `(2, 2)`.

**Visible values for the mental pass:**

```python
table = torch.tensor([
    [0.0, 1.0, 2.0],
    [3.0, 4.0, 5.0],
    [6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0],
    [12.0, 13.0, 14.0],
], dtype=DTYPE)
ids = torch.tensor([
    [2, 0],
    [4, 1],
], dtype=torch.long)
```

**Operation:** Replace every integer in `ids` with the corresponding row from `table`.

**Two required passes:** First write `ex015_manual` as a literal tensor containing either simplified values or explicit arithmetic made from the visible numeric values. Then compute `ex015_out` with the supplied PyTorch variables. Also predict the output shape.

**Ingredients:** Advanced integer indexing into the table's first axis.

**Required outputs:**

- `ex015_manual`: the selected embedding vectors written directly as a literal tensor; numeric arithmetic is allowed, but do not refer to supplied input variables
- `ex015_out`: the same embedding lookup computed with PyTorch indexing


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex015_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex015_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex015_out_shape`: predict the shape of `ex015_out` as a literal Python tuple

**Next concept:** Add a constant backward.


In [44]:
# Supplied inputs are visible so you can calculate the result mentally.
table = torch.tensor([
    [0.0, 1.0, 2.0],
    [3.0, 4.0, 5.0],
    [6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0],
    [12.0, 13.0, 14.0],
], dtype=DTYPE)
ids = torch.tensor([
    [2, 0],
    [4, 1],
], dtype=torch.long)

# Register the private expected result without displaying its calculation.
_run_forward_reference(15, globals())
print("Visible inputs ready.")


Visible inputs ready.


In [45]:
# Exercise 015: complete both the mental and PyTorch passes.
# Explicit pass: `ex015_manual` may use visible-number arithmetic or simplified values.
# Numeric arithmetic is allowed; do not use supplied input variable names.
# PyTorch pass: define `ex015_out` with the requested operator.
# Define `ex015_out` — compute the result with the requested PyTorch operator.
# Predict every visible tensor shape before computing values.
# Define `ex015_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex015_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex015_out_shape` — the shape of `ex015_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex015_table_shape = (5, 3)
ex015_ids_shape = (2, 2)
ex015_out_shape = (2, 2, 3)
ex015_manual = torch.tensor([
    [
        [6.0, 7.0, 8.0],
        [0.0, 1.0, 2.0],
    ],
    [
        [12.0, 13.0, 14.0],
        [3.0, 4.0, 5.0],
    ]
], dtype=DTYPE)
ex015_out = table[ids]

In [46]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex015_table_shape", "table")
_check_visible_tensor_shape("ex015_ids_shape", "ids")
_check_private_tensor_shape("ex015_out_shape", "ex015", "out")
_check_tensor("ex015_manual", "ex015", "out")
_check_tensor("ex015_out", "ex015", "out")


PASS: ex015_table_shape
PASS: ex015_ids_shape
PASS: ex015_out_shape
PASS: ex015_manual
PASS: ex015_out


## 2. Local derivatives with scalar and elementwise tensor operations

Now every supplied forward output receives an upstream gradient called `dout`. Derive gradients manually; do not call `.backward()`, `torch.autograd.grad`, or read `.grad`.

### What `dout` means

For a local forward operation that maps an input tensor `x` to an output tensor `y`, `dout` is the upstream gradient arriving from the rest of the graph:

$$
\mathrm{dout}
=
\frac{\partial L}{\partial y}
$$

Here, `L` is the final scalar loss and `y` is this local operation's output. Therefore, `dout` has the same shape as `y`. It tells you how a small change in every element of `y` would affect `L`.

To continue backward to `x`, combine `dout` with the local derivative of `y` with respect to `x`:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
$$

In the exercises, the resulting tensor is named `dx`. Although `dout` comes from an operation later in the forward graph, that later operation is processed earlier while moving backward from `L`.

For an elementwise operation, each output position initially depends on the corresponding input position. The chain rule is

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
$$

Here, `L` is the final scalar objective, `y` is this exercise's forward output, `x` is an input, and `dout` represents the first factor. The resulting gradient for an input must have exactly the same shape as that input.


### Two-pass backward rule for Exercises 016–033, except Exercise 022

The tensors are deliberately small enough to show every chain-rule factor explicitly.

1. **Explicit per-entry pass:** write each requested `_manual` gradient as a literal tensor. Each entry may be a fully simplified number or an unsimplified arithmetic expression made from the visible numeric values. Keep the upstream factor first when possible, followed by the local derivative factors. Do not refer to supplied tensor variable names in this pass.
2. **Scalable PyTorch pass:** derive the same gradient with the supplied tensors such as `dout`, `x`, and other visible forward values. This pass demonstrates elementwise operations and, where shapes differ compatibly, actual broadcasting.

For example, both of these manual entries are acceptable representations of the same mental calculation:

```python
# Expressive chain-rule factors.
manual = torch.tensor([0.25 * 3.0 * (-2.0) ** 2], dtype=DTYPE)

# Fully simplified value.
manual = torch.tensor([3.0], dtype=DTYPE)
```

The first form is often more informative because it preserves `upstream × local derivative`. Remember that arithmetic inside the literal tensor is scalar Python arithmetic; the scalable tensor pass is what demonstrates PyTorch vectorization and broadcasting across shapes.

**Exercise 022 is the deliberate exception.** Its inverse-square-root derivative is supplied in LaTeX, and the required work is only to translate that formula into one scalable PyTorch expression. It does not require an `_manual` tensor because evaluating the nested reciprocal and square-root arithmetic by hand would distract from the BatchNorm connection.

**Exercises 025 and 026 allow saved-output indexing in the explicit pass.** Their `_manual` tensors may use `y[i]` so each entry visibly multiplies its upstream value by the local derivative. This preserves the chain-rule factors without requiring hand evaluation of `tanh` or sigmoid exponentials; the second pass still expresses the same calculation as one scalable tensor operation.

The two variables are tested independently. After Exercise 033, gradients become larger and involve reductions or broadcasting, so the notebook keeps the same derivation principles but uses scalable tensor expressions instead of long per-entry lists.


### Exercise 016 — Add a constant backward

**Purpose:** Learn that adding a constant lets the incoming gradient pass back to `x` unchanged.

**Inputs:** `x` and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x + 4.0
dout = torch.tensor([0.2, -1.0, 3.0], dtype=DTYPE)
```

**Forward operation:** `y = x + 4.0`.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The local slope of adding a constant and the incoming `dout`.

**Required outputs:**

- `ex016_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex016_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex016_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex016_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex016_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex016_dx_shape`: predict the shape of `ex016_dx` as a literal Python tuple

**Next concept:** Multiply by a constant backward.


In [47]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x + 4.0
dout = torch.tensor([0.2, -1.0, 3.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(16, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [48]:
# Exercise 016: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex016_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex016_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex016_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex016_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex016_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex016_dx_shape` — the shape of `ex016_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
# Chain rule convention: write upstream dL/dy (`dout`) before local dy/dx.
ex016_x_shape = (3,)
ex016_y_shape = (3,)
ex016_dout_shape = (3,)
ex016_dx_shape = (3,)
ex016_dx_manual = torch.tensor([0.2 * 1.0, -1.0 * 1.0, 3.0 * 1.0], dtype=DTYPE)
ex016_dx = dout * 1

In [49]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex016_x_shape", "x")
_check_visible_tensor_shape("ex016_y_shape", "y")
_check_visible_tensor_shape("ex016_dout_shape", "dout")
_check_private_tensor_shape("ex016_dx_shape", "ex016", "dx")
_check_tensor("ex016_dx_manual", "ex016", "dx")
_check_tensor("ex016_dx", "ex016", "dx")


PASS: ex016_x_shape
PASS: ex016_y_shape
PASS: ex016_dout_shape
PASS: ex016_dx_shape
PASS: ex016_dx_manual
PASS: ex016_dx


### Exercise 017 — Multiply by a constant backward

**Purpose:** Learn that multiplying by a constant also multiplies the incoming gradient by that constant.

**Inputs:** `x` and `dout` have shape `(4,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([1.0, -2.0, 0.5, 4.0], dtype=DTYPE, requires_grad=True)
y = -3.0 * x
dout = torch.tensor([1.0, 2.0, -1.0, 0.25], dtype=DTYPE)
```

**Forward operation:** `y = -3.0 * x`.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The local derivative of multiplication by a fixed scalar.

**Required outputs:**

- `ex017_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex017_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex017_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex017_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex017_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex017_dx_shape`: predict the shape of `ex017_dx` as a literal Python tuple

**Next concept:** Negation backward.


In [50]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([1.0, -2.0, 0.5, 4.0], dtype=DTYPE, requires_grad=True)
y = -3.0 * x
dout = torch.tensor([1.0, 2.0, -1.0, 0.25], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(17, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [51]:
# Exercise 017: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex017_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex017_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex017_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex017_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex017_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex017_dx_shape` — the shape of `ex017_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
# Chain rule convention: write upstream dL/dy (`dout`) before local dy/dx.
ex017_x_shape = (4,)
ex017_y_shape = (4,)
ex017_dout_shape = (4,)
ex017_dx_shape = (4,)
ex017_dx_manual = torch.tensor([-3.0, -6.0, 3.0, -0.75], dtype=DTYPE)
ex017_dx = dout * -3.0

In [52]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex017_x_shape", "x")
_check_visible_tensor_shape("ex017_y_shape", "y")
_check_visible_tensor_shape("ex017_dout_shape", "dout")
_check_private_tensor_shape("ex017_dx_shape", "ex017", "dx")
_check_tensor("ex017_dx_manual", "ex017", "dx")
_check_tensor("ex017_dx", "ex017", "dx")


PASS: ex017_x_shape
PASS: ex017_y_shape
PASS: ex017_dout_shape
PASS: ex017_dx_shape
PASS: ex017_dx_manual
PASS: ex017_dx


### Exercise 018 — Negation backward

**Purpose:** Learn that a minus sign reverses the sign of the incoming gradient.

**Inputs:** `x` and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([2.0, -1.0, 5.0], dtype=DTYPE, requires_grad=True)
y = -x
dout = torch.tensor([0.5, 2.0, -3.0], dtype=DTYPE)
```

**Forward operation:** `y = -x`.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The local slope of negation.

**Required outputs:**

- `ex018_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex018_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex018_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex018_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex018_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex018_dx_shape`: predict the shape of `ex018_dx` as a literal Python tuple

**Next concept:** Square backward.


In [53]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([2.0, -1.0, 5.0], dtype=DTYPE, requires_grad=True)
y = -x
dout = torch.tensor([0.5, 2.0, -3.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(18, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [54]:
# Exercise 018: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex018_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex018_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex018_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex018_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex018_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex018_dx_shape` — the shape of `ex018_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
# Chain rule convention: write upstream dL/dy (`dout`) before local dy/dx.
ex018_x_shape = (3,)
ex018_y_shape = (3,)
ex018_dout_shape = (3,)
ex018_dx_shape = (3,)
ex018_dx_manual = torch.tensor([-0.5, -2.0, 3.0], dtype=DTYPE)
ex018_dx = dout * -1

In [55]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex018_x_shape", "x")
_check_visible_tensor_shape("ex018_y_shape", "y")
_check_visible_tensor_shape("ex018_dout_shape", "dout")
_check_private_tensor_shape("ex018_dx_shape", "ex018", "dx")
_check_tensor("ex018_dx_manual", "ex018", "dx")
_check_tensor("ex018_dx", "ex018", "dx")


PASS: ex018_x_shape
PASS: ex018_y_shape
PASS: ex018_dout_shape
PASS: ex018_dx_shape
PASS: ex018_dx_manual
PASS: ex018_dx


### Exercise 019 — Square backward

**Purpose:** Learn how the gradient of a square depends on the original input value.

**Inputs:** `x` and `dout` have shape `(4,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([-2.0, -0.5, 1.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x**2
dout = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
```

**Forward operation:** `y = x**2`.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The power rule and elementwise chain rule.

**Required outputs:**

- `ex019_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex019_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex019_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex019_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex019_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex019_dx_shape`: predict the shape of `ex019_dx` as a literal Python tuple

**Next concept:** Cube backward.


In [56]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([-2.0, -0.5, 1.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x**2
dout = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(19, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [57]:
ex019_x_shape = (4,)
ex019_y_shape = (4,)
ex019_dout_shape = (4,)
ex019_dx_shape = (4,)
 # Exercise 019: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex019_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex019_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex019_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex019_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex019_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex019_dx_shape` — the shape of `ex019_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
# Chain rule convention: write upstream dL/dy (`dout`) before local dy/dx.
ex019_dx_manual = torch.tensor([-4.0, 2.0, 1.5, 18.0], dtype=DTYPE)
ex019_dx = dout * 2 * x

In [58]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex019_x_shape", "x")
_check_visible_tensor_shape("ex019_y_shape", "y")
_check_visible_tensor_shape("ex019_dout_shape", "dout")
_check_private_tensor_shape("ex019_dx_shape", "ex019", "dx")
_check_tensor("ex019_dx_manual", "ex019", "dx")
_check_tensor("ex019_dx", "ex019", "dx")


PASS: ex019_x_shape
PASS: ex019_y_shape
PASS: ex019_dout_shape
PASS: ex019_dx_shape
PASS: ex019_dx_manual
PASS: ex019_dx


### Exercise 020 — Cube backward

**Purpose:** Learn to apply the same power-rule idea to a cube.

**Inputs:** `x` and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x**3
dout = torch.tensor([0.25, -1.0, 2.0], dtype=DTYPE)
```

**Forward operation:** `y = x**3`.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The power rule for exponent 3 and the upstream gradient.

**Required outputs:**

- `ex020_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex020_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex020_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex020_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex020_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex020_dx_shape`: predict the shape of `ex020_dx` as a literal Python tuple

**Next concept:** Reciprocal backward.


In [59]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x**3
dout = torch.tensor([0.25, -1.0, 2.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(20, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [60]:
# Exercise 020: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex020_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex020_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex020_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex020_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex020_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex020_dx_shape` — the shape of `ex020_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex020_x_shape = (3,)
ex020_y_shape = (3,)
ex020_dout_shape = (3,)
ex020_dx_shape = (3,)
ex020_dx_manual = torch.tensor([0.25 * 3.0 * (-2.0)**2, -1.0 * 3.0 * 0.5**2, 2.0 * 3.0 * 3.0**2 ], dtype=DTYPE)
ex020_dx = dout * 3 * x**2

In [61]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex020_x_shape", "x")
_check_visible_tensor_shape("ex020_y_shape", "y")
_check_visible_tensor_shape("ex020_dout_shape", "dout")
_check_private_tensor_shape("ex020_dx_shape", "ex020", "dx")
_check_tensor("ex020_dx_manual", "ex020", "dx")
_check_tensor("ex020_dx", "ex020", "dx")


PASS: ex020_x_shape
PASS: ex020_y_shape
PASS: ex020_dout_shape
PASS: ex020_dx_shape
PASS: ex020_dx_manual
PASS: ex020_dx


### Exercise 021 — Reciprocal backward

**Purpose:** Learn how to differentiate `1 / x` by viewing it as a negative power.

**Inputs:** Positive `x` and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([0.5, 2.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x**-1
dout = torch.tensor([1.0, -0.5, 3.0], dtype=DTYPE)
```

**Forward operation:** `y = x**-1`.

**Reciprocal identity:** For every nonzero value of `x`, exponent `-1` means reciprocal:

$$
x^{-1}
=
\frac{1}{x}
$$

Therefore, these floating-point tensor expressions describe the same forward operation for the supplied positive inputs:

```python
y = x**-1
y = 1 / x
```

More generally, a negative power moves the corresponding positive power into the denominator:

$$
x^{-k}
=
\frac{1}{x^k}
$$

The reciprocal is undefined at `x = 0`; this exercise deliberately supplies positive nonzero values. Use the negative-power form with the power rule, then write the backward chain in the established order: upstream `dout` first, followed by the local derivative.

**Worked backward rule:** No memorization is required. Start from the fixed-power rule:

$$
\frac{d}{dx}x^k
=
kx^{k-1}
$$

For the reciprocal, substitute `k = -1`:

$$
\frac{dy}{dx}
=
\frac{d}{dx}x^{-1}
=
-1 \cdot x^{-2}
=
-\frac{1}{x^2}
$$

Now apply the chain rule with the upstream gradient first. Since `dout` represents the derivative of the final scalar loss with respect to `y`, the gradient with respect to `x` is:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
=
\mathrm{dout}\left(-x^{-2}\right)
=
-\frac{\mathrm{dout}}{x^2}
$$

In scalable tensor form, this symbolic result corresponds to:

```python
ex021_dx = dout * (-1) * x**-2
```

For the mental pass, evaluate the same expression independently for each aligned pair `dout[i]` and `x[i]`, then write the resulting values literally in `ex021_dx_manual`. The exercise tests applying the supplied rule correctly; it does not require recalling the reciprocal derivative from memory.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** Rewrite the reciprocal as a power, apply the power rule, then chain with `dout`.

**Required outputs:**

- `ex021_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex021_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex021_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex021_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex021_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex021_dx_shape`: predict the shape of `ex021_dx` as a literal Python tuple

**Next concept:** Inverse square root backward.


In [62]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([0.5, 2.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x**-1  # Equivalent to 1 / x for these nonzero floating-point values.
dout = torch.tensor([1.0, -0.5, 3.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(21, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [63]:
# Exercise 021: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex021_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex021_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Supplied rule: dy/dx = (-1) * x**-2 = -1 / x**2.
# Chain-rule order: dx = dout * local derivative.
# Predict every visible tensor shape before computing values.
# Define `ex021_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex021_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex021_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex021_dx_shape` — the shape of `ex021_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex021_x_shape = (3,)
ex021_y_shape = (3,)
ex021_dout_shape = (3,)
ex021_dx_shape = (3,)
ex021_dx_manual = torch.tensor([1.0 * -(1/0.5**2), -0.5 * -(1/2.0**2), 3.0 * -(1/4.0**2)], dtype=DTYPE)
ex021_dx = dout * -(1/x**2)

In [64]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex021_x_shape", "x")
_check_visible_tensor_shape("ex021_y_shape", "y")
_check_visible_tensor_shape("ex021_dout_shape", "dout")
_check_private_tensor_shape("ex021_dx_shape", "ex021", "dx")
_check_tensor("ex021_dx_manual", "ex021", "dx")
_check_tensor("ex021_dx", "ex021", "dx")


PASS: ex021_x_shape
PASS: ex021_y_shape
PASS: ex021_dout_shape
PASS: ex021_dx_shape
PASS: ex021_dx_manual
PASS: ex021_dx


### Exercise 022 — Inverse square root backward

**Purpose:** Learn how to differentiate the inverse square root used later in BatchNorm.

**Inputs:** Positive `x` and `dout` have shape `(3,)`. Positivity matters because the real-valued square root is differentiable only for strictly positive inputs in this exercise.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([0.25, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
s = torch.sqrt(x)
y = 1 / s
dout = torch.tensor([0.5, -2.0, 1.5], dtype=DTYPE)
```

**Forward operations:** `s = torch.sqrt(x)`, followed by `y = 1 / s`. The supplied `s` is the square-root intermediate reused during backward.

**What inverse square root means:** First, the square root of a positive number `x` is the positive number that produces `x` when multiplied by itself. The **inverse square root** then takes the reciprocal of that square root. Here, “inverse” means multiplicative reciprocal—not the inverse function of the square-root operation.

For a strictly positive input, define the inverse-square-root function as:

$$
f(x)
=
\frac{1}{\sqrt{x}},
\qquad x > 0
$$

Its defining reciprocal relationship is:

$$
f(x)\sqrt{x}
=
\frac{1}{\sqrt{x}}\sqrt{x}
=
1
$$

As `x` increases, its positive square root increases, so the reciprocal becomes smaller. In BatchNorm, this operation converts a positive variance into the inverse standard deviation used to scale centered activations.

**Supplied derivation—reference only:** Introduce `s` as the square root of `x`, so the forward operation becomes a two-step chain:

$$
s
=
\sqrt{x}
$$

$$
y
=
\frac{1}{s}
$$

The local derivative of the square-root step is:

$$
\frac{\partial s}{\partial x}
=
\frac{1}{2\sqrt{x}}
$$

The local derivative of the reciprocal step is:

$$
\frac{\partial y}{\partial s}
=
-\frac{1}{s^2}
$$

Multiply those local derivatives and substitute `s` with the square root of `x`:

$$
\frac{\partial y}{\partial x}
=
\frac{\partial y}{\partial s}
\frac{\partial s}{\partial x}
=
\left(-\frac{1}{s^2}\right)
\left(\frac{1}{2\sqrt{x}}\right)
=
-\frac{1}{2x\sqrt{x}}
$$

Finally, put the upstream gradient first according to the course convention:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
=
\mathrm{dout}
\left(-\frac{1}{2x\sqrt{x}}\right)
$$

As a notation hint only, the same forward function can be written as:

$$
\frac{1}{\sqrt{x}}
=
x^{-\frac{1}{2}}
$$

The supplied forward graph uses `torch.sqrt` to create `s`; the fractional-power notation only explains why a power-rule derivation would produce the same derivative.

**Required PyTorch pass:** Use the supplied intermediate `s` and the supplied derivative to write a scalable PyTorch backward calculation for the gradient with respect to `x`. You may name the two local derivatives before combining them with `dout`. You do not need to reproduce the symbolic derivation or calculate a separate literal `_manual` tensor. Also predict the forward output shape.

**Ingredients:** The supplied square-root intermediate `s`, the two supplied local derivatives, and upstream `dout`.

**Required outputs:**

- `ex022_dx`: the gradient with respect to `x`, derived by combining `dout` with the two local derivatives using the supplied `s`


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex022_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex022_s_shape`: predict the shape of `s` as a literal Python tuple
- `ex022_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex022_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex022_dx_shape`: predict the shape of `ex022_dx` as a literal Python tuple

**Next concept:** Exponential backward.


In [65]:
# Supplied forward tensors, intermediate, and upstream gradient are visible.
x = torch.tensor([0.25, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
s = torch.sqrt(x)
y = 1 / s
dout = torch.tensor([0.5, -2.0, 1.5], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(22, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [66]:
# Exercise 022: translate the supplied derivative into PyTorch; do not use autograd.
# Reuse the supplied forward intermediate `s`; do not recompute `torch.sqrt(x)` here.
# You may name `ds_dx` and `dy_ds`, then combine them with upstream `dout`.
# Define `ex022_dx` — the gradient with respect to `x`; no `_manual` tensor is required.
# Predict every visible tensor shape before computing values.
# Define `ex022_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex022_s_shape` — the shape of `s` as a literal Python tuple.
# Define `ex022_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex022_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex022_dx_shape` — the shape of `ex022_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex022_x_shape = (3,)
ex022_s_shape = (3,)
ex022_y_shape = (3,)
ex022_dout_shape = (3,)
ex022_dx_shape = (3,)
ds_dx = 1 / (2 * s)
dy_ds = -1 / s**2
ex022_dx = dout * dy_ds * ds_dx

In [67]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex022_x_shape", "x")
_check_visible_tensor_shape("ex022_s_shape", "s")
_check_visible_tensor_shape("ex022_y_shape", "y")
_check_visible_tensor_shape("ex022_dout_shape", "dout")
_check_private_tensor_shape("ex022_dx_shape", "ex022", "dx")
_check_tensor("ex022_dx", "ex022", "dx")


PASS: ex022_x_shape
PASS: ex022_s_shape
PASS: ex022_y_shape
PASS: ex022_dout_shape
PASS: ex022_dx_shape
PASS: ex022_dx


### Exercise 023 — Exponential backward

**Purpose:** Learn that the exponential's backward calculation can reuse its forward output.

**Inputs:** `x`, `y`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([-1.0, 0.0, 1.5], dtype=DTYPE, requires_grad=True)
y = x.exp()
dout = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)
```

**Forward operation:** `y = x.exp()`.

**What exponential means:** The exponential function raises the mathematical constant `e` to each input value:

$$
\exp(x)
=
e^x
$$

In PyTorch, `x.exp()` is equivalent to `torch.exp(x)`. It applies the function elementwise and returns a tensor with the same shape. Exponentials later turn softmax logits into positive candidate weights.

**Derivative reference:** Here, `y` is the exponential forward output. The exponential is its own derivative:

$$
\frac{\partial y}{\partial x}
=
e^x
=
y
$$

Therefore, the upstream-first chain rule is:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\,y
$$

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The derivative of the exponential and the available forward tensor `y`.

**Required outputs:**

- `ex023_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex023_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex023_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex023_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex023_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex023_dx_shape`: predict the shape of `ex023_dx` as a literal Python tuple

**Next concept:** Logarithm backward.


In [68]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([-1.0, 0.0, 1.5], dtype=DTYPE, requires_grad=True)
y = x.exp()
dout = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(23, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [69]:
# Exercise 023: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex023_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex023_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex023_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex023_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex023_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex023_dx_shape` — the shape of `ex023_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex023_x_shape = (3,)
ex023_y_shape = (3,)
ex023_dout_shape = (3,)
ex023_dx_shape = (3,)
import math
ex023_dx_manual = torch.tensor([2.0 * math.e**-1.0, -1.0 * math.e**0.0, 0.5 * math.e**1.5], dtype=DTYPE)
ex023_dx = dout * y

In [70]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex023_x_shape", "x")
_check_visible_tensor_shape("ex023_y_shape", "y")
_check_visible_tensor_shape("ex023_dout_shape", "dout")
_check_private_tensor_shape("ex023_dx_shape", "ex023", "dx")
_check_tensor("ex023_dx_manual", "ex023", "dx")
_check_tensor("ex023_dx", "ex023", "dx")


PASS: ex023_x_shape
PASS: ex023_y_shape
PASS: ex023_dout_shape
PASS: ex023_dx_shape
PASS: ex023_dx_manual
PASS: ex023_dx


### Exercise 024 — Logarithm backward

**Purpose:** Learn that the gradient through `log(x)` divides the incoming gradient by `x`.

**Inputs:** Positive `x`, `y`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([0.5, 2.0, 5.0], dtype=DTYPE, requires_grad=True)
y = x.log()
dout = torch.tensor([-1.0, 0.25, 3.0], dtype=DTYPE)
```

**Forward operation:** `y = x.log()`.

**What logarithm means:** Here, `log` is the natural logarithm, the inverse of the exponential. For positive `x`:

$$
y
=
\log(x)
\quad\Longleftrightarrow\quad
e^y
=
x
$$

In PyTorch, `x.log()` is equivalent to `torch.log(x)`. It applies the natural logarithm elementwise and preserves the tensor shape. Logarithms later convert target probabilities into the values used by negative log-likelihood.

**Derivative reference:** For positive `x`, the local derivative of the natural logarithm is:

$$
\frac{\partial y}{\partial x}
=
\frac{1}{x}
$$

Therefore, the upstream-first chain rule is:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\frac{1}{x}
$$

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The derivative of natural logarithm and the elementwise chain rule.

**Required outputs:**

- `ex024_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex024_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex024_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex024_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex024_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex024_dx_shape`: predict the shape of `ex024_dx` as a literal Python tuple

**Next concept:** Tanh backward.


In [71]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([0.5, 2.0, 5.0], dtype=DTYPE, requires_grad=True)
y = x.log()
dout = torch.tensor([-1.0, 0.25, 3.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(24, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [72]:
# Exercise 024: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex024_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex024_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex024_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex024_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex024_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex024_dx_shape` — the shape of `ex024_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex024_x_shape = (3,)
ex024_y_shape = (3,)
ex024_dout_shape = (3,)
ex024_dx_shape = (3,)
ex024_dx_manual = torch.tensor([-1.0 * (1/0.5), 0.25 * (1/2.0), 3.0 * (1/5.0)], dtype=DTYPE)
ex024_dx = dout * (1/x)

In [73]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex024_x_shape", "x")
_check_visible_tensor_shape("ex024_y_shape", "y")
_check_visible_tensor_shape("ex024_dout_shape", "dout")
_check_private_tensor_shape("ex024_dx_shape", "ex024", "dx")
_check_tensor("ex024_dx_manual", "ex024", "dx")
_check_tensor("ex024_dx", "ex024", "dx")


PASS: ex024_x_shape
PASS: ex024_y_shape
PASS: ex024_dout_shape
PASS: ex024_dx_shape
PASS: ex024_dx_manual
PASS: ex024_dx


### Exercise 025 — Tanh backward

**Purpose:** Learn how an incoming gradient passes backward through `tanh`.

**Inputs:** `x`, `y`, and `dout` have shape `(4,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([-2.0, -0.5, 0.5, 2.0], dtype=DTYPE, requires_grad=True)
y = x.tanh()
dout = torch.tensor([1.0, 2.0, -1.0, 0.5], dtype=DTYPE)
```

**Forward operation:** `y = x.tanh()`.

**What `tanh` is:** The name means **hyperbolic tangent**. It is a smooth function built from exponentials, not the trigonometric tangent used with angles:

$$
\tanh(x)
=
\frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

It maps each real input to a value between `-1` and `1`, preserves the input's sign, and maps zero to zero:

$$
-1
<
\tanh(x)
<
1,
\qquad
\tanh(0)
=
0
$$

In PyTorch, `x.tanh()` is equivalent to `torch.tanh(x)`. It applies the function independently to each tensor element, returns a new tensor with the same shape, and does not require broadcasting. Neural networks use `tanh` as a bounded nonlinear activation function.

**Derivative reference:** Reuse the forward output `y`, where `y` is the hyperbolic tangent of `x`. Its local derivative is:

$$
\frac{\partial y}{\partial x}
=
1-y^2
$$

Therefore, the upstream-first chain rule is:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\left(1-y^2\right)
$$

**Two required backward passes:** First write the `_manual` gradient entry by entry, multiplying each visible upstream value by the local-derivative factors built from the corresponding saved `y[i]`. Then derive the same gradient with one scalable PyTorch tensor expression. Also predict the forward output shape.

**Ingredients:** The tanh local derivative expressed using `y`, then the chain rule.

**Required outputs:**

- `ex025_dx_manual`: the gradient written entry by entry; each entry may index the corresponding saved `y[i]` to show `upstream value × local derivative`
- `ex025_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex025_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex025_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex025_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex025_dx_shape`: predict the shape of `ex025_dx` as a literal Python tuple

**Next concept:** Sigmoid backward.


In [74]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([-2.0, -0.5, 0.5, 2.0], dtype=DTYPE, requires_grad=True)
y = x.tanh()
dout = torch.tensor([1.0, 2.0, -1.0, 0.5], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(25, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [75]:
# Exercise 025: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex025_dx_manual` may index saved `y[i]` to show each chain-rule product.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex025_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex025_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex025_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex025_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex025_dx_shape` — the shape of `ex025_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex025_x_shape = (4,)
ex025_y_shape = (4,)
ex025_dout_shape = (4,)
ex025_dx_shape = (4,)
y_d = y.detach()
ex025_dx_manual = torch.tensor([
    1.0 * (1-y_d[0]**2),
    2.0 * (1-y_d[1]**2),
    -1.0 * (1-y_d[2]**2),
    0.5 * (1-y_d[3]**2)
], dtype=DTYPE)
ex025_dx = dout * (1-y**2)

In [76]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex025_x_shape", "x")
_check_visible_tensor_shape("ex025_y_shape", "y")
_check_visible_tensor_shape("ex025_dout_shape", "dout")
_check_private_tensor_shape("ex025_dx_shape", "ex025", "dx")
_check_tensor("ex025_dx_manual", "ex025", "dx")
_check_tensor("ex025_dx", "ex025", "dx")


PASS: ex025_x_shape
PASS: ex025_y_shape
PASS: ex025_dout_shape
PASS: ex025_dx_shape
PASS: ex025_dx_manual
PASS: ex025_dx


### Exercise 026 — Sigmoid backward

**Purpose:** Learn how an incoming gradient passes backward through `sigmoid`.

**Inputs:** `x`, `y`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([-1.5, 0.0, 2.0], dtype=DTYPE, requires_grad=True)
y = x.sigmoid()
dout = torch.tensor([2.0, -1.0, 0.25], dtype=DTYPE)
```

**Forward operation:** `y = x.sigmoid()`.

**What sigmoid is:** Sigmoid is a smooth function that converts any real input into a value between `0` and `1`:

$$
\sigma(x)
=
\frac{1}{1+e^{-x}}
$$

It maps zero to one half:

$$
\sigma(0)
=
\frac{1}{2}
$$

In PyTorch, `x.sigmoid()` is equivalent to `torch.sigmoid(x)`. It applies sigmoid independently to every tensor element and returns a new tensor with the same shape. Neural networks commonly use sigmoid for gates and probability-like outputs.

**Derivative reference:** Reuse the forward output `y`, where `y` is the sigmoid of `x`. Its local derivative is:

$$
\frac{\partial y}{\partial x}
=
y(1-y)
$$

Therefore, the upstream-first chain rule is:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\,y(1-y)
$$

**Two required backward passes:** First write the `_manual` gradient entry by entry, multiplying each visible upstream value by the local-derivative factors built from the corresponding saved `y[i]`. Then derive the same gradient with one scalable PyTorch tensor expression. Also predict the forward output shape.

**Ingredients:** The sigmoid local slope written with `y` and `1 - y`.

**Required outputs:**

- `ex026_dx_manual`: the gradient written entry by entry; each entry may index the corresponding saved `y[i]` to show `upstream value × local derivative`
- `ex026_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex026_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex026_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex026_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex026_dx_shape`: predict the shape of `ex026_dx` as a literal Python tuple

**Next concept:** ReLU backward.


In [77]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([-1.5, 0.0, 2.0], dtype=DTYPE, requires_grad=True)
y = x.sigmoid()
dout = torch.tensor([2.0, -1.0, 0.25], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(26, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [78]:
# Exercise 026: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex026_dx_manual` may index saved `y[i]` to show each chain-rule product.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex026_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex026_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex026_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex026_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex026_dx_shape` — the shape of `ex026_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex026_x_shape = (3,)
ex026_y_shape = (3,)
ex026_dout_shape = (3,)
ex026_dx_shape = (3,)
y_d = y.detach()
ex026_dx_manual = torch.tensor([2.0 * y_d[0]*(1-y_d[0]), -1.0 * y_d[1]*(1-y_d[1]), 0.25 * y_d[2]*(1-y_d[2])], dtype=DTYPE)
ex026_dx = dout * y * (1-y)

In [79]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex026_x_shape", "x")
_check_visible_tensor_shape("ex026_y_shape", "y")
_check_visible_tensor_shape("ex026_dout_shape", "dout")
_check_private_tensor_shape("ex026_dx_shape", "ex026", "dx")
_check_tensor("ex026_dx_manual", "ex026", "dx")
_check_tensor("ex026_dx", "ex026", "dx")


PASS: ex026_x_shape
PASS: ex026_y_shape
PASS: ex026_dout_shape
PASS: ex026_dx_shape
PASS: ex026_dx_manual
PASS: ex026_dx


### Exercise 027 — ReLU backward

**Purpose:** Learn that ReLU passes gradients through positive inputs and blocks them at negative inputs.

**Inputs:** `x` contains no zero and has shape `(4,)`; `dout` matches it.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([-3.0, -0.5, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x.relu()
dout = torch.tensor([1.0, 2.0, -2.0, 0.5], dtype=DTYPE)
```

**Forward operation:** `y = x.relu()`.

**What ReLU is:** ReLU means **rectified linear unit**. It keeps positive values and replaces zero or negative values with zero:

$$
\operatorname{ReLU}(x)
=
\left\{
\begin{aligned}
x &\quad \text{if } x > 0, \\
0 &\quad \text{if } x \le 0.
\end{aligned}
\right.
$$

In PyTorch, `x.relu()` is equivalent to `torch.relu(x)`. It applies ReLU elementwise, returns a tensor with the same shape, and is commonly used as a neural-network activation function.

**Derivative reference:** PyTorch uses a zero local derivative at zero. The ReLU local derivative is:

$$
\frac{\partial y}{\partial x}
=
\left\{
\begin{aligned}
1 &\quad \text{if } x > 0, \\
0 &\quad \text{if } x \le 0.
\end{aligned}
\right.
$$

Apply the upstream-first chain rule:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}
\frac{\partial y}{\partial x}
$$

Therefore, positive entries pass their corresponding `dout` values backward, while zero or negative entries receive zero gradient.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** A Boolean mask for positive entries, converted by multiplication with `dout`.

**Required outputs:**

- `ex027_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex027_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex027_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex027_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex027_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex027_dx_shape`: predict the shape of `ex027_dx` as a literal Python tuple

**Next concept:** Elementwise addition backward.


In [80]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([-3.0, -0.5, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x.relu()
dout = torch.tensor([1.0, 2.0, -2.0, 0.5], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(27, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [81]:
# Exercise 027: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex027_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex027_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex027_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex027_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex027_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex027_dx_shape` — the shape of `ex027_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex027_x_shape = (4,)
ex027_y_shape = (4,)
ex027_dout_shape = (4,)
ex027_dx_shape = (4,)
ex027_dx_manual = torch.tensor([1.0 * 0.0, 2.0 * 0.0, -2.0 * 1.0, 0.5 * 1.0], dtype=DTYPE)
ex027_dx = dout * (x > 0).to(dtype=x.dtype)

In [82]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex027_x_shape", "x")
_check_visible_tensor_shape("ex027_y_shape", "y")
_check_visible_tensor_shape("ex027_dout_shape", "dout")
_check_private_tensor_shape("ex027_dx_shape", "ex027", "dx")
_check_tensor("ex027_dx_manual", "ex027", "dx")
_check_tensor("ex027_dx", "ex027", "dx")


PASS: ex027_x_shape
PASS: ex027_y_shape
PASS: ex027_dout_shape
PASS: ex027_dx_shape
PASS: ex027_dx_manual
PASS: ex027_dx


### Exercise 028 — Elementwise addition backward

**Purpose:** Learn that an addition sends the same incoming gradient to both of its inputs.

**Inputs:** `x`, `z`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([-2.0, 0.5, 4.0], dtype=DTYPE, requires_grad=True)
y = x + z
dout = torch.tensor([0.5, -1.0, 2.0], dtype=DTYPE)
```

**Forward operation:** `y = x + z`.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** Treat each input as changing while the other is held fixed.

**Required outputs:**

- `ex028_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex028_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations
- `ex028_dz_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex028_dz`: the same gradient with respect to `z`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex028_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex028_z_shape`: predict the shape of `z` as a literal Python tuple
- `ex028_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex028_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex028_dx_shape`: predict the shape of `ex028_dx` as a literal Python tuple
- `ex028_dz_shape`: predict the shape of `ex028_dz` as a literal Python tuple

**Next concept:** Elementwise subtraction backward.


In [83]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([-2.0, 0.5, 4.0], dtype=DTYPE, requires_grad=True)
y = x + z
dout = torch.tensor([0.5, -1.0, 2.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(28, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [84]:
# Exercise 028: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex028_dx_manual` may use visible-number arithmetic or simplified values.
# Explicit pass: `ex028_dz_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex028_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Define `ex028_dz` — derive the gradient with respect to `z` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex028_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex028_z_shape` — the shape of `z` as a literal Python tuple.
# Define `ex028_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex028_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex028_dx_shape` — the shape of `ex028_dx` as a literal Python tuple.
# Define `ex028_dz_shape` — the shape of `ex028_dz` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex028_x_shape = (3,)
ex028_z_shape = (3,)
ex028_y_shape = (3,)
ex028_dout_shape = (3,)
ex028_dx_shape = (3,)
ex028_dz_shape = (3,)
ex028_dx_manual = torch.tensor([0.5 * 1.0, -1.0 * 1.0, 2.0 * 1.0], dtype=DTYPE)
ex028_dx = dout * 1.0
ex028_dz_manual = torch.tensor([0.5 * 1.0, -1.0 * 1.0, 2.0 * 1.0], dtype=DTYPE)
ex028_dz = dout * 1.0

In [85]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex028_x_shape", "x")
_check_visible_tensor_shape("ex028_z_shape", "z")
_check_visible_tensor_shape("ex028_y_shape", "y")
_check_visible_tensor_shape("ex028_dout_shape", "dout")
_check_private_tensor_shape("ex028_dx_shape", "ex028", "dx")
_check_private_tensor_shape("ex028_dz_shape", "ex028", "dz")
_check_tensor("ex028_dx_manual", "ex028", "dx")
_check_tensor("ex028_dx", "ex028", "dx")
_check_tensor("ex028_dz_manual", "ex028", "dz")
_check_tensor("ex028_dz", "ex028", "dz")


PASS: ex028_x_shape
PASS: ex028_z_shape
PASS: ex028_y_shape
PASS: ex028_dout_shape
PASS: ex028_dx_shape
PASS: ex028_dz_shape
PASS: ex028_dx_manual
PASS: ex028_dx
PASS: ex028_dz_manual
PASS: ex028_dz


### Exercise 029 — Elementwise subtraction backward

**Purpose:** Learn that subtraction sends opposite-signed gradients to its left and right inputs.

**Inputs:** `x`, `z`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([3.0, 1.0, -2.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([1.0, -4.0, 0.5], dtype=DTYPE, requires_grad=True)
y = x - z
dout = torch.tensor([2.0, -0.5, 1.0], dtype=DTYPE)
```

**Forward operation:** `y = x - z`.

**Derivative reference:** Subtraction gives its two inputs different local slopes. Rewrite the operation as:

$$
y
=
(+1)x+(-1)z
$$

When differentiating with respect to `x`, hold `z` fixed. Increasing `x` by one increases `y` by one, so:

$$
\frac{\partial y}{\partial x}
=
1
$$

When differentiating with respect to `z`, hold `x` fixed. Increasing `z` by one decreases `y` by one, so:

$$
\frac{\partial y}{\partial z}
=
-1
$$

Apply `dout` separately to the two paths:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\cdot 1
$$

$$
\frac{\partial L}{\partial z}
=
\mathrm{dout}\cdot(-1)
$$

Therefore, the left input `x` receives `dout` unchanged, while the subtracted input `z` receives the sign-flipped gradient.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The local slope with respect to the left input and with respect to the right input.

**Required outputs:**

- `ex029_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex029_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations
- `ex029_dz_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex029_dz`: the same gradient with respect to `z`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex029_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex029_z_shape`: predict the shape of `z` as a literal Python tuple
- `ex029_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex029_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex029_dx_shape`: predict the shape of `ex029_dx` as a literal Python tuple
- `ex029_dz_shape`: predict the shape of `ex029_dz` as a literal Python tuple

**Next concept:** Elementwise multiplication backward.


In [86]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([3.0, 1.0, -2.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([1.0, -4.0, 0.5], dtype=DTYPE, requires_grad=True)
y = x - z
dout = torch.tensor([2.0, -0.5, 1.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(29, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [87]:
# Exercise 029: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex029_dx_manual` may use visible-number arithmetic or simplified values.
# Explicit pass: `ex029_dz_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex029_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Define `ex029_dz` — derive the gradient with respect to `z` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex029_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex029_z_shape` — the shape of `z` as a literal Python tuple.
# Define `ex029_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex029_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex029_dx_shape` — the shape of `ex029_dx` as a literal Python tuple.
# Define `ex029_dz_shape` — the shape of `ex029_dz` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex029_x_shape = (3,)
ex029_z_shape = (3,)
ex029_y_shape = (3,)
ex029_dout_shape = (3,)
ex029_dx_shape = (3,)
ex029_dz_shape = (3,)
ex029_dx_manual = torch.tensor([2.0 * 1.0, -0.5 * 1.0, 1.0 * 1.0], dtype=DTYPE)
ex029_dx = dout * 1.0
ex029_dz_manual = torch.tensor([2.0 * -1.0, -0.5 * -1.0, 1.0 * -1.0], dtype=DTYPE)
ex029_dz = dout * -1.0

In [88]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex029_x_shape", "x")
_check_visible_tensor_shape("ex029_z_shape", "z")
_check_visible_tensor_shape("ex029_y_shape", "y")
_check_visible_tensor_shape("ex029_dout_shape", "dout")
_check_private_tensor_shape("ex029_dx_shape", "ex029", "dx")
_check_private_tensor_shape("ex029_dz_shape", "ex029", "dz")
_check_tensor("ex029_dx_manual", "ex029", "dx")
_check_tensor("ex029_dx", "ex029", "dx")
_check_tensor("ex029_dz_manual", "ex029", "dz")
_check_tensor("ex029_dz", "ex029", "dz")


PASS: ex029_x_shape
PASS: ex029_z_shape
PASS: ex029_y_shape
PASS: ex029_dout_shape
PASS: ex029_dx_shape
PASS: ex029_dz_shape
PASS: ex029_dx_manual
PASS: ex029_dx
PASS: ex029_dz_manual
PASS: ex029_dz


### Exercise 030 — Elementwise multiplication backward

**Purpose:** Learn that each input of a multiplication uses the other input when computing its gradient.

**Inputs:** `x`, `z`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([2.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([4.0, 5.0, -2.0], dtype=DTYPE, requires_grad=True)
y = x * z
dout = torch.tensor([0.5, -2.0, 1.5], dtype=DTYPE)
```

**Forward operation:** `y = x * z`.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** For each input, identify the other forward factor, then apply `dout`.

**Required outputs:**

- `ex030_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex030_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations
- `ex030_dz_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex030_dz`: the same gradient with respect to `z`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex030_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex030_z_shape`: predict the shape of `z` as a literal Python tuple
- `ex030_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex030_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex030_dx_shape`: predict the shape of `ex030_dx` as a literal Python tuple
- `ex030_dz_shape`: predict the shape of `ex030_dz` as a literal Python tuple

**Next concept:** Elementwise division backward.


In [89]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([2.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([4.0, 5.0, -2.0], dtype=DTYPE, requires_grad=True)
y = x * z
dout = torch.tensor([0.5, -2.0, 1.5], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(30, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [90]:
# Exercise 030: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex030_dx_manual` may use visible-number arithmetic or simplified values.
# Explicit pass: `ex030_dz_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex030_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Define `ex030_dz` — derive the gradient with respect to `z` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex030_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex030_z_shape` — the shape of `z` as a literal Python tuple.
# Define `ex030_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex030_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex030_dx_shape` — the shape of `ex030_dx` as a literal Python tuple.
# Define `ex030_dz_shape` — the shape of `ex030_dz` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex030_x_shape = (3,)
ex030_z_shape = (3,)
ex030_y_shape = (3,)
ex030_dout_shape = (3,)
ex030_dx_shape = (3,)
ex030_dz_shape = (3,)
ex030_dx_manual = torch.tensor([0.5 * 4.0, -2.0 * 5.0, 1.5 * -2.0], dtype=DTYPE)
ex030_dx = dout * z
ex030_dz_manual = torch.tensor([0.5 * 2.0, -2.0 * -1.0, 1.5 * 3.0], dtype=DTYPE)
ex030_dz = dout * x

In [91]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex030_x_shape", "x")
_check_visible_tensor_shape("ex030_z_shape", "z")
_check_visible_tensor_shape("ex030_y_shape", "y")
_check_visible_tensor_shape("ex030_dout_shape", "dout")
_check_private_tensor_shape("ex030_dx_shape", "ex030", "dx")
_check_private_tensor_shape("ex030_dz_shape", "ex030", "dz")
_check_tensor("ex030_dx_manual", "ex030", "dx")
_check_tensor("ex030_dx", "ex030", "dx")
_check_tensor("ex030_dz_manual", "ex030", "dz")
_check_tensor("ex030_dz", "ex030", "dz")


PASS: ex030_x_shape
PASS: ex030_z_shape
PASS: ex030_y_shape
PASS: ex030_dout_shape
PASS: ex030_dx_shape
PASS: ex030_dz_shape
PASS: ex030_dx_manual
PASS: ex030_dx
PASS: ex030_dz_manual
PASS: ex030_dz


### Exercise 031 — Elementwise division backward

**Purpose:** Learn that division gives different gradient rules for the numerator and denominator.

**Inputs:** Positive `z`; `x`, `z`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([2.0, -3.0, 5.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([0.5, 2.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x / z
dout = torch.tensor([1.0, -0.5, 2.0], dtype=DTYPE)
```

**Forward operation:** `y = x / z`.

**Derivative reference:** Treat `x` as the numerator and `z` as the denominator. Their local derivatives are different:

$$
\frac{\partial y}{\partial x}
=
\frac{1}{z}
$$

$$
\frac{\partial y}{\partial z}
=
-\frac{x}{z^2}
$$

Apply the upstream gradient separately to each path:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\frac{1}{z}
$$

$$
\frac{\partial L}{\partial z}
=
\mathrm{dout}\left(-\frac{x}{z^2}\right)
$$

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** Rewrite division as multiplication by `z**-1`; differentiate each input separately.

**Required outputs:**

- `ex031_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex031_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations
- `ex031_dz_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex031_dz`: the same gradient with respect to `z`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex031_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex031_z_shape`: predict the shape of `z` as a literal Python tuple
- `ex031_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex031_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex031_dx_shape`: predict the shape of `ex031_dx` as a literal Python tuple
- `ex031_dz_shape`: predict the shape of `ex031_dz` as a literal Python tuple

**Next concept:** Fractional fixed power backward.


In [92]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([2.0, -3.0, 5.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([0.5, 2.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x / z
dout = torch.tensor([1.0, -0.5, 2.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(31, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [93]:
# Exercise 031: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex031_dx_manual` may use visible-number arithmetic or simplified values.
# Explicit pass: `ex031_dz_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex031_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Define `ex031_dz` — derive the gradient with respect to `z` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex031_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex031_z_shape` — the shape of `z` as a literal Python tuple.
# Define `ex031_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex031_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex031_dx_shape` — the shape of `ex031_dx` as a literal Python tuple.
# Define `ex031_dz_shape` — the shape of `ex031_dz` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex031_x_shape = (3,)
ex031_z_shape = (3,)
ex031_y_shape = (3,)
ex031_dout_shape = (3,)
ex031_dx_shape = (3,)
ex031_dz_shape = (3,)
ex031_dx_manual = torch.tensor([1.0 * (1/0.5), -0.5 * (1/2.0), 2.0 * (1/4.0)], dtype=DTYPE)
ex031_dx = dout * (1/z)
ex031_dz_manual = torch.tensor([1.0 * -(2.0/0.5**2), -0.5 * -(-3.0/2.0**2), 2.0 * -(5.0/4.0**2)], dtype=DTYPE)
ex031_dz = dout * -(x/z**2)

In [94]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex031_x_shape", "x")
_check_visible_tensor_shape("ex031_z_shape", "z")
_check_visible_tensor_shape("ex031_y_shape", "y")
_check_visible_tensor_shape("ex031_dout_shape", "dout")
_check_private_tensor_shape("ex031_dx_shape", "ex031", "dx")
_check_private_tensor_shape("ex031_dz_shape", "ex031", "dz")
_check_tensor("ex031_dx_manual", "ex031", "dx")
_check_tensor("ex031_dx", "ex031", "dx")
_check_tensor("ex031_dz_manual", "ex031", "dz")
_check_tensor("ex031_dz", "ex031", "dz")


PASS: ex031_x_shape
PASS: ex031_z_shape
PASS: ex031_y_shape
PASS: ex031_dout_shape
PASS: ex031_dx_shape
PASS: ex031_dz_shape
PASS: ex031_dx_manual
PASS: ex031_dx
PASS: ex031_dz_manual
PASS: ex031_dz


### Exercise 032 — Fractional fixed power backward

**Purpose:** Learn to differentiate a fixed power even when its exponent is not a whole number.

**Inputs:** Positive `x` and same-shape `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([0.25, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x**2.5
dout = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)
```

**Forward operation:** `y = x**2.5`.

**Derivative reference:** The fixed-power rule works for the fractional exponent in this exercise:

$$
\frac{d}{dx}x^k
=
kx^{k-1}
$$

Substitute `k = 2.5`:

$$
\frac{\partial y}{\partial x}
=
2.5x^{1.5}
$$

Therefore, the upstream-first chain rule is:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\left(2.5x^{1.5}\right)
$$

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** The fixed-exponent power rule and `dout`.

**Required outputs:**

- `ex032_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex032_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex032_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex032_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex032_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex032_dx_shape`: predict the shape of `ex032_dx` as a literal Python tuple

**Next concept:** Elementwise affine operation.


In [95]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([0.25, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x**2.5
dout = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(32, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [96]:
# Exercise 032: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex032_dx_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex032_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex032_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex032_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex032_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex032_dx_shape` — the shape of `ex032_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex032_x_shape = (3,)
ex032_y_shape = (3,)
ex032_dout_shape = (3,)
ex032_dx_shape = (3,)
ex032_dx_manual = torch.tensor([2.0 * (2.5*0.25**1.5), -1.0 * (2.5*1.0**1.5), 0.5 * (2.5*4.0**1.5)], dtype=DTYPE)
ex032_dx = dout * (2.5*x**1.5)

In [97]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex032_x_shape", "x")
_check_visible_tensor_shape("ex032_y_shape", "y")
_check_visible_tensor_shape("ex032_dout_shape", "dout")
_check_private_tensor_shape("ex032_dx_shape", "ex032", "dx")
_check_tensor("ex032_dx_manual", "ex032", "dx")
_check_tensor("ex032_dx", "ex032", "dx")


PASS: ex032_x_shape
PASS: ex032_y_shape
PASS: ex032_dout_shape
PASS: ex032_dx_shape
PASS: ex032_dx_manual
PASS: ex032_dx


### Exercise 033 — Elementwise affine operation

**Purpose:** Learn how `x * scale + shift` sends gradients to `x`, `scale`, and `shift`.

**Inputs:** `x`, `scale`, `shift`, and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Visible forward tensors and upstream gradient:**

```python
x = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE, requires_grad=True)
scale = torch.tensor([2.0, 3.0, -4.0], dtype=DTYPE, requires_grad=True)
shift = torch.tensor([0.5, 1.0, -1.0], dtype=DTYPE, requires_grad=True)
y = x * scale + shift
dout = torch.tensor([1.0, -0.5, 2.0], dtype=DTYPE)
```

**Forward operation:** `y = x * scale + shift`.

Here, `*` is elementwise multiplication between matching entries, not matrix multiplication with `@`. This learned elementwise scale-and-shift operation is used by BatchNorm after normalization.

**Derivative reference:** Let `s` represent `scale` and `b` represent `shift`, so the elementwise forward operation is:

$$
y
=
xs+b
$$

Its three local derivatives are:

$$
\frac{\partial y}{\partial x}
=
s
$$

$$
\frac{\partial y}{\partial s}
=
x
$$

$$
\frac{\partial y}{\partial b}
=
1
$$

Multiply each path by the same upstream gradient:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}\,s
$$

$$
\frac{\partial L}{\partial s}
=
\mathrm{dout}\,x
$$

$$
\frac{\partial L}{\partial b}
=
\mathrm{dout}
$$

Because all three forward tensors have the same shape here, these are elementwise paths; no reduction is needed in this exercise.

**Two required backward passes:** First write every requested `_manual` gradient as a literal tensor containing either simplified values or explicit chain-rule arithmetic made from the visible numeric values. Then derive the same gradients with scalable PyTorch tensor operations using the original variable names. Also predict the forward output shape.

**Ingredients:** Backward through addition first, then multiplication; each operation is elementwise here.

**Required outputs:**

- `ex033_dx_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex033_dx`: the same gradient with respect to `x`, derived with PyTorch tensor operations
- `ex033_dscale_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex033_dscale`: the same gradient with respect to `scale`, derived with PyTorch tensor operations
- `ex033_dshift_manual`: the same gradient written as a literal tensor of visible-number arithmetic or simplified values; do not refer to supplied tensor variable names
- `ex033_dshift`: the same gradient with respect to `shift`, derived with PyTorch tensor operations


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex033_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex033_scale_shape`: predict the shape of `scale` as a literal Python tuple
- `ex033_shift_shape`: predict the shape of `shift` as a literal Python tuple
- `ex033_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex033_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex033_dx_shape`: predict the shape of `ex033_dx` as a literal Python tuple
- `ex033_dscale_shape`: predict the shape of `ex033_dscale` as a literal Python tuple
- `ex033_dshift_shape`: predict the shape of `ex033_dshift` as a literal Python tuple

**Next concept:** Vector sum backward.


In [98]:
# Supplied forward tensors and upstream gradient are visible for mental calculation.
x = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE, requires_grad=True)
scale = torch.tensor([2.0, 3.0, -4.0], dtype=DTYPE, requires_grad=True)
shift = torch.tensor([0.5, 1.0, -1.0], dtype=DTYPE, requires_grad=True)
y = x * scale + shift
dout = torch.tensor([1.0, -0.5, 2.0], dtype=DTYPE)

# Register private expected gradients without displaying their calculation.
_run_gradient_reference(33, globals())
print("Visible gradient inputs ready.")


Visible gradient inputs ready.


In [99]:
# Exercise 033: complete mental and PyTorch backward passes; do not use autograd.
# Explicit pass: `ex033_dx_manual` may use visible-number arithmetic or simplified values.
# Explicit pass: `ex033_dscale_manual` may use visible-number arithmetic or simplified values.
# Explicit pass: `ex033_dshift_manual` may use visible-number arithmetic or simplified values.
# PyTorch pass: derive the existing non-suffixed gradient variables with tensor operations.
# Define `ex033_dx` — derive the gradient with respect to `x` using PyTorch tensor operations.
# Define `ex033_dscale` — derive the gradient with respect to `scale` using PyTorch tensor operations.
# Define `ex033_dshift` — derive the gradient with respect to `shift` using PyTorch tensor operations.
# Predict every visible tensor shape before computing values.
# Define `ex033_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex033_scale_shape` — the shape of `scale` as a literal Python tuple.
# Define `ex033_shift_shape` — the shape of `shift` as a literal Python tuple.
# Define `ex033_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex033_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex033_dx_shape` — the shape of `ex033_dx` as a literal Python tuple.
# Define `ex033_dscale_shape` — the shape of `ex033_dscale` as a literal Python tuple.
# Define `ex033_dshift_shape` — the shape of `ex033_dshift` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex033_x_shape = (3,)
ex033_scale_shape = (3,)
ex033_shift_shape = (3,)
ex033_y_shape = (3,)
ex033_dout_shape = (3,)
ex033_dx_shape = (3,)
ex033_dscale_shape = (3,)
ex033_dshift_shape = (3,)
ex033_dx_manual = torch.tensor([1.0 * 2.0, -0.5 * 3.0, 2.0 * -4.0], dtype=DTYPE)
ex033_dx = dout * scale
ex033_dscale_manual = torch.tensor([1.0 * 1.0, -0.5 * -2.0, 2.0 * 0.5], dtype=DTYPE)
ex033_dscale = dout * x
ex033_dshift_manual = torch.tensor([1.0 * 1.0, -0.5 * 1.0, 2.0 * 1.0], dtype=DTYPE)
ex033_dshift = dout * 1.0

In [100]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex033_x_shape", "x")
_check_visible_tensor_shape("ex033_scale_shape", "scale")
_check_visible_tensor_shape("ex033_shift_shape", "shift")
_check_visible_tensor_shape("ex033_y_shape", "y")
_check_visible_tensor_shape("ex033_dout_shape", "dout")
_check_private_tensor_shape("ex033_dx_shape", "ex033", "dx")
_check_private_tensor_shape("ex033_dscale_shape", "ex033", "dscale")
_check_private_tensor_shape("ex033_dshift_shape", "ex033", "dshift")
_check_tensor("ex033_dx_manual", "ex033", "dx")
_check_tensor("ex033_dx", "ex033", "dx")
_check_tensor("ex033_dscale_manual", "ex033", "dscale")
_check_tensor("ex033_dscale", "ex033", "dscale")
_check_tensor("ex033_dshift_manual", "ex033", "dshift")
_check_tensor("ex033_dshift", "ex033", "dshift")


PASS: ex033_x_shape
PASS: ex033_scale_shape
PASS: ex033_shift_shape
PASS: ex033_y_shape
PASS: ex033_dout_shape
PASS: ex033_dx_shape
PASS: ex033_dscale_shape
PASS: ex033_dshift_shape
PASS: ex033_dx_manual
PASS: ex033_dx
PASS: ex033_dscale_manual
PASS: ex033_dscale
PASS: ex033_dshift_manual
PASS: ex033_dshift


## 3. Reductions: forward combines entries, backward distributes gradients

A **reduction** combines several input entries into fewer output entries. Forward may remove every axis, as in `x.sum()`, or only one selected axis, as in `x.sum(dim=0)`. Using `keepdim=True` retains a reduced axis with size one.

Backward sends each output gradient to the input entries that contributed to that output:

- a sum gives every contributor local derivative `1`;
- a mean gives every contributor local derivative `1 / m`, where `m` is the number of averaged values;
- a unique maximum sends the gradient only to its winning entry.

Always track these shape relationships:

```text
shape(dout) = shape(forward output)
shape(dx)   = shape(x)
```

Each exercise below explains its own reduced axis, contributor paths, reduction size, and backward shape. Backward through a reduction does not recover the original input values; it distributes sensitivity through the paths that created the reduced output.


### Exercise 034 — Vector sum backward

**Purpose:** Learn that a scalar gradient from a vector sum is copied back to every vector entry.

**Inputs:** `x` has shape `(4,)`; `y` and `dout` are scalar tensors.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x.sum()`.

**Forward/backward idea:** Forward writes the scalar sum as:

$$
y
=
x_0+x_1+x_2+x_3
$$

If any one entry increases by a small amount, `y` increases by that same amount. Therefore, every input has local derivative `1`:

$$
\left[
\frac{\partial y}{\partial x_0},
\frac{\partial y}{\partial x_1},
\frac{\partial y}{\partial x_2},
\frac{\partial y}{\partial x_3}
\right]
=
[1,1,1,1]
$$

The vector of ones contains local slopes; it is not an attempt to reconstruct `x`. Apply scalar `dout` to each path:

$$
\frac{\partial L}{\partial x}
=
\mathrm{dout}[1,1,1,1]
$$

In PyTorch, `torch.ones_like(x)` represents those four local slopes, and scalar `dout` broadcasts across them to produce `dx` with shape `(4,)`.

**Derive/do:** Derive `dx`.

**Ingredients:** Each input contributes once to the total.

**Required outputs:**

- `ex034_dx`: the length-4 gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex034_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex034_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex034_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex034_dx_shape`: predict the shape of `ex034_dx` as a literal Python tuple

**Next concept:** Vector mean backward.


In [101]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([1.0, -2.0, 3.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x.sum()
dout = torch.tensor(2.5, dtype=DTYPE)
_capture("ex034", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


x (4,) y () dout ()


In [102]:
# Exercise 034: derive manually; do not use autograd in this cell.
# Define `ex034_dx` — the length-4 gradient with respect to `x`.
# A sum gives every input a local derivative of 1, so scalar `dout` is copied to each entry.
# Predict every visible tensor shape before computing values.
# Define `ex034_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex034_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex034_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex034_dx_shape` — the shape of `ex034_dx` as a literal Python tuple.
ex034_x_shape = (4,)
ex034_y_shape = ()
ex034_dout_shape = ()
ex034_dx_shape = (4,)
ex034_dx = dout * torch.ones_like(x)


In [103]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex034_x_shape", "x")
_check_visible_tensor_shape("ex034_y_shape", "y")
_check_visible_tensor_shape("ex034_dout_shape", "dout")
_check_private_tensor_shape("ex034_dx_shape", "ex034", "dx")
_check_tensor("ex034_dx", "ex034", "dx")


PASS: ex034_x_shape
PASS: ex034_y_shape
PASS: ex034_dout_shape
PASS: ex034_dx_shape
PASS: ex034_dx


### Exercise 035 — Vector mean backward

**Purpose:** Learn that a vector mean shares the incoming gradient equally among all vector entries.

**Inputs:** `x` has four entries; `y` and `dout` are scalar tensors.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x.mean()`.

**Forward/backward idea:** Forward adds four vector entries and divides by `4`. Backward gives every entry an equal share, so each position of `dx` receives `dout / 4`.

**Derive/do:** Derive `dx` and identify the number of averaged values.

**Ingredients:** Mean equals sum divided by element count.

**Required outputs:**

- `ex035_dx`: the length-4 gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex035_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex035_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex035_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex035_dx_shape`: predict the shape of `ex035_dx` as a literal Python tuple

**Next concept:** Weighted sum backward.


In [104]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([2.0, 4.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x.mean()
dout = torch.tensor(-2.0, dtype=DTYPE)
_capture("ex035", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


x (4,) y () dout ()


In [105]:
# Exercise 035: derive manually; do not use autograd in this cell.
# Define `ex035_dx` — the length-4 gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex035_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex035_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex035_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex035_dx_shape` — the shape of `ex035_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex035_x_shape = (4,)
ex035_y_shape = ()
ex035_dout_shape = ()
ex035_dx_shape = (4,)
ex035_dx = dout * (1/(len(x)) * torch.ones_like(x))

In [106]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex035_x_shape", "x")
_check_visible_tensor_shape("ex035_y_shape", "y")
_check_visible_tensor_shape("ex035_dout_shape", "dout")
_check_private_tensor_shape("ex035_dx_shape", "ex035", "dx")
_check_tensor("ex035_dx", "ex035", "dx")


PASS: ex035_x_shape
PASS: ex035_y_shape
PASS: ex035_dout_shape
PASS: ex035_dx_shape
PASS: ex035_dx


### Exercise 036 — Weighted sum backward

**Purpose:** Learn to reverse a weighted sum by undoing the sum first and the elementwise multiplication second.

**Inputs:** `x` and `weights` have shape `(4,)`; the output is scalar.

**Forward operation:** `y = (x * weights).sum()`.

**Forward/backward idea:** Forward first forms four elementwise products and then sums them into one scalar. Backward sends scalar `dout` to every product; each `x` gradient uses its matching `weight`, and each `weight` gradient uses its matching `x`.

**Derive/do:** Derive both gradients.

**Ingredients:** Reverse the sum, then reverse the elementwise product.

**Required outputs:**

- `ex036_dx`: the gradient with respect to `x`.
- `ex036_dweights`: the gradient with respect to `weights`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex036_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex036_weights_shape`: predict the shape of `weights` as a literal Python tuple
- `ex036_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex036_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex036_dx_shape`: predict the shape of `ex036_dx` as a literal Python tuple
- `ex036_dweights_shape`: predict the shape of `ex036_dweights` as a literal Python tuple

**Next concept:** Matrix total backward.


In [107]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([1.0, 2.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
weights = torch.tensor([0.5, -2.0, 4.0, 1.5], dtype=DTYPE, requires_grad=True)
y = (x * weights).sum()
dout = torch.tensor(1.25, dtype=DTYPE)
_capture("ex036", y, {"dx": x, "dweights": weights}, dout)
print("Inputs", tuple(x.shape), "output", tuple(y.shape))


Inputs (4,) output ()


In [108]:
# Exercise 036: derive manually; do not use autograd in this cell.
# Define `ex036_dx` — the gradient with respect to `x`.
# Define `ex036_dweights` — the gradient with respect to `weights`.
# Predict every visible tensor shape before computing values.
# Define `ex036_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex036_weights_shape` — the shape of `weights` as a literal Python tuple.
# Define `ex036_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex036_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex036_dx_shape` — the shape of `ex036_dx` as a literal Python tuple.
# Define `ex036_dweights_shape` — the shape of `ex036_dweights` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex036_x_shape = (4,)
ex036_weights_shape = (4,)
ex036_y_shape = ()
ex036_dout_shape = ()
ex036_dx_shape = (4,)
ex036_dweights_shape = (4,)
ex036_dx = dout * torch.ones_like(x * weights) * weights
ex036_dweights = dout * torch.ones_like(x * weights) * x

In [109]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex036_x_shape", "x")
_check_visible_tensor_shape("ex036_weights_shape", "weights")
_check_visible_tensor_shape("ex036_y_shape", "y")
_check_visible_tensor_shape("ex036_dout_shape", "dout")
_check_private_tensor_shape("ex036_dx_shape", "ex036", "dx")
_check_private_tensor_shape("ex036_dweights_shape", "ex036", "dweights")
_check_tensor("ex036_dx", "ex036", "dx")
_check_tensor("ex036_dweights", "ex036", "dweights")


PASS: ex036_x_shape
PASS: ex036_weights_shape
PASS: ex036_y_shape
PASS: ex036_dout_shape
PASS: ex036_dx_shape
PASS: ex036_dweights_shape
PASS: ex036_dx
PASS: ex036_dweights


### Exercise 037 — Matrix total backward

**Purpose:** Learn that the gradient of a matrix total must return to every row-column position.

**Inputs:** `x` has shape `(2, 3)`; the output and upstream gradient are scalar.

**Forward operation:** `y = x.sum()`.

**Forward/backward idea:** Forward combines all six matrix entries into one scalar. Backward gives every row-column position local slope `1`, so scalar `dout` is copied into a `(2, 3)` gradient.

**Derive/do:** Derive `dx` with shape `(2, 3)`.

**Ingredients:** Every matrix entry contributes once to the scalar total.

**Required outputs:**

- `ex037_dx`: the matrix gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex037_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex037_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex037_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex037_dx_shape`: predict the shape of `ex037_dx` as a literal Python tuple

**Next concept:** Matrix mean backward.


In [110]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum()
dout = torch.tensor(-0.75, dtype=DTYPE)
_capture("ex037", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


x (2, 3) y ()


In [111]:
# Exercise 037: derive manually; do not use autograd in this cell.
# Define `ex037_dx` — the matrix gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex037_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex037_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex037_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex037_dx_shape` — the shape of `ex037_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex037_x_shape = (2, 3)
ex037_y_shape = ()
ex037_dout_shape = ()
ex037_dx_shape = (2, 3)
ex037_dx = dout * torch.ones_like(x)

In [112]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex037_x_shape", "x")
_check_visible_tensor_shape("ex037_y_shape", "y")
_check_visible_tensor_shape("ex037_dout_shape", "dout")
_check_private_tensor_shape("ex037_dx_shape", "ex037", "dx")
_check_tensor("ex037_dx", "ex037", "dx")


PASS: ex037_x_shape
PASS: ex037_y_shape
PASS: ex037_dout_shape
PASS: ex037_dx_shape
PASS: ex037_dx


### Exercise 038 — Matrix mean backward

**Purpose:** Learn that a matrix mean divides the incoming gradient among every matrix entry.

**Inputs:** `x` has six entries in shape `(2, 3)`.

**Forward operation:** `y = x.mean()`.

**Forward/backward idea:** Forward averages all six matrix entries into one scalar. Backward distributes `dout` equally across those six contributors, so every input position receives `dout / 6`.

**Derive/do:** Derive `dx`; use the total number of averaged entries.

**Ingredients:** Mean reduction over all axes and a scalar upstream gradient.

**Required outputs:**

- `ex038_dx`: the matrix gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex038_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex038_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex038_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex038_dx_shape`: predict the shape of `ex038_dx` as a literal Python tuple

**Next concept:** Column sums backward.


In [113]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.mean()
dout = torch.tensor(3.0, dtype=DTYPE)
_capture("ex038", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


x (2, 3) y ()


In [114]:
# Exercise 038: derive manually; do not use autograd in this cell.
# Define `ex038_dx` — the matrix gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex038_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex038_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex038_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex038_dx_shape` — the shape of `ex038_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex038_x_shape = (2, 3)
ex038_y_shape = ()
ex038_dout_shape = ()
ex038_dx_shape = (2, 3)
ex038_dx = dout * (1 / x.numel()) * torch.ones_like(x)

In [115]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex038_x_shape", "x")
_check_visible_tensor_shape("ex038_y_shape", "y")
_check_visible_tensor_shape("ex038_dout_shape", "dout")
_check_private_tensor_shape("ex038_dx_shape", "ex038", "dx")
_check_tensor("ex038_dx", "ex038", "dx")


PASS: ex038_x_shape
PASS: ex038_y_shape
PASS: ex038_dout_shape
PASS: ex038_dx_shape
PASS: ex038_dx


### Exercise 039 — Column sums backward

**Purpose:** Learn that each column-sum gradient is copied back down all rows of that column.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x.sum(dim=0)`.

**Forward/backward idea:** Forward reduces dimension `0`, combining the two rows while keeping the three columns. Each output entry belongs to one column, so backward copies `dout[j]` down every row of column `j`.

**Natural misconception:** `dim=0` changes the grouping and routing, but it does not change the local derivative of addition. For column `j`, forward computes:

$$
y_j
=
\sum_i x_{i,j}
$$

Every contributing entry is added once, so:

$$
\frac{\partial y_j}{\partial x_{i,j}}
=
1
$$

The backward value at each original matrix position is therefore:

$$
\frac{\partial L}{\partial x_{i,j}}
=
\mathrm{dout}_j
$$

The `(2, 3)` tensor of ones represents these local slopes. The `(3,)` upstream tensor contains one gradient per retained column and broadcasts down the two rows that were reduced. In short: **sum determines the slope `1`; `dim=0` determines where each column gradient is copied.**

**Derive/do:** Derive `dx`.

**Ingredients:** Dimension 0 was reduced; determine which upstream entry belongs to each input column.

**Required outputs:**

- `ex039_dx`: the `(2, 3)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex039_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex039_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex039_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex039_dx_shape`: predict the shape of `ex039_dx` as a literal Python tuple

**Next concept:** Row sums backward.


In [116]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum(dim=0)
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex039", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


x (2, 3) y (3,) dout (3,)


In [117]:
# Exercise 039: derive manually; do not use autograd in this cell.
# Define `ex039_dx` — the `(2, 3)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex039_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex039_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex039_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex039_dx_shape` — the shape of `ex039_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex039_x_shape = (2, 3)
ex039_y_shape = (3,)
ex039_dout_shape = (3,)
ex039_dx_shape = (2, 3)
ex039_dx = dout * torch.ones_like(x)

In [118]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex039_x_shape", "x")
_check_visible_tensor_shape("ex039_y_shape", "y")
_check_visible_tensor_shape("ex039_dout_shape", "dout")
_check_private_tensor_shape("ex039_dx_shape", "ex039", "dx")
_check_tensor("ex039_dx", "ex039", "dx")


PASS: ex039_x_shape
PASS: ex039_y_shape
PASS: ex039_dout_shape
PASS: ex039_dx_shape
PASS: ex039_dx


### Exercise 040 — Row sums backward

**Purpose:** Learn that each row-sum gradient is copied back across all columns of that row.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(2,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x.sum(dim=1)`.

**Forward/backward idea:** Forward reduces dimension `1`, combining the three columns while keeping the two rows. Each output entry belongs to one row, so backward copies `dout[i]` across every column of row `i`.

**Derive/do:** Derive `dx`.

**Ingredients:** Dimension 1 was reduced; introduce an axis if needed to broadcast row gradients.

**Required outputs:**

- `ex040_dx`: the `(2, 3)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex040_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex040_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex040_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex040_dx_shape`: predict the shape of `ex040_dx` as a literal Python tuple

**Next concept:** Row sums with keepdim backward.


In [119]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum(dim=1)
dout = torch.tensor([2.0, -1.0], dtype=DTYPE)
_capture("ex040", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


x (2, 3) y (2,) dout (2,)


In [129]:
a = torch.tensor([2, 2])
b = torch.tensor([[1,2],[3,4]])
a * b


tensor([[2, 4],
        [6, 8]])

In [123]:
# Exercise 040: derive manually; do not use autograd in this cell.
# Define `ex040_dx` — the `(2, 3)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex040_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex040_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex040_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex040_dx_shape` — the shape of `ex040_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.
ex040_x_shape = (2, 3)
ex040_y_shape = (2,)
ex040_dout_shape = (2,)
ex040_dx_shape = (2, 3)
ex040_dx = dout * torch.ones_like(x)

RuntimeError: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 1

In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex040_x_shape", "x")
_check_visible_tensor_shape("ex040_y_shape", "y")
_check_visible_tensor_shape("ex040_dout_shape", "dout")
_check_private_tensor_shape("ex040_dx_shape", "ex040", "dx")
_check_tensor("ex040_dx", "ex040", "dx")


### Exercise 041 — Row sums with keepdim backward

**Purpose:** Learn how keeping a reduced axis at length 1 makes the backward shape easier to reuse.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(2, 1)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x.sum(dim=1, keepdim=True)`.

**Forward/backward idea:** Forward still sums each row, but `keepdim=True` keeps the reduced column axis with size `1`. Backward can therefore expand the `(2, 1)` upstream gradient directly across the three columns without first inserting a new axis.

**Derive/do:** Derive `dx`.

**Ingredients:** The upstream tensor already has the singleton column axis needed for expansion.

**Required outputs:**

- `ex041_dx`: the `(2, 3)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex041_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex041_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex041_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex041_dx_shape`: predict the shape of `ex041_dx` as a literal Python tuple

**Next concept:** Column means backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum(dim=1, keepdim=True)
dout = torch.tensor([[2.0], [-1.0]], dtype=DTYPE)
_capture("ex041", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 041: derive manually; do not use autograd in this cell.
# Define `ex041_dx` — the `(2, 3)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex041_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex041_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex041_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex041_dx_shape` — the shape of `ex041_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex041_x_shape", "x")
_check_visible_tensor_shape("ex041_y_shape", "y")
_check_visible_tensor_shape("ex041_dout_shape", "dout")
_check_private_tensor_shape("ex041_dx_shape", "ex041", "dx")
_check_tensor("ex041_dx", "ex041", "dx")


### Exercise 042 — Column means backward

**Purpose:** Learn that each column-mean gradient is copied down the column and divided by the number of rows.

**Inputs:** `x` has shape `(4, 3)`; `y` and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x.mean(dim=0)`.

**Forward/backward idea:** Forward averages the four rows independently for each of the three columns. Backward copies each feature gradient `dout[j]` down column `j` and divides every copy by `4`.

**Derive/do:** Derive `dx`.

**Ingredients:** There are four training-example rows in each column mean.

**Required outputs:**

- `ex042_dx`: the `(4, 3)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex042_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex042_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex042_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex042_dx_shape`: predict the shape of `ex042_dx` as a literal Python tuple

**Next concept:** Row means with keepdim backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(12, dtype=DTYPE).reshape(4, 3).requires_grad_()
y = x.mean(dim=0)
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex042", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 042: derive manually; do not use autograd in this cell.
# Define `ex042_dx` — the `(4, 3)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex042_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex042_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex042_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex042_dx_shape` — the shape of `ex042_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex042_x_shape", "x")
_check_visible_tensor_shape("ex042_y_shape", "y")
_check_visible_tensor_shape("ex042_dout_shape", "dout")
_check_private_tensor_shape("ex042_dx_shape", "ex042", "dx")
_check_tensor("ex042_dx", "ex042", "dx")


### Exercise 043 — Row means with keepdim backward

**Purpose:** Learn that each row-mean gradient is copied across the row and divided by the number of columns.

**Inputs:** `x` has shape `(3, 4)`; `y` and `dout` have shape `(3, 1)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x.mean(dim=1, keepdim=True)`.

**Forward/backward idea:** Forward averages four columns independently within each row and keeps the column axis at size `1`. Backward copies each row gradient across its four columns and divides every copy by `4`.

**Derive/do:** Derive `dx`.

**Ingredients:** Each row mean averages four feature entries.

**Required outputs:**

- `ex043_dx`: the `(3, 4)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex043_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex043_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex043_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex043_dx_shape`: predict the shape of `ex043_dx` as a literal Python tuple

**Next concept:** Sum of squares.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
y = x.mean(dim=1, keepdim=True)
dout = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE)
_capture("ex043", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 043: derive manually; do not use autograd in this cell.
# Define `ex043_dx` — the `(3, 4)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex043_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex043_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex043_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex043_dx_shape` — the shape of `ex043_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex043_x_shape", "x")
_check_visible_tensor_shape("ex043_y_shape", "y")
_check_visible_tensor_shape("ex043_dout_shape", "dout")
_check_private_tensor_shape("ex043_dx_shape", "ex043", "dx")
_check_tensor("ex043_dx", "ex043", "dx")


### Exercise 044 — Sum of squares

**Purpose:** Learn to reverse a scalar sum of squared tensor entries.

**Inputs:** `x` has shape `(4,)`; `y` is scalar.

**Forward operation:** `y = (x**2).sum()`.

**Forward/backward idea:** Forward squares each entry and then sums the four squares into one scalar. Backward reverses those steps: first copy scalar `dout` to every squared entry, then multiply by the square's local derivative `2 * x`.

**Derive/do:** Derive `dx` by reversing the sum and square.

**Ingredients:** A scalar upstream gradient and the square local derivative.

**Required outputs:**

- `ex044_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex044_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex044_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex044_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex044_dx_shape`: predict the shape of `ex044_dx` as a literal Python tuple

**Next concept:** Row-wise sums of squares.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-2.0, -0.5, 1.0, 3.0], dtype=DTYPE, requires_grad=True)
y = (x**2).sum()
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex044", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 044: derive manually; do not use autograd in this cell.
# Define `ex044_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex044_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex044_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex044_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex044_dx_shape` — the shape of `ex044_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex044_x_shape", "x")
_check_visible_tensor_shape("ex044_y_shape", "y")
_check_visible_tensor_shape("ex044_dout_shape", "dout")
_check_private_tensor_shape("ex044_dx_shape", "ex044", "dx")
_check_tensor("ex044_dx", "ex044", "dx")


### Exercise 045 — Row-wise sums of squares

**Purpose:** Learn to reverse one separate sum of squares for each row.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(2,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = (x**2).sum(dim=1)`.

**Forward/backward idea:** Forward squares every entry and produces one sum per row. Backward copies each row's `dout[i]` across that row, then multiplies elementwise by the square derivative `2 * x`.

**Derive/do:** Derive `dx`.

**Ingredients:** Expand each row's upstream gradient, then reverse the square.

**Required outputs:**

- `ex045_dx`: the `(2, 3)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex045_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex045_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex045_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex045_dx_shape`: predict the shape of `ex045_dx` as a literal Python tuple

**Next concept:** Column means of squares.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([[-2.0, 1.0, 3.0], [0.5, -1.0, 2.0]], dtype=DTYPE, requires_grad=True)
y = (x**2).sum(dim=1)
dout = torch.tensor([2.0, -0.5], dtype=DTYPE)
_capture("ex045", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 045: derive manually; do not use autograd in this cell.
# Define `ex045_dx` — the `(2, 3)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex045_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex045_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex045_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex045_dx_shape` — the shape of `ex045_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex045_x_shape", "x")
_check_visible_tensor_shape("ex045_y_shape", "y")
_check_visible_tensor_shape("ex045_dout_shape", "dout")
_check_private_tensor_shape("ex045_dx_shape", "ex045", "dx")
_check_tensor("ex045_dx", "ex045", "dx")


### Exercise 046 — Column means of squares

**Purpose:** Learn to reverse a square followed by one mean for each column.

**Inputs:** `x` has shape `(4, 3)`; `y` and `dout` have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = (x**2).mean(dim=0)`.

**Forward/backward idea:** Forward squares every entry and then averages the four rows separately for each column. Backward copies each `dout[j] / 4` down column `j`, then multiplies elementwise by `2 * x`.

**Derive/do:** Derive `dx`.

**Ingredients:** Reverse the mean over four rows, then reverse the square.

**Required outputs:**

- `ex046_dx`: the `(4, 3)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex046_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex046_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex046_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex046_dx_shape`: predict the shape of `ex046_dx` as a literal Python tuple

**Next concept:** Unique vector maximum backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
y = (x**2).mean(dim=0)
dout = torch.tensor([1.0, -1.5, 0.25], dtype=DTYPE)
_capture("ex046", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 046: derive manually; do not use autograd in this cell.
# Define `ex046_dx` — the `(4, 3)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex046_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex046_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex046_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex046_dx_shape` — the shape of `ex046_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex046_x_shape", "x")
_check_visible_tensor_shape("ex046_y_shape", "y")
_check_visible_tensor_shape("ex046_dout_shape", "dout")
_check_private_tensor_shape("ex046_dx_shape", "ex046", "dx")
_check_tensor("ex046_dx", "ex046", "dx")


### Exercise 047 — Unique vector maximum backward

**Purpose:** Learn that only the entry that won a unique maximum receives the gradient.

**Inputs:** `x` has shape `(4,)` with one unique maximum; `y` is scalar.

**Forward operation:** `y = x.max()`.

**Forward/backward idea:** Forward selects one scalar: the unique largest vector entry. Backward sends scalar `dout` only to that winning position; every non-winning position receives zero because it did not determine the maximum.

**Derive/do:** Derive `dx`.

**Ingredients:** Build a mask for the unique maximum and apply the scalar upstream gradient.

**Required outputs:**

- `ex047_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex047_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex047_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex047_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex047_dx_shape`: predict the shape of `ex047_dx` as a literal Python tuple

**Next concept:** Unique row maxima backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-1.0, 3.0, 2.0, 0.5], dtype=DTYPE, requires_grad=True)
y = x.max()
dout = torch.tensor(2.0, dtype=DTYPE)
_capture("ex047", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 047: derive manually; do not use autograd in this cell.
# Define `ex047_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex047_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex047_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex047_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex047_dx_shape` — the shape of `ex047_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex047_x_shape", "x")
_check_visible_tensor_shape("ex047_y_shape", "y")
_check_visible_tensor_shape("ex047_dout_shape", "dout")
_check_private_tensor_shape("ex047_dx_shape", "ex047", "dx")
_check_tensor("ex047_dx", "ex047", "dx")


### Exercise 048 — Unique row maxima backward

**Purpose:** Learn that a row-wise maximum sends each row's gradient only to that row's winning entry.

**Inputs:** `x` has shape `(3, 4)` and each row has one unique maximum; `y` has shape `(3,)`.

**Forward operation:** `y = x.max(dim=1).values`.

**Forward/backward idea:** Forward selects one maximum from each row, producing three outputs. Backward sends each `dout[i]` only to the winning position in row `i`; all other positions in that row receive zero.

**Derive/do:** Derive `dx`.

**Ingredients:** One maximum mask per row and one upstream value per row.

**Required outputs:**

- `ex048_dx`: the `(3, 4)` gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex048_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex048_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex048_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex048_dx_shape`: predict the shape of `ex048_dx` as a literal Python tuple

**Next concept:** Reduction chain.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([[1.0, 4.0, 2.0, 0.0], [3.0, -1.0, 5.0, 2.0], [7.0, 1.0, 0.0, 6.0]], dtype=DTYPE, requires_grad=True)
y = x.max(dim=1).values
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex048", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 048: derive manually; do not use autograd in this cell.
# Define `ex048_dx` — the `(3, 4)` gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex048_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex048_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex048_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex048_dx_shape` — the shape of `ex048_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex048_x_shape", "x")
_check_visible_tensor_shape("ex048_y_shape", "y")
_check_visible_tensor_shape("ex048_dout_shape", "dout")
_check_private_tensor_shape("ex048_dx_shape", "ex048", "dx")
_check_tensor("ex048_dx", "ex048", "dx")


### Exercise 049 — Reduction chain

**Purpose:** Learn to reverse two reductions in the opposite order from the forward pass.

**Inputs:** `x` has shape `(2, 3)`; the final output is scalar.

**Forward operation:** `columns = x.sum(dim=0)` and `y = columns.mean()`.

**Forward/backward idea:** Forward first sums the two rows into three column totals, then averages those three totals into one scalar. Backward reverses the order: distribute `dout / 3` to the three column totals, then copy each column-total gradient down the two rows that formed it.

**Derive/do:** Derive gradients for `columns` and `x`.

**Ingredients:** Reverse the mean over three column totals, then reverse the row reduction.

**Required outputs:**

- `ex049_dcolumns`: the gradient with respect to the three column totals.
- `ex049_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex049_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex049_columns_shape`: predict the shape of `columns` as a literal Python tuple
- `ex049_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex049_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex049_dcolumns_shape`: predict the shape of `ex049_dcolumns` as a literal Python tuple
- `ex049_dx_shape`: predict the shape of `ex049_dx` as a literal Python tuple

**Next concept:** Add a row bias.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
columns = x.sum(dim=0)
y = columns.mean()
dout = torch.tensor(-1.5, dtype=DTYPE)
_capture("ex049", y, {"dcolumns": columns, "dx": x}, dout)
print("x", tuple(x.shape), "columns", tuple(columns.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 049: derive manually; do not use autograd in this cell.
# Define `ex049_dcolumns` — the gradient with respect to the three column totals.
# Define `ex049_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex049_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex049_columns_shape` — the shape of `columns` as a literal Python tuple.
# Define `ex049_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex049_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex049_dcolumns_shape` — the shape of `ex049_dcolumns` as a literal Python tuple.
# Define `ex049_dx_shape` — the shape of `ex049_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex049_x_shape", "x")
_check_visible_tensor_shape("ex049_columns_shape", "columns")
_check_visible_tensor_shape("ex049_y_shape", "y")
_check_visible_tensor_shape("ex049_dout_shape", "dout")
_check_private_tensor_shape("ex049_dcolumns_shape", "ex049", "dcolumns")
_check_private_tensor_shape("ex049_dx_shape", "ex049", "dx")
_check_tensor("ex049_dcolumns", "ex049", "dcolumns")
_check_tensor("ex049_dx", "ex049", "dx")


## 4. Broadcasting: backward must unbroadcast

Broadcasting reuses a smaller tensor at many output positions. Each reuse creates a path to the objective. Backward must add all path contributions until the gradient has the original input shape.

Use this rule:

> If an axis had size 1 before forward broadcasting, sum the gradient over that expanded axis in backward and keep the axis when the original input kept it.

Never stop at a broadcast output shape when computing the gradient of a smaller input.


### Exercise 050 — Add a row bias

**Purpose:** Learn that one feature bias is reused by every example, so its gradient adds contributions from all rows.

**Inputs:** `x` has shape `(3, 4)`, `bias` has shape `(4,)`, and `dout` has shape `(3, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x + bias`.

**Derive/do:** Derive `dx` and `dbias`.

**Ingredients:** The bias is reused down the three-row axis; its gradient must return to shape `(4,)`.

**Required outputs:**

- `ex050_dx`: the gradient with respect to `x`.
- `ex050_dbias`: the unbroadcast gradient with respect to `bias`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex050_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex050_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex050_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex050_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex050_dx_shape`: predict the shape of `ex050_dx` as a literal Python tuple
- `ex050_dbias_shape`: predict the shape of `ex050_dbias` as a literal Python tuple

**Next concept:** Add a column bias.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x + bias
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex050", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 050: derive manually; do not use autograd in this cell.
# Define `ex050_dx` — the gradient with respect to `x`.
# Define `ex050_dbias` — the unbroadcast gradient with respect to `bias`.
# Predict every visible tensor shape before computing values.
# Define `ex050_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex050_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex050_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex050_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex050_dx_shape` — the shape of `ex050_dx` as a literal Python tuple.
# Define `ex050_dbias_shape` — the shape of `ex050_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex050_x_shape", "x")
_check_visible_tensor_shape("ex050_bias_shape", "bias")
_check_visible_tensor_shape("ex050_y_shape", "y")
_check_visible_tensor_shape("ex050_dout_shape", "dout")
_check_private_tensor_shape("ex050_dx_shape", "ex050", "dx")
_check_private_tensor_shape("ex050_dbias_shape", "ex050", "dbias")
_check_tensor("ex050_dx", "ex050", "dx")
_check_tensor("ex050_dbias", "ex050", "dbias")


### Exercise 051 — Add a column bias

**Purpose:** Learn that a `(3, 1)` value reused across columns receives one summed gradient per row.

**Inputs:** `x` has shape `(3, 4)`, `bias` has shape `(3, 1)`, and `dout` has shape `(3, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x + bias`.

**Derive/do:** Derive `dx` and `dbias`.

**Ingredients:** The size-one column axis expanded from 1 to 4; restore it in backward.

**Required outputs:**

- `ex051_dx`: the gradient with respect to `x`.
- `ex051_dbias`: the `(3, 1)` gradient with respect to `bias`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex051_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex051_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex051_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex051_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex051_dx_shape`: predict the shape of `ex051_dx` as a literal Python tuple
- `ex051_dbias_shape`: predict the shape of `ex051_dbias` as a literal Python tuple

**Next concept:** Add a scalar parameter.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([[0.5], [-1.0], [2.0]], dtype=DTYPE, requires_grad=True)
y = x + bias
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex051", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 051: derive manually; do not use autograd in this cell.
# Define `ex051_dx` — the gradient with respect to `x`.
# Define `ex051_dbias` — the `(3, 1)` gradient with respect to `bias`.
# Predict every visible tensor shape before computing values.
# Define `ex051_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex051_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex051_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex051_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex051_dx_shape` — the shape of `ex051_dx` as a literal Python tuple.
# Define `ex051_dbias_shape` — the shape of `ex051_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex051_x_shape", "x")
_check_visible_tensor_shape("ex051_bias_shape", "bias")
_check_visible_tensor_shape("ex051_y_shape", "y")
_check_visible_tensor_shape("ex051_dout_shape", "dout")
_check_private_tensor_shape("ex051_dx_shape", "ex051", "dx")
_check_private_tensor_shape("ex051_dbias_shape", "ex051", "dbias")
_check_tensor("ex051_dx", "ex051", "dx")
_check_tensor("ex051_dbias", "ex051", "dbias")


### Exercise 052 — Add a scalar parameter

**Purpose:** Learn that a scalar reused at every matrix position receives all of those gradient contributions added together.

**Inputs:** `x` has shape `(2, 3)`, `shift` has shape `()`, and `dout` has shape `(2, 3)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x + shift`.

**Derive/do:** Derive `dx` and scalar `dshift`.

**Ingredients:** The scalar was reused at all six output positions.

**Required outputs:**

- `ex052_dx`: the gradient with respect to `x`.
- `ex052_dshift`: the scalar gradient with respect to `shift`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex052_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex052_shift_shape`: predict the shape of `shift` as a literal Python tuple
- `ex052_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex052_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex052_dx_shape`: predict the shape of `ex052_dx` as a literal Python tuple
- `ex052_dshift_shape`: predict the shape of `ex052_dshift` as a literal Python tuple

**Next concept:** Multiply by a row scale.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
shift = torch.tensor(1.5, dtype=DTYPE, requires_grad=True)
y = x + shift
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex052", y, {"dx": x, "dshift": shift}, dout)
print("x", tuple(x.shape), "shift", tuple(shift.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 052: derive manually; do not use autograd in this cell.
# Define `ex052_dx` — the gradient with respect to `x`.
# Define `ex052_dshift` — the scalar gradient with respect to `shift`.
# Predict every visible tensor shape before computing values.
# Define `ex052_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex052_shift_shape` — the shape of `shift` as a literal Python tuple.
# Define `ex052_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex052_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex052_dx_shape` — the shape of `ex052_dx` as a literal Python tuple.
# Define `ex052_dshift_shape` — the shape of `ex052_dshift` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex052_x_shape", "x")
_check_visible_tensor_shape("ex052_shift_shape", "shift")
_check_visible_tensor_shape("ex052_y_shape", "y")
_check_visible_tensor_shape("ex052_dout_shape", "dout")
_check_private_tensor_shape("ex052_dx_shape", "ex052", "dx")
_check_private_tensor_shape("ex052_dshift_shape", "ex052", "dshift")
_check_tensor("ex052_dx", "ex052", "dx")
_check_tensor("ex052_dshift", "ex052", "dshift")


### Exercise 053 — Multiply by a row scale

**Purpose:** Learn that one feature scale reused by every row receives contributions from all rows.

**Inputs:** `x` has shape `(3, 4)`, `scale` has shape `(4,)`, and `dout` has shape `(3, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x * scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** First derive per-output product contributions; then sum scale contributions over examples.

**Required outputs:**

- `ex053_dx`: the gradient with respect to `x`.
- `ex053_dscale`: the unbroadcast gradient with respect to `scale`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex053_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex053_scale_shape`: predict the shape of `scale` as a literal Python tuple
- `ex053_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex053_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex053_dx_shape`: predict the shape of `ex053_dx` as a literal Python tuple
- `ex053_dscale_shape`: predict the shape of `ex053_dscale` as a literal Python tuple

**Next concept:** Multiply by a column scale.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x * scale
dout = torch.linspace(-0.5, 1.7, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex053", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 053: derive manually; do not use autograd in this cell.
# Define `ex053_dx` — the gradient with respect to `x`.
# Define `ex053_dscale` — the unbroadcast gradient with respect to `scale`.
# Predict every visible tensor shape before computing values.
# Define `ex053_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex053_scale_shape` — the shape of `scale` as a literal Python tuple.
# Define `ex053_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex053_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex053_dx_shape` — the shape of `ex053_dx` as a literal Python tuple.
# Define `ex053_dscale_shape` — the shape of `ex053_dscale` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex053_x_shape", "x")
_check_visible_tensor_shape("ex053_scale_shape", "scale")
_check_visible_tensor_shape("ex053_y_shape", "y")
_check_visible_tensor_shape("ex053_dout_shape", "dout")
_check_private_tensor_shape("ex053_dx_shape", "ex053", "dx")
_check_private_tensor_shape("ex053_dscale_shape", "ex053", "dscale")
_check_tensor("ex053_dx", "ex053", "dx")
_check_tensor("ex053_dscale", "ex053", "dscale")


### Exercise 054 — Multiply by a column scale

**Purpose:** Learn that one scale per row receives contributions from every column in that row.

**Inputs:** `x` has shape `(3, 4)`, `scale` has shape `(3, 1)`, and `dout` has shape `(3, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x * scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** Sum scale contributions over the four-column axis with `keepdim=True`.

**Required outputs:**

- `ex054_dx`: the gradient with respect to `x`.
- `ex054_dscale`: the `(3, 1)` gradient with respect to `scale`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex054_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex054_scale_shape`: predict the shape of `scale` as a literal Python tuple
- `ex054_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex054_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex054_dx_shape`: predict the shape of `ex054_dx` as a literal Python tuple
- `ex054_dscale_shape`: predict the shape of `ex054_dscale` as a literal Python tuple

**Next concept:** Two-way broadcasted addition.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([[0.5], [-1.0], [2.0]], dtype=DTYPE, requires_grad=True)
y = x * scale
dout = torch.linspace(-0.5, 1.7, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex054", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 054: derive manually; do not use autograd in this cell.
# Define `ex054_dx` — the gradient with respect to `x`.
# Define `ex054_dscale` — the `(3, 1)` gradient with respect to `scale`.
# Predict every visible tensor shape before computing values.
# Define `ex054_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex054_scale_shape` — the shape of `scale` as a literal Python tuple.
# Define `ex054_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex054_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex054_dx_shape` — the shape of `ex054_dx` as a literal Python tuple.
# Define `ex054_dscale_shape` — the shape of `ex054_dscale` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex054_x_shape", "x")
_check_visible_tensor_shape("ex054_scale_shape", "scale")
_check_visible_tensor_shape("ex054_y_shape", "y")
_check_visible_tensor_shape("ex054_dout_shape", "dout")
_check_private_tensor_shape("ex054_dx_shape", "ex054", "dx")
_check_private_tensor_shape("ex054_dscale_shape", "ex054", "dscale")
_check_tensor("ex054_dx", "ex054", "dx")
_check_tensor("ex054_dscale", "ex054", "dscale")


### Exercise 055 — Two-way broadcasted addition

**Purpose:** Learn how tensors shaped `(3, 1)` and `(1, 4)` are reused to make a `(3, 4)` sum, and how their gradients shrink back.

**Inputs:** `left` has shape `(3, 1)`, `right` has shape `(1, 4)`, and `y` has shape `(3, 4)`.

**Forward operation:** `y = left + right`.

**Derive/do:** Derive gradients with each operand's original shape.

**Ingredients:** `left` repeats across columns; `right` repeats across rows.

**Required outputs:**

- `ex055_dleft`: the `(3, 1)` gradient with respect to `left`.
- `ex055_dright`: the `(1, 4)` gradient with respect to `right`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex055_left_shape`: predict the shape of `left` as a literal Python tuple
- `ex055_right_shape`: predict the shape of `right` as a literal Python tuple
- `ex055_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex055_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex055_dleft_shape`: predict the shape of `ex055_dleft` as a literal Python tuple
- `ex055_dright_shape`: predict the shape of `ex055_dright` as a literal Python tuple

**Next concept:** Two-way broadcasted multiplication.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
left = torch.tensor([[1.0], [2.0], [3.0]], dtype=DTYPE, requires_grad=True)
right = torch.tensor([[0.5, -1.0, 2.0, 4.0]], dtype=DTYPE, requires_grad=True)
y = left + right
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex055", y, {"dleft": left, "dright": right}, dout)
print("left", tuple(left.shape), "right", tuple(right.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 055: derive manually; do not use autograd in this cell.
# Define `ex055_dleft` — the `(3, 1)` gradient with respect to `left`.
# Define `ex055_dright` — the `(1, 4)` gradient with respect to `right`.
# Predict every visible tensor shape before computing values.
# Define `ex055_left_shape` — the shape of `left` as a literal Python tuple.
# Define `ex055_right_shape` — the shape of `right` as a literal Python tuple.
# Define `ex055_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex055_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex055_dleft_shape` — the shape of `ex055_dleft` as a literal Python tuple.
# Define `ex055_dright_shape` — the shape of `ex055_dright` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex055_left_shape", "left")
_check_visible_tensor_shape("ex055_right_shape", "right")
_check_visible_tensor_shape("ex055_y_shape", "y")
_check_visible_tensor_shape("ex055_dout_shape", "dout")
_check_private_tensor_shape("ex055_dleft_shape", "ex055", "dleft")
_check_private_tensor_shape("ex055_dright_shape", "ex055", "dright")
_check_tensor("ex055_dleft", "ex055", "dleft")
_check_tensor("ex055_dright", "ex055", "dright")


### Exercise 056 — Two-way broadcasted multiplication

**Purpose:** Learn how two differently shaped tensors are reused in multiplication and how each gradient returns to its original shape.

**Inputs:** `left` has shape `(3, 1)`, `right` has shape `(1, 4)`, and `dout` has shape `(3, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = left * right`.

**Derive/do:** Derive both gradients.

**Ingredients:** Use the other factor locally, then reduce the axis on which each input was reused.

**Required outputs:**

- `ex056_dleft`: the `(3, 1)` gradient with respect to `left`.
- `ex056_dright`: the `(1, 4)` gradient with respect to `right`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex056_left_shape`: predict the shape of `left` as a literal Python tuple
- `ex056_right_shape`: predict the shape of `right` as a literal Python tuple
- `ex056_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex056_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex056_dleft_shape`: predict the shape of `ex056_dleft` as a literal Python tuple
- `ex056_dright_shape`: predict the shape of `ex056_dright` as a literal Python tuple

**Next concept:** Broadcast a channel bias in rank 3.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
left = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE, requires_grad=True)
right = torch.tensor([[0.5, -1.0, 2.0, 4.0]], dtype=DTYPE, requires_grad=True)
y = left * right
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex056", y, {"dleft": left, "dright": right}, dout)
print("left", tuple(left.shape), "right", tuple(right.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 056: derive manually; do not use autograd in this cell.
# Define `ex056_dleft` — the `(3, 1)` gradient with respect to `left`.
# Define `ex056_dright` — the `(1, 4)` gradient with respect to `right`.
# Predict every visible tensor shape before computing values.
# Define `ex056_left_shape` — the shape of `left` as a literal Python tuple.
# Define `ex056_right_shape` — the shape of `right` as a literal Python tuple.
# Define `ex056_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex056_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex056_dleft_shape` — the shape of `ex056_dleft` as a literal Python tuple.
# Define `ex056_dright_shape` — the shape of `ex056_dright` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex056_left_shape", "left")
_check_visible_tensor_shape("ex056_right_shape", "right")
_check_visible_tensor_shape("ex056_y_shape", "y")
_check_visible_tensor_shape("ex056_dout_shape", "dout")
_check_private_tensor_shape("ex056_dleft_shape", "ex056", "dleft")
_check_private_tensor_shape("ex056_dright_shape", "ex056", "dright")
_check_tensor("ex056_dleft", "ex056", "dleft")
_check_tensor("ex056_dright", "ex056", "dright")


### Exercise 057 — Broadcast a channel bias in rank 3

**Purpose:** Learn to track batch, channel, and position axes when one channel bias is reused in a rank-3 tensor.

**Inputs:** `x` has shape `(2, 3, 4)` and `bias` has shape `(1, 3, 1)`.

**Forward operation:** `y = x + bias`.

**Derive/do:** Derive `dx` and `dbias`.

**Ingredients:** The bias is reused over batch and position, but not over channel.

**Required outputs:**

- `ex057_dx`: the gradient with respect to `x`.
- `ex057_dbias`: the `(1, 3, 1)` channel-bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex057_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex057_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex057_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex057_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex057_dx_shape`: predict the shape of `ex057_dx` as a literal Python tuple
- `ex057_dbias_shape`: predict the shape of `ex057_dbias` as a literal Python tuple

**Next concept:** Broadcast a feature scale in rank 3.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4).requires_grad_()
bias = torch.tensor([[[0.5], [-1.0], [2.0]]], dtype=DTYPE, requires_grad=True)
y = x + bias
dout = torch.linspace(-1.0, 1.3, steps=24, dtype=DTYPE).reshape(2, 3, 4)
_capture("ex057", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 057: derive manually; do not use autograd in this cell.
# Define `ex057_dx` — the gradient with respect to `x`.
# Define `ex057_dbias` — the `(1, 3, 1)` channel-bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex057_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex057_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex057_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex057_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex057_dx_shape` — the shape of `ex057_dx` as a literal Python tuple.
# Define `ex057_dbias_shape` — the shape of `ex057_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex057_x_shape", "x")
_check_visible_tensor_shape("ex057_bias_shape", "bias")
_check_visible_tensor_shape("ex057_y_shape", "y")
_check_visible_tensor_shape("ex057_dout_shape", "dout")
_check_private_tensor_shape("ex057_dx_shape", "ex057", "dx")
_check_private_tensor_shape("ex057_dbias_shape", "ex057", "dbias")
_check_tensor("ex057_dx", "ex057", "dx")
_check_tensor("ex057_dbias", "ex057", "dbias")


### Exercise 058 — Broadcast a feature scale in rank 3

**Purpose:** Learn that a feature scale on the last axis is reused at every batch and position location.

**Inputs:** `x` has shape `(2, 3, 4)` and `scale` has shape `(4,)`.

**Forward operation:** `y = x * scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** The feature scale is reused over both leading axes.

**Required outputs:**

- `ex058_dx`: the gradient with respect to `x`.
- `ex058_dscale`: the length-4 feature-scale gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex058_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex058_scale_shape`: predict the shape of `scale` as a literal Python tuple
- `ex058_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex058_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex058_dx_shape`: predict the shape of `ex058_dx` as a literal Python tuple
- `ex058_dscale_shape`: predict the shape of `ex058_dscale` as a literal Python tuple

**Next concept:** Broadcasted bias followed by a total.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.linspace(-2.0, 3.0, steps=24, dtype=DTYPE).reshape(2, 3, 4).requires_grad_()
scale = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x * scale
dout = torch.linspace(1.0, -1.3, steps=24, dtype=DTYPE).reshape(2, 3, 4)
_capture("ex058", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 058: derive manually; do not use autograd in this cell.
# Define `ex058_dx` — the gradient with respect to `x`.
# Define `ex058_dscale` — the length-4 feature-scale gradient.
# Predict every visible tensor shape before computing values.
# Define `ex058_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex058_scale_shape` — the shape of `scale` as a literal Python tuple.
# Define `ex058_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex058_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex058_dx_shape` — the shape of `ex058_dx` as a literal Python tuple.
# Define `ex058_dscale_shape` — the shape of `ex058_dscale` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex058_x_shape", "x")
_check_visible_tensor_shape("ex058_scale_shape", "scale")
_check_visible_tensor_shape("ex058_y_shape", "y")
_check_visible_tensor_shape("ex058_dout_shape", "dout")
_check_private_tensor_shape("ex058_dx_shape", "ex058", "dx")
_check_private_tensor_shape("ex058_dscale_shape", "ex058", "dscale")
_check_tensor("ex058_dx", "ex058", "dx")
_check_tensor("ex058_dscale", "ex058", "dscale")


### Exercise 059 — Broadcasted bias followed by a total

**Purpose:** Learn why summing all biased outputs makes the bias gradient count every example that used it.

**Inputs:** `x` has shape `(3, 4)` and `bias` has shape `(4,)`.

**Forward operation:** `pre = x + bias` and `y = pre.sum()`.

**Derive/do:** Derive `dpre`, `dx`, and `dbias`.

**Ingredients:** Reverse the scalar reduction before unbroadcasting the bias path.

**Required outputs:**

- `ex059_dpre`: the gradient entering the broadcasted addition.
- `ex059_dx`: the gradient with respect to `x`.
- `ex059_dbias`: the accumulated bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex059_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex059_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex059_pre_shape`: predict the shape of `pre` as a literal Python tuple
- `ex059_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex059_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex059_dpre_shape`: predict the shape of `ex059_dpre` as a literal Python tuple
- `ex059_dx_shape`: predict the shape of `ex059_dx` as a literal Python tuple
- `ex059_dbias_shape`: predict the shape of `ex059_dbias` as a literal Python tuple

**Next concept:** Broadcasted scale followed by a mean.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x + bias
y = pre.sum()
dout = torch.tensor(1.75, dtype=DTYPE)
_capture("ex059", y, {"dpre": pre, "dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "pre", tuple(pre.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 059: derive manually; do not use autograd in this cell.
# Define `ex059_dpre` — the gradient entering the broadcasted addition.
# Define `ex059_dx` — the gradient with respect to `x`.
# Define `ex059_dbias` — the accumulated bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex059_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex059_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex059_pre_shape` — the shape of `pre` as a literal Python tuple.
# Define `ex059_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex059_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex059_dpre_shape` — the shape of `ex059_dpre` as a literal Python tuple.
# Define `ex059_dx_shape` — the shape of `ex059_dx` as a literal Python tuple.
# Define `ex059_dbias_shape` — the shape of `ex059_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex059_x_shape", "x")
_check_visible_tensor_shape("ex059_bias_shape", "bias")
_check_visible_tensor_shape("ex059_pre_shape", "pre")
_check_visible_tensor_shape("ex059_y_shape", "y")
_check_visible_tensor_shape("ex059_dout_shape", "dout")
_check_private_tensor_shape("ex059_dpre_shape", "ex059", "dpre")
_check_private_tensor_shape("ex059_dx_shape", "ex059", "dx")
_check_private_tensor_shape("ex059_dbias_shape", "ex059", "dbias")
_check_tensor("ex059_dpre", "ex059", "dpre")
_check_tensor("ex059_dx", "ex059", "dx")
_check_tensor("ex059_dbias", "ex059", "dbias")


### Exercise 060 — Broadcasted scale followed by a mean

**Purpose:** Learn how a final mean changes gradients before they return through a reused scale.

**Inputs:** `x` has shape `(3, 4)` and `scale` has shape `(4,)`.

**Forward operation:** `pre = x * scale` and `y = pre.mean()`.

**Derive/do:** Derive `dpre`, `dx`, and `dscale`.

**Ingredients:** The mean averages all 12 output entries before product backward.

**Required outputs:**

- `ex060_dpre`: the gradient entering the broadcasted multiplication.
- `ex060_dx`: the gradient with respect to `x`.
- `ex060_dscale`: the accumulated scale gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex060_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex060_scale_shape`: predict the shape of `scale` as a literal Python tuple
- `ex060_pre_shape`: predict the shape of `pre` as a literal Python tuple
- `ex060_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex060_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex060_dpre_shape`: predict the shape of `ex060_dpre` as a literal Python tuple
- `ex060_dx_shape`: predict the shape of `ex060_dx` as a literal Python tuple
- `ex060_dscale_shape`: predict the shape of `ex060_dscale` as a literal Python tuple

**Next concept:** Center columns by their means.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x * scale
y = pre.mean()
dout = torch.tensor(-2.0, dtype=DTYPE)
_capture("ex060", y, {"dpre": pre, "dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "pre", tuple(pre.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 060: derive manually; do not use autograd in this cell.
# Define `ex060_dpre` — the gradient entering the broadcasted multiplication.
# Define `ex060_dx` — the gradient with respect to `x`.
# Define `ex060_dscale` — the accumulated scale gradient.
# Predict every visible tensor shape before computing values.
# Define `ex060_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex060_scale_shape` — the shape of `scale` as a literal Python tuple.
# Define `ex060_pre_shape` — the shape of `pre` as a literal Python tuple.
# Define `ex060_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex060_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex060_dpre_shape` — the shape of `ex060_dpre` as a literal Python tuple.
# Define `ex060_dx_shape` — the shape of `ex060_dx` as a literal Python tuple.
# Define `ex060_dscale_shape` — the shape of `ex060_dscale` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex060_x_shape", "x")
_check_visible_tensor_shape("ex060_scale_shape", "scale")
_check_visible_tensor_shape("ex060_pre_shape", "pre")
_check_visible_tensor_shape("ex060_y_shape", "y")
_check_visible_tensor_shape("ex060_dout_shape", "dout")
_check_private_tensor_shape("ex060_dpre_shape", "ex060", "dpre")
_check_private_tensor_shape("ex060_dx_shape", "ex060", "dx")
_check_private_tensor_shape("ex060_dscale_shape", "ex060", "dscale")
_check_tensor("ex060_dpre", "ex060", "dpre")
_check_tensor("ex060_dx", "ex060", "dx")
_check_tensor("ex060_dscale", "ex060", "dscale")


### Exercise 061 — Center columns by their means

**Purpose:** Learn that `x` affects centered values both directly and indirectly through the mean computed from `x`.

**Inputs:** `x` has shape `(4, 3)`; `mean` has shape `(1, 3)`; `y` and `dout` have shape `(4, 3)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `mean = x.mean(dim=0, keepdim=True)` and `y = x - mean`.

**Derive/do:** Derive `dmean` and the total `dx` from both paths.

**Ingredients:** Backward through subtraction gives a direct `x` path and a mean path; add contributions at `x`.

**Required outputs:**

- `ex061_dmean`: the `(1, 3)` gradient with respect to the column means.
- `ex061_dx`: the total gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex061_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex061_mean_shape`: predict the shape of `mean` as a literal Python tuple
- `ex061_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex061_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex061_dmean_shape`: predict the shape of `ex061_dmean` as a literal Python tuple
- `ex061_dx_shape`: predict the shape of `ex061_dx` as a literal Python tuple

**Next concept:** Divide by a row scale.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
mean = x.mean(dim=0, keepdim=True)
y = x - mean
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex061", y, {"dmean": mean, "dx": x}, dout)
print("x", tuple(x.shape), "mean", tuple(mean.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 061: derive manually; do not use autograd in this cell.
# Define `ex061_dmean` — the `(1, 3)` gradient with respect to the column means.
# Define `ex061_dx` — the total gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex061_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex061_mean_shape` — the shape of `mean` as a literal Python tuple.
# Define `ex061_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex061_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex061_dmean_shape` — the shape of `ex061_dmean` as a literal Python tuple.
# Define `ex061_dx_shape` — the shape of `ex061_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex061_x_shape", "x")
_check_visible_tensor_shape("ex061_mean_shape", "mean")
_check_visible_tensor_shape("ex061_y_shape", "y")
_check_visible_tensor_shape("ex061_dout_shape", "dout")
_check_private_tensor_shape("ex061_dmean_shape", "ex061", "dmean")
_check_private_tensor_shape("ex061_dx_shape", "ex061", "dx")
_check_tensor("ex061_dmean", "ex061", "dmean")
_check_tensor("ex061_dx", "ex061", "dx")


### Exercise 062 — Divide by a row scale

**Purpose:** Learn how a denominator reused across a row collects gradient contributions from every column.

**Inputs:** `x` has shape `(3, 4)`, positive `scale` has shape `(3, 1)`, and `dout` has shape `(3, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `y = x / scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** Use the division local derivatives, then sum denominator contributions across columns.

**Required outputs:**

- `ex062_dx`: the gradient with respect to `x`.
- `ex062_dscale`: the `(3, 1)` denominator gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex062_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex062_scale_shape`: predict the shape of `scale` as a literal Python tuple
- `ex062_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex062_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex062_dx_shape`: predict the shape of `ex062_dx` as a literal Python tuple
- `ex062_dscale_shape`: predict the shape of `ex062_dscale` as a literal Python tuple

**Next concept:** Normalize rows by their sums.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([[1.0], [2.0], [4.0]], dtype=DTYPE, requires_grad=True)
y = x / scale
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex062", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 062: derive manually; do not use autograd in this cell.
# Define `ex062_dx` — the gradient with respect to `x`.
# Define `ex062_dscale` — the `(3, 1)` denominator gradient.
# Predict every visible tensor shape before computing values.
# Define `ex062_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex062_scale_shape` — the shape of `scale` as a literal Python tuple.
# Define `ex062_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex062_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex062_dx_shape` — the shape of `ex062_dx` as a literal Python tuple.
# Define `ex062_dscale_shape` — the shape of `ex062_dscale` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex062_x_shape", "x")
_check_visible_tensor_shape("ex062_scale_shape", "scale")
_check_visible_tensor_shape("ex062_y_shape", "y")
_check_visible_tensor_shape("ex062_dout_shape", "dout")
_check_private_tensor_shape("ex062_dx_shape", "ex062", "dx")
_check_private_tensor_shape("ex062_dscale_shape", "ex062", "dscale")
_check_tensor("ex062_dx", "ex062", "dx")
_check_tensor("ex062_dscale", "ex062", "dscale")


### Exercise 063 — Normalize rows by their sums

**Purpose:** Learn that each entry affects row normalization both as a numerator and as part of the shared row sum.

**Inputs:** Positive `x` has shape `(3, 4)`; `row_sum` has shape `(3, 1)`.

**Forward operation:** `row_sum = x.sum(dim=1, keepdim=True)` and `y = x / row_sum`.

**Derive/do:** Derive `drow_sum` and total `dx`.

**Ingredients:** Each `x` entry has a direct numerator path and an indirect path through its row total.

**Required outputs:**

- `ex063_drow_sum`: the `(3, 1)` gradient of the shared row denominators.
- `ex063_dx`: the total gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex063_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex063_row_sum_shape`: predict the shape of `row_sum` as a literal Python tuple
- `ex063_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex063_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex063_drow_sum_shape`: predict the shape of `ex063_drow_sum` as a literal Python tuple
- `ex063_dx_shape`: predict the shape of `ex063_dx` as a literal Python tuple

**Next concept:** Broadcasted where with a scalar fallback.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([[1.0, 2.0, 3.0, 4.0], [2.0, 1.0, 4.0, 3.0], [3.0, 5.0, 2.0, 1.0]], dtype=DTYPE, requires_grad=True)
row_sum = x.sum(dim=1, keepdim=True)
y = x / row_sum
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex063", y, {"drow_sum": row_sum, "dx": x}, dout)
print("x", tuple(x.shape), "row_sum", tuple(row_sum.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 063: derive manually; do not use autograd in this cell.
# Define `ex063_drow_sum` — the `(3, 1)` gradient of the shared row denominators.
# Define `ex063_dx` — the total gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex063_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex063_row_sum_shape` — the shape of `row_sum` as a literal Python tuple.
# Define `ex063_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex063_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex063_drow_sum_shape` — the shape of `ex063_drow_sum` as a literal Python tuple.
# Define `ex063_dx_shape` — the shape of `ex063_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex063_x_shape", "x")
_check_visible_tensor_shape("ex063_row_sum_shape", "row_sum")
_check_visible_tensor_shape("ex063_y_shape", "y")
_check_visible_tensor_shape("ex063_dout_shape", "dout")
_check_private_tensor_shape("ex063_drow_sum_shape", "ex063", "drow_sum")
_check_private_tensor_shape("ex063_dx_shape", "ex063", "dx")
_check_tensor("ex063_drow_sum", "ex063", "drow_sum")
_check_tensor("ex063_dx", "ex063", "dx")


### Exercise 064 — Broadcasted where with a scalar fallback

**Purpose:** Learn that `where` sends each output gradient only through the branch chosen by its Boolean mask.

**Inputs:** `x` has shape `(2, 3)`, `fallback` is scalar, and `mask` is Boolean `(2, 3)`.

**Forward operation:** `y = torch.where(mask, x, fallback)`.

**Derive/do:** Derive `dx` and scalar `dfallback`.

**Ingredients:** The mask chooses one active source at each output; the scalar is reused at every false position.

**Required outputs:**

- `ex064_dx`: the masked gradient with respect to `x`.
- `ex064_dfallback`: the accumulated scalar fallback gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex064_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex064_fallback_shape`: predict the shape of `fallback` as a literal Python tuple
- `ex064_mask_shape`: predict the shape of `mask` as a literal Python tuple
- `ex064_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex064_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex064_dx_shape`: predict the shape of `ex064_dx` as a literal Python tuple
- `ex064_dfallback_shape`: predict the shape of `ex064_dfallback` as a literal Python tuple

**Next concept:** A shared bias on two branches.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
fallback = torch.tensor(-2.0, dtype=DTYPE, requires_grad=True)
mask = torch.tensor([[True, False, True], [False, False, True]])
y = torch.where(mask, x, fallback)
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex064", y, {"dx": x, "dfallback": fallback}, dout)
print("x", tuple(x.shape), "fallback", tuple(fallback.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 064: derive manually; do not use autograd in this cell.
# Define `ex064_dx` — the masked gradient with respect to `x`.
# Define `ex064_dfallback` — the accumulated scalar fallback gradient.
# Predict every visible tensor shape before computing values.
# Define `ex064_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex064_fallback_shape` — the shape of `fallback` as a literal Python tuple.
# Define `ex064_mask_shape` — the shape of `mask` as a literal Python tuple.
# Define `ex064_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex064_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex064_dx_shape` — the shape of `ex064_dx` as a literal Python tuple.
# Define `ex064_dfallback_shape` — the shape of `ex064_dfallback` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex064_x_shape", "x")
_check_visible_tensor_shape("ex064_fallback_shape", "fallback")
_check_visible_tensor_shape("ex064_mask_shape", "mask")
_check_visible_tensor_shape("ex064_y_shape", "y")
_check_visible_tensor_shape("ex064_dout_shape", "dout")
_check_private_tensor_shape("ex064_dx_shape", "ex064", "dx")
_check_private_tensor_shape("ex064_dfallback_shape", "ex064", "dfallback")
_check_tensor("ex064_dx", "ex064", "dx")
_check_tensor("ex064_dfallback", "ex064", "dfallback")


### Exercise 065 — A shared bias on two branches

**Purpose:** Learn that when one bias is used in two branches, its final gradient is the sum of both branch contributions.

**Inputs:** `x` has shape `(2, 3)` and `bias` has shape `(3,)`.

**Forward operation:** `left = x + bias`, `right = 2 * bias`, and `y = left + right`.

**Derive/do:** Derive total `dbias` and `dx`.

**Ingredients:** The bias receives a broadcasted path through `left` and a separate path through `right`; add them.

**Required outputs:**

- `ex065_dx`: the gradient with respect to `x`.
- `ex065_dbias`: the total gradient from both bias uses.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex065_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex065_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex065_left_shape`: predict the shape of `left` as a literal Python tuple
- `ex065_right_shape`: predict the shape of `right` as a literal Python tuple
- `ex065_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex065_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex065_dx_shape`: predict the shape of `ex065_dx` as a literal Python tuple
- `ex065_dbias_shape`: predict the shape of `ex065_dbias` as a literal Python tuple

**Next concept:** Backward through expand.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0], dtype=DTYPE, requires_grad=True)
left = x + bias
right = 2.0 * bias
y = left + right
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex065", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 065: derive manually; do not use autograd in this cell.
# Define `ex065_dx` — the gradient with respect to `x`.
# Define `ex065_dbias` — the total gradient from both bias uses.
# Predict every visible tensor shape before computing values.
# Define `ex065_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex065_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex065_left_shape` — the shape of `left` as a literal Python tuple.
# Define `ex065_right_shape` — the shape of `right` as a literal Python tuple.
# Define `ex065_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex065_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex065_dx_shape` — the shape of `ex065_dx` as a literal Python tuple.
# Define `ex065_dbias_shape` — the shape of `ex065_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex065_x_shape", "x")
_check_visible_tensor_shape("ex065_bias_shape", "bias")
_check_visible_tensor_shape("ex065_left_shape", "left")
_check_visible_tensor_shape("ex065_right_shape", "right")
_check_visible_tensor_shape("ex065_y_shape", "y")
_check_visible_tensor_shape("ex065_dout_shape", "dout")
_check_private_tensor_shape("ex065_dx_shape", "ex065", "dx")
_check_private_tensor_shape("ex065_dbias_shape", "ex065", "dbias")
_check_tensor("ex065_dx", "ex065", "dx")
_check_tensor("ex065_dbias", "ex065", "dbias")


### Exercise 066 — Backward through expand

**Purpose:** Learn that expanding a tensor reuses its values, so backward adds gradients back into the smaller original shape.

**Inputs:** `bias` has shape `(1, 3)` and `expanded` has shape `(2, 3)`.

**Forward operation:** `expanded = bias.expand(2, 3)` and `y = expanded * x`.

**Derive/do:** Derive `dexpanded`, `dbias`, and `dx`.

**Ingredients:** Reverse multiplication, then sum expanded contributions back to the original size-one row axis.

**Required outputs:**

- `ex066_dexpanded`: the gradient at the expanded view.
- `ex066_dbias`: the unexpanded gradient with respect to `bias`.
- `ex066_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex066_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex066_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex066_expanded_shape`: predict the shape of `expanded` as a literal Python tuple
- `ex066_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex066_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex066_dexpanded_shape`: predict the shape of `ex066_dexpanded` as a literal Python tuple
- `ex066_dbias_shape`: predict the shape of `ex066_dbias` as a literal Python tuple
- `ex066_dx_shape`: predict the shape of `ex066_dx` as a literal Python tuple

**Next concept:** Broadcast, square, then backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
bias = torch.tensor([[0.5, -1.0, 2.0]], dtype=DTYPE, requires_grad=True)
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
expanded = bias.expand(2, 3)
y = expanded * x
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex066", y, {"dexpanded": expanded, "dbias": bias, "dx": x}, dout)
print("bias", tuple(bias.shape), "expanded", tuple(expanded.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 066: derive manually; do not use autograd in this cell.
# Define `ex066_dexpanded` — the gradient at the expanded view.
# Define `ex066_dbias` — the unexpanded gradient with respect to `bias`.
# Define `ex066_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex066_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex066_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex066_expanded_shape` — the shape of `expanded` as a literal Python tuple.
# Define `ex066_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex066_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex066_dexpanded_shape` — the shape of `ex066_dexpanded` as a literal Python tuple.
# Define `ex066_dbias_shape` — the shape of `ex066_dbias` as a literal Python tuple.
# Define `ex066_dx_shape` — the shape of `ex066_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex066_bias_shape", "bias")
_check_visible_tensor_shape("ex066_x_shape", "x")
_check_visible_tensor_shape("ex066_expanded_shape", "expanded")
_check_visible_tensor_shape("ex066_y_shape", "y")
_check_visible_tensor_shape("ex066_dout_shape", "dout")
_check_private_tensor_shape("ex066_dexpanded_shape", "ex066", "dexpanded")
_check_private_tensor_shape("ex066_dbias_shape", "ex066", "dbias")
_check_private_tensor_shape("ex066_dx_shape", "ex066", "dx")
_check_tensor("ex066_dexpanded", "ex066", "dexpanded")
_check_tensor("ex066_dbias", "ex066", "dbias")
_check_tensor("ex066_dx", "ex066", "dx")


### Exercise 067 — Broadcast, square, then backward

**Purpose:** Learn to reverse a square first and then reverse the broadcasted addition that produced its input.

**Inputs:** `x` has shape `(3, 4)`, `bias` has shape `(4,)`, and `pre` has shape `(3, 4)`.

**Forward operation:** `pre = x + bias` and `y = pre**2`.

**Derive/do:** Derive `dpre`, `dx`, and `dbias`.

**Ingredients:** Reverse the square first; then split addition paths and unbroadcast the bias path.

**Required outputs:**

- `ex067_dpre`: the gradient after reversing the square.
- `ex067_dx`: the gradient with respect to `x`.
- `ex067_dbias`: the unbroadcast bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex067_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex067_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex067_pre_shape`: predict the shape of `pre` as a literal Python tuple
- `ex067_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex067_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex067_dpre_shape`: predict the shape of `ex067_dpre` as a literal Python tuple
- `ex067_dx_shape`: predict the shape of `ex067_dx` as a literal Python tuple
- `ex067_dbias_shape`: predict the shape of `ex067_dbias` as a literal Python tuple

**Next concept:** Vector dot product.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x + bias
y = pre**2
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex067", y, {"dpre": pre, "dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "pre", tuple(pre.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 067: derive manually; do not use autograd in this cell.
# Define `ex067_dpre` — the gradient after reversing the square.
# Define `ex067_dx` — the gradient with respect to `x`.
# Define `ex067_dbias` — the unbroadcast bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex067_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex067_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex067_pre_shape` — the shape of `pre` as a literal Python tuple.
# Define `ex067_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex067_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex067_dpre_shape` — the shape of `ex067_dpre` as a literal Python tuple.
# Define `ex067_dx_shape` — the shape of `ex067_dx` as a literal Python tuple.
# Define `ex067_dbias_shape` — the shape of `ex067_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex067_x_shape", "x")
_check_visible_tensor_shape("ex067_bias_shape", "bias")
_check_visible_tensor_shape("ex067_pre_shape", "pre")
_check_visible_tensor_shape("ex067_y_shape", "y")
_check_visible_tensor_shape("ex067_dout_shape", "dout")
_check_private_tensor_shape("ex067_dpre_shape", "ex067", "dpre")
_check_private_tensor_shape("ex067_dx_shape", "ex067", "dx")
_check_private_tensor_shape("ex067_dbias_shape", "ex067", "dbias")
_check_tensor("ex067_dpre", "ex067", "dpre")
_check_tensor("ex067_dx", "ex067", "dx")
_check_tensor("ex067_dbias", "ex067", "dbias")


## 5. Matrix multiplication and affine layers

Matrix multiplication contracts an inner axis. Its backward pass must restore each operand's original shape.

For a batch matrix `X`, weights `W`, and output `Y`, the forward relationship is

$$
Y = XW
$$

Here, `X` has shape `(B, I)`, `W` has shape `(I, O)`, and `Y` has shape `(B, O)`. In backward, reason with transposes and verify that every matrix product's inner dimensions match. A bias added to `Y` is broadcast across the `B` examples, so its gradient accumulates over the batch axis.


### Exercise 068 — Vector dot product

**Purpose:** Learn how gradients flow through a dot product of two vectors.

**Inputs:** `x` and `w` have shape `(4,)`; `y = x @ w` is scalar.

**Forward operation:** `y = x @ w`.

**Derive/do:** Derive `dx` and `dw`.

**Ingredients:** Expand the dot product as a sum of elementwise products.

**Required outputs:**

- `ex068_dx`: the gradient with respect to `x`.
- `ex068_dw`: the gradient with respect to `w`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex068_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex068_w_shape`: predict the shape of `w` as a literal Python tuple
- `ex068_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex068_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex068_dx_shape`: predict the shape of `ex068_dx` as a literal Python tuple
- `ex068_dw_shape`: predict the shape of `ex068_dw` as a literal Python tuple

**Next concept:** Matrix-vector product.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([1.0, -2.0, 3.0, 0.5], dtype=DTYPE, requires_grad=True)
w = torch.tensor([0.5, 4.0, -1.0, 2.0], dtype=DTYPE, requires_grad=True)
y = x @ w
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex068", y, {"dx": x, "dw": w}, dout)
print("x", tuple(x.shape), "w", tuple(w.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 068: derive manually; do not use autograd in this cell.
# Define `ex068_dx` — the gradient with respect to `x`.
# Define `ex068_dw` — the gradient with respect to `w`.
# Predict every visible tensor shape before computing values.
# Define `ex068_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex068_w_shape` — the shape of `w` as a literal Python tuple.
# Define `ex068_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex068_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex068_dx_shape` — the shape of `ex068_dx` as a literal Python tuple.
# Define `ex068_dw_shape` — the shape of `ex068_dw` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex068_x_shape", "x")
_check_visible_tensor_shape("ex068_w_shape", "w")
_check_visible_tensor_shape("ex068_y_shape", "y")
_check_visible_tensor_shape("ex068_dout_shape", "dout")
_check_private_tensor_shape("ex068_dx_shape", "ex068", "dx")
_check_private_tensor_shape("ex068_dw_shape", "ex068", "dw")
_check_tensor("ex068_dx", "ex068", "dx")
_check_tensor("ex068_dw", "ex068", "dw")


### Exercise 069 — Matrix-vector product

**Purpose:** Learn how a matrix-vector product sends gradients back to both the matrix and the vector.

**Inputs:** `A` has shape `(3, 4)`, `x` has shape `(4,)`, and `y` has shape `(3,)`.

**Forward operation:** `y = A @ x`.

**Derive/do:** Derive `dA` and `dx`.

**Ingredients:** Write one output as a dot product, account for all three outputs, and use shape checks.

**Required outputs:**

- `ex069_dA`: the `(3, 4)` gradient with respect to `A`.
- `ex069_dx`: the length-4 gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex069_A_shape`: predict the shape of `A` as a literal Python tuple
- `ex069_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex069_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex069_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex069_dA_shape`: predict the shape of `ex069_dA` as a literal Python tuple
- `ex069_dx_shape`: predict the shape of `ex069_dx` as a literal Python tuple

**Next concept:** Row vector times weight matrix.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
A = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
x = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = A @ x
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex069", y, {"dA": A, "dx": x}, dout)
print("A", tuple(A.shape), "x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 069: derive manually; do not use autograd in this cell.
# Define `ex069_dA` — the `(3, 4)` gradient with respect to `A`.
# Define `ex069_dx` — the length-4 gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex069_A_shape` — the shape of `A` as a literal Python tuple.
# Define `ex069_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex069_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex069_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex069_dA_shape` — the shape of `ex069_dA` as a literal Python tuple.
# Define `ex069_dx_shape` — the shape of `ex069_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex069_A_shape", "A")
_check_visible_tensor_shape("ex069_x_shape", "x")
_check_visible_tensor_shape("ex069_y_shape", "y")
_check_visible_tensor_shape("ex069_dout_shape", "dout")
_check_private_tensor_shape("ex069_dA_shape", "ex069", "dA")
_check_private_tensor_shape("ex069_dx_shape", "ex069", "dx")
_check_tensor("ex069_dA", "ex069", "dA")
_check_tensor("ex069_dx", "ex069", "dx")


### Exercise 070 — Row vector times weight matrix

**Purpose:** Learn how one input vector produces several outputs through a weight matrix and receives gradients from all of them.

**Inputs:** `x` has shape `(3,)`, `W` has shape `(3, 2)`, and `y` has shape `(2,)`.

**Forward operation:** `y = x @ W`.

**Derive/do:** Derive `dx` and `dW`.

**Ingredients:** Each output is a dot product with one weight column; use transposes to satisfy shapes.

**Required outputs:**

- `ex070_dx`: the length-3 gradient with respect to `x`.
- `ex070_dW`: the `(3, 2)` gradient with respect to `W`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex070_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex070_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex070_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex070_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex070_dx_shape`: predict the shape of `ex070_dx` as a literal Python tuple
- `ex070_dW_shape`: predict the shape of `ex070_dW` as a literal Python tuple

**Next concept:** Matrix-matrix product.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE, requires_grad=True)
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
y = x @ W
dout = torch.tensor([2.0, -1.5], dtype=DTYPE)
_capture("ex070", y, {"dx": x, "dW": W}, dout)
print("x", tuple(x.shape), "W", tuple(W.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 070: derive manually; do not use autograd in this cell.
# Define `ex070_dx` — the length-3 gradient with respect to `x`.
# Define `ex070_dW` — the `(3, 2)` gradient with respect to `W`.
# Predict every visible tensor shape before computing values.
# Define `ex070_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex070_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex070_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex070_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex070_dx_shape` — the shape of `ex070_dx` as a literal Python tuple.
# Define `ex070_dW_shape` — the shape of `ex070_dW` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex070_x_shape", "x")
_check_visible_tensor_shape("ex070_W_shape", "W")
_check_visible_tensor_shape("ex070_y_shape", "y")
_check_visible_tensor_shape("ex070_dout_shape", "dout")
_check_private_tensor_shape("ex070_dx_shape", "ex070", "dx")
_check_private_tensor_shape("ex070_dW_shape", "ex070", "dW")
_check_tensor("ex070_dx", "ex070", "dx")
_check_tensor("ex070_dW", "ex070", "dW")


### Exercise 071 — Matrix-matrix product

**Purpose:** Learn how a matrix-matrix product sends gradients back to both input matrices.

**Inputs:** `A` has shape `(2, 3)`, `B` has shape `(3, 4)`, and `dout` has shape `(2, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `Y = A @ B`.

**Derive/do:** Derive `dA` and `dB`.

**Ingredients:** Use the upstream matrix and the appropriate transpose of the other operand; audit all inner dimensions.

**Required outputs:**

- `ex071_dA`: the `(2, 3)` gradient with respect to `A`.
- `ex071_dB`: the `(3, 4)` gradient with respect to `B`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex071_A_shape`: predict the shape of `A` as a literal Python tuple
- `ex071_B_shape`: predict the shape of `B` as a literal Python tuple
- `ex071_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex071_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex071_dA_shape`: predict the shape of `ex071_dA` as a literal Python tuple
- `ex071_dB_shape`: predict the shape of `ex071_dB` as a literal Python tuple

**Next concept:** A batch through one weight matrix.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
A = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
B = torch.linspace(-1.0, 2.3, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
Y = A @ B
dout = torch.linspace(0.5, -1.0, steps=8, dtype=DTYPE).reshape(2, 4)
_capture("ex071", Y, {"dA": A, "dB": B}, dout)
print("A", tuple(A.shape), "B", tuple(B.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 071: derive manually; do not use autograd in this cell.
# Define `ex071_dA` — the `(2, 3)` gradient with respect to `A`.
# Define `ex071_dB` — the `(3, 4)` gradient with respect to `B`.
# Predict every visible tensor shape before computing values.
# Define `ex071_A_shape` — the shape of `A` as a literal Python tuple.
# Define `ex071_B_shape` — the shape of `B` as a literal Python tuple.
# Define `ex071_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex071_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex071_dA_shape` — the shape of `ex071_dA` as a literal Python tuple.
# Define `ex071_dB_shape` — the shape of `ex071_dB` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex071_A_shape", "A")
_check_visible_tensor_shape("ex071_B_shape", "B")
_check_visible_tensor_shape("ex071_Y_shape", "Y")
_check_visible_tensor_shape("ex071_dout_shape", "dout")
_check_private_tensor_shape("ex071_dA_shape", "ex071", "dA")
_check_private_tensor_shape("ex071_dB_shape", "ex071", "dB")
_check_tensor("ex071_dA", "ex071", "dA")
_check_tensor("ex071_dB", "ex071", "dB")


### Exercise 072 — A batch through one weight matrix

**Purpose:** Learn that a weight matrix shared by all batch rows collects a contribution from every training example.

**Inputs:** `X` has shape `(4, 3)`, `W` has shape `(3, 2)`, and `Y` has shape `(4, 2)`.

**Forward operation:** `Y = X @ W`.

**Derive/do:** Derive `dX` and `dW`.

**Ingredients:** The same `W` is reused for all four rows; matrix multiplication performs the needed accumulation.

**Required outputs:**

- `ex072_dX`: the `(4, 3)` gradient with respect to `X`.
- `ex072_dW`: the `(3, 2)` gradient accumulated over the batch.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex072_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex072_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex072_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex072_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex072_dX_shape`: predict the shape of `ex072_dX` as a literal Python tuple
- `ex072_dW_shape`: predict the shape of `ex072_dW` as a literal Python tuple

**Next concept:** Output bias backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
Y = X @ W
dout = torch.linspace(1.0, -0.75, steps=8, dtype=DTYPE).reshape(4, 2)
_capture("ex072", Y, {"dX": X, "dW": W}, dout)
print("X", tuple(X.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 072: derive manually; do not use autograd in this cell.
# Define `ex072_dX` — the `(4, 3)` gradient with respect to `X`.
# Define `ex072_dW` — the `(3, 2)` gradient accumulated over the batch.
# Predict every visible tensor shape before computing values.
# Define `ex072_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex072_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex072_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex072_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex072_dX_shape` — the shape of `ex072_dX` as a literal Python tuple.
# Define `ex072_dW_shape` — the shape of `ex072_dW` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex072_X_shape", "X")
_check_visible_tensor_shape("ex072_W_shape", "W")
_check_visible_tensor_shape("ex072_Y_shape", "Y")
_check_visible_tensor_shape("ex072_dout_shape", "dout")
_check_private_tensor_shape("ex072_dX_shape", "ex072", "dX")
_check_private_tensor_shape("ex072_dW_shape", "ex072", "dW")
_check_tensor("ex072_dX", "ex072", "dX")
_check_tensor("ex072_dW", "ex072", "dW")


### Exercise 073 — Output bias backward

**Purpose:** Learn that an output bias shared by all examples adds its gradient contributions across the batch rows.

**Inputs:** `Z` has shape `(4, 2)`, `bias` has shape `(2,)`, and `dout` has shape `(4, 2)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `Y = Z + bias`.

**Derive/do:** Derive `dZ` and `dbias`.

**Ingredients:** The bias is reused for four examples; sum only the example axis.

**Required outputs:**

- `ex073_dZ`: the gradient with respect to `Z`.
- `ex073_dbias`: the length-2 bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex073_Z_shape`: predict the shape of `Z` as a literal Python tuple
- `ex073_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex073_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex073_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex073_dZ_shape`: predict the shape of `ex073_dZ` as a literal Python tuple
- `ex073_dbias_shape`: predict the shape of `ex073_dbias` as a literal Python tuple

**Next concept:** Complete affine layer.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
Z = torch.linspace(-2.0, 1.5, steps=8, dtype=DTYPE).reshape(4, 2).requires_grad_()
bias = torch.tensor([0.5, -1.0], dtype=DTYPE, requires_grad=True)
Y = Z + bias
dout = torch.linspace(1.0, -0.75, steps=8, dtype=DTYPE).reshape(4, 2)
_capture("ex073", Y, {"dZ": Z, "dbias": bias}, dout)
print("Z", tuple(Z.shape), "bias", tuple(bias.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 073: derive manually; do not use autograd in this cell.
# Define `ex073_dZ` — the gradient with respect to `Z`.
# Define `ex073_dbias` — the length-2 bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex073_Z_shape` — the shape of `Z` as a literal Python tuple.
# Define `ex073_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex073_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex073_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex073_dZ_shape` — the shape of `ex073_dZ` as a literal Python tuple.
# Define `ex073_dbias_shape` — the shape of `ex073_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex073_Z_shape", "Z")
_check_visible_tensor_shape("ex073_bias_shape", "bias")
_check_visible_tensor_shape("ex073_Y_shape", "Y")
_check_visible_tensor_shape("ex073_dout_shape", "dout")
_check_private_tensor_shape("ex073_dZ_shape", "ex073", "dZ")
_check_private_tensor_shape("ex073_dbias_shape", "ex073", "dbias")
_check_tensor("ex073_dZ", "ex073", "dZ")
_check_tensor("ex073_dbias", "ex073", "dbias")


### Exercise 074 — Complete affine layer

**Purpose:** Learn to backpropagate through the complete operation `X @ W + bias`.

**Inputs:** `X` is `(4, 3)`, `W` is `(3, 2)`, `bias` is `(2,)`, and `Y` is `(4, 2)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `Y = X @ W + bias`.

**Derive/do:** Derive `dX`, `dW`, and `dbias`.

**Ingredients:** Start from `dout`; handle the addition and matrix product while checking each gradient shape.

**Required outputs:**

- `ex074_dX`: the input gradient.
- `ex074_dW`: the weight gradient.
- `ex074_dbias`: the bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex074_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex074_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex074_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex074_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex074_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex074_dX_shape`: predict the shape of `ex074_dX` as a literal Python tuple
- `ex074_dW_shape`: predict the shape of `ex074_dW` as a literal Python tuple
- `ex074_dbias_shape`: predict the shape of `ex074_dbias` as a literal Python tuple

**Next concept:** Affine layer followed by a total.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
dout = torch.linspace(1.0, -0.75, steps=8, dtype=DTYPE).reshape(4, 2)
_capture("ex074", Y, {"dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "W", tuple(W.shape), "bias", tuple(bias.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 074: derive manually; do not use autograd in this cell.
# Define `ex074_dX` — the input gradient.
# Define `ex074_dW` — the weight gradient.
# Define `ex074_dbias` — the bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex074_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex074_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex074_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex074_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex074_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex074_dX_shape` — the shape of `ex074_dX` as a literal Python tuple.
# Define `ex074_dW_shape` — the shape of `ex074_dW` as a literal Python tuple.
# Define `ex074_dbias_shape` — the shape of `ex074_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex074_X_shape", "X")
_check_visible_tensor_shape("ex074_W_shape", "W")
_check_visible_tensor_shape("ex074_bias_shape", "bias")
_check_visible_tensor_shape("ex074_Y_shape", "Y")
_check_visible_tensor_shape("ex074_dout_shape", "dout")
_check_private_tensor_shape("ex074_dX_shape", "ex074", "dX")
_check_private_tensor_shape("ex074_dW_shape", "ex074", "dW")
_check_private_tensor_shape("ex074_dbias_shape", "ex074", "dbias")
_check_tensor("ex074_dX", "ex074", "dX")
_check_tensor("ex074_dW", "ex074", "dW")
_check_tensor("ex074_dbias", "ex074", "dbias")


### Exercise 075 — Affine layer followed by a total

**Purpose:** Learn how a scalar sum creates the first gradient tensor for an affine layer's outputs.

**Inputs:** `Y = X @ W + bias` has shape `(3, 2)`; `loss = Y.sum()` is scalar.

**Forward operation:** Compute `Y`, then `loss = Y.sum()`.

**Derive/do:** Derive `dY`, `dX`, `dW`, and `dbias`.

**Ingredients:** Reverse the sum to create `dY`, then apply affine backward.

**Required outputs:**

- `ex075_dY`: the gradient after reversing the total.
- `ex075_dX`: the input gradient.
- `ex075_dW`: the weight gradient.
- `ex075_dbias`: the bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex075_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex075_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex075_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex075_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex075_loss_local_shape`: predict the shape of `loss_local` as a literal Python tuple
- `ex075_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex075_dY_shape`: predict the shape of `ex075_dY` as a literal Python tuple
- `ex075_dX_shape`: predict the shape of `ex075_dX` as a literal Python tuple
- `ex075_dW_shape`: predict the shape of `ex075_dW` as a literal Python tuple
- `ex075_dbias_shape`: predict the shape of `ex075_dbias` as a literal Python tuple

**Next concept:** Affine layer followed by a mean.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-1.0, 2.0, steps=9, dtype=DTYPE).reshape(3, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
loss_local = Y.sum()
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex075", loss_local, {"dY": Y, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 075: derive manually; do not use autograd in this cell.
# Define `ex075_dY` — the gradient after reversing the total.
# Define `ex075_dX` — the input gradient.
# Define `ex075_dW` — the weight gradient.
# Define `ex075_dbias` — the bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex075_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex075_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex075_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex075_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex075_loss_local_shape` — the shape of `loss_local` as a literal Python tuple.
# Define `ex075_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex075_dY_shape` — the shape of `ex075_dY` as a literal Python tuple.
# Define `ex075_dX_shape` — the shape of `ex075_dX` as a literal Python tuple.
# Define `ex075_dW_shape` — the shape of `ex075_dW` as a literal Python tuple.
# Define `ex075_dbias_shape` — the shape of `ex075_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex075_X_shape", "X")
_check_visible_tensor_shape("ex075_W_shape", "W")
_check_visible_tensor_shape("ex075_bias_shape", "bias")
_check_visible_tensor_shape("ex075_Y_shape", "Y")
_check_visible_tensor_shape("ex075_loss_local_shape", "loss_local")
_check_visible_tensor_shape("ex075_dout_shape", "dout")
_check_private_tensor_shape("ex075_dY_shape", "ex075", "dY")
_check_private_tensor_shape("ex075_dX_shape", "ex075", "dX")
_check_private_tensor_shape("ex075_dW_shape", "ex075", "dW")
_check_private_tensor_shape("ex075_dbias_shape", "ex075", "dbias")
_check_tensor("ex075_dY", "ex075", "dY")
_check_tensor("ex075_dX", "ex075", "dX")
_check_tensor("ex075_dW", "ex075", "dW")
_check_tensor("ex075_dbias", "ex075", "dbias")


### Exercise 076 — Affine layer followed by a mean

**Purpose:** Learn how averaging affine outputs changes the gradient scale compared with summing them.

**Inputs:** `Y` has shape `(3, 2)`, so its mean averages six values.

**Forward operation:** `Y = X @ W + bias` and `loss = Y.mean()`.

**Derive/do:** Derive `dY`, `dX`, `dW`, and `dbias`.

**Ingredients:** Seed all six `Y` entries with the scalar upstream gradient divided by six.

**Required outputs:**

- `ex076_dY`: the gradient after reversing the six-value mean.
- `ex076_dX`: the input gradient.
- `ex076_dW`: the weight gradient.
- `ex076_dbias`: the bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex076_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex076_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex076_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex076_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex076_loss_local_shape`: predict the shape of `loss_local` as a literal Python tuple
- `ex076_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex076_dY_shape`: predict the shape of `ex076_dY` as a literal Python tuple
- `ex076_dX_shape`: predict the shape of `ex076_dX` as a literal Python tuple
- `ex076_dW_shape`: predict the shape of `ex076_dW` as a literal Python tuple
- `ex076_dbias_shape`: predict the shape of `ex076_dbias` as a literal Python tuple

**Next concept:** Squared affine objective.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-1.0, 2.0, steps=9, dtype=DTYPE).reshape(3, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
loss_local = Y.mean()
dout = torch.tensor(-2.0, dtype=DTYPE)
_capture("ex076", loss_local, {"dY": Y, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 076: derive manually; do not use autograd in this cell.
# Define `ex076_dY` — the gradient after reversing the six-value mean.
# Define `ex076_dX` — the input gradient.
# Define `ex076_dW` — the weight gradient.
# Define `ex076_dbias` — the bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex076_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex076_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex076_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex076_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex076_loss_local_shape` — the shape of `loss_local` as a literal Python tuple.
# Define `ex076_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex076_dY_shape` — the shape of `ex076_dY` as a literal Python tuple.
# Define `ex076_dX_shape` — the shape of `ex076_dX` as a literal Python tuple.
# Define `ex076_dW_shape` — the shape of `ex076_dW` as a literal Python tuple.
# Define `ex076_dbias_shape` — the shape of `ex076_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex076_X_shape", "X")
_check_visible_tensor_shape("ex076_W_shape", "W")
_check_visible_tensor_shape("ex076_bias_shape", "bias")
_check_visible_tensor_shape("ex076_Y_shape", "Y")
_check_visible_tensor_shape("ex076_loss_local_shape", "loss_local")
_check_visible_tensor_shape("ex076_dout_shape", "dout")
_check_private_tensor_shape("ex076_dY_shape", "ex076", "dY")
_check_private_tensor_shape("ex076_dX_shape", "ex076", "dX")
_check_private_tensor_shape("ex076_dW_shape", "ex076", "dW")
_check_private_tensor_shape("ex076_dbias_shape", "ex076", "dbias")
_check_tensor("ex076_dY", "ex076", "dY")
_check_tensor("ex076_dX", "ex076", "dX")
_check_tensor("ex076_dW", "ex076", "dW")
_check_tensor("ex076_dbias", "ex076", "dbias")


### Exercise 077 — Squared affine objective

**Purpose:** Learn to reverse a mean of squared affine outputs before computing parameter gradients.

**Inputs:** `Y` has shape `(3, 2)` and `loss = (Y**2).mean()` is scalar.

**Forward operation:** `Y = X @ W + bias`; average the six squared outputs.

**Derive/do:** Derive `dY`, `dX`, `dW`, and `dbias`.

**Ingredients:** Reverse mean, square, addition, and matrix multiplication in that order.

**Required outputs:**

- `ex077_dY`: the gradient with respect to affine outputs.
- `ex077_dX`: the input gradient.
- `ex077_dW`: the weight gradient.
- `ex077_dbias`: the bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex077_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex077_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex077_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex077_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex077_loss_local_shape`: predict the shape of `loss_local` as a literal Python tuple
- `ex077_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex077_dY_shape`: predict the shape of `ex077_dY` as a literal Python tuple
- `ex077_dX_shape`: predict the shape of `ex077_dX` as a literal Python tuple
- `ex077_dW_shape`: predict the shape of `ex077_dW` as a literal Python tuple
- `ex077_dbias_shape`: predict the shape of `ex077_dbias` as a literal Python tuple

**Next concept:** Transpose backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-1.0, 2.0, steps=9, dtype=DTYPE).reshape(3, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
loss_local = (Y**2).mean()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex077", loss_local, {"dY": Y, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 077: derive manually; do not use autograd in this cell.
# Define `ex077_dY` — the gradient with respect to affine outputs.
# Define `ex077_dX` — the input gradient.
# Define `ex077_dW` — the weight gradient.
# Define `ex077_dbias` — the bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex077_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex077_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex077_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex077_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex077_loss_local_shape` — the shape of `loss_local` as a literal Python tuple.
# Define `ex077_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex077_dY_shape` — the shape of `ex077_dY` as a literal Python tuple.
# Define `ex077_dX_shape` — the shape of `ex077_dX` as a literal Python tuple.
# Define `ex077_dW_shape` — the shape of `ex077_dW` as a literal Python tuple.
# Define `ex077_dbias_shape` — the shape of `ex077_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex077_X_shape", "X")
_check_visible_tensor_shape("ex077_W_shape", "W")
_check_visible_tensor_shape("ex077_bias_shape", "bias")
_check_visible_tensor_shape("ex077_Y_shape", "Y")
_check_visible_tensor_shape("ex077_loss_local_shape", "loss_local")
_check_visible_tensor_shape("ex077_dout_shape", "dout")
_check_private_tensor_shape("ex077_dY_shape", "ex077", "dY")
_check_private_tensor_shape("ex077_dX_shape", "ex077", "dX")
_check_private_tensor_shape("ex077_dW_shape", "ex077", "dW")
_check_private_tensor_shape("ex077_dbias_shape", "ex077", "dbias")
_check_tensor("ex077_dY", "ex077", "dY")
_check_tensor("ex077_dX", "ex077", "dX")
_check_tensor("ex077_dW", "ex077", "dW")
_check_tensor("ex077_dbias", "ex077", "dbias")


### Exercise 078 — Transpose backward

**Purpose:** Learn that backward through a transpose swaps the axes back to their original order.

**Inputs:** `X` has shape `(2, 3)`, `Y = X.T` has shape `(3, 2)`.

**Forward operation:** `Y = X.T`.

**Derive/do:** Derive `dX` from a `(3, 2)` upstream tensor.

**Ingredients:** A transpose only swaps axes; backward must restore their original order.

**Required outputs:**

- `ex078_dX`: the `(2, 3)` gradient with respect to `X`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex078_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex078_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex078_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex078_dX_shape`: predict the shape of `ex078_dX` as a literal Python tuple

**Next concept:** Outer product.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
Y = X.T
dout = torch.tensor([[1.0, -1.0], [2.0, 0.5], [-2.0, 3.0]], dtype=DTYPE)
_capture("ex078", Y, {"dX": X}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 078: derive manually; do not use autograd in this cell.
# Define `ex078_dX` — the `(2, 3)` gradient with respect to `X`.
# Predict every visible tensor shape before computing values.
# Define `ex078_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex078_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex078_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex078_dX_shape` — the shape of `ex078_dX` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex078_X_shape", "X")
_check_visible_tensor_shape("ex078_Y_shape", "Y")
_check_visible_tensor_shape("ex078_dout_shape", "dout")
_check_private_tensor_shape("ex078_dX_shape", "ex078", "dX")
_check_tensor("ex078_dX", "ex078", "dX")


### Exercise 079 — Outer product

**Purpose:** Learn how every pair of entries from two vectors forms a matrix and contributes to both vector gradients.

**Inputs:** `x` has shape `(3,)`, `z` has shape `(4,)`, and `Y` has shape `(3, 4)`.

**Forward operation:** `Y = x[:, None] * z[None, :]`.

**Derive/do:** Derive `dx` and `dz`.

**Ingredients:** This is two-way broadcasted multiplication; reduce the axis introduced for each vector.

**Required outputs:**

- `ex079_dx`: the length-3 gradient with respect to `x`.
- `ex079_dz`: the length-4 gradient with respect to `z`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex079_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex079_z_shape`: predict the shape of `z` as a literal Python tuple
- `ex079_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex079_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex079_dx_shape`: predict the shape of `ex079_dx` as a literal Python tuple
- `ex079_dz_shape`: predict the shape of `ex079_dz` as a literal Python tuple

**Next concept:** Batched matrix multiplication.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE, requires_grad=True)
z = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
Y = x[:, None] * z[None, :]
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex079", Y, {"dx": x, "dz": z}, dout)
print("x", tuple(x.shape), "z", tuple(z.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 079: derive manually; do not use autograd in this cell.
# Define `ex079_dx` — the length-3 gradient with respect to `x`.
# Define `ex079_dz` — the length-4 gradient with respect to `z`.
# Predict every visible tensor shape before computing values.
# Define `ex079_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex079_z_shape` — the shape of `z` as a literal Python tuple.
# Define `ex079_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex079_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex079_dx_shape` — the shape of `ex079_dx` as a literal Python tuple.
# Define `ex079_dz_shape` — the shape of `ex079_dz` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex079_x_shape", "x")
_check_visible_tensor_shape("ex079_z_shape", "z")
_check_visible_tensor_shape("ex079_Y_shape", "Y")
_check_visible_tensor_shape("ex079_dout_shape", "dout")
_check_private_tensor_shape("ex079_dx_shape", "ex079", "dx")
_check_private_tensor_shape("ex079_dz_shape", "ex079", "dz")
_check_tensor("ex079_dx", "ex079", "dx")
_check_tensor("ex079_dz", "ex079", "dz")


### Exercise 080 — Batched matrix multiplication

**Purpose:** Learn to backpropagate through several independent matrix products stored along a batch axis.

**Inputs:** `A` has shape `(2, 2, 3)`, `B` has shape `(2, 3, 2)`, and `Y` has shape `(2, 2, 2)`.

**Forward operation:** `Y = torch.bmm(A, B)`.

**Derive/do:** Derive `dA` and `dB`.

**Ingredients:** Apply matrix-product backward within each of the two batch entries; `transpose(1, 2)` swaps matrix axes only.

**Required outputs:**

- `ex080_dA`: the `(2, 2, 3)` gradient with respect to `A`.
- `ex080_dB`: the `(2, 3, 2)` gradient with respect to `B`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex080_A_shape`: predict the shape of `A` as a literal Python tuple
- `ex080_B_shape`: predict the shape of `B` as a literal Python tuple
- `ex080_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex080_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex080_dA_shape`: predict the shape of `ex080_dA` as a literal Python tuple
- `ex080_dB_shape`: predict the shape of `ex080_dB` as a literal Python tuple

**Next concept:** Transform the last axis of rank 3.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
A = torch.linspace(-1.0, 2.3, steps=12, dtype=DTYPE).reshape(2, 2, 3).requires_grad_()
B = torch.linspace(0.5, -1.8, steps=12, dtype=DTYPE).reshape(2, 3, 2).requires_grad_()
Y = torch.bmm(A, B)
dout = torch.linspace(1.0, -0.4, steps=8, dtype=DTYPE).reshape(2, 2, 2)
_capture("ex080", Y, {"dA": A, "dB": B}, dout)
print("A", tuple(A.shape), "B", tuple(B.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 080: derive manually; do not use autograd in this cell.
# Define `ex080_dA` — the `(2, 2, 3)` gradient with respect to `A`.
# Define `ex080_dB` — the `(2, 3, 2)` gradient with respect to `B`.
# Predict every visible tensor shape before computing values.
# Define `ex080_A_shape` — the shape of `A` as a literal Python tuple.
# Define `ex080_B_shape` — the shape of `B` as a literal Python tuple.
# Define `ex080_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex080_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex080_dA_shape` — the shape of `ex080_dA` as a literal Python tuple.
# Define `ex080_dB_shape` — the shape of `ex080_dB` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex080_A_shape", "A")
_check_visible_tensor_shape("ex080_B_shape", "B")
_check_visible_tensor_shape("ex080_Y_shape", "Y")
_check_visible_tensor_shape("ex080_dout_shape", "dout")
_check_private_tensor_shape("ex080_dA_shape", "ex080", "dA")
_check_private_tensor_shape("ex080_dB_shape", "ex080", "dB")
_check_tensor("ex080_dA", "ex080", "dA")
_check_tensor("ex080_dB", "ex080", "dB")


### Exercise 081 — Transform the last axis of rank 3

**Purpose:** Learn that `X @ W` transforms the last axis of a rank-3 tensor at every leading position.

**Inputs:** `X` has shape `(2, 3, 4)`, `W` has shape `(4, 2)`, and `Y` has shape `(2, 3, 2)`.

**Forward operation:** `Y = X @ W`.

**Derive/do:** Derive `dX` and `dW`.

**Ingredients:** Flatten leading positions conceptually into six examples, or use last-two-axis matrix rules and sum shared-weight contributions.

**Required outputs:**

- `ex081_dX`: the `(2, 3, 4)` gradient with respect to `X`.
- `ex081_dW`: the `(4, 2)` shared-weight gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex081_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex081_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex081_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex081_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex081_dX_shape`: predict the shape of `ex081_dX` as a literal Python tuple
- `ex081_dW_shape`: predict the shape of `ex081_dW` as a literal Python tuple

**Next concept:** Shared weights on two input branches.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-2.0, 3.5, steps=24, dtype=DTYPE).reshape(2, 3, 4).requires_grad_()
W = torch.linspace(-1.0, 1.5, steps=8, dtype=DTYPE).reshape(4, 2).requires_grad_()
Y = X @ W
dout = torch.linspace(1.0, -0.7, steps=12, dtype=DTYPE).reshape(2, 3, 2)
_capture("ex081", Y, {"dX": X, "dW": W}, dout)
print("X", tuple(X.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 081: derive manually; do not use autograd in this cell.
# Define `ex081_dX` — the `(2, 3, 4)` gradient with respect to `X`.
# Define `ex081_dW` — the `(4, 2)` shared-weight gradient.
# Predict every visible tensor shape before computing values.
# Define `ex081_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex081_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex081_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex081_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex081_dX_shape` — the shape of `ex081_dX` as a literal Python tuple.
# Define `ex081_dW_shape` — the shape of `ex081_dW` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex081_X_shape", "X")
_check_visible_tensor_shape("ex081_W_shape", "W")
_check_visible_tensor_shape("ex081_Y_shape", "Y")
_check_visible_tensor_shape("ex081_dout_shape", "dout")
_check_private_tensor_shape("ex081_dX_shape", "ex081", "dX")
_check_private_tensor_shape("ex081_dW_shape", "ex081", "dW")
_check_tensor("ex081_dX", "ex081", "dX")
_check_tensor("ex081_dW", "ex081", "dW")


### Exercise 082 — Shared weights on two input branches

**Purpose:** Learn that weights used by two matrix-product branches receive the sum of both branches' gradients.

**Inputs:** `X1` and `X2` are `(2, 3)`; shared `W` is `(3, 2)`.

**Forward operation:** `Y = X1 @ W + X2 @ W`.

**Derive/do:** Derive `dX1`, `dX2`, and total `dW`.

**Ingredients:** Reverse the final addition, backpropagate through both products, then add the two weight contributions.

**Required outputs:**

- `ex082_dX1`: the first input gradient.
- `ex082_dX2`: the second input gradient.
- `ex082_dW`: the total shared-weight gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex082_X1_shape`: predict the shape of `X1` as a literal Python tuple
- `ex082_X2_shape`: predict the shape of `X2` as a literal Python tuple
- `ex082_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex082_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex082_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex082_dX1_shape`: predict the shape of `ex082_dX1` as a literal Python tuple
- `ex082_dX2_shape`: predict the shape of `ex082_dX2` as a literal Python tuple
- `ex082_dW_shape`: predict the shape of `ex082_dW` as a literal Python tuple

**Next concept:** Weight gradient under a summed objective.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X1 = torch.linspace(-1.0, 1.5, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
X2 = torch.linspace(2.0, -0.5, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
Y = X1 @ W + X2 @ W
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex082", Y, {"dX1": X1, "dX2": X2, "dW": W}, dout)
print("X1", tuple(X1.shape), "X2", tuple(X2.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 082: derive manually; do not use autograd in this cell.
# Define `ex082_dX1` — the first input gradient.
# Define `ex082_dX2` — the second input gradient.
# Define `ex082_dW` — the total shared-weight gradient.
# Predict every visible tensor shape before computing values.
# Define `ex082_X1_shape` — the shape of `X1` as a literal Python tuple.
# Define `ex082_X2_shape` — the shape of `X2` as a literal Python tuple.
# Define `ex082_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex082_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex082_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex082_dX1_shape` — the shape of `ex082_dX1` as a literal Python tuple.
# Define `ex082_dX2_shape` — the shape of `ex082_dX2` as a literal Python tuple.
# Define `ex082_dW_shape` — the shape of `ex082_dW` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex082_X1_shape", "X1")
_check_visible_tensor_shape("ex082_X2_shape", "X2")
_check_visible_tensor_shape("ex082_W_shape", "W")
_check_visible_tensor_shape("ex082_Y_shape", "Y")
_check_visible_tensor_shape("ex082_dout_shape", "dout")
_check_private_tensor_shape("ex082_dX1_shape", "ex082", "dX1")
_check_private_tensor_shape("ex082_dX2_shape", "ex082", "dX2")
_check_private_tensor_shape("ex082_dW_shape", "ex082", "dW")
_check_tensor("ex082_dX1", "ex082", "dX1")
_check_tensor("ex082_dX2", "ex082", "dX2")
_check_tensor("ex082_dW", "ex082", "dW")


### Exercise 083 — Weight gradient under a summed objective

**Purpose:** Learn how a total over all outputs makes every output contribute equally to the weight gradient.

**Inputs:** `X` is `(4, 3)`, `W` is `(3, 2)`, and the objective totals all eight outputs.

**Forward operation:** `Y = X @ W` and `loss = Y.sum()`.

**Derive/do:** Derive `dY` and `dW`.

**Ingredients:** A total seeds each output with one times the scalar upstream gradient; then apply matrix backward.

**Required outputs:**

- `ex083_dY`: the gradient at all eight outputs.
- `ex083_dW`: the summed-objective weight gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex083_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex083_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex083_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex083_loss_local_shape`: predict the shape of `loss_local` as a literal Python tuple
- `ex083_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex083_dY_shape`: predict the shape of `ex083_dY` as a literal Python tuple
- `ex083_dW_shape`: predict the shape of `ex083_dW` as a literal Python tuple

**Next concept:** Weight gradient under an averaged objective.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-1.0, 2.0, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.linspace(-0.5, 1.0, steps=6, dtype=DTYPE).reshape(3, 2).requires_grad_()
Y = X @ W
loss_local = Y.sum()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex083", loss_local, {"dY": Y, "dW": W}, dout)
print("Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 083: derive manually; do not use autograd in this cell.
# Define `ex083_dY` — the gradient at all eight outputs.
# Define `ex083_dW` — the summed-objective weight gradient.
# Predict every visible tensor shape before computing values.
# Define `ex083_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex083_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex083_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex083_loss_local_shape` — the shape of `loss_local` as a literal Python tuple.
# Define `ex083_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex083_dY_shape` — the shape of `ex083_dY` as a literal Python tuple.
# Define `ex083_dW_shape` — the shape of `ex083_dW` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex083_X_shape", "X")
_check_visible_tensor_shape("ex083_W_shape", "W")
_check_visible_tensor_shape("ex083_Y_shape", "Y")
_check_visible_tensor_shape("ex083_loss_local_shape", "loss_local")
_check_visible_tensor_shape("ex083_dout_shape", "dout")
_check_private_tensor_shape("ex083_dY_shape", "ex083", "dY")
_check_private_tensor_shape("ex083_dW_shape", "ex083", "dW")
_check_tensor("ex083_dY", "ex083", "dY")
_check_tensor("ex083_dW", "ex083", "dW")


### Exercise 084 — Weight gradient under an averaged objective

**Purpose:** Learn how replacing that total with a mean divides the output and weight gradients by the number of outputs.

**Inputs:** `X` is `(4, 3)`, `W` is `(3, 2)`, and the objective averages all eight outputs.

**Forward operation:** `Y = X @ W` and `loss = Y.mean()`.

**Derive/do:** Derive `dY` and `dW`.

**Ingredients:** Determine the exact number of averaged output entries before matrix backward.

**Required outputs:**

- `ex084_dY`: the gradient at all eight averaged outputs.
- `ex084_dW`: the averaged-objective weight gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex084_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex084_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex084_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex084_loss_local_shape`: predict the shape of `loss_local` as a literal Python tuple
- `ex084_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex084_dY_shape`: predict the shape of `ex084_dY` as a literal Python tuple
- `ex084_dW_shape`: predict the shape of `ex084_dW` as a literal Python tuple

**Next concept:** Two linear layers.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-1.0, 2.0, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.linspace(-0.5, 1.0, steps=6, dtype=DTYPE).reshape(3, 2).requires_grad_()
Y = X @ W
loss_local = Y.mean()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex084", loss_local, {"dY": Y, "dW": W}, dout)
print("Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 084: derive manually; do not use autograd in this cell.
# Define `ex084_dY` — the gradient at all eight averaged outputs.
# Define `ex084_dW` — the averaged-objective weight gradient.
# Predict every visible tensor shape before computing values.
# Define `ex084_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex084_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex084_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex084_loss_local_shape` — the shape of `loss_local` as a literal Python tuple.
# Define `ex084_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex084_dY_shape` — the shape of `ex084_dY` as a literal Python tuple.
# Define `ex084_dW_shape` — the shape of `ex084_dW` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex084_X_shape", "X")
_check_visible_tensor_shape("ex084_W_shape", "W")
_check_visible_tensor_shape("ex084_Y_shape", "Y")
_check_visible_tensor_shape("ex084_loss_local_shape", "loss_local")
_check_visible_tensor_shape("ex084_dout_shape", "dout")
_check_private_tensor_shape("ex084_dY_shape", "ex084", "dY")
_check_private_tensor_shape("ex084_dW_shape", "ex084", "dW")
_check_tensor("ex084_dY", "ex084", "dY")
_check_tensor("ex084_dW", "ex084", "dW")


### Exercise 085 — Two linear layers

**Purpose:** Learn to work backward through two linear layers in the correct reverse order.

**Inputs:** `X` is `(2, 3)`, `W1` is `(3, 4)`, hidden `H` is `(2, 4)`, and `W2` is `(4, 2)`.

**Forward operation:** `H = X @ W1` and `Y = H @ W2`.

**Derive/do:** Derive `dH`, `dX`, `dW1`, and `dW2`.

**Ingredients:** Reverse the second product first; use its `dH` as the upstream gradient for the first product.

**Required outputs:**

- `ex085_dH`: the hidden-state gradient.
- `ex085_dX`: the input gradient.
- `ex085_dW1`: the first weight gradient.
- `ex085_dW2`: the second weight gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex085_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex085_W1_shape`: predict the shape of `W1` as a literal Python tuple
- `ex085_H_shape`: predict the shape of `H` as a literal Python tuple
- `ex085_W2_shape`: predict the shape of `W2` as a literal Python tuple
- `ex085_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex085_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex085_dH_shape`: predict the shape of `ex085_dH` as a literal Python tuple
- `ex085_dX_shape`: predict the shape of `ex085_dX` as a literal Python tuple
- `ex085_dW1_shape`: predict the shape of `ex085_dW1` as a literal Python tuple
- `ex085_dW2_shape`: predict the shape of `ex085_dW2` as a literal Python tuple

**Next concept:** Multiply then add.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-1.0, 1.5, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
W1 = torch.linspace(-0.8, 1.4, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
H = X @ W1
W2 = torch.linspace(0.7, -1.0, steps=8, dtype=DTYPE).reshape(4, 2).requires_grad_()
Y = H @ W2
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex085", Y, {"dH": H, "dX": X, "dW1": W1, "dW2": W2}, dout)
print("X", tuple(X.shape), "H", tuple(H.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 085: derive manually; do not use autograd in this cell.
# Define `ex085_dH` — the hidden-state gradient.
# Define `ex085_dX` — the input gradient.
# Define `ex085_dW1` — the first weight gradient.
# Define `ex085_dW2` — the second weight gradient.
# Predict every visible tensor shape before computing values.
# Define `ex085_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex085_W1_shape` — the shape of `W1` as a literal Python tuple.
# Define `ex085_H_shape` — the shape of `H` as a literal Python tuple.
# Define `ex085_W2_shape` — the shape of `W2` as a literal Python tuple.
# Define `ex085_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex085_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex085_dH_shape` — the shape of `ex085_dH` as a literal Python tuple.
# Define `ex085_dX_shape` — the shape of `ex085_dX` as a literal Python tuple.
# Define `ex085_dW1_shape` — the shape of `ex085_dW1` as a literal Python tuple.
# Define `ex085_dW2_shape` — the shape of `ex085_dW2` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex085_X_shape", "X")
_check_visible_tensor_shape("ex085_W1_shape", "W1")
_check_visible_tensor_shape("ex085_H_shape", "H")
_check_visible_tensor_shape("ex085_W2_shape", "W2")
_check_visible_tensor_shape("ex085_Y_shape", "Y")
_check_visible_tensor_shape("ex085_dout_shape", "dout")
_check_private_tensor_shape("ex085_dH_shape", "ex085", "dH")
_check_private_tensor_shape("ex085_dX_shape", "ex085", "dX")
_check_private_tensor_shape("ex085_dW1_shape", "ex085", "dW1")
_check_private_tensor_shape("ex085_dW2_shape", "ex085", "dW2")
_check_tensor("ex085_dH", "ex085", "dH")
_check_tensor("ex085_dX", "ex085", "dX")
_check_tensor("ex085_dW1", "ex085", "dW1")
_check_tensor("ex085_dW2", "ex085", "dW2")


## 6. Shallow chains, deeper chains, and fan-out

A chain passes the upstream gradient through operations in reverse order. A branch sends one value along multiple paths; backward adds the contributions when those paths meet again.

For a branch from `x` to intermediate values `u` and `v`, the total gradient is

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial u}
\frac{\partial u}{\partial x}
+
\frac{\partial L}{\partial v}
\frac{\partial v}{\partial x}
$$

Here, both terms represent distinct paths from `x` to the same scalar objective `L`. A zero gradient blocks only its own path; another path may still contribute.


### Exercise 086 — Multiply then add

**Purpose:** Learn to reverse a two-step chain by undoing the last operation first.

**Inputs:** `x`, `a`, and `y` all have shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `a = 3 * x` and `y = a + 2`.

**Derive/do:** Derive `da` and `dx`.

**Ingredients:** Start from `dout`, reverse addition, then reverse multiplication.

**Required outputs:**

- `ex086_da`: the intermediate gradient with respect to `a`.
- `ex086_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex086_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex086_a_shape`: predict the shape of `a` as a literal Python tuple
- `ex086_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex086_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex086_da_shape`: predict the shape of `ex086_da` as a literal Python tuple
- `ex086_dx_shape`: predict the shape of `ex086_dx` as a literal Python tuple

**Next concept:** Square then sum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
a = 3.0 * x
y = a + 2.0
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex086", y, {"da": a, "dx": x}, dout)
print("x", tuple(x.shape), "a", tuple(a.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 086: derive manually; do not use autograd in this cell.
# Define `ex086_da` — the intermediate gradient with respect to `a`.
# Define `ex086_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex086_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex086_a_shape` — the shape of `a` as a literal Python tuple.
# Define `ex086_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex086_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex086_da_shape` — the shape of `ex086_da` as a literal Python tuple.
# Define `ex086_dx_shape` — the shape of `ex086_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex086_x_shape", "x")
_check_visible_tensor_shape("ex086_a_shape", "a")
_check_visible_tensor_shape("ex086_y_shape", "y")
_check_visible_tensor_shape("ex086_dout_shape", "dout")
_check_private_tensor_shape("ex086_da_shape", "ex086", "da")
_check_private_tensor_shape("ex086_dx_shape", "ex086", "dx")
_check_tensor("ex086_da", "ex086", "da")
_check_tensor("ex086_dx", "ex086", "dx")


### Exercise 087 — Square then sum

**Purpose:** Learn to reverse a scalar sum first and then the square that produced its entries.

**Inputs:** `x` and `squared` have shape `(4,)`; `y` is scalar.

**Forward operation:** `squared = x**2` and `y = squared.sum()`.

**Derive/do:** Derive `dsquared` and `dx`.

**Ingredients:** Reverse the total first, then the square.

**Required outputs:**

- `ex087_dsquared`: the gradient after reversing the sum.
- `ex087_dx`: the gradient after also reversing the square.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex087_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex087_squared_shape`: predict the shape of `squared` as a literal Python tuple
- `ex087_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex087_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex087_dsquared_shape`: predict the shape of `ex087_dsquared` as a literal Python tuple
- `ex087_dx_shape`: predict the shape of `ex087_dx` as a literal Python tuple

**Next concept:** Exponential then sum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-2.0, -0.5, 1.0, 3.0], dtype=DTYPE, requires_grad=True)
squared = x**2
y = squared.sum()
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex087", y, {"dsquared": squared, "dx": x}, dout)
print("x", tuple(x.shape), "squared", tuple(squared.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 087: derive manually; do not use autograd in this cell.
# Define `ex087_dsquared` — the gradient after reversing the sum.
# Define `ex087_dx` — the gradient after also reversing the square.
# Predict every visible tensor shape before computing values.
# Define `ex087_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex087_squared_shape` — the shape of `squared` as a literal Python tuple.
# Define `ex087_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex087_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex087_dsquared_shape` — the shape of `ex087_dsquared` as a literal Python tuple.
# Define `ex087_dx_shape` — the shape of `ex087_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex087_x_shape", "x")
_check_visible_tensor_shape("ex087_squared_shape", "squared")
_check_visible_tensor_shape("ex087_y_shape", "y")
_check_visible_tensor_shape("ex087_dout_shape", "dout")
_check_private_tensor_shape("ex087_dsquared_shape", "ex087", "dsquared")
_check_private_tensor_shape("ex087_dx_shape", "ex087", "dx")
_check_tensor("ex087_dsquared", "ex087", "dsquared")
_check_tensor("ex087_dx", "ex087", "dx")


### Exercise 088 — Exponential then sum

**Purpose:** Learn to reverse a sum and then use the saved exponential values to continue backward.

**Inputs:** `x` and `exp_x` have shape `(3,)`; `y` is scalar.

**Forward operation:** `exp_x = x.exp()` and `y = exp_x.sum()`.

**Derive/do:** Derive `dexp_x` and `dx`.

**Ingredients:** Reverse the sum, then use the exponential forward value.

**Required outputs:**

- `ex088_dexp_x`: the gradient with respect to `exp_x`.
- `ex088_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex088_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex088_exp_x_shape`: predict the shape of `exp_x` as a literal Python tuple
- `ex088_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex088_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex088_dexp_x_shape`: predict the shape of `ex088_dexp_x` as a literal Python tuple
- `ex088_dx_shape`: predict the shape of `ex088_dx` as a literal Python tuple

**Next concept:** Log then weighted sum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-1.0, 0.0, 1.5], dtype=DTYPE, requires_grad=True)
exp_x = x.exp()
y = exp_x.sum()
dout = torch.tensor(-0.75, dtype=DTYPE)
_capture("ex088", y, {"dexp_x": exp_x, "dx": x}, dout)
print("x", tuple(x.shape), "exp_x", tuple(exp_x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 088: derive manually; do not use autograd in this cell.
# Define `ex088_dexp_x` — the gradient with respect to `exp_x`.
# Define `ex088_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex088_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex088_exp_x_shape` — the shape of `exp_x` as a literal Python tuple.
# Define `ex088_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex088_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex088_dexp_x_shape` — the shape of `ex088_dexp_x` as a literal Python tuple.
# Define `ex088_dx_shape` — the shape of `ex088_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex088_x_shape", "x")
_check_visible_tensor_shape("ex088_exp_x_shape", "exp_x")
_check_visible_tensor_shape("ex088_y_shape", "y")
_check_visible_tensor_shape("ex088_dout_shape", "dout")
_check_private_tensor_shape("ex088_dexp_x_shape", "ex088", "dexp_x")
_check_private_tensor_shape("ex088_dx_shape", "ex088", "dx")
_check_tensor("ex088_dexp_x", "ex088", "dexp_x")
_check_tensor("ex088_dx", "ex088", "dx")


### Exercise 089 — Log then weighted sum

**Purpose:** Learn to reverse a weighted sum before passing its gradient through a logarithm.

**Inputs:** Positive `x`, `log_x`, and `weights` have shape `(3,)`; `y` is scalar.

**Forward operation:** `log_x = x.log()` and `y = (log_x * weights).sum()`.

**Derive/do:** Derive `dlog_x` and `dx`.

**Ingredients:** Reverse sum, multiplication by weights, and logarithm.

**Required outputs:**

- `ex089_dlog_x`: the gradient entering the logarithm.
- `ex089_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex089_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex089_weights_shape`: predict the shape of `weights` as a literal Python tuple
- `ex089_log_x_shape`: predict the shape of `log_x` as a literal Python tuple
- `ex089_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex089_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex089_dlog_x_shape`: predict the shape of `ex089_dlog_x` as a literal Python tuple
- `ex089_dx_shape`: predict the shape of `ex089_dx` as a literal Python tuple

**Next concept:** Tanh then weighted sum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([0.5, 2.0, 5.0], dtype=DTYPE, requires_grad=True)
weights = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)
log_x = x.log()
y = (log_x * weights).sum()
dout = torch.tensor(1.25, dtype=DTYPE)
_capture("ex089", y, {"dlog_x": log_x, "dx": x}, dout)
print("x", tuple(x.shape), "log_x", tuple(log_x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 089: derive manually; do not use autograd in this cell.
# Define `ex089_dlog_x` — the gradient entering the logarithm.
# Define `ex089_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex089_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex089_weights_shape` — the shape of `weights` as a literal Python tuple.
# Define `ex089_log_x_shape` — the shape of `log_x` as a literal Python tuple.
# Define `ex089_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex089_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex089_dlog_x_shape` — the shape of `ex089_dlog_x` as a literal Python tuple.
# Define `ex089_dx_shape` — the shape of `ex089_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex089_x_shape", "x")
_check_visible_tensor_shape("ex089_weights_shape", "weights")
_check_visible_tensor_shape("ex089_log_x_shape", "log_x")
_check_visible_tensor_shape("ex089_y_shape", "y")
_check_visible_tensor_shape("ex089_dout_shape", "dout")
_check_private_tensor_shape("ex089_dlog_x_shape", "ex089", "dlog_x")
_check_private_tensor_shape("ex089_dx_shape", "ex089", "dx")
_check_tensor("ex089_dlog_x", "ex089", "dlog_x")
_check_tensor("ex089_dx", "ex089", "dx")


### Exercise 090 — Tanh then weighted sum

**Purpose:** Learn to pass a weighted-sum gradient backward through the MLP's `tanh` activation.

**Inputs:** `x`, `h`, and `weights` have shape `(4,)`; `y` is scalar.

**Forward operation:** `h = x.tanh()` and `y = (h * weights).sum()`.

**Derive/do:** Derive `dh` and `dx`.

**Ingredients:** Reverse the weighted sum, then apply tanh's local derivative using `h`.

**Required outputs:**

- `ex090_dh`: the gradient entering tanh.
- `ex090_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex090_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex090_weights_shape`: predict the shape of `weights` as a literal Python tuple
- `ex090_h_shape`: predict the shape of `h` as a literal Python tuple
- `ex090_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex090_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex090_dh_shape`: predict the shape of `ex090_dh` as a literal Python tuple
- `ex090_dx_shape`: predict the shape of `ex090_dx` as a literal Python tuple

**Next concept:** Product, square, and sum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-2.0, -0.5, 0.5, 2.0], dtype=DTYPE, requires_grad=True)
weights = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
h = x.tanh()
y = (h * weights).sum()
dout = torch.tensor(-1.5, dtype=DTYPE)
_capture("ex090", y, {"dh": h, "dx": x}, dout)
print("x", tuple(x.shape), "h", tuple(h.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 090: derive manually; do not use autograd in this cell.
# Define `ex090_dh` — the gradient entering tanh.
# Define `ex090_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex090_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex090_weights_shape` — the shape of `weights` as a literal Python tuple.
# Define `ex090_h_shape` — the shape of `h` as a literal Python tuple.
# Define `ex090_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex090_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex090_dh_shape` — the shape of `ex090_dh` as a literal Python tuple.
# Define `ex090_dx_shape` — the shape of `ex090_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex090_x_shape", "x")
_check_visible_tensor_shape("ex090_weights_shape", "weights")
_check_visible_tensor_shape("ex090_h_shape", "h")
_check_visible_tensor_shape("ex090_y_shape", "y")
_check_visible_tensor_shape("ex090_dout_shape", "dout")
_check_private_tensor_shape("ex090_dh_shape", "ex090", "dh")
_check_private_tensor_shape("ex090_dx_shape", "ex090", "dx")
_check_tensor("ex090_dh", "ex090", "dh")
_check_tensor("ex090_dx", "ex090", "dx")


### Exercise 091 — Product, square, and sum

**Purpose:** Learn to track two inputs backward through multiplication, squaring, and summation.

**Inputs:** `x`, `z`, and `product` have shape `(3,)`; `y` is scalar.

**Forward operation:** `product = x * z`, `squared = product**2`, and `y = squared.sum()`.

**Derive/do:** Derive `dsquared`, `dproduct`, `dx`, and `dz`.

**Ingredients:** Reverse one operation at a time; product backward sends gradients to both inputs.

**Required outputs:**

- `ex091_dsquared`: the gradient after reversing the sum.
- `ex091_dproduct`: the gradient after reversing the square.
- `ex091_dx`: the gradient with respect to `x`.
- `ex091_dz`: the gradient with respect to `z`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex091_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex091_z_shape`: predict the shape of `z` as a literal Python tuple
- `ex091_product_shape`: predict the shape of `product` as a literal Python tuple
- `ex091_squared_shape`: predict the shape of `squared` as a literal Python tuple
- `ex091_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex091_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex091_dsquared_shape`: predict the shape of `ex091_dsquared` as a literal Python tuple
- `ex091_dproduct_shape`: predict the shape of `ex091_dproduct` as a literal Python tuple
- `ex091_dx_shape`: predict the shape of `ex091_dx` as a literal Python tuple
- `ex091_dz_shape`: predict the shape of `ex091_dz` as a literal Python tuple

**Next concept:** Square and linear fan-out.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([2.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([0.5, 4.0, -2.0], dtype=DTYPE, requires_grad=True)
product = x * z
squared = product**2
y = squared.sum()
dout = torch.tensor(0.75, dtype=DTYPE)
_capture("ex091", y, {"dsquared": squared, "dproduct": product, "dx": x, "dz": z}, dout)
print("product", tuple(product.shape), "squared", tuple(squared.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 091: derive manually; do not use autograd in this cell.
# Define `ex091_dsquared` — the gradient after reversing the sum.
# Define `ex091_dproduct` — the gradient after reversing the square.
# Define `ex091_dx` — the gradient with respect to `x`.
# Define `ex091_dz` — the gradient with respect to `z`.
# Predict every visible tensor shape before computing values.
# Define `ex091_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex091_z_shape` — the shape of `z` as a literal Python tuple.
# Define `ex091_product_shape` — the shape of `product` as a literal Python tuple.
# Define `ex091_squared_shape` — the shape of `squared` as a literal Python tuple.
# Define `ex091_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex091_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex091_dsquared_shape` — the shape of `ex091_dsquared` as a literal Python tuple.
# Define `ex091_dproduct_shape` — the shape of `ex091_dproduct` as a literal Python tuple.
# Define `ex091_dx_shape` — the shape of `ex091_dx` as a literal Python tuple.
# Define `ex091_dz_shape` — the shape of `ex091_dz` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex091_x_shape", "x")
_check_visible_tensor_shape("ex091_z_shape", "z")
_check_visible_tensor_shape("ex091_product_shape", "product")
_check_visible_tensor_shape("ex091_squared_shape", "squared")
_check_visible_tensor_shape("ex091_y_shape", "y")
_check_visible_tensor_shape("ex091_dout_shape", "dout")
_check_private_tensor_shape("ex091_dsquared_shape", "ex091", "dsquared")
_check_private_tensor_shape("ex091_dproduct_shape", "ex091", "dproduct")
_check_private_tensor_shape("ex091_dx_shape", "ex091", "dx")
_check_private_tensor_shape("ex091_dz_shape", "ex091", "dz")
_check_tensor("ex091_dsquared", "ex091", "dsquared")
_check_tensor("ex091_dproduct", "ex091", "dproduct")
_check_tensor("ex091_dx", "ex091", "dx")
_check_tensor("ex091_dz", "ex091", "dz")


### Exercise 092 — Square and linear fan-out

**Purpose:** Learn that when `x` reaches the output through a square branch and a linear branch, both gradients must be added.

**Inputs:** `x`, `square`, `linear`, and `y` have shape `(3,)`.

**Forward operation:** `square = x**2`, `linear = 3*x`, and `y = square + linear`.

**Derive/do:** Derive `dsquare`, `dlinear`, and total `dx`.

**Ingredients:** Reverse the final addition; compute each branch contribution to `x`; add them.

**Required outputs:**

- `ex092_dsquare`: the upstream gradient on the square branch.
- `ex092_dlinear`: the upstream gradient on the linear branch.
- `ex092_dx`: the sum of both contributions at `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex092_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex092_square_shape`: predict the shape of `square` as a literal Python tuple
- `ex092_linear_shape`: predict the shape of `linear` as a literal Python tuple
- `ex092_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex092_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex092_dsquare_shape`: predict the shape of `ex092_dsquare` as a literal Python tuple
- `ex092_dlinear_shape`: predict the shape of `ex092_dlinear` as a literal Python tuple
- `ex092_dx_shape`: predict the shape of `ex092_dx` as a literal Python tuple

**Next concept:** Exponential and square fan-out.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
square = x**2
linear = 3.0 * x
y = square + linear
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex092", y, {"dsquare": square, "dlinear": linear, "dx": x}, dout)
print("All forward branch tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 092: derive manually; do not use autograd in this cell.
# Define `ex092_dsquare` — the upstream gradient on the square branch.
# Define `ex092_dlinear` — the upstream gradient on the linear branch.
# Define `ex092_dx` — the sum of both contributions at `x`.
# Predict every visible tensor shape before computing values.
# Define `ex092_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex092_square_shape` — the shape of `square` as a literal Python tuple.
# Define `ex092_linear_shape` — the shape of `linear` as a literal Python tuple.
# Define `ex092_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex092_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex092_dsquare_shape` — the shape of `ex092_dsquare` as a literal Python tuple.
# Define `ex092_dlinear_shape` — the shape of `ex092_dlinear` as a literal Python tuple.
# Define `ex092_dx_shape` — the shape of `ex092_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex092_x_shape", "x")
_check_visible_tensor_shape("ex092_square_shape", "square")
_check_visible_tensor_shape("ex092_linear_shape", "linear")
_check_visible_tensor_shape("ex092_y_shape", "y")
_check_visible_tensor_shape("ex092_dout_shape", "dout")
_check_private_tensor_shape("ex092_dsquare_shape", "ex092", "dsquare")
_check_private_tensor_shape("ex092_dlinear_shape", "ex092", "dlinear")
_check_private_tensor_shape("ex092_dx_shape", "ex092", "dx")
_check_tensor("ex092_dsquare", "ex092", "dsquare")
_check_tensor("ex092_dlinear", "ex092", "dlinear")
_check_tensor("ex092_dx", "ex092", "dx")


### Exercise 093 — Exponential and square fan-out

**Purpose:** Learn to add the two gradients that return to `x` through exponential and square branches.

**Inputs:** `x`, both branches, and `y` have shape `(3,)`.

**Forward operation:** `left = x.exp()`, `right = x**2`, and `y = left + right`.

**Derive/do:** Derive branch gradients and total `dx`.

**Ingredients:** Reverse addition; use each branch's local derivative; add at the shared source.

**Required outputs:**

- `ex093_dleft`: the upstream gradient on the exponential branch.
- `ex093_dright`: the upstream gradient on the square branch.
- `ex093_dx`: the total gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex093_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex093_left_shape`: predict the shape of `left` as a literal Python tuple
- `ex093_right_shape`: predict the shape of `right` as a literal Python tuple
- `ex093_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex093_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex093_dleft_shape`: predict the shape of `ex093_dleft` as a literal Python tuple
- `ex093_dright_shape`: predict the shape of `ex093_dright` as a literal Python tuple
- `ex093_dx_shape`: predict the shape of `ex093_dx` as a literal Python tuple

**Next concept:** One tensor in both product slots.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-1.0, 0.5, 2.0], dtype=DTYPE, requires_grad=True)
left = x.exp()
right = x**2
y = left + right
dout = torch.tensor([2.0, -1.0, 0.25], dtype=DTYPE)
_capture("ex093", y, {"dleft": left, "dright": right, "dx": x}, dout)
print("All forward branch tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 093: derive manually; do not use autograd in this cell.
# Define `ex093_dleft` — the upstream gradient on the exponential branch.
# Define `ex093_dright` — the upstream gradient on the square branch.
# Define `ex093_dx` — the total gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex093_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex093_left_shape` — the shape of `left` as a literal Python tuple.
# Define `ex093_right_shape` — the shape of `right` as a literal Python tuple.
# Define `ex093_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex093_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex093_dleft_shape` — the shape of `ex093_dleft` as a literal Python tuple.
# Define `ex093_dright_shape` — the shape of `ex093_dright` as a literal Python tuple.
# Define `ex093_dx_shape` — the shape of `ex093_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex093_x_shape", "x")
_check_visible_tensor_shape("ex093_left_shape", "left")
_check_visible_tensor_shape("ex093_right_shape", "right")
_check_visible_tensor_shape("ex093_y_shape", "y")
_check_visible_tensor_shape("ex093_dout_shape", "dout")
_check_private_tensor_shape("ex093_dleft_shape", "ex093", "dleft")
_check_private_tensor_shape("ex093_dright_shape", "ex093", "dright")
_check_private_tensor_shape("ex093_dx_shape", "ex093", "dx")
_check_tensor("ex093_dleft", "ex093", "dleft")
_check_tensor("ex093_dright", "ex093", "dright")
_check_tensor("ex093_dx", "ex093", "dx")


### Exercise 094 — One tensor in both product slots

**Purpose:** Learn that `x * x` uses `x` twice, so backward has two contributions even though both inputs have the same name.

**Inputs:** `x` and `y` have shape `(3,)`.

**Forward operation:** `y = x * x`.

**Derive/do:** Derive total `dx` without treating the two uses as one path.

**Ingredients:** Imagine temporarily naming the left and right copies separately; compute and add both contributions.

**Required outputs:**

- `ex094_dx`: the total gradient from both uses of `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex094_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex094_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex094_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex094_dx_shape`: predict the shape of `ex094_dx` as a literal Python tuple

**Next concept:** Broadcast, square, and average.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x * x
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex094", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 094: derive manually; do not use autograd in this cell.
# Define `ex094_dx` — the total gradient from both uses of `x`.
# Predict every visible tensor shape before computing values.
# Define `ex094_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex094_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex094_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex094_dx_shape` — the shape of `ex094_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex094_x_shape", "x")
_check_visible_tensor_shape("ex094_y_shape", "y")
_check_visible_tensor_shape("ex094_dout_shape", "dout")
_check_private_tensor_shape("ex094_dx_shape", "ex094", "dx")
_check_tensor("ex094_dx", "ex094", "dx")


### Exercise 095 — Broadcast, square, and average

**Purpose:** Learn to combine a mean, a square, and a reused bias in one backward chain.

**Inputs:** `x` is `(3, 4)`, `bias` is `(4,)`, and the scalar loss averages 12 squared entries.

**Forward operation:** `pre = x + bias`, `squared = pre**2`, and `loss = squared.mean()`.

**Derive/do:** Derive `dsquared`, `dpre`, `dx`, and `dbias`.

**Ingredients:** Reverse mean, square, and broadcast addition in order.

**Required outputs:**

- `ex095_dsquared`: the gradient after reversing the mean.
- `ex095_dpre`: the gradient after reversing the square.
- `ex095_dx`: the input gradient.
- `ex095_dbias`: the accumulated bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex095_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex095_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex095_pre_shape`: predict the shape of `pre` as a literal Python tuple
- `ex095_squared_shape`: predict the shape of `squared` as a literal Python tuple
- `ex095_loss_local_shape`: predict the shape of `loss_local` as a literal Python tuple
- `ex095_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex095_dsquared_shape`: predict the shape of `ex095_dsquared` as a literal Python tuple
- `ex095_dpre_shape`: predict the shape of `ex095_dpre` as a literal Python tuple
- `ex095_dx_shape`: predict the shape of `ex095_dx` as a literal Python tuple
- `ex095_dbias_shape`: predict the shape of `ex095_dbias` as a literal Python tuple

**Next concept:** Row totals, square, then total.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x + bias
squared = pre**2
loss_local = squared.mean()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex095", loss_local, {"dsquared": squared, "dpre": pre, "dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "pre", tuple(pre.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 095: derive manually; do not use autograd in this cell.
# Define `ex095_dsquared` — the gradient after reversing the mean.
# Define `ex095_dpre` — the gradient after reversing the square.
# Define `ex095_dx` — the input gradient.
# Define `ex095_dbias` — the accumulated bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex095_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex095_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex095_pre_shape` — the shape of `pre` as a literal Python tuple.
# Define `ex095_squared_shape` — the shape of `squared` as a literal Python tuple.
# Define `ex095_loss_local_shape` — the shape of `loss_local` as a literal Python tuple.
# Define `ex095_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex095_dsquared_shape` — the shape of `ex095_dsquared` as a literal Python tuple.
# Define `ex095_dpre_shape` — the shape of `ex095_dpre` as a literal Python tuple.
# Define `ex095_dx_shape` — the shape of `ex095_dx` as a literal Python tuple.
# Define `ex095_dbias_shape` — the shape of `ex095_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex095_x_shape", "x")
_check_visible_tensor_shape("ex095_bias_shape", "bias")
_check_visible_tensor_shape("ex095_pre_shape", "pre")
_check_visible_tensor_shape("ex095_squared_shape", "squared")
_check_visible_tensor_shape("ex095_loss_local_shape", "loss_local")
_check_visible_tensor_shape("ex095_dout_shape", "dout")
_check_private_tensor_shape("ex095_dsquared_shape", "ex095", "dsquared")
_check_private_tensor_shape("ex095_dpre_shape", "ex095", "dpre")
_check_private_tensor_shape("ex095_dx_shape", "ex095", "dx")
_check_private_tensor_shape("ex095_dbias_shape", "ex095", "dbias")
_check_tensor("ex095_dsquared", "ex095", "dsquared")
_check_tensor("ex095_dpre", "ex095", "dpre")
_check_tensor("ex095_dx", "ex095", "dx")
_check_tensor("ex095_dbias", "ex095", "dbias")


### Exercise 096 — Row totals, square, then total

**Purpose:** Learn to restore matrix-shaped gradients after row sums are squared and then summed again.

**Inputs:** `x` is `(3, 4)`, `rows` is `(3,)`, and the final output is scalar.

**Forward operation:** `rows = x.sum(dim=1)`, `squared = rows**2`, and `y = squared.sum()`.

**Derive/do:** Derive `dsquared`, `drows`, and `dx`.

**Ingredients:** Reverse the final total, square, then row reduction; restore four columns per row.

**Required outputs:**

- `ex096_dsquared`: the gradient after reversing the final sum.
- `ex096_drows`: the gradient with respect to row totals.
- `ex096_dx`: the matrix gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex096_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex096_rows_shape`: predict the shape of `rows` as a literal Python tuple
- `ex096_squared_shape`: predict the shape of `squared` as a literal Python tuple
- `ex096_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex096_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex096_dsquared_shape`: predict the shape of `ex096_dsquared` as a literal Python tuple
- `ex096_drows_shape`: predict the shape of `ex096_drows` as a literal Python tuple
- `ex096_dx_shape`: predict the shape of `ex096_dx` as a literal Python tuple

**Next concept:** Affine then tanh.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
rows = x.sum(dim=1)
squared = rows**2
y = squared.sum()
dout = torch.tensor(-0.5, dtype=DTYPE)
_capture("ex096", y, {"dsquared": squared, "drows": rows, "dx": x}, dout)
print("x", tuple(x.shape), "rows", tuple(rows.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 096: derive manually; do not use autograd in this cell.
# Define `ex096_dsquared` — the gradient after reversing the final sum.
# Define `ex096_drows` — the gradient with respect to row totals.
# Define `ex096_dx` — the matrix gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex096_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex096_rows_shape` — the shape of `rows` as a literal Python tuple.
# Define `ex096_squared_shape` — the shape of `squared` as a literal Python tuple.
# Define `ex096_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex096_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex096_dsquared_shape` — the shape of `ex096_dsquared` as a literal Python tuple.
# Define `ex096_drows_shape` — the shape of `ex096_drows` as a literal Python tuple.
# Define `ex096_dx_shape` — the shape of `ex096_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex096_x_shape", "x")
_check_visible_tensor_shape("ex096_rows_shape", "rows")
_check_visible_tensor_shape("ex096_squared_shape", "squared")
_check_visible_tensor_shape("ex096_y_shape", "y")
_check_visible_tensor_shape("ex096_dout_shape", "dout")
_check_private_tensor_shape("ex096_dsquared_shape", "ex096", "dsquared")
_check_private_tensor_shape("ex096_drows_shape", "ex096", "drows")
_check_private_tensor_shape("ex096_dx_shape", "ex096", "dx")
_check_tensor("ex096_dsquared", "ex096", "dsquared")
_check_tensor("ex096_drows", "ex096", "drows")
_check_tensor("ex096_dx", "ex096", "dx")


### Exercise 097 — Affine then tanh

**Purpose:** Learn to reverse `tanh` and then the matrix multiplication and bias addition before it.

**Inputs:** `X` is `(3, 2)`, `W` is `(2, 4)`, `bias` is `(4,)`, and `h` is `(3, 4)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `pre = X @ W + bias` and `h = pre.tanh()`.

**Derive/do:** Derive `dpre`, `dX`, `dW`, and `dbias` from supplied `dout`.

**Ingredients:** Reverse tanh first, then the affine layer; preserve batch and hidden axes.

**Required outputs:**

- `ex097_dpre`: the gradient entering the affine operation.
- `ex097_dX`: the input gradient.
- `ex097_dW`: the weight gradient.
- `ex097_dbias`: the bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex097_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex097_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex097_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex097_pre_shape`: predict the shape of `pre` as a literal Python tuple
- `ex097_h_shape`: predict the shape of `h` as a literal Python tuple
- `ex097_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex097_dpre_shape`: predict the shape of `ex097_dpre` as a literal Python tuple
- `ex097_dX_shape`: predict the shape of `ex097_dX` as a literal Python tuple
- `ex097_dW_shape`: predict the shape of `ex097_dW` as a literal Python tuple
- `ex097_dbias_shape`: predict the shape of `ex097_dbias` as a literal Python tuple

**Next concept:** Affine then ReLU then total.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.linspace(-1.0, 1.5, steps=6, dtype=DTYPE).reshape(3, 2).requires_grad_()
W = torch.linspace(-0.8, 1.3, steps=8, dtype=DTYPE).reshape(2, 4).requires_grad_()
bias = torch.tensor([0.25, -0.5, 1.0, -1.5], dtype=DTYPE, requires_grad=True)
pre = X @ W + bias
h = pre.tanh()
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex097", h, {"dpre": pre, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "pre", tuple(pre.shape), "h", tuple(h.shape))


In [ ]:
# Exercise 097: derive manually; do not use autograd in this cell.
# Define `ex097_dpre` — the gradient entering the affine operation.
# Define `ex097_dX` — the input gradient.
# Define `ex097_dW` — the weight gradient.
# Define `ex097_dbias` — the bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex097_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex097_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex097_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex097_pre_shape` — the shape of `pre` as a literal Python tuple.
# Define `ex097_h_shape` — the shape of `h` as a literal Python tuple.
# Define `ex097_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex097_dpre_shape` — the shape of `ex097_dpre` as a literal Python tuple.
# Define `ex097_dX_shape` — the shape of `ex097_dX` as a literal Python tuple.
# Define `ex097_dW_shape` — the shape of `ex097_dW` as a literal Python tuple.
# Define `ex097_dbias_shape` — the shape of `ex097_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex097_X_shape", "X")
_check_visible_tensor_shape("ex097_W_shape", "W")
_check_visible_tensor_shape("ex097_bias_shape", "bias")
_check_visible_tensor_shape("ex097_pre_shape", "pre")
_check_visible_tensor_shape("ex097_h_shape", "h")
_check_visible_tensor_shape("ex097_dout_shape", "dout")
_check_private_tensor_shape("ex097_dpre_shape", "ex097", "dpre")
_check_private_tensor_shape("ex097_dX_shape", "ex097", "dX")
_check_private_tensor_shape("ex097_dW_shape", "ex097", "dW")
_check_private_tensor_shape("ex097_dbias_shape", "ex097", "dbias")
_check_tensor("ex097_dpre", "ex097", "dpre")
_check_tensor("ex097_dX", "ex097", "dX")
_check_tensor("ex097_dW", "ex097", "dW")
_check_tensor("ex097_dbias", "ex097", "dbias")


### Exercise 098 — Affine then ReLU then total

**Purpose:** Learn to reverse a scalar total, then a ReLU mask, and then an affine layer.

**Inputs:** `pre` and `h` are `(3, 4)`; no pre-activation is exactly zero.

**Forward operation:** `pre = X @ W + bias`, `h = pre.relu()`, and `loss = h.sum()`.

**Derive/do:** Derive `dh`, `dpre`, `dX`, `dW`, and `dbias`.

**Ingredients:** Reverse total, ReLU mask, and affine layer.

**Required outputs:**

- `ex098_dh`: the gradient after reversing the total.
- `ex098_dpre`: the gradient after reversing ReLU.
- `ex098_dX`: the input gradient.
- `ex098_dW`: the weight gradient.
- `ex098_dbias`: the bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex098_X_shape`: predict the shape of `X` as a literal Python tuple
- `ex098_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex098_bias_shape`: predict the shape of `bias` as a literal Python tuple
- `ex098_pre_shape`: predict the shape of `pre` as a literal Python tuple
- `ex098_h_shape`: predict the shape of `h` as a literal Python tuple
- `ex098_loss_local_shape`: predict the shape of `loss_local` as a literal Python tuple
- `ex098_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex098_dh_shape`: predict the shape of `ex098_dh` as a literal Python tuple
- `ex098_dpre_shape`: predict the shape of `ex098_dpre` as a literal Python tuple
- `ex098_dX_shape`: predict the shape of `ex098_dX` as a literal Python tuple
- `ex098_dW_shape`: predict the shape of `ex098_dW` as a literal Python tuple
- `ex098_dbias_shape`: predict the shape of `ex098_dbias` as a literal Python tuple

**Next concept:** Exp, sum, then log.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
X = torch.tensor([[-1.0, 0.5], [1.5, -2.0], [0.25, 3.0]], dtype=DTYPE, requires_grad=True)
W = torch.tensor([[0.7, -1.2, 2.1, 0.3], [-0.4, 1.3, 0.6, -2.0]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.11, -0.37, 0.83, -1.19], dtype=DTYPE, requires_grad=True)
pre = X @ W + bias
h = pre.relu()
loss_local = h.sum()
dout = torch.tensor(0.75, dtype=DTYPE)
_capture("ex098", loss_local, {"dh": h, "dpre": pre, "dX": X, "dW": W, "dbias": bias}, dout)
print("pre", tuple(pre.shape), "h", tuple(h.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 098: derive manually; do not use autograd in this cell.
# Define `ex098_dh` — the gradient after reversing the total.
# Define `ex098_dpre` — the gradient after reversing ReLU.
# Define `ex098_dX` — the input gradient.
# Define `ex098_dW` — the weight gradient.
# Define `ex098_dbias` — the bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex098_X_shape` — the shape of `X` as a literal Python tuple.
# Define `ex098_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex098_bias_shape` — the shape of `bias` as a literal Python tuple.
# Define `ex098_pre_shape` — the shape of `pre` as a literal Python tuple.
# Define `ex098_h_shape` — the shape of `h` as a literal Python tuple.
# Define `ex098_loss_local_shape` — the shape of `loss_local` as a literal Python tuple.
# Define `ex098_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex098_dh_shape` — the shape of `ex098_dh` as a literal Python tuple.
# Define `ex098_dpre_shape` — the shape of `ex098_dpre` as a literal Python tuple.
# Define `ex098_dX_shape` — the shape of `ex098_dX` as a literal Python tuple.
# Define `ex098_dW_shape` — the shape of `ex098_dW` as a literal Python tuple.
# Define `ex098_dbias_shape` — the shape of `ex098_dbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex098_X_shape", "X")
_check_visible_tensor_shape("ex098_W_shape", "W")
_check_visible_tensor_shape("ex098_bias_shape", "bias")
_check_visible_tensor_shape("ex098_pre_shape", "pre")
_check_visible_tensor_shape("ex098_h_shape", "h")
_check_visible_tensor_shape("ex098_loss_local_shape", "loss_local")
_check_visible_tensor_shape("ex098_dout_shape", "dout")
_check_private_tensor_shape("ex098_dh_shape", "ex098", "dh")
_check_private_tensor_shape("ex098_dpre_shape", "ex098", "dpre")
_check_private_tensor_shape("ex098_dX_shape", "ex098", "dX")
_check_private_tensor_shape("ex098_dW_shape", "ex098", "dW")
_check_private_tensor_shape("ex098_dbias_shape", "ex098", "dbias")
_check_tensor("ex098_dh", "ex098", "dh")
_check_tensor("ex098_dpre", "ex098", "dpre")
_check_tensor("ex098_dX", "ex098", "dX")
_check_tensor("ex098_dW", "ex098", "dW")
_check_tensor("ex098_dbias", "ex098", "dbias")


### Exercise 099 — Exp, sum, then log

**Purpose:** Learn to work backward through exponential, sum, and logarithm operations one step at a time.

**Inputs:** `x` and `exp_x` are `(4,)`; `total` and `y` are scalar.

**Forward operation:** `exp_x = x.exp()`, `total = exp_x.sum()`, and `y = total.log()`.

**Derive/do:** Derive `dtotal`, `dexp_x`, and `dx`.

**Ingredients:** Reverse log, scalar sum, and exp in order.

**Required outputs:**

- `ex099_dtotal`: the gradient entering the sum.
- `ex099_dexp_x`: the gradient entering exp.
- `ex099_dx`: the gradient with respect to `x`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex099_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex099_exp_x_shape`: predict the shape of `exp_x` as a literal Python tuple
- `ex099_total_shape`: predict the shape of `total` as a literal Python tuple
- `ex099_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex099_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex099_dtotal_shape`: predict the shape of `ex099_dtotal` as a literal Python tuple
- `ex099_dexp_x_shape`: predict the shape of `ex099_dexp_x` as a literal Python tuple
- `ex099_dx_shape`: predict the shape of `ex099_dx` as a literal Python tuple

**Next concept:** Subtract a computed maximum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-1.0, 0.0, 0.5, 1.5], dtype=DTYPE, requires_grad=True)
exp_x = x.exp()
total = exp_x.sum()
y = total.log()
dout = torch.tensor(1.25, dtype=DTYPE)
_capture("ex099", y, {"dtotal": total, "dexp_x": exp_x, "dx": x}, dout)
print("x", tuple(x.shape), "total", tuple(total.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 099: derive manually; do not use autograd in this cell.
# Define `ex099_dtotal` — the gradient entering the sum.
# Define `ex099_dexp_x` — the gradient entering exp.
# Define `ex099_dx` — the gradient with respect to `x`.
# Predict every visible tensor shape before computing values.
# Define `ex099_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex099_exp_x_shape` — the shape of `exp_x` as a literal Python tuple.
# Define `ex099_total_shape` — the shape of `total` as a literal Python tuple.
# Define `ex099_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex099_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex099_dtotal_shape` — the shape of `ex099_dtotal` as a literal Python tuple.
# Define `ex099_dexp_x_shape` — the shape of `ex099_dexp_x` as a literal Python tuple.
# Define `ex099_dx_shape` — the shape of `ex099_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex099_x_shape", "x")
_check_visible_tensor_shape("ex099_exp_x_shape", "exp_x")
_check_visible_tensor_shape("ex099_total_shape", "total")
_check_visible_tensor_shape("ex099_y_shape", "y")
_check_visible_tensor_shape("ex099_dout_shape", "dout")
_check_private_tensor_shape("ex099_dtotal_shape", "ex099", "dtotal")
_check_private_tensor_shape("ex099_dexp_x_shape", "ex099", "dexp_x")
_check_private_tensor_shape("ex099_dx_shape", "ex099", "dx")
_check_tensor("ex099_dtotal", "ex099", "dtotal")
_check_tensor("ex099_dexp_x", "ex099", "dexp_x")
_check_tensor("ex099_dx", "ex099", "dx")


### Exercise 100 — Subtract a computed maximum

**Purpose:** Learn that subtracting a maximum gives `x` a direct path and another path through the winning maximum position.

**Inputs:** `x` is `(4,)`, `maximum` is scalar, and `shifted` is `(4,)`; the maximum is unique.

**Forward operation:** `maximum = x.max()` and `shifted = x - maximum`.

**Derive/do:** Derive `dmaximum` and total `dx`.

**Ingredients:** The direct subtraction path reaches every `x`; the maximum path returns only to the winning index; add them.

**Required outputs:**

- `ex100_dmaximum`: the scalar gradient with respect to the computed maximum.
- `ex100_dx`: the total gradient from direct and maximum paths.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex100_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex100_maximum_shape`: predict the shape of `maximum` as a literal Python tuple
- `ex100_shifted_shape`: predict the shape of `shifted` as a literal Python tuple
- `ex100_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex100_dmaximum_shape`: predict the shape of `ex100_dmaximum` as a literal Python tuple
- `ex100_dx_shape`: predict the shape of `ex100_dx` as a literal Python tuple

**Next concept:** Two scalar objectives from one tensor.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.tensor([-1.0, 3.0, 2.0, 0.5], dtype=DTYPE, requires_grad=True)
maximum = x.max()
shifted = x - maximum
dout = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
_capture("ex100", shifted, {"dmaximum": maximum, "dx": x}, dout)
print("x", tuple(x.shape), "maximum", tuple(maximum.shape), "shifted", tuple(shifted.shape))


In [ ]:
# Exercise 100: derive manually; do not use autograd in this cell.
# Define `ex100_dmaximum` — the scalar gradient with respect to the computed maximum.
# Define `ex100_dx` — the total gradient from direct and maximum paths.
# Predict every visible tensor shape before computing values.
# Define `ex100_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex100_maximum_shape` — the shape of `maximum` as a literal Python tuple.
# Define `ex100_shifted_shape` — the shape of `shifted` as a literal Python tuple.
# Define `ex100_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex100_dmaximum_shape` — the shape of `ex100_dmaximum` as a literal Python tuple.
# Define `ex100_dx_shape` — the shape of `ex100_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex100_x_shape", "x")
_check_visible_tensor_shape("ex100_maximum_shape", "maximum")
_check_visible_tensor_shape("ex100_shifted_shape", "shifted")
_check_visible_tensor_shape("ex100_dout_shape", "dout")
_check_private_tensor_shape("ex100_dmaximum_shape", "ex100", "dmaximum")
_check_private_tensor_shape("ex100_dx_shape", "ex100", "dx")
_check_tensor("ex100_dmaximum", "ex100", "dmaximum")
_check_tensor("ex100_dx", "ex100", "dx")


### Exercise 101 — Two scalar objectives from one tensor

**Purpose:** Learn that one tensor used by two scalar objectives receives the sum of both objectives' gradients.

**Inputs:** `x` has shape `(2, 3)`; both branch outputs are scalar.

**Forward operation:** `left = (x**2).sum()`, `right = x.mean()`, and `y = left + right`.

**Derive/do:** Derive `dleft`, `dright`, and total `dx`.

**Ingredients:** Reverse the scalar addition; independently backpropagate each branch; add both `x` contributions.

**Required outputs:**

- `ex101_dleft`: the scalar gradient entering the square-sum branch.
- `ex101_dright`: the scalar gradient entering the mean branch.
- `ex101_dx`: the total gradient from both branches.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex101_x_shape`: predict the shape of `x` as a literal Python tuple
- `ex101_left_shape`: predict the shape of `left` as a literal Python tuple
- `ex101_right_shape`: predict the shape of `right` as a literal Python tuple
- `ex101_y_shape`: predict the shape of `y` as a literal Python tuple
- `ex101_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex101_dleft_shape`: predict the shape of `ex101_dleft` as a literal Python tuple
- `ex101_dright_shape`: predict the shape of `ex101_dright` as a literal Python tuple
- `ex101_dx_shape`: predict the shape of `ex101_dx` as a literal Python tuple

**Next concept:** One embedding row.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
x = torch.linspace(-2.0, 3.0, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
left = (x**2).sum()
right = x.mean()
y = left + right
dout = torch.tensor(-0.75, dtype=DTYPE)
_capture("ex101", y, {"dleft": left, "dright": right, "dx": x}, dout)
print("x", tuple(x.shape), "left", tuple(left.shape), "right", tuple(right.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 101: derive manually; do not use autograd in this cell.
# Define `ex101_dleft` — the scalar gradient entering the square-sum branch.
# Define `ex101_dright` — the scalar gradient entering the mean branch.
# Define `ex101_dx` — the total gradient from both branches.
# Predict every visible tensor shape before computing values.
# Define `ex101_x_shape` — the shape of `x` as a literal Python tuple.
# Define `ex101_left_shape` — the shape of `left` as a literal Python tuple.
# Define `ex101_right_shape` — the shape of `right` as a literal Python tuple.
# Define `ex101_y_shape` — the shape of `y` as a literal Python tuple.
# Define `ex101_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex101_dleft_shape` — the shape of `ex101_dleft` as a literal Python tuple.
# Define `ex101_dright_shape` — the shape of `ex101_dright` as a literal Python tuple.
# Define `ex101_dx_shape` — the shape of `ex101_dx` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex101_x_shape", "x")
_check_visible_tensor_shape("ex101_left_shape", "left")
_check_visible_tensor_shape("ex101_right_shape", "right")
_check_visible_tensor_shape("ex101_y_shape", "y")
_check_visible_tensor_shape("ex101_dout_shape", "dout")
_check_private_tensor_shape("ex101_dleft_shape", "ex101", "dleft")
_check_private_tensor_shape("ex101_dright_shape", "ex101", "dright")
_check_private_tensor_shape("ex101_dx_shape", "ex101", "dx")
_check_tensor("ex101_dleft", "ex101", "dleft")
_check_tensor("ex101_dright", "ex101", "dright")
_check_tensor("ex101_dx", "ex101", "dx")


## 7. Embedding lookups and indexed gradient accumulation

An embedding lookup uses integer IDs to select rows. Forward indexing replaces each ID with a vector. Backward sends each selected vector gradient back to its source row. Unselected rows receive zero. If an ID occurs more than once, all of its gradient contributions add into the same embedding-table row.

This is the backward counterpart of the lookup shape rule practiced in the indexing workbook.


### Exercise 102 — One embedding row

**Purpose:** Learn that selecting one embedding row sends a vector gradient to that row and zero to every unselected row.

**Inputs:** `table` is `(5, 3)` and `token_id` is one integer.

**Forward operation:** `emb = table[token_id]`.

**Derive/do:** Derive the full table gradient `dtable`.

**Ingredients:** Start with zeros shaped like the table and place the upstream vector at the selected row.

**Required outputs:**

- `ex102_dtable`: the `(5, 3)` table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex102_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex102_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex102_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex102_dtable_shape`: predict the shape of `ex102_dtable` as a literal Python tuple

**Next concept:** Several unique embedding rows.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(15, dtype=DTYPE).reshape(5, 3) / 10).requires_grad_()
token_id = 2
emb = table[token_id]
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex102", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "emb", tuple(emb.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 102: derive manually; do not use autograd in this cell.
# Define `ex102_dtable` — the `(5, 3)` table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex102_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex102_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex102_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex102_dtable_shape` — the shape of `ex102_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex102_table_shape", "table")
_check_visible_tensor_shape("ex102_emb_shape", "emb")
_check_visible_tensor_shape("ex102_dout_shape", "dout")
_check_private_tensor_shape("ex102_dtable_shape", "ex102", "dtable")
_check_tensor("ex102_dtable", "ex102", "dtable")


### Exercise 103 — Several unique embedding rows

**Purpose:** Learn to send several lookup gradients back to their corresponding embedding-table rows.

**Inputs:** `table` is `(6, 3)` and `ids = [4, 1, 3]` has shape `(3,)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `emb = table[ids]`.

**Derive/do:** Derive `dtable`.

**Ingredients:** Each row of `dout` belongs to the corresponding ID; all other table rows stay zero.

**Required outputs:**

- `ex103_dtable`: the `(6, 3)` table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex103_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex103_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex103_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex103_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex103_dtable_shape`: predict the shape of `ex103_dtable` as a literal Python tuple

**Next concept:** Repeated embedding IDs.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(18, dtype=DTYPE).reshape(6, 3) / 10).requires_grad_()
ids = torch.tensor([4, 1, 3], dtype=torch.long)
emb = table[ids]
dout = torch.tensor([[1.0, 0.5, -1.0], [-2.0, 3.0, 0.25], [0.5, -0.75, 2.0]], dtype=DTYPE)
_capture("ex103", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 103: derive manually; do not use autograd in this cell.
# Define `ex103_dtable` — the `(6, 3)` table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex103_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex103_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex103_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex103_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex103_dtable_shape` — the shape of `ex103_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex103_table_shape", "table")
_check_visible_tensor_shape("ex103_ids_shape", "ids")
_check_visible_tensor_shape("ex103_emb_shape", "emb")
_check_visible_tensor_shape("ex103_dout_shape", "dout")
_check_private_tensor_shape("ex103_dtable_shape", "ex103", "dtable")
_check_tensor("ex103_dtable", "ex103", "dtable")


### Exercise 104 — Repeated embedding IDs

**Purpose:** Learn that repeated token IDs make several gradients add into the same embedding-table row.

**Inputs:** `table` is `(6, 3)` and `ids = [2, 4, 2, 2]`.

**Forward operation:** `emb = table[ids]`.

**Derive/do:** Derive `dtable`, paying special attention to row 2.

**Ingredients:** Use an accumulating indexed write such as `index_add_`; plain replacement loses repeated contributions.

**Required outputs:**

- `ex104_dtable`: the table gradient with repeated-ID accumulation.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex104_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex104_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex104_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex104_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex104_dtable_shape`: predict the shape of `ex104_dtable` as a literal Python tuple

**Next concept:** A rank-2 grid of IDs.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(18, dtype=DTYPE).reshape(6, 3) / 10).requires_grad_()
ids = torch.tensor([2, 4, 2, 2], dtype=torch.long)
emb = table[ids]
dout = torch.tensor([[1.0, 0.5, -1.0], [-2.0, 3.0, 0.25], [0.5, -0.75, 2.0], [3.0, 1.0, -0.5]], dtype=DTYPE)
_capture("ex104", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 104: derive manually; do not use autograd in this cell.
# Define `ex104_dtable` — the table gradient with repeated-ID accumulation.
# Predict every visible tensor shape before computing values.
# Define `ex104_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex104_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex104_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex104_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex104_dtable_shape` — the shape of `ex104_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex104_table_shape", "table")
_check_visible_tensor_shape("ex104_ids_shape", "ids")
_check_visible_tensor_shape("ex104_emb_shape", "emb")
_check_visible_tensor_shape("ex104_dout_shape", "dout")
_check_private_tensor_shape("ex104_dtable_shape", "ex104", "dtable")
_check_tensor("ex104_dtable", "ex104", "dtable")


### Exercise 105 — A rank-2 grid of IDs

**Purpose:** Learn to map a two-dimensional grid of lookup gradients back into embedding-table rows.

**Inputs:** `table` is `(6, 2)`, `ids` is `(2, 3)`, and `emb` is `(2, 3, 2)`.

**Forward operation:** `emb = table[ids]`.

**Derive/do:** Derive `dtable` across all six batch-position lookups.

**Ingredients:** Flatten ID positions and embedding gradients conceptually, then accumulate by ID.

**Required outputs:**

- `ex105_dtable`: the `(6, 2)` table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex105_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex105_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex105_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex105_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex105_dtable_shape`: predict the shape of `ex105_dtable` as a literal Python tuple

**Next concept:** Weighted embedding outputs.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(2, 3, 2)
_capture("ex105", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 105: derive manually; do not use autograd in this cell.
# Define `ex105_dtable` — the `(6, 2)` table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex105_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex105_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex105_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex105_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex105_dtable_shape` — the shape of `ex105_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex105_table_shape", "table")
_check_visible_tensor_shape("ex105_ids_shape", "ids")
_check_visible_tensor_shape("ex105_emb_shape", "emb")
_check_visible_tensor_shape("ex105_dout_shape", "dout")
_check_private_tensor_shape("ex105_dtable_shape", "ex105", "dtable")
_check_tensor("ex105_dtable", "ex105", "dtable")


### Exercise 106 — Weighted embedding outputs

**Purpose:** Learn that every lookup position can send a different vector gradient back to its selected table row.

**Inputs:** `table` is `(7, 3)`, `ids` is `(2, 2)`, and `emb` is `(2, 2, 3)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `emb = table[ids]` with the supplied `dout`.

**Derive/do:** Derive `dtable`.

**Ingredients:** Every trailing length-3 vector is one lookup contribution; accumulate repeated IDs.

**Required outputs:**

- `ex106_dtable`: the table gradient from arbitrary lookup-vector gradients.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex106_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex106_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex106_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex106_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex106_dtable_shape`: predict the shape of `ex106_dtable` as a literal Python tuple

**Next concept:** Lookup then flatten.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(21, dtype=DTYPE).reshape(7, 3) / 10).requires_grad_()
ids = torch.tensor([[5, 2], [5, 1]], dtype=torch.long)
emb = table[ids]
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(2, 2, 3)
_capture("ex106", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 106: derive manually; do not use autograd in this cell.
# Define `ex106_dtable` — the table gradient from arbitrary lookup-vector gradients.
# Predict every visible tensor shape before computing values.
# Define `ex106_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex106_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex106_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex106_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex106_dtable_shape` — the shape of `ex106_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex106_table_shape", "table")
_check_visible_tensor_shape("ex106_ids_shape", "ids")
_check_visible_tensor_shape("ex106_emb_shape", "emb")
_check_visible_tensor_shape("ex106_dout_shape", "dout")
_check_private_tensor_shape("ex106_dtable_shape", "ex106", "dtable")
_check_tensor("ex106_dtable", "ex106", "dtable")


### Exercise 107 — Lookup then flatten

**Purpose:** Learn to restore an embedding tensor's shape before sending its gradients back through the lookup.

**Inputs:** `ids` is `(2, 3)`, `emb` is `(2, 3, 2)`, and `flat` is `(2, 6)`.

**Forward operation:** `emb = table[ids]` and `flat = emb.reshape(2, 6)`.

**Derive/do:** Derive `demb` and `dtable` from the supplied `(2, 6)` upstream tensor.

**Ingredients:** A reshape backward restores the old shape without changing element order; then accumulate by ID.

**Required outputs:**

- `ex107_demb`: the upstream gradient restored to `(2, 3, 2)`.
- `ex107_dtable`: the accumulated embedding-table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex107_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex107_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex107_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex107_flat_shape`: predict the shape of `flat` as a literal Python tuple
- `ex107_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex107_demb_shape`: predict the shape of `ex107_demb` as a literal Python tuple
- `ex107_dtable_shape`: predict the shape of `ex107_dtable` as a literal Python tuple

**Next concept:** Pool context embeddings by sum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
flat = emb.reshape(2, 6)
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(2, 6)
_capture("ex107", flat, {"demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "flat", tuple(flat.shape))


In [ ]:
# Exercise 107: derive manually; do not use autograd in this cell.
# Define `ex107_demb` — the upstream gradient restored to `(2, 3, 2)`.
# Define `ex107_dtable` — the accumulated embedding-table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex107_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex107_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex107_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex107_flat_shape` — the shape of `flat` as a literal Python tuple.
# Define `ex107_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex107_demb_shape` — the shape of `ex107_demb` as a literal Python tuple.
# Define `ex107_dtable_shape` — the shape of `ex107_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex107_table_shape", "table")
_check_visible_tensor_shape("ex107_ids_shape", "ids")
_check_visible_tensor_shape("ex107_emb_shape", "emb")
_check_visible_tensor_shape("ex107_flat_shape", "flat")
_check_visible_tensor_shape("ex107_dout_shape", "dout")
_check_private_tensor_shape("ex107_demb_shape", "ex107", "demb")
_check_private_tensor_shape("ex107_dtable_shape", "ex107", "dtable")
_check_tensor("ex107_demb", "ex107", "demb")
_check_tensor("ex107_dtable", "ex107", "dtable")


### Exercise 108 — Pool context embeddings by sum

**Purpose:** Learn to copy each pooled-sum gradient back to every context position before updating embedding rows.

**Inputs:** `emb` is `(2, 3, 2)` and `pooled = emb.sum(dim=1)` is `(2, 2)`.

**Forward operation:** Look up the IDs, then sum three context positions per example.

**Derive/do:** Derive `demb` and `dtable`.

**Ingredients:** Broadcast each example's pooled gradient across its three positions, then accumulate rows by token ID.

**Required outputs:**

- `ex108_demb`: the `(2, 3, 2)` gradient before lookup backward.
- `ex108_dtable`: the accumulated table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex108_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex108_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex108_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex108_pooled_shape`: predict the shape of `pooled` as a literal Python tuple
- `ex108_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex108_demb_shape`: predict the shape of `ex108_demb` as a literal Python tuple
- `ex108_dtable_shape`: predict the shape of `ex108_dtable` as a literal Python tuple

**Next concept:** Pool context embeddings by mean.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
pooled = emb.sum(dim=1)
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex108", pooled, {"demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "pooled", tuple(pooled.shape))


In [ ]:
# Exercise 108: derive manually; do not use autograd in this cell.
# Define `ex108_demb` — the `(2, 3, 2)` gradient before lookup backward.
# Define `ex108_dtable` — the accumulated table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex108_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex108_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex108_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex108_pooled_shape` — the shape of `pooled` as a literal Python tuple.
# Define `ex108_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex108_demb_shape` — the shape of `ex108_demb` as a literal Python tuple.
# Define `ex108_dtable_shape` — the shape of `ex108_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex108_table_shape", "table")
_check_visible_tensor_shape("ex108_ids_shape", "ids")
_check_visible_tensor_shape("ex108_emb_shape", "emb")
_check_visible_tensor_shape("ex108_pooled_shape", "pooled")
_check_visible_tensor_shape("ex108_dout_shape", "dout")
_check_private_tensor_shape("ex108_demb_shape", "ex108", "demb")
_check_private_tensor_shape("ex108_dtable_shape", "ex108", "dtable")
_check_tensor("ex108_demb", "ex108", "demb")
_check_tensor("ex108_dtable", "ex108", "dtable")


### Exercise 109 — Pool context embeddings by mean

**Purpose:** Learn that mean pooling divides each example's gradient among its context positions before lookup backward.

**Inputs:** `emb` is `(2, 3, 2)` and `pooled = emb.mean(dim=1)` is `(2, 2)`.

**Forward operation:** Average the three context embeddings per example.

**Derive/do:** Derive `demb` and `dtable`.

**Ingredients:** Each pooled gradient is divided across three positions before ID accumulation.

**Required outputs:**

- `ex109_demb`: the position-wise gradient after reversing the mean.
- `ex109_dtable`: the accumulated table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex109_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex109_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex109_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex109_pooled_shape`: predict the shape of `pooled` as a literal Python tuple
- `ex109_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex109_demb_shape`: predict the shape of `ex109_demb` as a literal Python tuple
- `ex109_dtable_shape`: predict the shape of `ex109_dtable` as a literal Python tuple

**Next concept:** Lookup, flatten, then linear.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
pooled = emb.mean(dim=1)
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex109", pooled, {"demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "pooled", tuple(pooled.shape))


In [ ]:
# Exercise 109: derive manually; do not use autograd in this cell.
# Define `ex109_demb` — the position-wise gradient after reversing the mean.
# Define `ex109_dtable` — the accumulated table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex109_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex109_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex109_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex109_pooled_shape` — the shape of `pooled` as a literal Python tuple.
# Define `ex109_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex109_demb_shape` — the shape of `ex109_demb` as a literal Python tuple.
# Define `ex109_dtable_shape` — the shape of `ex109_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex109_table_shape", "table")
_check_visible_tensor_shape("ex109_ids_shape", "ids")
_check_visible_tensor_shape("ex109_emb_shape", "emb")
_check_visible_tensor_shape("ex109_pooled_shape", "pooled")
_check_visible_tensor_shape("ex109_dout_shape", "dout")
_check_private_tensor_shape("ex109_demb_shape", "ex109", "demb")
_check_private_tensor_shape("ex109_dtable_shape", "ex109", "dtable")
_check_tensor("ex109_demb", "ex109", "demb")
_check_tensor("ex109_dtable", "ex109", "dtable")


### Exercise 110 — Lookup, flatten, then linear

**Purpose:** Learn to reverse a linear layer, restore the embedding shape, and then accumulate embedding-table gradients.

**Inputs:** `flat` is `(2, 6)`, `W` is `(6, 4)`, and `Y` is `(2, 4)`.

**Forward operation:** `emb = table[ids]`, `flat = emb.reshape(2, 6)`, and `Y = flat @ W`.

**Derive/do:** Derive `dW`, `dflat`, `demb`, and `dtable`.

**Ingredients:** Reverse matrix multiplication, restore the embedding shape, then accumulate repeated IDs.

**Required outputs:**

- `ex110_dW`: the projection-weight gradient.
- `ex110_dflat`: the flattened embedding gradient.
- `ex110_demb`: the restored embedding-grid gradient.
- `ex110_dtable`: the accumulated table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex110_table_shape`: predict the shape of `table` as a literal Python tuple
- `ex110_ids_shape`: predict the shape of `ids` as a literal Python tuple
- `ex110_emb_shape`: predict the shape of `emb` as a literal Python tuple
- `ex110_flat_shape`: predict the shape of `flat` as a literal Python tuple
- `ex110_W_shape`: predict the shape of `W` as a literal Python tuple
- `ex110_Y_shape`: predict the shape of `Y` as a literal Python tuple
- `ex110_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex110_dW_shape`: predict the shape of `ex110_dW` as a literal Python tuple
- `ex110_dflat_shape`: predict the shape of `ex110_dflat` as a literal Python tuple
- `ex110_demb_shape`: predict the shape of `ex110_demb` as a literal Python tuple
- `ex110_dtable_shape`: predict the shape of `ex110_dtable` as a literal Python tuple

**Next concept:** Paired target selection backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
flat = emb.reshape(2, 6)
W = torch.linspace(-1.0, 1.3, steps=24, dtype=DTYPE).reshape(6, 4).requires_grad_()
Y = flat @ W
dout = torch.linspace(1.0, -0.7, steps=8, dtype=DTYPE).reshape(2, 4)
_capture("ex110", Y, {"dW": W, "dflat": flat, "demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "flat", tuple(flat.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 110: derive manually; do not use autograd in this cell.
# Define `ex110_dW` — the projection-weight gradient.
# Define `ex110_dflat` — the flattened embedding gradient.
# Define `ex110_demb` — the restored embedding-grid gradient.
# Define `ex110_dtable` — the accumulated table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex110_table_shape` — the shape of `table` as a literal Python tuple.
# Define `ex110_ids_shape` — the shape of `ids` as a literal Python tuple.
# Define `ex110_emb_shape` — the shape of `emb` as a literal Python tuple.
# Define `ex110_flat_shape` — the shape of `flat` as a literal Python tuple.
# Define `ex110_W_shape` — the shape of `W` as a literal Python tuple.
# Define `ex110_Y_shape` — the shape of `Y` as a literal Python tuple.
# Define `ex110_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex110_dW_shape` — the shape of `ex110_dW` as a literal Python tuple.
# Define `ex110_dflat_shape` — the shape of `ex110_dflat` as a literal Python tuple.
# Define `ex110_demb_shape` — the shape of `ex110_demb` as a literal Python tuple.
# Define `ex110_dtable_shape` — the shape of `ex110_dtable` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex110_table_shape", "table")
_check_visible_tensor_shape("ex110_ids_shape", "ids")
_check_visible_tensor_shape("ex110_emb_shape", "emb")
_check_visible_tensor_shape("ex110_flat_shape", "flat")
_check_visible_tensor_shape("ex110_W_shape", "W")
_check_visible_tensor_shape("ex110_Y_shape", "Y")
_check_visible_tensor_shape("ex110_dout_shape", "dout")
_check_private_tensor_shape("ex110_dW_shape", "ex110", "dW")
_check_private_tensor_shape("ex110_dflat_shape", "ex110", "dflat")
_check_private_tensor_shape("ex110_demb_shape", "ex110", "demb")
_check_private_tensor_shape("ex110_dtable_shape", "ex110", "dtable")
_check_tensor("ex110_dW", "ex110", "dW")
_check_tensor("ex110_dflat", "ex110", "dflat")
_check_tensor("ex110_demb", "ex110", "demb")
_check_tensor("ex110_dtable", "ex110", "dtable")


### Exercise 111 — Paired target selection backward

**Purpose:** Learn that selecting one target candidate per example creates one nonzero direct gradient in each row.

**Inputs:** `log_values` is `(3, 5)` and `targets` is `(3,)`, one candidate index per example.

**Forward operation:** `selected = log_values[range(3), targets]`.

**Derive/do:** Derive the full `(3, 5)` gradient `dlog_values` from a length-3 upstream gradient.

**Ingredients:** Start with zeros and place one upstream entry at each paired example-target coordinate.

**Required outputs:**

- `ex111_dlog_values`: the sparse `(3, 5)` gradient at paired target positions.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex111_log_values_shape`: predict the shape of `log_values` as a literal Python tuple
- `ex111_targets_shape`: predict the shape of `targets` as a literal Python tuple
- `ex111_selected_shape`: predict the shape of `selected` as a literal Python tuple
- `ex111_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex111_dlog_values_shape`: predict the shape of `ex111_dlog_values` as a literal Python tuple

**Next concept:** Stable row maxima.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
log_values = torch.linspace(-3.0, -0.1, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
targets = torch.tensor([2, 0, 4], dtype=torch.long)
selected = log_values[range(3), targets]
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex111", selected, {"dlog_values": log_values}, dout)
print("log_values", tuple(log_values.shape), "targets", tuple(targets.shape), "selected", tuple(selected.shape))


In [ ]:
# Exercise 111: derive manually; do not use autograd in this cell.
# Define `ex111_dlog_values` — the sparse `(3, 5)` gradient at paired target positions.
# Predict every visible tensor shape before computing values.
# Define `ex111_log_values_shape` — the shape of `log_values` as a literal Python tuple.
# Define `ex111_targets_shape` — the shape of `targets` as a literal Python tuple.
# Define `ex111_selected_shape` — the shape of `selected` as a literal Python tuple.
# Define `ex111_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex111_dlog_values_shape` — the shape of `ex111_dlog_values` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex111_log_values_shape", "log_values")
_check_visible_tensor_shape("ex111_targets_shape", "targets")
_check_visible_tensor_shape("ex111_selected_shape", "selected")
_check_visible_tensor_shape("ex111_dout_shape", "dout")
_check_private_tensor_shape("ex111_dlog_values_shape", "ex111", "dlog_values")
_check_tensor("ex111_dlog_values", "ex111", "dlog_values")


## 8. Stable softmax and mean negative log-likelihood

For each training example `i`, the model produces one logit for every candidate `j`. Stable softmax is

$$
p_{i,j}
=
\frac{\exp(z_{i,j}-m_i)}{\sum_k \exp(z_{i,k}-m_i)}
$$

Here, `z` is `logits`, `m_i` is the maximum logit in example row `i`, `k` ranges over all candidates, and `p` is `probs`. Subtracting the row maximum changes numerical scale but not probabilities.

For one example, negative log-likelihood is

$$
\ell_i
=
-\log p_{i,y_i}
$$

Here, `y_i` is the expected target candidate. For a batch of `B` examples, the notebook uses the average

$$
L
=
\frac{1}{B}\sum_{i=1}^{B}\ell_i
$$

Only the indexed target log-probability contributes directly to each example's loss. Other candidates still influence that probability through the shared softmax denominator.

The lecture's variable name `counts` does **not** mean observed character occurrences here. It is simply `norm_logits.exp()`: positive, unnormalized candidate weights. Likewise, `counts_sum_inv` is the reciprocal of one row total.


### Exercise 112 — Stable row maxima

**Purpose:** Learn to find one maximum per example so exponentiation can use safer numbers.

**Inputs:** `logits` has shape `(3, 5)`: three examples and five candidates.

**Forward operation:** Compute `logits.max(dim=1, keepdim=True).values` manually in your answer cell.

**Derive/do:** Find the maximum of each row while preserving the candidate axis as size one.

**Ingredients:** `max(dim=1, keepdim=True).values`.

**Required outputs:**

- `ex112_out`: the `(3, 1)` row maxima.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex112_logits_shape`: predict the shape of `logits` as a literal Python tuple
- `ex112_out_shape`: predict the shape of `ex112_out` as a literal Python tuple

**Next concept:** Subtract an independent row maximum.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
logits = torch.tensor([[1.0, 3.0, -2.0, 0.5, 2.0], [4.0, -1.0, 2.0, 0.0, 1.0], [-2.0, 0.5, 3.5, 1.0, 2.0]], dtype=DTYPE)
_store_forward("ex112", logits.max(dim=1, keepdim=True).values)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 112: derive manually; do not use autograd in this cell.
# Define `ex112_out` — the `(3, 1)` row maxima.
# Predict every visible tensor shape before computing values.
# Define `ex112_logits_shape` — the shape of `logits` as a literal Python tuple.
# Define `ex112_out_shape` — the shape of `ex112_out` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex112_logits_shape", "logits")
_check_private_tensor_shape("ex112_out_shape", "ex112", "out")
_check_tensor("ex112_out", "ex112", "out")


### Exercise 113 — Subtract an independent row maximum

**Purpose:** Learn how subtracting one row value from every candidate sends a summed gradient back to that row value.

**Inputs:** `logits` is `(3, 5)`, independent `row_max` is `(3, 1)`, and `dout` is `(3, 5)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `norm = logits - row_max`.

**Derive/do:** Derive `dlogits` and `drow_max`.

**Ingredients:** The maximum tensor is reused across five candidates in its row.

**Required outputs:**

- `ex113_dlogits`: the direct gradient with respect to `logits`.
- `ex113_drow_max`: the `(3, 1)` unbroadcast subtraction gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex113_logits_shape`: predict the shape of `logits` as a literal Python tuple
- `ex113_row_max_shape`: predict the shape of `row_max` as a literal Python tuple
- `ex113_norm_shape`: predict the shape of `norm` as a literal Python tuple
- `ex113_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex113_dlogits_shape`: predict the shape of `ex113_dlogits` as a literal Python tuple
- `ex113_drow_max_shape`: predict the shape of `ex113_drow_max` as a literal Python tuple

**Next concept:** Exponentiate normalized logits.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
logits = torch.linspace(-2.0, 2.2, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
row_max = torch.tensor([[2.2], [1.1], [3.4]], dtype=DTYPE, requires_grad=True)
norm = logits - row_max
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex113", norm, {"dlogits": logits, "drow_max": row_max}, dout)
print("logits", tuple(logits.shape), "row_max", tuple(row_max.shape), "norm", tuple(norm.shape))


In [ ]:
# Exercise 113: derive manually; do not use autograd in this cell.
# Define `ex113_dlogits` — the direct gradient with respect to `logits`.
# Define `ex113_drow_max` — the `(3, 1)` unbroadcast subtraction gradient.
# Predict every visible tensor shape before computing values.
# Define `ex113_logits_shape` — the shape of `logits` as a literal Python tuple.
# Define `ex113_row_max_shape` — the shape of `row_max` as a literal Python tuple.
# Define `ex113_norm_shape` — the shape of `norm` as a literal Python tuple.
# Define `ex113_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex113_dlogits_shape` — the shape of `ex113_dlogits` as a literal Python tuple.
# Define `ex113_drow_max_shape` — the shape of `ex113_drow_max` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex113_logits_shape", "logits")
_check_visible_tensor_shape("ex113_row_max_shape", "row_max")
_check_visible_tensor_shape("ex113_norm_shape", "norm")
_check_visible_tensor_shape("ex113_dout_shape", "dout")
_check_private_tensor_shape("ex113_dlogits_shape", "ex113", "dlogits")
_check_private_tensor_shape("ex113_drow_max_shape", "ex113", "drow_max")
_check_tensor("ex113_dlogits", "ex113", "dlogits")
_check_tensor("ex113_drow_max", "ex113", "drow_max")


### Exercise 114 — Exponentiate normalized logits

**Purpose:** Learn to pass gradients backward through the exponentiation that creates positive candidate weights.

**Inputs:** `norm_logits`, `counts`, and `dout` have shape `(3, 5)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `counts = norm_logits.exp()`.

**Derive/do:** Derive `dnorm_logits`.

**Ingredients:** Use the stored `counts` forward value as exp's local derivative.

**Required outputs:**

- `ex114_dnorm_logits`: the gradient with respect to normalized logits.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex114_norm_logits_shape`: predict the shape of `norm_logits` as a literal Python tuple
- `ex114_counts_shape`: predict the shape of `counts` as a literal Python tuple
- `ex114_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex114_dnorm_logits_shape`: predict the shape of `ex114_dnorm_logits` as a literal Python tuple

**Next concept:** Sum candidate weights per row.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
norm_logits = torch.linspace(-4.0, 0.0, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
counts = norm_logits.exp()
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex114", counts, {"dnorm_logits": norm_logits}, dout)
print("norm_logits", tuple(norm_logits.shape), "counts", tuple(counts.shape))


In [ ]:
# Exercise 114: derive manually; do not use autograd in this cell.
# Define `ex114_dnorm_logits` — the gradient with respect to normalized logits.
# Predict every visible tensor shape before computing values.
# Define `ex114_norm_logits_shape` — the shape of `norm_logits` as a literal Python tuple.
# Define `ex114_counts_shape` — the shape of `counts` as a literal Python tuple.
# Define `ex114_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex114_dnorm_logits_shape` — the shape of `ex114_dnorm_logits` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex114_norm_logits_shape", "norm_logits")
_check_visible_tensor_shape("ex114_counts_shape", "counts")
_check_visible_tensor_shape("ex114_dout_shape", "dout")
_check_private_tensor_shape("ex114_dnorm_logits_shape", "ex114", "dnorm_logits")
_check_tensor("ex114_dnorm_logits", "ex114", "dnorm_logits")


### Exercise 115 — Sum candidate weights per row

**Purpose:** Learn that one row-total gradient is copied back to every candidate weight in that row.

**Inputs:** `counts` is `(3, 5)` and `counts_sum` is `(3, 1)`.

**Forward operation:** `counts_sum = counts.sum(dim=1, keepdim=True)`.

**Derive/do:** Derive `dcounts` from `(3, 1)` upstream gradients.

**Ingredients:** Each row total depends once on all five candidate weights in that row.

**Required outputs:**

- `ex115_dcounts`: the `(3, 5)` gradient after reversing the row sums.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex115_counts_shape`: predict the shape of `counts` as a literal Python tuple
- `ex115_counts_sum_shape`: predict the shape of `counts_sum` as a literal Python tuple
- `ex115_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex115_dcounts_shape`: predict the shape of `ex115_dcounts` as a literal Python tuple

**Next concept:** Reciprocal row totals.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
counts = torch.linspace(0.1, 1.5, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
counts_sum = counts.sum(dim=1, keepdim=True)
dout = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE)
_capture("ex115", counts_sum, {"dcounts": counts}, dout)
print("counts", tuple(counts.shape), "counts_sum", tuple(counts_sum.shape))


In [ ]:
# Exercise 115: derive manually; do not use autograd in this cell.
# Define `ex115_dcounts` — the `(3, 5)` gradient after reversing the row sums.
# Predict every visible tensor shape before computing values.
# Define `ex115_counts_shape` — the shape of `counts` as a literal Python tuple.
# Define `ex115_counts_sum_shape` — the shape of `counts_sum` as a literal Python tuple.
# Define `ex115_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex115_dcounts_shape` — the shape of `ex115_dcounts` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex115_counts_shape", "counts")
_check_visible_tensor_shape("ex115_counts_sum_shape", "counts_sum")
_check_visible_tensor_shape("ex115_dout_shape", "dout")
_check_private_tensor_shape("ex115_dcounts_shape", "ex115", "dcounts")
_check_tensor("ex115_dcounts", "ex115", "dcounts")


### Exercise 116 — Reciprocal row totals

**Purpose:** Learn to pass gradients backward through the reciprocal of each row total.

**Inputs:** Positive `counts_sum` and `counts_sum_inv` have shape `(3, 1)`.

**Forward operation:** `counts_sum_inv = counts_sum**-1`.

**Derive/do:** Derive `dcounts_sum`.

**Ingredients:** Use the fixed-power rule with exponent `-1`.

**Required outputs:**

- `ex116_dcounts_sum`: the `(3, 1)` gradient with respect to row totals.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex116_counts_sum_shape`: predict the shape of `counts_sum` as a literal Python tuple
- `ex116_counts_sum_inv_shape`: predict the shape of `counts_sum_inv` as a literal Python tuple
- `ex116_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex116_dcounts_sum_shape`: predict the shape of `ex116_dcounts_sum` as a literal Python tuple

**Next concept:** Normalize weights by reciprocal totals.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
counts_sum = torch.tensor([[2.0], [4.0], [0.5]], dtype=DTYPE, requires_grad=True)
counts_sum_inv = counts_sum**-1
dout = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE)
_capture("ex116", counts_sum_inv, {"dcounts_sum": counts_sum}, dout)
print("counts_sum", tuple(counts_sum.shape), "counts_sum_inv", tuple(counts_sum_inv.shape))


In [ ]:
# Exercise 116: derive manually; do not use autograd in this cell.
# Define `ex116_dcounts_sum` — the `(3, 1)` gradient with respect to row totals.
# Predict every visible tensor shape before computing values.
# Define `ex116_counts_sum_shape` — the shape of `counts_sum` as a literal Python tuple.
# Define `ex116_counts_sum_inv_shape` — the shape of `counts_sum_inv` as a literal Python tuple.
# Define `ex116_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex116_dcounts_sum_shape` — the shape of `ex116_dcounts_sum` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex116_counts_sum_shape", "counts_sum")
_check_visible_tensor_shape("ex116_counts_sum_inv_shape", "counts_sum_inv")
_check_visible_tensor_shape("ex116_dout_shape", "dout")
_check_private_tensor_shape("ex116_dcounts_sum_shape", "ex116", "dcounts_sum")
_check_tensor("ex116_dcounts_sum", "ex116", "dcounts_sum")


### Exercise 117 — Normalize weights by reciprocal totals

**Purpose:** Learn how probability normalization sends gradients to both candidate weights and their shared row denominator.

**Inputs:** `counts` is `(3, 5)`, `counts_sum_inv` is `(3, 1)`, and `probs` is `(3, 5)`.

**Forward operation:** `probs = counts * counts_sum_inv`.

**Derive/do:** Derive the direct `dcounts` and accumulated `dcounts_sum_inv`.

**Ingredients:** Product backward plus summation over five candidates for the broadcast inverse total.

**Required outputs:**

- `ex117_dcounts`: the direct numerator gradient.
- `ex117_dcounts_sum_inv`: the `(3, 1)` gradient accumulated across candidates.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex117_counts_shape`: predict the shape of `counts` as a literal Python tuple
- `ex117_counts_sum_inv_shape`: predict the shape of `counts_sum_inv` as a literal Python tuple
- `ex117_probs_shape`: predict the shape of `probs` as a literal Python tuple
- `ex117_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex117_dcounts_shape`: predict the shape of `ex117_dcounts` as a literal Python tuple
- `ex117_dcounts_sum_inv_shape`: predict the shape of `ex117_dcounts_sum_inv` as a literal Python tuple

**Next concept:** Log probabilities.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
counts = torch.linspace(0.1, 1.5, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
counts_sum_inv = torch.tensor([[0.25], [0.5], [0.125]], dtype=DTYPE, requires_grad=True)
probs = counts * counts_sum_inv
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex117", probs, {"dcounts": counts, "dcounts_sum_inv": counts_sum_inv}, dout)
print("counts", tuple(counts.shape), "counts_sum_inv", tuple(counts_sum_inv.shape), "probs", tuple(probs.shape))


In [ ]:
# Exercise 117: derive manually; do not use autograd in this cell.
# Define `ex117_dcounts` — the direct numerator gradient.
# Define `ex117_dcounts_sum_inv` — the `(3, 1)` gradient accumulated across candidates.
# Predict every visible tensor shape before computing values.
# Define `ex117_counts_shape` — the shape of `counts` as a literal Python tuple.
# Define `ex117_counts_sum_inv_shape` — the shape of `counts_sum_inv` as a literal Python tuple.
# Define `ex117_probs_shape` — the shape of `probs` as a literal Python tuple.
# Define `ex117_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex117_dcounts_shape` — the shape of `ex117_dcounts` as a literal Python tuple.
# Define `ex117_dcounts_sum_inv_shape` — the shape of `ex117_dcounts_sum_inv` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex117_counts_shape", "counts")
_check_visible_tensor_shape("ex117_counts_sum_inv_shape", "counts_sum_inv")
_check_visible_tensor_shape("ex117_probs_shape", "probs")
_check_visible_tensor_shape("ex117_dout_shape", "dout")
_check_private_tensor_shape("ex117_dcounts_shape", "ex117", "dcounts")
_check_private_tensor_shape("ex117_dcounts_sum_inv_shape", "ex117", "dcounts_sum_inv")
_check_tensor("ex117_dcounts", "ex117", "dcounts")
_check_tensor("ex117_dcounts_sum_inv", "ex117", "dcounts_sum_inv")


### Exercise 118 — Log probabilities

**Purpose:** Learn to pass gradients from log-probabilities back into probabilities.

**Inputs:** Positive `probs`, `logprobs`, and `dout` have shape `(3, 5)`.

**Upstream gradient:** `dout` represents $\partial L / \partial y$ for this exercise—the gradient of the final scalar loss with respect to this operation's output `y`.

**Forward operation:** `logprobs = probs.log()`.

**Derive/do:** Derive `dprobs`.

**Ingredients:** The reciprocal local slope of log and the supplied upstream tensor.

**Required outputs:**

- `ex118_dprobs`: the gradient with respect to probabilities.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex118_probs_shape`: predict the shape of `probs` as a literal Python tuple
- `ex118_logprobs_shape`: predict the shape of `logprobs` as a literal Python tuple
- `ex118_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex118_dprobs_shape`: predict the shape of `ex118_dprobs` as a literal Python tuple

**Next concept:** Select one target from one example.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
probs = torch.tensor([[0.1, 0.2, 0.3, 0.15, 0.25], [0.4, 0.1, 0.2, 0.2, 0.1], [0.05, 0.15, 0.5, 0.2, 0.1]], dtype=DTYPE, requires_grad=True)
logprobs = probs.log()
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex118", logprobs, {"dprobs": probs}, dout)
print("probs", tuple(probs.shape), "logprobs", tuple(logprobs.shape))


In [ ]:
# Exercise 118: derive manually; do not use autograd in this cell.
# Define `ex118_dprobs` — the gradient with respect to probabilities.
# Predict every visible tensor shape before computing values.
# Define `ex118_probs_shape` — the shape of `probs` as a literal Python tuple.
# Define `ex118_logprobs_shape` — the shape of `logprobs` as a literal Python tuple.
# Define `ex118_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex118_dprobs_shape` — the shape of `ex118_dprobs` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex118_probs_shape", "probs")
_check_visible_tensor_shape("ex118_logprobs_shape", "logprobs")
_check_visible_tensor_shape("ex118_dout_shape", "dout")
_check_private_tensor_shape("ex118_dprobs_shape", "ex118", "dprobs")
_check_tensor("ex118_dprobs", "ex118", "dprobs")


### Exercise 119 — Select one target from one example

**Purpose:** Learn that selecting one target from one example gives a direct gradient only to that target position.

**Inputs:** `logprobs` has shape `(5,)` and the expected target is candidate 3.

**Forward operation:** `selected = logprobs[target]`.

**Derive/do:** Derive the full length-5 `dlogprobs`.

**Ingredients:** Only the indexed candidate has a direct path to the selected scalar.

**Required outputs:**

- `ex119_dlogprobs`: the sparse length-5 gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex119_logprobs_shape`: predict the shape of `logprobs` as a literal Python tuple
- `ex119_selected_shape`: predict the shape of `selected` as a literal Python tuple
- `ex119_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex119_dlogprobs_shape`: predict the shape of `ex119_dlogprobs` as a literal Python tuple

**Next concept:** Select one target per batch row.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
logprobs = torch.tensor([-2.0, -1.5, -3.0, -0.2, -2.5], dtype=DTYPE, requires_grad=True)
target = 3
selected = logprobs[target]
dout = torch.tensor(-1.25, dtype=DTYPE)
_capture("ex119", selected, {"dlogprobs": logprobs}, dout)
print("logprobs", tuple(logprobs.shape), "selected", tuple(selected.shape))


In [ ]:
# Exercise 119: derive manually; do not use autograd in this cell.
# Define `ex119_dlogprobs` — the sparse length-5 gradient.
# Predict every visible tensor shape before computing values.
# Define `ex119_logprobs_shape` — the shape of `logprobs` as a literal Python tuple.
# Define `ex119_selected_shape` — the shape of `selected` as a literal Python tuple.
# Define `ex119_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex119_dlogprobs_shape` — the shape of `ex119_dlogprobs` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex119_logprobs_shape", "logprobs")
_check_visible_tensor_shape("ex119_selected_shape", "selected")
_check_visible_tensor_shape("ex119_dout_shape", "dout")
_check_private_tensor_shape("ex119_dlogprobs_shape", "ex119", "dlogprobs")
_check_tensor("ex119_dlogprobs", "ex119", "dlogprobs")


### Exercise 120 — Select one target per batch row

**Purpose:** Learn to place one direct target gradient in every training-example row.

**Inputs:** `logprobs` is `(4, 5)` and `targets` is `(4,)`.

**Forward operation:** `selected = logprobs[range(4), targets]`.

**Derive/do:** Derive the full `(4, 5)` `dlogprobs` from a length-4 upstream tensor.

**Ingredients:** Place one upstream value at the target coordinate in each training-example row.

**Required outputs:**

- `ex120_dlogprobs`: the sparse `(4, 5)` target gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex120_logprobs_shape`: predict the shape of `logprobs` as a literal Python tuple
- `ex120_targets_shape`: predict the shape of `targets` as a literal Python tuple
- `ex120_selected_shape`: predict the shape of `selected` as a literal Python tuple
- `ex120_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex120_dlogprobs_shape`: predict the shape of `ex120_dlogprobs` as a literal Python tuple

**Next concept:** Mean NLL seed.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
logprobs = torch.linspace(-3.0, -0.1, steps=20, dtype=DTYPE).reshape(4, 5).requires_grad_()
targets = torch.tensor([3, 0, 4, 2], dtype=torch.long)
selected = logprobs[range(4), targets]
dout = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
_capture("ex120", selected, {"dlogprobs": logprobs}, dout)
print("logprobs", tuple(logprobs.shape), "targets", tuple(targets.shape), "selected", tuple(selected.shape))


In [ ]:
# Exercise 120: derive manually; do not use autograd in this cell.
# Define `ex120_dlogprobs` — the sparse `(4, 5)` target gradient.
# Predict every visible tensor shape before computing values.
# Define `ex120_logprobs_shape` — the shape of `logprobs` as a literal Python tuple.
# Define `ex120_targets_shape` — the shape of `targets` as a literal Python tuple.
# Define `ex120_selected_shape` — the shape of `selected` as a literal Python tuple.
# Define `ex120_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex120_dlogprobs_shape` — the shape of `ex120_dlogprobs` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex120_logprobs_shape", "logprobs")
_check_visible_tensor_shape("ex120_targets_shape", "targets")
_check_visible_tensor_shape("ex120_selected_shape", "selected")
_check_visible_tensor_shape("ex120_dout_shape", "dout")
_check_private_tensor_shape("ex120_dlogprobs_shape", "ex120", "dlogprobs")
_check_tensor("ex120_dlogprobs", "ex120", "dlogprobs")


### Exercise 121 — Mean NLL seed

**Purpose:** Learn why mean negative log-likelihood starts with `-1 / batch_size` at each selected target and zero elsewhere.

**Inputs:** Four training examples, five candidates each, and one expected target per example.

**Forward operation:** The supplied fixture computes `sm_loss = -sm_logprobs[range(sm_n), sm_targets].mean()`.

**Derive/do:** Derive `sm_dlogprobs` manually.

**Ingredients:** Start with zeros shaped like `sm_logprobs`; combine the minus sign and four-example mean only at paired target coordinates.

**Required outputs:**

- `sm_dlogprobs`: the `(4, 5)` gradient with respect to `sm_logprobs`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex121_sm_logits_shape`: predict the shape of `sm_logits` as a literal Python tuple
- `ex121_sm_targets_shape`: predict the shape of `sm_targets` as a literal Python tuple
- `ex121_sm_logit_maxes_shape`: predict the shape of `sm_logit_maxes` as a literal Python tuple
- `ex121_sm_norm_logits_shape`: predict the shape of `sm_norm_logits` as a literal Python tuple
- `ex121_sm_counts_shape`: predict the shape of `sm_counts` as a literal Python tuple
- `ex121_sm_counts_sum_shape`: predict the shape of `sm_counts_sum` as a literal Python tuple
- `ex121_sm_counts_sum_inv_shape`: predict the shape of `sm_counts_sum_inv` as a literal Python tuple
- `ex121_sm_probs_shape`: predict the shape of `sm_probs` as a literal Python tuple
- `ex121_sm_logprobs_shape`: predict the shape of `sm_logprobs` as a literal Python tuple
- `ex121_sm_loss_shape`: predict the shape of `sm_loss` as a literal Python tuple
- `ex121_sm_dlogprobs_shape`: predict the shape of `sm_dlogprobs` as a literal Python tuple

**Next concept:** Backward through log.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
# Supplied mini-batch: four examples and five candidate classes per example.
sm_logits = torch.tensor([
    [1.2, -0.7, 2.1, 0.3, -1.4],
    [-0.2, 1.7, 0.4, 2.5, -0.9],
    [2.2, 0.1, -1.1, 0.8, 1.5],
    [0.6, 2.3, -0.4, 1.1, -1.7],
], dtype=DTYPE, requires_grad=True)
sm_targets = torch.tensor([2, 3, 0, 1], dtype=torch.long)
sm_n = sm_logits.shape[0]

# Expanded stable softmax and mean NLL; this scores all four supplied examples.
sm_logit_maxes = sm_logits.max(dim=1, keepdim=True).values
sm_norm_logits = sm_logits - sm_logit_maxes
sm_counts = sm_norm_logits.exp()
sm_counts_sum = sm_counts.sum(dim=1, keepdim=True)
sm_counts_sum_inv = sm_counts_sum**-1
sm_probs = sm_counts * sm_counts_sum_inv
sm_logprobs = sm_probs.log()
sm_loss = -sm_logprobs[range(sm_n), sm_targets].mean()

_capture(
    "softmax_capstone",
    sm_loss,
    {
        "dlogprobs": sm_logprobs,
        "dprobs": sm_probs,
        "dcounts_sum_inv": sm_counts_sum_inv,
        "dcounts_sum": sm_counts_sum,
        "dcounts": sm_counts,
        "dnorm_logits": sm_norm_logits,
        "dlogit_maxes": sm_logit_maxes,
        "dlogits": sm_logits,
    },
)
print("Softmax capstone ready:", tuple(sm_logits.shape), "-> loss", tuple(sm_loss.shape))


In [ ]:
# Exercise 121: derive manually; do not use autograd in this cell.
# Define `sm_dlogprobs` — the `(4, 5)` gradient with respect to `sm_logprobs`.
# Predict every visible tensor shape before computing values.
# Define `ex121_sm_logits_shape` — the shape of `sm_logits` as a literal Python tuple.
# Define `ex121_sm_targets_shape` — the shape of `sm_targets` as a literal Python tuple.
# Define `ex121_sm_logit_maxes_shape` — the shape of `sm_logit_maxes` as a literal Python tuple.
# Define `ex121_sm_norm_logits_shape` — the shape of `sm_norm_logits` as a literal Python tuple.
# Define `ex121_sm_counts_shape` — the shape of `sm_counts` as a literal Python tuple.
# Define `ex121_sm_counts_sum_shape` — the shape of `sm_counts_sum` as a literal Python tuple.
# Define `ex121_sm_counts_sum_inv_shape` — the shape of `sm_counts_sum_inv` as a literal Python tuple.
# Define `ex121_sm_probs_shape` — the shape of `sm_probs` as a literal Python tuple.
# Define `ex121_sm_logprobs_shape` — the shape of `sm_logprobs` as a literal Python tuple.
# Define `ex121_sm_loss_shape` — the shape of `sm_loss` as a literal Python tuple.
# Define `ex121_sm_dlogprobs_shape` — the shape of `sm_dlogprobs` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex121_sm_logits_shape", "sm_logits")
_check_visible_tensor_shape("ex121_sm_targets_shape", "sm_targets")
_check_visible_tensor_shape("ex121_sm_logit_maxes_shape", "sm_logit_maxes")
_check_visible_tensor_shape("ex121_sm_norm_logits_shape", "sm_norm_logits")
_check_visible_tensor_shape("ex121_sm_counts_shape", "sm_counts")
_check_visible_tensor_shape("ex121_sm_counts_sum_shape", "sm_counts_sum")
_check_visible_tensor_shape("ex121_sm_counts_sum_inv_shape", "sm_counts_sum_inv")
_check_visible_tensor_shape("ex121_sm_probs_shape", "sm_probs")
_check_visible_tensor_shape("ex121_sm_logprobs_shape", "sm_logprobs")
_check_visible_tensor_shape("ex121_sm_loss_shape", "sm_loss")
_check_private_tensor_shape("ex121_sm_dlogprobs_shape", "softmax_capstone", "dlogprobs")
_check_tensor("sm_dlogprobs", "softmax_capstone", "dlogprobs")


### Exercise 122 — Backward through log

**Purpose:** Learn to pass the sparse target gradient backward through the logarithm.

**Inputs:** `sm_probs` and `sm_dlogprobs` are `(4, 5)`.

**Forward operation:** `sm_logprobs = sm_probs.log()`.

**Derive/do:** Using your previous `sm_dlogprobs`, derive `sm_dprobs`.

**Ingredients:** Natural-log local derivative; this operation is elementwise.

**Required outputs:**

- `sm_dprobs`: the `(4, 5)` gradient with respect to `sm_probs`.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex122_sm_logprobs_shape`: predict the shape of `sm_logprobs` as a literal Python tuple
- `ex122_sm_probs_shape`: predict the shape of `sm_probs` as a literal Python tuple
- `ex122_sm_dprobs_shape`: predict the shape of `sm_dprobs` as a literal Python tuple

**Next concept:** Backward into the shared inverse row totals.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "softmax_capstone" in _REFS

# Forward relation under study:
# sm_logprobs = sm_probs.log()
# Visible forward tensors and shapes:
# - `sm_probs` is already defined by the visible forward graph.
# - `sm_logprobs` is already defined by the visible forward graph.
# Upstream gradient for this stage: sm_dlogprobs from Exercise 121.
print("sm_probs", tuple(sm_probs.shape), "sm_logprobs", tuple(sm_logprobs.shape))


In [ ]:
# Exercise 122: derive manually; do not use autograd in this cell.
# Define `sm_dprobs` — the `(4, 5)` gradient with respect to `sm_probs`.
# Predict every visible tensor shape before computing values.
# Define `ex122_sm_logprobs_shape` — the shape of `sm_logprobs` as a literal Python tuple.
# Define `ex122_sm_probs_shape` — the shape of `sm_probs` as a literal Python tuple.
# Define `ex122_sm_dprobs_shape` — the shape of `sm_dprobs` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex122_sm_logprobs_shape", "sm_logprobs")
_check_visible_tensor_shape("ex122_sm_probs_shape", "sm_probs")
_check_private_tensor_shape("ex122_sm_dprobs_shape", "softmax_capstone", "dprobs")
_check_tensor("sm_dprobs", "softmax_capstone", "dprobs")


### Exercise 123 — Backward into the shared inverse row totals

**Purpose:** Learn why one shared inverse row total receives contributions from all five candidates in its example.

**Inputs:** `sm_counts` and `sm_dprobs` are `(4, 5)`; `sm_counts_sum_inv` is `(4, 1)`.

**Forward operation:** `sm_probs = sm_counts * sm_counts_sum_inv`.

**Derive/do:** Derive `sm_dcounts_sum_inv`.

**Ingredients:** Use product backward, then sum candidate contributions along dimension 1 while preserving it.

**Required outputs:**

- `sm_dcounts_sum_inv`: the `(4, 1)` gradient with respect to inverse row totals.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex123_sm_probs_shape`: predict the shape of `sm_probs` as a literal Python tuple
- `ex123_sm_counts_shape`: predict the shape of `sm_counts` as a literal Python tuple
- `ex123_sm_counts_sum_inv_shape`: predict the shape of `sm_counts_sum_inv` as a literal Python tuple
- `ex123_sm_dcounts_sum_inv_shape`: predict the shape of `sm_dcounts_sum_inv` as a literal Python tuple

**Next concept:** Backward through reciprocal row totals.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "softmax_capstone" in _REFS

# Forward relation under study:
# sm_probs = sm_counts * sm_counts_sum_inv
# Visible forward tensors and shapes:
# - `sm_counts` is already defined by the visible forward graph.
# - `sm_counts_sum_inv` is already defined by the visible forward graph.
# - `sm_probs` is already defined by the visible forward graph.
# Upstream gradient for this stage: sm_dprobs from Exercise 122.
print("sm_counts", tuple(sm_counts.shape), "sm_counts_sum_inv", tuple(sm_counts_sum_inv.shape), "sm_probs", tuple(sm_probs.shape))


In [ ]:
# Exercise 123: derive manually; do not use autograd in this cell.
# Define `sm_dcounts_sum_inv` — the `(4, 1)` gradient with respect to inverse row totals.
# Predict every visible tensor shape before computing values.
# Define `ex123_sm_probs_shape` — the shape of `sm_probs` as a literal Python tuple.
# Define `ex123_sm_counts_shape` — the shape of `sm_counts` as a literal Python tuple.
# Define `ex123_sm_counts_sum_inv_shape` — the shape of `sm_counts_sum_inv` as a literal Python tuple.
# Define `ex123_sm_dcounts_sum_inv_shape` — the shape of `sm_dcounts_sum_inv` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex123_sm_probs_shape", "sm_probs")
_check_visible_tensor_shape("ex123_sm_counts_shape", "sm_counts")
_check_visible_tensor_shape("ex123_sm_counts_sum_inv_shape", "sm_counts_sum_inv")
_check_private_tensor_shape("ex123_sm_dcounts_sum_inv_shape", "softmax_capstone", "dcounts_sum_inv")
_check_tensor("sm_dcounts_sum_inv", "softmax_capstone", "dcounts_sum_inv")


### Exercise 124 — Backward through reciprocal row totals

**Purpose:** Learn to pass each inverse-total gradient backward to the original row total.

**Inputs:** `sm_counts_sum` and its inverse have shape `(4, 1)`.

**Forward operation:** `sm_counts_sum_inv = sm_counts_sum**-1`.

**Derive/do:** Using `sm_dcounts_sum_inv`, derive `sm_dcounts_sum`.

**Ingredients:** The fixed-power rule with exponent `-1`.

**Required outputs:**

- `sm_dcounts_sum`: the `(4, 1)` gradient with respect to row totals.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex124_sm_counts_sum_inv_shape`: predict the shape of `sm_counts_sum_inv` as a literal Python tuple
- `ex124_sm_counts_sum_shape`: predict the shape of `sm_counts_sum` as a literal Python tuple
- `ex124_sm_dcounts_sum_shape`: predict the shape of `sm_dcounts_sum` as a literal Python tuple

**Next concept:** Combine both paths into counts.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "softmax_capstone" in _REFS

# Forward relation under study:
# sm_counts_sum_inv = sm_counts_sum**-1
# Visible forward tensors and shapes:
# - `sm_counts_sum` is already defined by the visible forward graph.
# - `sm_counts_sum_inv` is already defined by the visible forward graph.
# Upstream gradient for this stage: sm_dcounts_sum_inv from Exercise 123.
print("sm_counts_sum", tuple(sm_counts_sum.shape), "sm_counts_sum_inv", tuple(sm_counts_sum_inv.shape))


In [ ]:
# Exercise 124: derive manually; do not use autograd in this cell.
# Define `sm_dcounts_sum` — the `(4, 1)` gradient with respect to row totals.
# Predict every visible tensor shape before computing values.
# Define `ex124_sm_counts_sum_inv_shape` — the shape of `sm_counts_sum_inv` as a literal Python tuple.
# Define `ex124_sm_counts_sum_shape` — the shape of `sm_counts_sum` as a literal Python tuple.
# Define `ex124_sm_dcounts_sum_shape` — the shape of `sm_dcounts_sum` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex124_sm_counts_sum_inv_shape", "sm_counts_sum_inv")
_check_visible_tensor_shape("ex124_sm_counts_sum_shape", "sm_counts_sum")
_check_private_tensor_shape("ex124_sm_dcounts_sum_shape", "softmax_capstone", "dcounts_sum")
_check_tensor("sm_dcounts_sum", "softmax_capstone", "dcounts_sum")


### Exercise 125 — Combine both paths into counts

**Purpose:** Learn that exponentiated logits receive one gradient as numerators and another through the shared denominator.

**Inputs:** `sm_counts` is `(4, 5)` and `sm_counts_sum` is `(4, 1)`.

**Forward operation:** `sm_probs` uses `sm_counts` directly, while `sm_counts_sum` also depends on every `sm_counts` entry.

**Derive/do:** Derive total `sm_dcounts`.

**Ingredients:** Compute the direct multiplication path from `sm_dprobs`; broadcast the row-sum path from `sm_dcounts_sum`; add them.

**Required outputs:**

- `sm_dcounts`: the total `(4, 5)` gradient from both count paths.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex125_sm_counts_sum_shape`: predict the shape of `sm_counts_sum` as a literal Python tuple
- `ex125_sm_counts_shape`: predict the shape of `sm_counts` as a literal Python tuple
- `ex125_sm_probs_shape`: predict the shape of `sm_probs` as a literal Python tuple
- `ex125_sm_dcounts_shape`: predict the shape of `sm_dcounts` as a literal Python tuple

**Next concept:** Backward through exponentiation.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "softmax_capstone" in _REFS

# Forward relation under study:
# sm_counts_sum = sm_counts.sum(dim=1, keepdim=True), while sm_probs also uses sm_counts directly
# Visible forward tensors and shapes:
# - `sm_counts` is already defined by the visible forward graph.
# - `sm_counts_sum` is already defined by the visible forward graph.
# - `sm_probs` is already defined by the visible forward graph.
# Upstream gradient for this stage: sm_dcounts_sum from Exercise 124 and sm_dprobs from Exercise 122.
print("sm_counts", tuple(sm_counts.shape), "sm_counts_sum", tuple(sm_counts_sum.shape), "sm_probs", tuple(sm_probs.shape))


In [ ]:
# Exercise 125: derive manually; do not use autograd in this cell.
# Define `sm_dcounts` — the total `(4, 5)` gradient from both count paths.
# Predict every visible tensor shape before computing values.
# Define `ex125_sm_counts_sum_shape` — the shape of `sm_counts_sum` as a literal Python tuple.
# Define `ex125_sm_counts_shape` — the shape of `sm_counts` as a literal Python tuple.
# Define `ex125_sm_probs_shape` — the shape of `sm_probs` as a literal Python tuple.
# Define `ex125_sm_dcounts_shape` — the shape of `sm_dcounts` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex125_sm_counts_sum_shape", "sm_counts_sum")
_check_visible_tensor_shape("ex125_sm_counts_shape", "sm_counts")
_check_visible_tensor_shape("ex125_sm_probs_shape", "sm_probs")
_check_private_tensor_shape("ex125_sm_dcounts_shape", "softmax_capstone", "dcounts")
_check_tensor("sm_dcounts", "softmax_capstone", "dcounts")


### Exercise 126 — Backward through exponentiation

**Purpose:** Learn to pass the combined candidate-weight gradients backward through exponentiation.

**Inputs:** `sm_counts` and `sm_dcounts` are `(4, 5)`.

**Forward operation:** `sm_counts = sm_norm_logits.exp()`.

**Derive/do:** Derive `sm_dnorm_logits`.

**Ingredients:** Use the exponentiated forward values as the local derivative.

**Required outputs:**

- `sm_dnorm_logits`: the `(4, 5)` gradient with respect to normalized logits.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex126_sm_counts_shape`: predict the shape of `sm_counts` as a literal Python tuple
- `ex126_sm_norm_logits_shape`: predict the shape of `sm_norm_logits` as a literal Python tuple
- `ex126_sm_dnorm_logits_shape`: predict the shape of `sm_dnorm_logits` as a literal Python tuple

**Next concept:** Backward through maximum shifting.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "softmax_capstone" in _REFS

# Forward relation under study:
# sm_counts = sm_norm_logits.exp()
# Visible forward tensors and shapes:
# - `sm_norm_logits` is already defined by the visible forward graph.
# - `sm_counts` is already defined by the visible forward graph.
# Upstream gradient for this stage: sm_dcounts from Exercise 125.
print("sm_norm_logits", tuple(sm_norm_logits.shape), "sm_counts", tuple(sm_counts.shape))


In [ ]:
# Exercise 126: derive manually; do not use autograd in this cell.
# Define `sm_dnorm_logits` — the `(4, 5)` gradient with respect to normalized logits.
# Predict every visible tensor shape before computing values.
# Define `ex126_sm_counts_shape` — the shape of `sm_counts` as a literal Python tuple.
# Define `ex126_sm_norm_logits_shape` — the shape of `sm_norm_logits` as a literal Python tuple.
# Define `ex126_sm_dnorm_logits_shape` — the shape of `sm_dnorm_logits` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex126_sm_counts_shape", "sm_counts")
_check_visible_tensor_shape("ex126_sm_norm_logits_shape", "sm_norm_logits")
_check_private_tensor_shape("ex126_sm_dnorm_logits_shape", "softmax_capstone", "dnorm_logits")
_check_tensor("sm_dnorm_logits", "softmax_capstone", "dnorm_logits")


### Exercise 127 — Backward through maximum shifting

**Purpose:** Learn to combine the direct logit path with the path through each row's maximum.

**Inputs:** `sm_logits` and `sm_dnorm_logits` are `(4, 5)`; `sm_logit_maxes` is `(4, 1)`.

**Forward operation:** `sm_logit_maxes = sm_logits.max(dim=1, keepdim=True).values` and `sm_norm_logits = sm_logits - sm_logit_maxes`.

**Derive/do:** Derive `sm_dlogit_maxes` and total `sm_dlogits`.

**Ingredients:** First unbroadcast the subtraction's maximum path. Then route each row's maximum gradient to that row's unique argmax and add the direct logits path.

**Required outputs:**

- `sm_dlogit_maxes`: the `(4, 1)` gradient of row maxima.
- `sm_dlogits`: the final `(4, 5)` gradient with respect to logits.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex127_sm_logit_maxes_shape`: predict the shape of `sm_logit_maxes` as a literal Python tuple
- `ex127_sm_logits_shape`: predict the shape of `sm_logits` as a literal Python tuple
- `ex127_sm_norm_logits_shape`: predict the shape of `sm_norm_logits` as a literal Python tuple
- `ex127_sm_dlogit_maxes_shape`: predict the shape of `sm_dlogit_maxes` as a literal Python tuple
- `ex127_sm_dlogits_shape`: predict the shape of `sm_dlogits` as a literal Python tuple

**Next concept:** BatchNorm column means.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "softmax_capstone" in _REFS

# Forward relation under study:
# sm_logit_maxes = sm_logits.max(dim=1, keepdim=True).values; sm_norm_logits = sm_logits - sm_logit_maxes
# Visible forward tensors and shapes:
# - `sm_logits` is already defined by the visible forward graph.
# - `sm_logit_maxes` is already defined by the visible forward graph.
# - `sm_norm_logits` is already defined by the visible forward graph.
# Upstream gradient for this stage: sm_dnorm_logits from Exercise 126.
print("sm_logits", tuple(sm_logits.shape), "sm_logit_maxes", tuple(sm_logit_maxes.shape), "sm_norm_logits", tuple(sm_norm_logits.shape))


In [ ]:
# Exercise 127: derive manually; do not use autograd in this cell.
# Define `sm_dlogit_maxes` — the `(4, 1)` gradient of row maxima.
# Define `sm_dlogits` — the final `(4, 5)` gradient with respect to logits.
# Predict every visible tensor shape before computing values.
# Define `ex127_sm_logit_maxes_shape` — the shape of `sm_logit_maxes` as a literal Python tuple.
# Define `ex127_sm_logits_shape` — the shape of `sm_logits` as a literal Python tuple.
# Define `ex127_sm_norm_logits_shape` — the shape of `sm_norm_logits` as a literal Python tuple.
# Define `ex127_sm_dlogit_maxes_shape` — the shape of `sm_dlogit_maxes` as a literal Python tuple.
# Define `ex127_sm_dlogits_shape` — the shape of `sm_dlogits` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex127_sm_logit_maxes_shape", "sm_logit_maxes")
_check_visible_tensor_shape("ex127_sm_logits_shape", "sm_logits")
_check_visible_tensor_shape("ex127_sm_norm_logits_shape", "sm_norm_logits")
_check_private_tensor_shape("ex127_sm_dlogit_maxes_shape", "softmax_capstone", "dlogit_maxes")
_check_private_tensor_shape("ex127_sm_dlogits_shape", "softmax_capstone", "dlogits")
_check_tensor("sm_dlogit_maxes", "softmax_capstone", "dlogit_maxes")
_check_tensor("sm_dlogits", "softmax_capstone", "dlogits")


## 9. BatchNorm as small tensor operations

BatchNorm treats rows as training examples and columns as hidden neurons. In this section, `B` is the batch size and `H` is the number of hidden neurons. Each neuron gets one mean and one variance computed across the `B` examples.

The forward relationships are

$$
\mu_j
=
\frac{1}{B}\sum_i a_{i,j}
$$

$$
d_{i,j}
=
a_{i,j}-\mu_j
$$

$$
v_j
=
\frac{1}{B-1}\sum_i d_{i,j}^2
$$

$$
r_j
=
(v_j+\varepsilon)^{-1/2}
$$

$$
\widehat{a}_{i,j}
=
d_{i,j}r_j
$$

$$
\widetilde{a}_{i,j}
=
\gamma_j\widehat{a}_{i,j}+\beta_j
$$

Here, `a` is `hprebn`, `mu` is `bnmeani`, `d` is `bndiff`, `v` is `bnvar`, `r` is `bnvar_inv`, normalized values are `bnraw`, scale is `bngain`, shift is `bnbias`, and the final result is `hpreact`. Despite its historical name, `bnvar_inv` is the inverse **standard deviation**, not the inverse variance. The variance is a sample variance, so it divides by `B - 1`, not `B`.


### Exercise 128 — BatchNorm column means

**Purpose:** Learn that BatchNorm computes one mean per hidden-neuron column across all examples.

**Inputs:** `hprebn` has shape `(4, 3)` and `bnmeani` has shape `(1, 3)`.

**Forward operation:** `bnmeani = hprebn.mean(dim=0, keepdim=True)`.

**Derive/do:** Derive `dhprebn` from a `(1, 3)` upstream gradient.

**Ingredients:** Each neuron mean averages four example rows.

**Required outputs:**

- `ex128_dhprebn`: the `(4, 3)` gradient after reversing the column means.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex128_hprebn_shape`: predict the shape of `hprebn` as a literal Python tuple
- `ex128_bnmeani_shape`: predict the shape of `bnmeani` as a literal Python tuple
- `ex128_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex128_dhprebn_shape`: predict the shape of `ex128_dhprebn` as a literal Python tuple

**Next concept:** BatchNorm centering with independent means.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
hprebn = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bnmeani = hprebn.mean(dim=0, keepdim=True)
dout = torch.tensor([[1.0, -2.0, 0.5]], dtype=DTYPE)
_capture("ex127", bnmeani, {"dhprebn": hprebn}, dout)
print("hprebn", tuple(hprebn.shape), "bnmeani", tuple(bnmeani.shape))


In [ ]:
# Exercise 128: derive manually; do not use autograd in this cell.
# Define `ex128_dhprebn` — the `(4, 3)` gradient after reversing the column means.
# Predict every visible tensor shape before computing values.
# Define `ex128_hprebn_shape` — the shape of `hprebn` as a literal Python tuple.
# Define `ex128_bnmeani_shape` — the shape of `bnmeani` as a literal Python tuple.
# Define `ex128_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex128_dhprebn_shape` — the shape of `ex128_dhprebn` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex128_hprebn_shape", "hprebn")
_check_visible_tensor_shape("ex128_bnmeani_shape", "bnmeani")
_check_visible_tensor_shape("ex128_dout_shape", "dout")
_check_private_tensor_shape("ex128_dhprebn_shape", "ex127", "dhprebn")
_check_tensor("ex128_dhprebn", "ex127", "dhprebn")


### Exercise 129 — BatchNorm centering with independent means

**Purpose:** Learn how subtracting one hidden-neuron mean from every example sends gradients to both values and the mean.

**Inputs:** `hprebn` is `(4, 3)`, independent `bnmeani` is `(1, 3)`, and `bndiff` is `(4, 3)`.

**Forward operation:** `bndiff = hprebn - bnmeani`.

**Derive/do:** Derive the direct `dhprebn` and `dbnmeani`.

**Ingredients:** Subtraction sends a direct path to `hprebn`; sum the negative mean path over examples.

**Required outputs:**

- `ex129_dhprebn`: the direct centered-value gradient.
- `ex129_dbnmeani`: the `(1, 3)` unbroadcast mean gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex129_hprebn_shape`: predict the shape of `hprebn` as a literal Python tuple
- `ex129_bnmeani_shape`: predict the shape of `bnmeani` as a literal Python tuple
- `ex129_bndiff_shape`: predict the shape of `bndiff` as a literal Python tuple
- `ex129_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex129_dhprebn_shape`: predict the shape of `ex129_dhprebn` as a literal Python tuple
- `ex129_dbnmeani_shape`: predict the shape of `ex129_dbnmeani` as a literal Python tuple

**Next concept:** Square centered deviations.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
hprebn = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bnmeani = torch.tensor([[0.5, -1.0, 2.0]], dtype=DTYPE, requires_grad=True)
bndiff = hprebn - bnmeani
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex128", bndiff, {"dhprebn": hprebn, "dbnmeani": bnmeani}, dout)
print("hprebn", tuple(hprebn.shape), "bnmeani", tuple(bnmeani.shape), "bndiff", tuple(bndiff.shape))


In [ ]:
# Exercise 129: derive manually; do not use autograd in this cell.
# Define `ex129_dhprebn` — the direct centered-value gradient.
# Define `ex129_dbnmeani` — the `(1, 3)` unbroadcast mean gradient.
# Predict every visible tensor shape before computing values.
# Define `ex129_hprebn_shape` — the shape of `hprebn` as a literal Python tuple.
# Define `ex129_bnmeani_shape` — the shape of `bnmeani` as a literal Python tuple.
# Define `ex129_bndiff_shape` — the shape of `bndiff` as a literal Python tuple.
# Define `ex129_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex129_dhprebn_shape` — the shape of `ex129_dhprebn` as a literal Python tuple.
# Define `ex129_dbnmeani_shape` — the shape of `ex129_dbnmeani` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex129_hprebn_shape", "hprebn")
_check_visible_tensor_shape("ex129_bnmeani_shape", "bnmeani")
_check_visible_tensor_shape("ex129_bndiff_shape", "bndiff")
_check_visible_tensor_shape("ex129_dout_shape", "dout")
_check_private_tensor_shape("ex129_dhprebn_shape", "ex128", "dhprebn")
_check_private_tensor_shape("ex129_dbnmeani_shape", "ex128", "dbnmeani")
_check_tensor("ex129_dhprebn", "ex128", "dhprebn")
_check_tensor("ex129_dbnmeani", "ex128", "dbnmeani")


### Exercise 130 — Square centered deviations

**Purpose:** Learn to pass variance-related gradients backward through squared deviations.

**Inputs:** `bndiff` and `bndiff2` have shape `(4, 3)`.

**Forward operation:** `bndiff2 = bndiff**2`.

**Derive/do:** Derive `dbndiff`.

**Ingredients:** The square power rule with an arbitrary upstream tensor.

**Required outputs:**

- `ex130_dbndiff`: the gradient with respect to centered deviations.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex130_bndiff_shape`: predict the shape of `bndiff` as a literal Python tuple
- `ex130_bndiff2_shape`: predict the shape of `bndiff2` as a literal Python tuple
- `ex130_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex130_dbndiff_shape`: predict the shape of `ex130_dbndiff` as a literal Python tuple

**Next concept:** Unbiased variance reduction.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
bndiff = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bndiff2 = bndiff**2
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex129", bndiff2, {"dbndiff": bndiff}, dout)
print("bndiff", tuple(bndiff.shape), "bndiff2", tuple(bndiff2.shape))


In [ ]:
# Exercise 130: derive manually; do not use autograd in this cell.
# Define `ex130_dbndiff` — the gradient with respect to centered deviations.
# Predict every visible tensor shape before computing values.
# Define `ex130_bndiff_shape` — the shape of `bndiff` as a literal Python tuple.
# Define `ex130_bndiff2_shape` — the shape of `bndiff2` as a literal Python tuple.
# Define `ex130_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex130_dbndiff_shape` — the shape of `ex130_dbndiff` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex130_bndiff_shape", "bndiff")
_check_visible_tensor_shape("ex130_bndiff2_shape", "bndiff2")
_check_visible_tensor_shape("ex130_dout_shape", "dout")
_check_private_tensor_shape("ex130_dbndiff_shape", "ex129", "dbndiff")
_check_tensor("ex130_dbndiff", "ex129", "dbndiff")


### Exercise 131 — Unbiased variance reduction

**Purpose:** Learn that sample variance spreads one gradient per hidden neuron across examples and divides by `B - 1`.

**Inputs:** `bndiff2` is `(4, 3)`, `B = 4`, and `bnvar` is `(1, 3)`.

**Forward operation:** `bnvar = bndiff2.sum(dim=0, keepdim=True) / (B - 1)`.

**Derive/do:** Derive `dbndiff2`.

**Ingredients:** Broadcast each neuron variance gradient over four rows and include division by 3.

**Required outputs:**

- `ex131_dbndiff2`: the `(4, 3)` gradient after reversing sample variance.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex131_bndiff2_shape`: predict the shape of `bndiff2` as a literal Python tuple
- `ex131_bnvar_shape`: predict the shape of `bnvar` as a literal Python tuple
- `ex131_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex131_dbndiff2_shape`: predict the shape of `ex131_dbndiff2` as a literal Python tuple

**Next concept:** Stabilized inverse standard deviation.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
bndiff2 = torch.linspace(0.1, 2.4, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
B = bndiff2.shape[0]
bnvar = bndiff2.sum(dim=0, keepdim=True) / (B - 1)
dout = torch.tensor([[1.0, -2.0, 0.5]], dtype=DTYPE)
_capture("ex130", bnvar, {"dbndiff2": bndiff2}, dout)
print("bndiff2", tuple(bndiff2.shape), "bnvar", tuple(bnvar.shape), "B", B)


In [ ]:
# Exercise 131: derive manually; do not use autograd in this cell.
# Define `ex131_dbndiff2` — the `(4, 3)` gradient after reversing sample variance.
# Predict every visible tensor shape before computing values.
# Define `ex131_bndiff2_shape` — the shape of `bndiff2` as a literal Python tuple.
# Define `ex131_bnvar_shape` — the shape of `bnvar` as a literal Python tuple.
# Define `ex131_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex131_dbndiff2_shape` — the shape of `ex131_dbndiff2` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex131_bndiff2_shape", "bndiff2")
_check_visible_tensor_shape("ex131_bnvar_shape", "bnvar")
_check_visible_tensor_shape("ex131_dout_shape", "dout")
_check_private_tensor_shape("ex131_dbndiff2_shape", "ex130", "dbndiff2")
_check_tensor("ex131_dbndiff2", "ex130", "dbndiff2")


### Exercise 132 — Stabilized inverse standard deviation

**Purpose:** Learn to pass gradients backward through epsilon addition and inverse square root.

**Inputs:** `bnvar` and `bnvar_inv` have shape `(1, 3)`.

**Forward operation:** `bnvar_inv = (bnvar + eps)**-0.5`.

**Derive/do:** Derive `dbnvar`.

**Ingredients:** Epsilon is constant; apply the fixed-power rule to the stabilized variance.

**Required outputs:**

- `ex132_dbnvar`: the `(1, 3)` variance gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex132_bnvar_shape`: predict the shape of `bnvar` as a literal Python tuple
- `ex132_bnvar_inv_shape`: predict the shape of `bnvar_inv` as a literal Python tuple
- `ex132_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex132_dbnvar_shape`: predict the shape of `ex132_dbnvar` as a literal Python tuple

**Next concept:** Normalize centered values.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
bnvar = torch.tensor([[0.25, 1.0, 4.0]], dtype=DTYPE, requires_grad=True)
eps = 1e-5
bnvar_inv = (bnvar + eps) ** -0.5
dout = torch.tensor([[1.0, -2.0, 0.5]], dtype=DTYPE)
_capture("ex131", bnvar_inv, {"dbnvar": bnvar}, dout)
print("bnvar", tuple(bnvar.shape), "bnvar_inv", tuple(bnvar_inv.shape))


In [121]:
# Exercise 132: derive manually; do not use autograd in this cell.
# Define `ex132_dbnvar` — the `(1, 3)` variance gradient.
# Predict every visible tensor shape before computing values.
# Define `ex132_bnvar_shape` — the shape of `bnvar` as a literal Python tuple.
# Define `ex132_bnvar_inv_shape` — the shape of `bnvar_inv` as a literal Python tuple.
# Define `ex132_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex132_dbnvar_shape` — the shape of `ex132_dbnvar` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [122]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex132_bnvar_shape", "bnvar")
_check_visible_tensor_shape("ex132_bnvar_inv_shape", "bnvar_inv")
_check_visible_tensor_shape("ex132_dout_shape", "dout")
_check_private_tensor_shape("ex132_dbnvar_shape", "ex131", "dbnvar")
_check_tensor("ex132_dbnvar", "ex131", "dbnvar")


AssertionError: Visible tensor `bnvar` is not defined.

### Exercise 133 — Normalize centered values

**Purpose:** Learn how one inverse standard deviation reused across examples receives all of their contributions.

**Inputs:** `bndiff` is `(4, 3)`, `bnvar_inv` is `(1, 3)`, and `bnraw` is `(4, 3)`.

**Forward operation:** `bnraw = bndiff * bnvar_inv`.

**Derive/do:** Derive direct `dbndiff` and accumulated `dbnvar_inv`.

**Ingredients:** Product backward; sum inverse-standard-deviation contributions over four examples.

**Required outputs:**

- `ex133_dbndiff`: the direct centered-value gradient.
- `ex133_dbnvar_inv`: the `(1, 3)` gradient accumulated over examples.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex133_bndiff_shape`: predict the shape of `bndiff` as a literal Python tuple
- `ex133_bnvar_inv_shape`: predict the shape of `bnvar_inv` as a literal Python tuple
- `ex133_bnraw_shape`: predict the shape of `bnraw` as a literal Python tuple
- `ex133_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex133_dbndiff_shape`: predict the shape of `ex133_dbndiff` as a literal Python tuple
- `ex133_dbnvar_inv_shape`: predict the shape of `ex133_dbnvar_inv` as a literal Python tuple

**Next concept:** BatchNorm learned scale and shift.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
bndiff = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bnvar_inv = torch.tensor([[2.0, 1.0, 0.5]], dtype=DTYPE, requires_grad=True)
bnraw = bndiff * bnvar_inv
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex132", bnraw, {"dbndiff": bndiff, "dbnvar_inv": bnvar_inv}, dout)
print("bndiff", tuple(bndiff.shape), "bnvar_inv", tuple(bnvar_inv.shape), "bnraw", tuple(bnraw.shape))


In [ ]:
# Exercise 133: derive manually; do not use autograd in this cell.
# Define `ex133_dbndiff` — the direct centered-value gradient.
# Define `ex133_dbnvar_inv` — the `(1, 3)` gradient accumulated over examples.
# Predict every visible tensor shape before computing values.
# Define `ex133_bndiff_shape` — the shape of `bndiff` as a literal Python tuple.
# Define `ex133_bnvar_inv_shape` — the shape of `bnvar_inv` as a literal Python tuple.
# Define `ex133_bnraw_shape` — the shape of `bnraw` as a literal Python tuple.
# Define `ex133_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex133_dbndiff_shape` — the shape of `ex133_dbndiff` as a literal Python tuple.
# Define `ex133_dbnvar_inv_shape` — the shape of `ex133_dbnvar_inv` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex133_bndiff_shape", "bndiff")
_check_visible_tensor_shape("ex133_bnvar_inv_shape", "bnvar_inv")
_check_visible_tensor_shape("ex133_bnraw_shape", "bnraw")
_check_visible_tensor_shape("ex133_dout_shape", "dout")
_check_private_tensor_shape("ex133_dbndiff_shape", "ex132", "dbndiff")
_check_private_tensor_shape("ex133_dbnvar_inv_shape", "ex132", "dbnvar_inv")
_check_tensor("ex133_dbndiff", "ex132", "dbndiff")
_check_tensor("ex133_dbnvar_inv", "ex132", "dbnvar_inv")


### Exercise 134 — BatchNorm learned scale and shift

**Purpose:** Learn how BatchNorm's learned gain and bias collect gradients from every example.

**Inputs:** `bnraw` is `(4, 3)`; `bngain` and `bnbias` are `(1, 3)`.

**Forward operation:** `hpreact = bngain * bnraw + bnbias`.

**Derive/do:** Derive `dbnraw`, `dbngain`, and `dbnbias`.

**Ingredients:** Elementwise affine backward plus accumulation over four examples for gain and bias.

**Required outputs:**

- `ex134_dbnraw`: the gradient entering normalized values.
- `ex134_dbngain`: the `(1, 3)` learned-gain gradient.
- `ex134_dbnbias`: the `(1, 3)` learned-bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex134_bnraw_shape`: predict the shape of `bnraw` as a literal Python tuple
- `ex134_bngain_shape`: predict the shape of `bngain` as a literal Python tuple
- `ex134_bnbias_shape`: predict the shape of `bnbias` as a literal Python tuple
- `ex134_hpreact_shape`: predict the shape of `hpreact` as a literal Python tuple
- `ex134_dout_shape`: predict the shape of `dout` as a literal Python tuple
- `ex134_dbnraw_shape`: predict the shape of `ex134_dbnraw` as a literal Python tuple
- `ex134_dbngain_shape`: predict the shape of `ex134_dbngain` as a literal Python tuple
- `ex134_dbnbias_shape`: predict the shape of `ex134_dbnbias` as a literal Python tuple

**Next concept:** Full BatchNorm: affine backward.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
bnraw = torch.linspace(-1.5, 1.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bngain = torch.tensor([[1.2, 0.8, -1.1]], dtype=DTYPE, requires_grad=True)
bnbias = torch.tensor([[0.1, -0.2, 0.3]], dtype=DTYPE, requires_grad=True)
hpreact = bngain * bnraw + bnbias
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex133", hpreact, {"dbnraw": bnraw, "dbngain": bngain, "dbnbias": bnbias}, dout)
print("bnraw", tuple(bnraw.shape), "gain", tuple(bngain.shape), "hpreact", tuple(hpreact.shape))


In [ ]:
# Exercise 134: derive manually; do not use autograd in this cell.
# Define `ex134_dbnraw` — the gradient entering normalized values.
# Define `ex134_dbngain` — the `(1, 3)` learned-gain gradient.
# Define `ex134_dbnbias` — the `(1, 3)` learned-bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex134_bnraw_shape` — the shape of `bnraw` as a literal Python tuple.
# Define `ex134_bngain_shape` — the shape of `bngain` as a literal Python tuple.
# Define `ex134_bnbias_shape` — the shape of `bnbias` as a literal Python tuple.
# Define `ex134_hpreact_shape` — the shape of `hpreact` as a literal Python tuple.
# Define `ex134_dout_shape` — the shape of `dout` as a literal Python tuple.
# Define `ex134_dbnraw_shape` — the shape of `ex134_dbnraw` as a literal Python tuple.
# Define `ex134_dbngain_shape` — the shape of `ex134_dbngain` as a literal Python tuple.
# Define `ex134_dbnbias_shape` — the shape of `ex134_dbnbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex134_bnraw_shape", "bnraw")
_check_visible_tensor_shape("ex134_bngain_shape", "bngain")
_check_visible_tensor_shape("ex134_bnbias_shape", "bnbias")
_check_visible_tensor_shape("ex134_hpreact_shape", "hpreact")
_check_visible_tensor_shape("ex134_dout_shape", "dout")
_check_private_tensor_shape("ex134_dbnraw_shape", "ex133", "dbnraw")
_check_private_tensor_shape("ex134_dbngain_shape", "ex133", "dbngain")
_check_private_tensor_shape("ex134_dbnbias_shape", "ex133", "dbnbias")
_check_tensor("ex134_dbnraw", "ex133", "dbnraw")
_check_tensor("ex134_dbngain", "ex133", "dbngain")
_check_tensor("ex134_dbnbias", "ex133", "dbnbias")


### Exercise 135 — Full BatchNorm: affine backward

**Purpose:** Learn to begin the complete BatchNorm backward pass at its final learned scale-and-shift operation.

**Inputs:** `bncap_hpreact` and `bncap_dhpreact` are `(4, 3)`; gain and bias are `(1, 3)`.

**Forward operation:** The full fixture ends with `bncap_hpreact = gain * bnraw + bias`.

**Derive/do:** Derive `bncap_dbnraw`, `bncap_dbngain`, and `bncap_dbnbias`.

**Ingredients:** Reverse the learned affine step and unbroadcast parameter gradients over examples.

**Required outputs:**

- `bncap_dbnraw`: the `(4, 3)` gradient entering normalized values.
- `bncap_dbngain`: the `(1, 3)` gain gradient.
- `bncap_dbnbias`: the `(1, 3)` bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex135_bncap_hprebn_shape`: predict the shape of `bncap_hprebn` as a literal Python tuple
- `ex135_bncap_bngain_shape`: predict the shape of `bncap_bngain` as a literal Python tuple
- `ex135_bncap_bnbias_shape`: predict the shape of `bncap_bnbias` as a literal Python tuple
- `ex135_bncap_bnmeani_shape`: predict the shape of `bncap_bnmeani` as a literal Python tuple
- `ex135_bncap_bndiff_shape`: predict the shape of `bncap_bndiff` as a literal Python tuple
- `ex135_bncap_bndiff2_shape`: predict the shape of `bncap_bndiff2` as a literal Python tuple
- `ex135_bncap_bnvar_shape`: predict the shape of `bncap_bnvar` as a literal Python tuple
- `ex135_bncap_bnvar_inv_shape`: predict the shape of `bncap_bnvar_inv` as a literal Python tuple
- `ex135_bncap_bnraw_shape`: predict the shape of `bncap_bnraw` as a literal Python tuple
- `ex135_bncap_hpreact_shape`: predict the shape of `bncap_hpreact` as a literal Python tuple
- `ex135_bncap_dhpreact_shape`: predict the shape of `bncap_dhpreact` as a literal Python tuple
- `ex135_bncap_objective_shape`: predict the shape of `bncap_objective` as a literal Python tuple
- `ex135_bncap_dbnraw_shape`: predict the shape of `bncap_dbnraw` as a literal Python tuple
- `ex135_bncap_dbngain_shape`: predict the shape of `bncap_dbngain` as a literal Python tuple
- `ex135_bncap_dbnbias_shape`: predict the shape of `bncap_dbnbias` as a literal Python tuple

**Next concept:** Full BatchNorm: normalization product.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
# Supplied full BatchNorm graph: four examples and three hidden neurons.
bncap_hprebn = torch.tensor([
    [-1.0, 0.5, 2.0],
    [0.0, -1.5, 1.0],
    [2.0, 1.5, -0.5],
    [1.0, 0.0, 3.0],
], dtype=DTYPE, requires_grad=True)
bncap_n = bncap_hprebn.shape[0]
bncap_bngain = torch.tensor([[1.2, 0.8, -1.1]], dtype=DTYPE, requires_grad=True)
bncap_bnbias = torch.tensor([[0.1, -0.2, 0.3]], dtype=DTYPE, requires_grad=True)
bncap_eps = 1e-5

bncap_bnmeani = bncap_hprebn.mean(dim=0, keepdim=True)
bncap_bndiff = bncap_hprebn - bncap_bnmeani
bncap_bndiff2 = bncap_bndiff**2
bncap_bnvar = bncap_bndiff2.sum(dim=0, keepdim=True) / (bncap_n - 1)
bncap_bnvar_inv = (bncap_bnvar + bncap_eps) ** -0.5
bncap_bnraw = bncap_bndiff * bncap_bnvar_inv
bncap_hpreact = bncap_bngain * bncap_bnraw + bncap_bnbias

# This supplied tensor is the gradient arriving from the next graph operation.
bncap_dhpreact = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
bncap_objective = (bncap_hpreact * bncap_dhpreact).sum()
_capture(
    "batchnorm_capstone",
    bncap_objective,
    {
        "dbngain": bncap_bngain,
        "dbnbias": bncap_bnbias,
        "dbnraw": bncap_bnraw,
        "dbnvar_inv": bncap_bnvar_inv,
        "dbnvar": bncap_bnvar,
        "dbndiff2": bncap_bndiff2,
        "dbndiff": bncap_bndiff,
        "dbnmeani": bncap_bnmeani,
        "dhprebn": bncap_hprebn,
    },
    retain_graph=True,
)
print("BatchNorm capstone ready:", tuple(bncap_hprebn.shape), "->", tuple(bncap_hpreact.shape))
# Private staged references for paths that merge later.
_capture_product_path(
    "batchnorm_capstone",
    "dbndiff_direct",
    bncap_bndiff,
    bncap_bnvar_inv,
    _REFS["batchnorm_capstone"]["dbnraw"],
)
_capture_path(
    "batchnorm_capstone",
    "dbndiff_variance",
    bncap_bnvar,
    bncap_bndiff,
    _REFS["batchnorm_capstone"]["dbnvar"],
)


In [ ]:
# Exercise 135: derive manually; do not use autograd in this cell.
# Define `bncap_dbnraw` — the `(4, 3)` gradient entering normalized values.
# Define `bncap_dbngain` — the `(1, 3)` gain gradient.
# Define `bncap_dbnbias` — the `(1, 3)` bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex135_bncap_hprebn_shape` — the shape of `bncap_hprebn` as a literal Python tuple.
# Define `ex135_bncap_bngain_shape` — the shape of `bncap_bngain` as a literal Python tuple.
# Define `ex135_bncap_bnbias_shape` — the shape of `bncap_bnbias` as a literal Python tuple.
# Define `ex135_bncap_bnmeani_shape` — the shape of `bncap_bnmeani` as a literal Python tuple.
# Define `ex135_bncap_bndiff_shape` — the shape of `bncap_bndiff` as a literal Python tuple.
# Define `ex135_bncap_bndiff2_shape` — the shape of `bncap_bndiff2` as a literal Python tuple.
# Define `ex135_bncap_bnvar_shape` — the shape of `bncap_bnvar` as a literal Python tuple.
# Define `ex135_bncap_bnvar_inv_shape` — the shape of `bncap_bnvar_inv` as a literal Python tuple.
# Define `ex135_bncap_bnraw_shape` — the shape of `bncap_bnraw` as a literal Python tuple.
# Define `ex135_bncap_hpreact_shape` — the shape of `bncap_hpreact` as a literal Python tuple.
# Define `ex135_bncap_dhpreact_shape` — the shape of `bncap_dhpreact` as a literal Python tuple.
# Define `ex135_bncap_objective_shape` — the shape of `bncap_objective` as a literal Python tuple.
# Define `ex135_bncap_dbnraw_shape` — the shape of `bncap_dbnraw` as a literal Python tuple.
# Define `ex135_bncap_dbngain_shape` — the shape of `bncap_dbngain` as a literal Python tuple.
# Define `ex135_bncap_dbnbias_shape` — the shape of `bncap_dbnbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex135_bncap_hprebn_shape", "bncap_hprebn")
_check_visible_tensor_shape("ex135_bncap_bngain_shape", "bncap_bngain")
_check_visible_tensor_shape("ex135_bncap_bnbias_shape", "bncap_bnbias")
_check_visible_tensor_shape("ex135_bncap_bnmeani_shape", "bncap_bnmeani")
_check_visible_tensor_shape("ex135_bncap_bndiff_shape", "bncap_bndiff")
_check_visible_tensor_shape("ex135_bncap_bndiff2_shape", "bncap_bndiff2")
_check_visible_tensor_shape("ex135_bncap_bnvar_shape", "bncap_bnvar")
_check_visible_tensor_shape("ex135_bncap_bnvar_inv_shape", "bncap_bnvar_inv")
_check_visible_tensor_shape("ex135_bncap_bnraw_shape", "bncap_bnraw")
_check_visible_tensor_shape("ex135_bncap_hpreact_shape", "bncap_hpreact")
_check_visible_tensor_shape("ex135_bncap_dhpreact_shape", "bncap_dhpreact")
_check_visible_tensor_shape("ex135_bncap_objective_shape", "bncap_objective")
_check_private_tensor_shape("ex135_bncap_dbnraw_shape", "batchnorm_capstone", "dbnraw")
_check_private_tensor_shape("ex135_bncap_dbngain_shape", "batchnorm_capstone", "dbngain")
_check_private_tensor_shape("ex135_bncap_dbnbias_shape", "batchnorm_capstone", "dbnbias")
_check_tensor("bncap_dbnraw", "batchnorm_capstone", "dbnraw")
_check_tensor("bncap_dbngain", "batchnorm_capstone", "dbngain")
_check_tensor("bncap_dbnbias", "batchnorm_capstone", "dbnbias")


### Exercise 136 — Full BatchNorm: normalization product

**Purpose:** Learn to split the normalized-value gradient between centered values and the shared inverse standard deviation.

**Inputs:** `bncap_bndiff` is `(4, 3)` and `bncap_bnvar_inv` is `(1, 3)`.

**Forward operation:** `bncap_bnraw = bncap_bndiff * bncap_bnvar_inv`.

**Derive/do:** Using `bncap_dbnraw`, derive direct `bncap_dbndiff_direct` and `bncap_dbnvar_inv`.

**Ingredients:** Product backward; sum the inverse-standard-deviation path over examples.

**Required outputs:**

- `bncap_dbndiff_direct`: the direct `(4, 3)` centered-value contribution.
- `bncap_dbnvar_inv`: the `(1, 3)` inverse-standard-deviation gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex136_bncap_bnraw_shape`: predict the shape of `bncap_bnraw` as a literal Python tuple
- `ex136_bncap_bndiff_shape`: predict the shape of `bncap_bndiff` as a literal Python tuple
- `ex136_bncap_bnvar_inv_shape`: predict the shape of `bncap_bnvar_inv` as a literal Python tuple
- `ex136_bncap_dbndiff_direct_shape`: predict the shape of `bncap_dbndiff_direct` as a literal Python tuple
- `ex136_bncap_dbnvar_inv_shape`: predict the shape of `bncap_dbnvar_inv` as a literal Python tuple

**Next concept:** Full BatchNorm: inverse square root.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "batchnorm_capstone" in _REFS

# Forward relation under study:
# bncap_bnraw = bncap_bndiff * bncap_bnvar_inv
# Visible forward tensors and shapes:
# - `bncap_bndiff` is already defined by the visible forward graph.
# - `bncap_bnvar_inv` is already defined by the visible forward graph.
# - `bncap_bnraw` is already defined by the visible forward graph.
# Upstream gradient for this stage: bncap_dbnraw from Exercise 135.
print("bncap_bndiff", tuple(bncap_bndiff.shape), "bncap_bnvar_inv", tuple(bncap_bnvar_inv.shape), "bncap_bnraw", tuple(bncap_bnraw.shape))


In [ ]:
# Exercise 136: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff_direct` — the direct `(4, 3)` centered-value contribution.
# Define `bncap_dbnvar_inv` — the `(1, 3)` inverse-standard-deviation gradient.
# Predict every visible tensor shape before computing values.
# Define `ex136_bncap_bnraw_shape` — the shape of `bncap_bnraw` as a literal Python tuple.
# Define `ex136_bncap_bndiff_shape` — the shape of `bncap_bndiff` as a literal Python tuple.
# Define `ex136_bncap_bnvar_inv_shape` — the shape of `bncap_bnvar_inv` as a literal Python tuple.
# Define `ex136_bncap_dbndiff_direct_shape` — the shape of `bncap_dbndiff_direct` as a literal Python tuple.
# Define `ex136_bncap_dbnvar_inv_shape` — the shape of `bncap_dbnvar_inv` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex136_bncap_bnraw_shape", "bncap_bnraw")
_check_visible_tensor_shape("ex136_bncap_bndiff_shape", "bncap_bndiff")
_check_visible_tensor_shape("ex136_bncap_bnvar_inv_shape", "bncap_bnvar_inv")
_check_private_tensor_shape("ex136_bncap_dbndiff_direct_shape", "batchnorm_capstone", "dbndiff_direct")
_check_private_tensor_shape("ex136_bncap_dbnvar_inv_shape", "batchnorm_capstone", "dbnvar_inv")
_check_tensor("bncap_dbndiff_direct", "batchnorm_capstone", "dbndiff_direct")
_check_tensor("bncap_dbnvar_inv", "batchnorm_capstone", "dbnvar_inv")


### Exercise 137 — Full BatchNorm: inverse square root

**Purpose:** Learn to pass the inverse-standard-deviation gradient back to variance.

**Inputs:** `bncap_bnvar` and `bncap_bnvar_inv` are `(1, 3)`.

**Forward operation:** `bncap_bnvar_inv = (bncap_bnvar + eps)**-0.5`.

**Derive/do:** Using `bncap_dbnvar_inv`, derive `bncap_dbnvar`.

**Ingredients:** The fixed-power rule with exponent `-0.5`; epsilon is constant.

**Required outputs:**

- `bncap_dbnvar`: the `(1, 3)` variance gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex137_bncap_bnvar_inv_shape`: predict the shape of `bncap_bnvar_inv` as a literal Python tuple
- `ex137_bncap_bnvar_shape`: predict the shape of `bncap_bnvar` as a literal Python tuple
- `ex137_bncap_dbnvar_shape`: predict the shape of `bncap_dbnvar` as a literal Python tuple

**Next concept:** Full BatchNorm: variance reduction.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "batchnorm_capstone" in _REFS

# Forward relation under study:
# bncap_bnvar_inv = (bncap_bnvar + bncap_eps) ** -0.5
# Visible forward tensors and shapes:
# - `bncap_bnvar` is already defined by the visible forward graph.
# - `bncap_bnvar_inv` is already defined by the visible forward graph.
# Upstream gradient for this stage: bncap_dbnvar_inv from Exercise 136.
print("bncap_bnvar", tuple(bncap_bnvar.shape), "bncap_bnvar_inv", tuple(bncap_bnvar_inv.shape))


In [ ]:
# Exercise 137: derive manually; do not use autograd in this cell.
# Define `bncap_dbnvar` — the `(1, 3)` variance gradient.
# Predict every visible tensor shape before computing values.
# Define `ex137_bncap_bnvar_inv_shape` — the shape of `bncap_bnvar_inv` as a literal Python tuple.
# Define `ex137_bncap_bnvar_shape` — the shape of `bncap_bnvar` as a literal Python tuple.
# Define `ex137_bncap_dbnvar_shape` — the shape of `bncap_dbnvar` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex137_bncap_bnvar_inv_shape", "bncap_bnvar_inv")
_check_visible_tensor_shape("ex137_bncap_bnvar_shape", "bncap_bnvar")
_check_private_tensor_shape("ex137_bncap_dbnvar_shape", "batchnorm_capstone", "dbnvar")
_check_tensor("bncap_dbnvar", "batchnorm_capstone", "dbnvar")


### Exercise 138 — Full BatchNorm: variance reduction

**Purpose:** Learn to spread each variance gradient back across all squared deviations for that hidden neuron.

**Inputs:** `bncap_bndiff2` is `(4, 3)` and `bncap_bnvar` is `(1, 3)`.

**Forward operation:** `bncap_bnvar = bncap_bndiff2.sum(dim=0, keepdim=True) / (bncap_n - 1)`.

**Derive/do:** Derive `bncap_dbndiff2`.

**Ingredients:** Broadcast across examples and include Bessel's divisor `bncap_n - 1`.

**Required outputs:**

- `bncap_dbndiff2`: the `(4, 3)` squared-deviation gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex138_bncap_bnvar_shape`: predict the shape of `bncap_bnvar` as a literal Python tuple
- `ex138_bncap_bndiff2_shape`: predict the shape of `bncap_bndiff2` as a literal Python tuple
- `ex138_bncap_dbndiff2_shape`: predict the shape of `bncap_dbndiff2` as a literal Python tuple

**Next concept:** Full BatchNorm: square backward.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "batchnorm_capstone" in _REFS

# Forward relation under study:
# bncap_bnvar = bncap_bndiff2.sum(dim=0, keepdim=True) / (bncap_n - 1)
# Visible forward tensors and shapes:
# - `bncap_bndiff2` is already defined by the visible forward graph.
# - `bncap_bnvar` is already defined by the visible forward graph.
# Upstream gradient for this stage: bncap_dbnvar from Exercise 137.
print("bncap_bndiff2", tuple(bncap_bndiff2.shape), "bncap_bnvar", tuple(bncap_bnvar.shape))


In [ ]:
# Exercise 138: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff2` — the `(4, 3)` squared-deviation gradient.
# Predict every visible tensor shape before computing values.
# Define `ex138_bncap_bnvar_shape` — the shape of `bncap_bnvar` as a literal Python tuple.
# Define `ex138_bncap_bndiff2_shape` — the shape of `bncap_bndiff2` as a literal Python tuple.
# Define `ex138_bncap_dbndiff2_shape` — the shape of `bncap_dbndiff2` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex138_bncap_bnvar_shape", "bncap_bnvar")
_check_visible_tensor_shape("ex138_bncap_bndiff2_shape", "bncap_bndiff2")
_check_private_tensor_shape("ex138_bncap_dbndiff2_shape", "batchnorm_capstone", "dbndiff2")
_check_tensor("bncap_dbndiff2", "batchnorm_capstone", "dbndiff2")


### Exercise 139 — Full BatchNorm: square backward

**Purpose:** Learn to turn squared-deviation gradients into the variance branch's centered-value gradients.

**Inputs:** `bncap_bndiff` and `bncap_bndiff2` are `(4, 3)`.

**Forward operation:** `bncap_bndiff2 = bncap_bndiff**2`.

**Derive/do:** Using `bncap_dbndiff2`, derive `bncap_dbndiff_variance`.

**Ingredients:** The square local derivative; this is only the variance-path contribution.

**Required outputs:**

- `bncap_dbndiff_variance`: the variance-path contribution to centered values.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex139_bncap_bndiff2_shape`: predict the shape of `bncap_bndiff2` as a literal Python tuple
- `ex139_bncap_bndiff_shape`: predict the shape of `bncap_bndiff` as a literal Python tuple
- `ex139_bncap_dbndiff_variance_shape`: predict the shape of `bncap_dbndiff_variance` as a literal Python tuple

**Next concept:** Full BatchNorm: merge centered-value paths.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "batchnorm_capstone" in _REFS

# Forward relation under study:
# bncap_bndiff2 = bncap_bndiff**2
# Visible forward tensors and shapes:
# - `bncap_bndiff` is already defined by the visible forward graph.
# - `bncap_bndiff2` is already defined by the visible forward graph.
# Upstream gradient for this stage: bncap_dbndiff2 from Exercise 138.
print("bncap_bndiff", tuple(bncap_bndiff.shape), "bncap_bndiff2", tuple(bncap_bndiff2.shape))


In [ ]:
# Exercise 139: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff_variance` — the variance-path contribution to centered values.
# Predict every visible tensor shape before computing values.
# Define `ex139_bncap_bndiff2_shape` — the shape of `bncap_bndiff2` as a literal Python tuple.
# Define `ex139_bncap_bndiff_shape` — the shape of `bncap_bndiff` as a literal Python tuple.
# Define `ex139_bncap_dbndiff_variance_shape` — the shape of `bncap_dbndiff_variance` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex139_bncap_bndiff2_shape", "bncap_bndiff2")
_check_visible_tensor_shape("ex139_bncap_bndiff_shape", "bncap_bndiff")
_check_private_tensor_shape("ex139_bncap_dbndiff_variance_shape", "batchnorm_capstone", "dbndiff_variance")
_check_tensor("bncap_dbndiff_variance", "batchnorm_capstone", "dbndiff_variance")


### Exercise 140 — Full BatchNorm: merge centered-value paths

**Purpose:** Learn to add the direct normalization path and variance path when they meet at centered values.

**Inputs:** Both centered-value contributions have shape `(4, 3)`.

**Forward operation:** `bncap_bndiff` feeds both `bnraw` directly and the variance branch.

**Derive/do:** Derive total `bncap_dbndiff` by combining your two path gradients.

**Ingredients:** Fan-out backward adds contributions where branches meet.

**Required outputs:**

- `bncap_dbndiff`: the total `(4, 3)` centered-value gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex140_bncap_bndiff_shape`: predict the shape of `bncap_bndiff` as a literal Python tuple
- `ex140_bncap_bnraw_shape`: predict the shape of `bncap_bnraw` as a literal Python tuple
- `ex140_bncap_bnvar_shape`: predict the shape of `bncap_bnvar` as a literal Python tuple
- `ex140_bncap_dbndiff_shape`: predict the shape of `bncap_dbndiff` as a literal Python tuple

**Next concept:** Full BatchNorm: centering and mean.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "batchnorm_capstone" in _REFS

# Forward relation under study:
# bncap_bndiff feeds bncap_bnraw directly and bncap_bnvar through the square branch
# Visible forward tensors and shapes:
# - `bncap_bndiff` is already defined by the visible forward graph.
# - `bncap_bnraw` is already defined by the visible forward graph.
# - `bncap_bnvar` is already defined by the visible forward graph.
# Upstream gradient for this stage: bncap_dbndiff_direct from Exercise 136 and bncap_dbndiff_variance from Exercise 139.
print("bncap_bndiff", tuple(bncap_bndiff.shape), "bncap_bnraw", tuple(bncap_bnraw.shape), "bncap_bnvar", tuple(bncap_bnvar.shape))


In [ ]:
# Exercise 140: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff` — the total `(4, 3)` centered-value gradient.
# Predict every visible tensor shape before computing values.
# Define `ex140_bncap_bndiff_shape` — the shape of `bncap_bndiff` as a literal Python tuple.
# Define `ex140_bncap_bnraw_shape` — the shape of `bncap_bnraw` as a literal Python tuple.
# Define `ex140_bncap_bnvar_shape` — the shape of `bncap_bnvar` as a literal Python tuple.
# Define `ex140_bncap_dbndiff_shape` — the shape of `bncap_dbndiff` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex140_bncap_bndiff_shape", "bncap_bndiff")
_check_visible_tensor_shape("ex140_bncap_bnraw_shape", "bncap_bnraw")
_check_visible_tensor_shape("ex140_bncap_bnvar_shape", "bncap_bnvar")
_check_private_tensor_shape("ex140_bncap_dbndiff_shape", "batchnorm_capstone", "dbndiff")
_check_tensor("bncap_dbndiff", "batchnorm_capstone", "dbndiff")


### Exercise 141 — Full BatchNorm: centering and mean

**Purpose:** Learn to finish BatchNorm by returning through mean subtraction and the mean computed from the same inputs.

**Inputs:** `bncap_bnmeani` is `(1, 3)` and `bncap_hprebn` is `(4, 3)`.

**Forward operation:** `bncap_bndiff = bncap_hprebn - bncap_bnmeani` and `bncap_bnmeani = bncap_hprebn.mean(dim=0, keepdim=True)`.

**Derive/do:** Derive `bncap_dbnmeani` and total `bncap_dhprebn`.

**Ingredients:** Unbroadcast subtraction into the mean, reverse the four-example mean, then add its contribution to the direct path.

**Required outputs:**

- `bncap_dbnmeani`: the `(1, 3)` mean gradient.
- `bncap_dhprebn`: the final `(4, 3)` pre-BatchNorm gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex141_bncap_bnmeani_shape`: predict the shape of `bncap_bnmeani` as a literal Python tuple
- `ex141_bncap_hprebn_shape`: predict the shape of `bncap_hprebn` as a literal Python tuple
- `ex141_bncap_bndiff_shape`: predict the shape of `bncap_bndiff` as a literal Python tuple
- `ex141_bncap_dbnmeani_shape`: predict the shape of `bncap_dbnmeani` as a literal Python tuple
- `ex141_bncap_dhprebn_shape`: predict the shape of `bncap_dhprebn` as a literal Python tuple

**Next concept:** MLP forward shape audit.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "batchnorm_capstone" in _REFS

# Forward relation under study:
# bncap_bnmeani = bncap_hprebn.mean(dim=0, keepdim=True); bncap_bndiff = bncap_hprebn - bncap_bnmeani
# Visible forward tensors and shapes:
# - `bncap_hprebn` is already defined by the visible forward graph.
# - `bncap_bnmeani` is already defined by the visible forward graph.
# - `bncap_bndiff` is already defined by the visible forward graph.
# Upstream gradient for this stage: bncap_dbndiff from Exercise 140.
print("bncap_hprebn", tuple(bncap_hprebn.shape), "bncap_bnmeani", tuple(bncap_bnmeani.shape), "bncap_bndiff", tuple(bncap_bndiff.shape))


In [ ]:
# Exercise 141: derive manually; do not use autograd in this cell.
# Define `bncap_dbnmeani` — the `(1, 3)` mean gradient.
# Define `bncap_dhprebn` — the final `(4, 3)` pre-BatchNorm gradient.
# Predict every visible tensor shape before computing values.
# Define `ex141_bncap_bnmeani_shape` — the shape of `bncap_bnmeani` as a literal Python tuple.
# Define `ex141_bncap_hprebn_shape` — the shape of `bncap_hprebn` as a literal Python tuple.
# Define `ex141_bncap_bndiff_shape` — the shape of `bncap_bndiff` as a literal Python tuple.
# Define `ex141_bncap_dbnmeani_shape` — the shape of `bncap_dbnmeani` as a literal Python tuple.
# Define `ex141_bncap_dhprebn_shape` — the shape of `bncap_dhprebn` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex141_bncap_bnmeani_shape", "bncap_bnmeani")
_check_visible_tensor_shape("ex141_bncap_hprebn_shape", "bncap_hprebn")
_check_visible_tensor_shape("ex141_bncap_bndiff_shape", "bncap_bndiff")
_check_private_tensor_shape("ex141_bncap_dbnmeani_shape", "batchnorm_capstone", "dbnmeani")
_check_private_tensor_shape("ex141_bncap_dhprebn_shape", "batchnorm_capstone", "dhprebn")
_check_tensor("bncap_dbnmeani", "batchnorm_capstone", "dbnmeani")
_check_tensor("bncap_dhprebn", "batchnorm_capstone", "dhprebn")


## 10. Capstone: manually backpropagate a next-character MLP

The final graph mirrors the lecture at a deliberately small scale. It uses four training examples, context length three, embedding size two, five hidden neurons, and six vocabulary candidates.

Forward path:

1. look up three embeddings per example;
2. flatten each context;
3. apply a hidden affine layer;
4. apply BatchNorm;
5. apply tanh;
6. apply an output affine layer;
7. compute stable softmax;
8. select each example's expected target;
9. average all four negative log-likelihoods.

The loss scores the complete supplied four-example mini-batch. Each row is a training example, each output column is a vocabulary candidate, and only the indexed expected target probability contributes directly to that row's loss. The other candidates influence it through softmax normalization.

Do not call autograd in answer cells. Work backward one named forward tensor at a time and keep a shape ledger beside you.


### Exercise 142 — MLP forward shape audit

**Purpose:** Learn to name the meaning and size of every tensor axis in the complete MLP before deriving gradients.

**Inputs:** The supplied fixture defines the complete tiny MLP and all dimension constants.

**Forward operation:** Read the forward pass without using stored reference shapes.

**Derive/do:** Write a Python tuple for every requested intermediate shape.

**Ingredients:** Track examples, context positions, embedding features, hidden neurons, and vocabulary candidates independently.

**Required outputs:**



**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex142_mlp_Xb_shape`: predict the shape of `mlp_Xb` as a literal Python tuple
- `ex142_mlp_Yb_shape`: predict the shape of `mlp_Yb` as a literal Python tuple
- `ex142_mlp_C_shape`: predict the shape of `mlp_C` as a literal Python tuple
- `ex142_mlp_W1_shape`: predict the shape of `mlp_W1` as a literal Python tuple
- `ex142_mlp_b1_shape`: predict the shape of `mlp_b1` as a literal Python tuple
- `ex142_mlp_bngain_shape`: predict the shape of `mlp_bngain` as a literal Python tuple
- `ex142_mlp_bnbias_shape`: predict the shape of `mlp_bnbias` as a literal Python tuple
- `ex142_mlp_W2_shape`: predict the shape of `mlp_W2` as a literal Python tuple
- `ex142_mlp_b2_shape`: predict the shape of `mlp_b2` as a literal Python tuple
- `ex142_mlp_emb_shape`: predict the shape of `mlp_emb` as a literal Python tuple
- `ex142_mlp_embcat_shape`: predict the shape of `mlp_embcat` as a literal Python tuple
- `ex142_mlp_hprebn_shape`: predict the shape of `mlp_hprebn` as a literal Python tuple
- `ex142_mlp_bnmeani_shape`: predict the shape of `mlp_bnmeani` as a literal Python tuple
- `ex142_mlp_bndiff_shape`: predict the shape of `mlp_bndiff` as a literal Python tuple
- `ex142_mlp_bndiff2_shape`: predict the shape of `mlp_bndiff2` as a literal Python tuple
- `ex142_mlp_bnvar_shape`: predict the shape of `mlp_bnvar` as a literal Python tuple
- `ex142_mlp_bnvar_inv_shape`: predict the shape of `mlp_bnvar_inv` as a literal Python tuple
- `ex142_mlp_bnraw_shape`: predict the shape of `mlp_bnraw` as a literal Python tuple
- `ex142_mlp_hpreact_shape`: predict the shape of `mlp_hpreact` as a literal Python tuple
- `ex142_mlp_h_shape`: predict the shape of `mlp_h` as a literal Python tuple
- `ex142_mlp_logits_shape`: predict the shape of `mlp_logits` as a literal Python tuple
- `ex142_mlp_logit_maxes_shape`: predict the shape of `mlp_logit_maxes` as a literal Python tuple
- `ex142_mlp_norm_logits_shape`: predict the shape of `mlp_norm_logits` as a literal Python tuple
- `ex142_mlp_counts_shape`: predict the shape of `mlp_counts` as a literal Python tuple
- `ex142_mlp_counts_sum_shape`: predict the shape of `mlp_counts_sum` as a literal Python tuple
- `ex142_mlp_counts_sum_inv_shape`: predict the shape of `mlp_counts_sum_inv` as a literal Python tuple
- `ex142_mlp_probs_shape`: predict the shape of `mlp_probs` as a literal Python tuple
- `ex142_mlp_logprobs_shape`: predict the shape of `mlp_logprobs` as a literal Python tuple
- `ex142_mlp_loss_shape`: predict the shape of `mlp_loss` as a literal Python tuple

**Next concept:** MLP: mean NLL seed.


In [ ]:
# Supplied visible fixture: forward tensors and operations are shown; expected gradients remain private.
# Supplied tiny next-character MLP dimensions.
mlp_B = 4
mlp_BLOCK = 3
mlp_E = 2
mlp_H = 5
mlp_V = 6
mlp_eps = 1e-5

# Four complete training examples; repeated IDs make embedding accumulation observable.
mlp_Xb = torch.tensor([
    [0, 1, 2],
    [1, 2, 1],
    [3, 1, 0],
    [2, 4, 1],
], dtype=torch.long)
mlp_Yb = torch.tensor([2, 3, 1, 5], dtype=torch.long)

# Deterministic parameters; no training update occurs in this notebook.
mlp_g = torch.Generator().manual_seed(2147483647)
mlp_C = (torch.randn((mlp_V, mlp_E), generator=mlp_g, dtype=DTYPE) * 0.4).requires_grad_()
mlp_W1 = (torch.randn((mlp_BLOCK * mlp_E, mlp_H), generator=mlp_g, dtype=DTYPE) * 0.3).requires_grad_()
mlp_b1 = (torch.randn(mlp_H, generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()
mlp_bngain = (1.0 + torch.randn((1, mlp_H), generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()
mlp_bnbias = (torch.randn((1, mlp_H), generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()
mlp_W2 = (torch.randn((mlp_H, mlp_V), generator=mlp_g, dtype=DTYPE) * 0.2).requires_grad_()
mlp_b2 = (torch.randn(mlp_V, generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()

# Embedding and hidden affine layer.
mlp_emb = mlp_C[mlp_Xb]
mlp_embcat = mlp_emb.reshape(mlp_B, mlp_BLOCK * mlp_E)
mlp_hprebn = mlp_embcat @ mlp_W1 + mlp_b1

# Expanded BatchNorm across the four training-example rows.
mlp_bnmeani = mlp_hprebn.mean(dim=0, keepdim=True)
mlp_bndiff = mlp_hprebn - mlp_bnmeani
mlp_bndiff2 = mlp_bndiff**2
mlp_bnvar = mlp_bndiff2.sum(dim=0, keepdim=True) / (mlp_B - 1)
mlp_bnvar_inv = (mlp_bnvar + mlp_eps) ** -0.5
mlp_bnraw = mlp_bndiff * mlp_bnvar_inv
mlp_hpreact = mlp_bngain * mlp_bnraw + mlp_bnbias
mlp_h = mlp_hpreact.tanh()

# Output layer and expanded stable softmax cross-entropy.
mlp_logits = mlp_h @ mlp_W2 + mlp_b2
mlp_logit_maxes = mlp_logits.max(dim=1, keepdim=True).values
mlp_norm_logits = mlp_logits - mlp_logit_maxes
mlp_counts = mlp_norm_logits.exp()
mlp_counts_sum = mlp_counts.sum(dim=1, keepdim=True)
mlp_counts_sum_inv = mlp_counts_sum**-1
mlp_probs = mlp_counts * mlp_counts_sum_inv
mlp_logprobs = mlp_probs.log()
mlp_loss = -mlp_logprobs[range(mlp_B), mlp_Yb].mean()

_capture(
    "mlp_capstone",
    mlp_loss,
    {
        "dlogprobs": mlp_logprobs,
        "dprobs": mlp_probs,
        "dcounts_sum_inv": mlp_counts_sum_inv,
        "dcounts_sum": mlp_counts_sum,
        "dcounts": mlp_counts,
        "dnorm_logits": mlp_norm_logits,
        "dlogit_maxes": mlp_logit_maxes,
        "dlogits": mlp_logits,
        "dh": mlp_h,
        "dW2": mlp_W2,
        "db2": mlp_b2,
        "dhpreact": mlp_hpreact,
        "dbngain": mlp_bngain,
        "dbnbias": mlp_bnbias,
        "dbnraw": mlp_bnraw,
        "dbnvar_inv": mlp_bnvar_inv,
        "dbnvar": mlp_bnvar,
        "dbndiff2": mlp_bndiff2,
        "dbndiff": mlp_bndiff,
        "dbnmeani": mlp_bnmeani,
        "dhprebn": mlp_hprebn,
        "dembcat": mlp_embcat,
        "dW1": mlp_W1,
        "db1": mlp_b1,
        "demb": mlp_emb,
        "dC": mlp_C,
    },
    retain_graph=True,
)
for mlp_name, mlp_tensor in {
    "emb_shape": mlp_emb,
    "embcat_shape": mlp_embcat,
    "hprebn_shape": mlp_hprebn,
    "bnmeani_shape": mlp_bnmeani,
    "bnvar_shape": mlp_bnvar,
    "bnraw_shape": mlp_bnraw,
    "hpreact_shape": mlp_hpreact,
    "h_shape": mlp_h,
    "logits_shape": mlp_logits,
    "counts_sum_shape": mlp_counts_sum,
    "probs_shape": mlp_probs,
    "logprobs_shape": mlp_logprobs,
    "loss_shape": mlp_loss,
}.items():
    _store_shape("mlp_capstone", mlp_name, mlp_tensor)
print("MLP capstone ready:", tuple(mlp_Xb.shape), "-> loss", tuple(mlp_loss.shape))
# Private staged references for paths that merge later.
_capture_product_path(
    "mlp_capstone",
    "dcounts_direct",
    mlp_counts,
    mlp_counts_sum_inv,
    _REFS["mlp_capstone"]["dprobs"],
)
_capture_product_path(
    "mlp_capstone",
    "dbndiff_direct",
    mlp_bndiff,
    mlp_bnvar_inv,
    _REFS["mlp_capstone"]["dbnraw"],
)
_capture_path(
    "mlp_capstone",
    "dbndiff_variance",
    mlp_bnvar,
    mlp_bndiff,
    _REFS["mlp_capstone"]["dbnvar"],
)
_REFS["mlp_capstone"]["logit_row_grad_sums"] = (
    _REFS["mlp_capstone"]["dlogits"].sum(dim=1)
).detach().clone()
_REFS["mlp_capstone"]["hprebn_column_grad_sums"] = (
    _REFS["mlp_capstone"]["dhprebn"].sum(dim=0)
).detach().clone()


In [ ]:
# Exercise 142: derive manually; do not use autograd in this cell.
# Predict every visible tensor shape before computing values.
# Define `ex142_mlp_Xb_shape` — the shape of `mlp_Xb` as a literal Python tuple.
# Define `ex142_mlp_Yb_shape` — the shape of `mlp_Yb` as a literal Python tuple.
# Define `ex142_mlp_C_shape` — the shape of `mlp_C` as a literal Python tuple.
# Define `ex142_mlp_W1_shape` — the shape of `mlp_W1` as a literal Python tuple.
# Define `ex142_mlp_b1_shape` — the shape of `mlp_b1` as a literal Python tuple.
# Define `ex142_mlp_bngain_shape` — the shape of `mlp_bngain` as a literal Python tuple.
# Define `ex142_mlp_bnbias_shape` — the shape of `mlp_bnbias` as a literal Python tuple.
# Define `ex142_mlp_W2_shape` — the shape of `mlp_W2` as a literal Python tuple.
# Define `ex142_mlp_b2_shape` — the shape of `mlp_b2` as a literal Python tuple.
# Define `ex142_mlp_emb_shape` — the shape of `mlp_emb` as a literal Python tuple.
# Define `ex142_mlp_embcat_shape` — the shape of `mlp_embcat` as a literal Python tuple.
# Define `ex142_mlp_hprebn_shape` — the shape of `mlp_hprebn` as a literal Python tuple.
# Define `ex142_mlp_bnmeani_shape` — the shape of `mlp_bnmeani` as a literal Python tuple.
# Define `ex142_mlp_bndiff_shape` — the shape of `mlp_bndiff` as a literal Python tuple.
# Define `ex142_mlp_bndiff2_shape` — the shape of `mlp_bndiff2` as a literal Python tuple.
# Define `ex142_mlp_bnvar_shape` — the shape of `mlp_bnvar` as a literal Python tuple.
# Define `ex142_mlp_bnvar_inv_shape` — the shape of `mlp_bnvar_inv` as a literal Python tuple.
# Define `ex142_mlp_bnraw_shape` — the shape of `mlp_bnraw` as a literal Python tuple.
# Define `ex142_mlp_hpreact_shape` — the shape of `mlp_hpreact` as a literal Python tuple.
# Define `ex142_mlp_h_shape` — the shape of `mlp_h` as a literal Python tuple.
# Define `ex142_mlp_logits_shape` — the shape of `mlp_logits` as a literal Python tuple.
# Define `ex142_mlp_logit_maxes_shape` — the shape of `mlp_logit_maxes` as a literal Python tuple.
# Define `ex142_mlp_norm_logits_shape` — the shape of `mlp_norm_logits` as a literal Python tuple.
# Define `ex142_mlp_counts_shape` — the shape of `mlp_counts` as a literal Python tuple.
# Define `ex142_mlp_counts_sum_shape` — the shape of `mlp_counts_sum` as a literal Python tuple.
# Define `ex142_mlp_counts_sum_inv_shape` — the shape of `mlp_counts_sum_inv` as a literal Python tuple.
# Define `ex142_mlp_probs_shape` — the shape of `mlp_probs` as a literal Python tuple.
# Define `ex142_mlp_logprobs_shape` — the shape of `mlp_logprobs` as a literal Python tuple.
# Define `ex142_mlp_loss_shape` — the shape of `mlp_loss` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex142_mlp_Xb_shape", "mlp_Xb")
_check_visible_tensor_shape("ex142_mlp_Yb_shape", "mlp_Yb")
_check_visible_tensor_shape("ex142_mlp_C_shape", "mlp_C")
_check_visible_tensor_shape("ex142_mlp_W1_shape", "mlp_W1")
_check_visible_tensor_shape("ex142_mlp_b1_shape", "mlp_b1")
_check_visible_tensor_shape("ex142_mlp_bngain_shape", "mlp_bngain")
_check_visible_tensor_shape("ex142_mlp_bnbias_shape", "mlp_bnbias")
_check_visible_tensor_shape("ex142_mlp_W2_shape", "mlp_W2")
_check_visible_tensor_shape("ex142_mlp_b2_shape", "mlp_b2")
_check_visible_tensor_shape("ex142_mlp_emb_shape", "mlp_emb")
_check_visible_tensor_shape("ex142_mlp_embcat_shape", "mlp_embcat")
_check_visible_tensor_shape("ex142_mlp_hprebn_shape", "mlp_hprebn")
_check_visible_tensor_shape("ex142_mlp_bnmeani_shape", "mlp_bnmeani")
_check_visible_tensor_shape("ex142_mlp_bndiff_shape", "mlp_bndiff")
_check_visible_tensor_shape("ex142_mlp_bndiff2_shape", "mlp_bndiff2")
_check_visible_tensor_shape("ex142_mlp_bnvar_shape", "mlp_bnvar")
_check_visible_tensor_shape("ex142_mlp_bnvar_inv_shape", "mlp_bnvar_inv")
_check_visible_tensor_shape("ex142_mlp_bnraw_shape", "mlp_bnraw")
_check_visible_tensor_shape("ex142_mlp_hpreact_shape", "mlp_hpreact")
_check_visible_tensor_shape("ex142_mlp_h_shape", "mlp_h")
_check_visible_tensor_shape("ex142_mlp_logits_shape", "mlp_logits")
_check_visible_tensor_shape("ex142_mlp_logit_maxes_shape", "mlp_logit_maxes")
_check_visible_tensor_shape("ex142_mlp_norm_logits_shape", "mlp_norm_logits")
_check_visible_tensor_shape("ex142_mlp_counts_shape", "mlp_counts")
_check_visible_tensor_shape("ex142_mlp_counts_sum_shape", "mlp_counts_sum")
_check_visible_tensor_shape("ex142_mlp_counts_sum_inv_shape", "mlp_counts_sum_inv")
_check_visible_tensor_shape("ex142_mlp_probs_shape", "mlp_probs")
_check_visible_tensor_shape("ex142_mlp_logprobs_shape", "mlp_logprobs")
_check_visible_tensor_shape("ex142_mlp_loss_shape", "mlp_loss")


### Exercise 143 — MLP: mean NLL seed

**Purpose:** Learn to start the MLP backward pass with one direct target gradient per example.

**Inputs:** `mlp_logprobs` is `(4, 6)`, with four examples and six candidates; `mlp_Yb` has one target per example.

**Forward operation:** `mlp_loss = -mlp_logprobs[range(mlp_B), mlp_Yb].mean()`.

**Derive/do:** Derive `mlp_dlogprobs`.

**Ingredients:** Zeros everywhere except paired targets; account for negation and the four-example average.

**Required outputs:**

- `mlp_dlogprobs`: the sparse `(4, 6)` gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex143_mlp_loss_shape`: predict the shape of `mlp_loss` as a literal Python tuple
- `ex143_mlp_logprobs_shape`: predict the shape of `mlp_logprobs` as a literal Python tuple
- `ex143_mlp_Yb_shape`: predict the shape of `mlp_Yb` as a literal Python tuple
- `ex143_mlp_dlogprobs_shape`: predict the shape of `mlp_dlogprobs` as a literal Python tuple

**Next concept:** MLP: log backward.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_loss = -mlp_logprobs[range(mlp_B), mlp_Yb].mean()
# Visible forward tensors and shapes:
# - `mlp_logprobs` is already defined by the visible forward graph.
# - `mlp_Yb` is already defined by the visible forward graph.
# - `mlp_loss` is already defined by the visible forward graph.
# Upstream gradient for this stage: the scalar loss seed 1.
print("mlp_logprobs", tuple(mlp_logprobs.shape), "mlp_Yb", tuple(mlp_Yb.shape), "mlp_loss", tuple(mlp_loss.shape))


In [ ]:
# Exercise 143: derive manually; do not use autograd in this cell.
# Define `mlp_dlogprobs` — the sparse `(4, 6)` gradient.
# Predict every visible tensor shape before computing values.
# Define `ex143_mlp_loss_shape` — the shape of `mlp_loss` as a literal Python tuple.
# Define `ex143_mlp_logprobs_shape` — the shape of `mlp_logprobs` as a literal Python tuple.
# Define `ex143_mlp_Yb_shape` — the shape of `mlp_Yb` as a literal Python tuple.
# Define `ex143_mlp_dlogprobs_shape` — the shape of `mlp_dlogprobs` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex143_mlp_loss_shape", "mlp_loss")
_check_visible_tensor_shape("ex143_mlp_logprobs_shape", "mlp_logprobs")
_check_visible_tensor_shape("ex143_mlp_Yb_shape", "mlp_Yb")
_check_private_tensor_shape("ex143_mlp_dlogprobs_shape", "mlp_capstone", "dlogprobs")
_check_tensor("mlp_dlogprobs", "mlp_capstone", "dlogprobs")


### Exercise 144 — MLP: log backward

**Purpose:** Learn to pass the MLP's target gradients from log-probabilities back to probabilities.

**Inputs:** `mlp_probs` and `mlp_dlogprobs` are `(4, 6)`.

**Forward operation:** `mlp_logprobs = mlp_probs.log()`.

**Derive/do:** Derive `mlp_dprobs`.

**Ingredients:** Natural-log local derivative, elementwise.

**Required outputs:**

- `mlp_dprobs`: the `(4, 6)` probability gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex144_mlp_logprobs_shape`: predict the shape of `mlp_logprobs` as a literal Python tuple
- `ex144_mlp_probs_shape`: predict the shape of `mlp_probs` as a literal Python tuple
- `ex144_mlp_dprobs_shape`: predict the shape of `mlp_dprobs` as a literal Python tuple

**Next concept:** MLP: normalization multiplication.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_logprobs = mlp_probs.log()
# Visible forward tensors and shapes:
# - `mlp_probs` is already defined by the visible forward graph.
# - `mlp_logprobs` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dlogprobs from Exercise 143.
print("mlp_probs", tuple(mlp_probs.shape), "mlp_logprobs", tuple(mlp_logprobs.shape))


In [ ]:
# Exercise 144: derive manually; do not use autograd in this cell.
# Define `mlp_dprobs` — the `(4, 6)` probability gradient.
# Predict every visible tensor shape before computing values.
# Define `ex144_mlp_logprobs_shape` — the shape of `mlp_logprobs` as a literal Python tuple.
# Define `ex144_mlp_probs_shape` — the shape of `mlp_probs` as a literal Python tuple.
# Define `ex144_mlp_dprobs_shape` — the shape of `mlp_dprobs` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex144_mlp_logprobs_shape", "mlp_logprobs")
_check_visible_tensor_shape("ex144_mlp_probs_shape", "mlp_probs")
_check_private_tensor_shape("ex144_mlp_dprobs_shape", "mlp_capstone", "dprobs")
_check_tensor("mlp_dprobs", "mlp_capstone", "dprobs")


### Exercise 145 — MLP: normalization multiplication

**Purpose:** Learn to split each probability gradient between its candidate weight and its shared row denominator.

**Inputs:** `mlp_counts` is `(4, 6)` and `mlp_counts_sum_inv` is `(4, 1)`.

**Forward operation:** `mlp_probs = mlp_counts * mlp_counts_sum_inv`.

**Derive/do:** Derive direct `mlp_dcounts_direct` and `mlp_dcounts_sum_inv`.

**Ingredients:** Product backward; accumulate the broadcast denominator factor over six candidates.

**Required outputs:**

- `mlp_dcounts_direct`: the direct `(4, 6)` numerator contribution.
- `mlp_dcounts_sum_inv`: the `(4, 1)` reciprocal-denominator gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex145_mlp_probs_shape`: predict the shape of `mlp_probs` as a literal Python tuple
- `ex145_mlp_counts_shape`: predict the shape of `mlp_counts` as a literal Python tuple
- `ex145_mlp_counts_sum_inv_shape`: predict the shape of `mlp_counts_sum_inv` as a literal Python tuple
- `ex145_mlp_dcounts_direct_shape`: predict the shape of `mlp_dcounts_direct` as a literal Python tuple
- `ex145_mlp_dcounts_sum_inv_shape`: predict the shape of `mlp_dcounts_sum_inv` as a literal Python tuple

**Next concept:** MLP: denominator branch and count merge.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_probs = mlp_counts * mlp_counts_sum_inv
# Visible forward tensors and shapes:
# - `mlp_counts` is already defined by the visible forward graph.
# - `mlp_counts_sum_inv` is already defined by the visible forward graph.
# - `mlp_probs` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dprobs from Exercise 144.
print("mlp_counts", tuple(mlp_counts.shape), "mlp_counts_sum_inv", tuple(mlp_counts_sum_inv.shape), "mlp_probs", tuple(mlp_probs.shape))


In [ ]:
# Exercise 145: derive manually; do not use autograd in this cell.
# Define `mlp_dcounts_direct` — the direct `(4, 6)` numerator contribution.
# Define `mlp_dcounts_sum_inv` — the `(4, 1)` reciprocal-denominator gradient.
# Predict every visible tensor shape before computing values.
# Define `ex145_mlp_probs_shape` — the shape of `mlp_probs` as a literal Python tuple.
# Define `ex145_mlp_counts_shape` — the shape of `mlp_counts` as a literal Python tuple.
# Define `ex145_mlp_counts_sum_inv_shape` — the shape of `mlp_counts_sum_inv` as a literal Python tuple.
# Define `ex145_mlp_dcounts_direct_shape` — the shape of `mlp_dcounts_direct` as a literal Python tuple.
# Define `ex145_mlp_dcounts_sum_inv_shape` — the shape of `mlp_dcounts_sum_inv` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex145_mlp_probs_shape", "mlp_probs")
_check_visible_tensor_shape("ex145_mlp_counts_shape", "mlp_counts")
_check_visible_tensor_shape("ex145_mlp_counts_sum_inv_shape", "mlp_counts_sum_inv")
_check_private_tensor_shape("ex145_mlp_dcounts_direct_shape", "mlp_capstone", "dcounts_direct")
_check_private_tensor_shape("ex145_mlp_dcounts_sum_inv_shape", "mlp_capstone", "dcounts_sum_inv")
_check_tensor("mlp_dcounts_direct", "mlp_capstone", "dcounts_direct")
_check_tensor("mlp_dcounts_sum_inv", "mlp_capstone", "dcounts_sum_inv")


### Exercise 146 — MLP: denominator branch and count merge

**Purpose:** Learn to reverse the reciprocal and row sum, then add the denominator path to the direct candidate-weight path.

**Inputs:** `mlp_counts_sum` is `(4, 1)` and `mlp_counts` is `(4, 6)`.

**Forward operation:** The denominator branch is reciprocal after a row-wise sum.

**Derive/do:** Derive `mlp_dcounts_sum` and total `mlp_dcounts`.

**Ingredients:** Reverse power `-1`, broadcast through the candidate sum, then add to the direct count contribution.

**Required outputs:**

- `mlp_dcounts_sum`: the `(4, 1)` row-total gradient.
- `mlp_dcounts`: the total `(4, 6)` count gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex146_mlp_counts_sum_shape`: predict the shape of `mlp_counts_sum` as a literal Python tuple
- `ex146_mlp_counts_shape`: predict the shape of `mlp_counts` as a literal Python tuple
- `ex146_mlp_counts_sum_inv_shape`: predict the shape of `mlp_counts_sum_inv` as a literal Python tuple
- `ex146_mlp_dcounts_sum_shape`: predict the shape of `mlp_dcounts_sum` as a literal Python tuple
- `ex146_mlp_dcounts_shape`: predict the shape of `mlp_dcounts` as a literal Python tuple

**Next concept:** MLP: exp and maximum shift.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_counts_sum = mlp_counts.sum(dim=1, keepdim=True); mlp_counts_sum_inv = mlp_counts_sum**-1
# Visible forward tensors and shapes:
# - `mlp_counts` is already defined by the visible forward graph.
# - `mlp_counts_sum` is already defined by the visible forward graph.
# - `mlp_counts_sum_inv` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dcounts_sum_inv and mlp_dcounts_direct from Exercise 145.
print("mlp_counts", tuple(mlp_counts.shape), "mlp_counts_sum", tuple(mlp_counts_sum.shape), "mlp_counts_sum_inv", tuple(mlp_counts_sum_inv.shape))


In [ ]:
# Exercise 146: derive manually; do not use autograd in this cell.
# Define `mlp_dcounts_sum` — the `(4, 1)` row-total gradient.
# Define `mlp_dcounts` — the total `(4, 6)` count gradient.
# Predict every visible tensor shape before computing values.
# Define `ex146_mlp_counts_sum_shape` — the shape of `mlp_counts_sum` as a literal Python tuple.
# Define `ex146_mlp_counts_shape` — the shape of `mlp_counts` as a literal Python tuple.
# Define `ex146_mlp_counts_sum_inv_shape` — the shape of `mlp_counts_sum_inv` as a literal Python tuple.
# Define `ex146_mlp_dcounts_sum_shape` — the shape of `mlp_dcounts_sum` as a literal Python tuple.
# Define `ex146_mlp_dcounts_shape` — the shape of `mlp_dcounts` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex146_mlp_counts_sum_shape", "mlp_counts_sum")
_check_visible_tensor_shape("ex146_mlp_counts_shape", "mlp_counts")
_check_visible_tensor_shape("ex146_mlp_counts_sum_inv_shape", "mlp_counts_sum_inv")
_check_private_tensor_shape("ex146_mlp_dcounts_sum_shape", "mlp_capstone", "dcounts_sum")
_check_private_tensor_shape("ex146_mlp_dcounts_shape", "mlp_capstone", "dcounts")
_check_tensor("mlp_dcounts_sum", "mlp_capstone", "dcounts_sum")
_check_tensor("mlp_dcounts", "mlp_capstone", "dcounts")


### Exercise 147 — MLP: exp and maximum shift

**Purpose:** Learn to return through exponentiation and maximum shifting to obtain gradients for every logit.

**Inputs:** Normalized logits and logits are `(4, 6)`; row maxima are `(4, 1)`.

**Forward operation:** Counts are exponentials of max-shifted logits.

**Derive/do:** Derive `mlp_dnorm_logits`, `mlp_dlogit_maxes`, and total `mlp_dlogits`.

**Ingredients:** Reverse exp; reverse broadcast subtraction; route max gradients to unique row argmax positions and add the direct path.

**Required outputs:**

- `mlp_dnorm_logits`: the normalized-logit gradient.
- `mlp_dlogit_maxes`: the row-maximum gradient.
- `mlp_dlogits`: the final logit gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex147_mlp_logit_maxes_shape`: predict the shape of `mlp_logit_maxes` as a literal Python tuple
- `ex147_mlp_logits_shape`: predict the shape of `mlp_logits` as a literal Python tuple
- `ex147_mlp_norm_logits_shape`: predict the shape of `mlp_norm_logits` as a literal Python tuple
- `ex147_mlp_counts_shape`: predict the shape of `mlp_counts` as a literal Python tuple
- `ex147_mlp_dnorm_logits_shape`: predict the shape of `mlp_dnorm_logits` as a literal Python tuple
- `ex147_mlp_dlogit_maxes_shape`: predict the shape of `mlp_dlogit_maxes` as a literal Python tuple
- `ex147_mlp_dlogits_shape`: predict the shape of `mlp_dlogits` as a literal Python tuple

**Next concept:** MLP: output affine layer.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_logit_maxes = mlp_logits.max(dim=1, keepdim=True).values; mlp_norm_logits = mlp_logits - mlp_logit_maxes; mlp_counts = mlp_norm_logits.exp()
# Visible forward tensors and shapes:
# - `mlp_logits` is already defined by the visible forward graph.
# - `mlp_logit_maxes` is already defined by the visible forward graph.
# - `mlp_norm_logits` is already defined by the visible forward graph.
# - `mlp_counts` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dcounts from Exercise 146.
print("mlp_logits", tuple(mlp_logits.shape), "mlp_logit_maxes", tuple(mlp_logit_maxes.shape), "mlp_norm_logits", tuple(mlp_norm_logits.shape), "mlp_counts", tuple(mlp_counts.shape))


In [ ]:
# Exercise 147: derive manually; do not use autograd in this cell.
# Define `mlp_dnorm_logits` — the normalized-logit gradient.
# Define `mlp_dlogit_maxes` — the row-maximum gradient.
# Define `mlp_dlogits` — the final logit gradient.
# Predict every visible tensor shape before computing values.
# Define `ex147_mlp_logit_maxes_shape` — the shape of `mlp_logit_maxes` as a literal Python tuple.
# Define `ex147_mlp_logits_shape` — the shape of `mlp_logits` as a literal Python tuple.
# Define `ex147_mlp_norm_logits_shape` — the shape of `mlp_norm_logits` as a literal Python tuple.
# Define `ex147_mlp_counts_shape` — the shape of `mlp_counts` as a literal Python tuple.
# Define `ex147_mlp_dnorm_logits_shape` — the shape of `mlp_dnorm_logits` as a literal Python tuple.
# Define `ex147_mlp_dlogit_maxes_shape` — the shape of `mlp_dlogit_maxes` as a literal Python tuple.
# Define `ex147_mlp_dlogits_shape` — the shape of `mlp_dlogits` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex147_mlp_logit_maxes_shape", "mlp_logit_maxes")
_check_visible_tensor_shape("ex147_mlp_logits_shape", "mlp_logits")
_check_visible_tensor_shape("ex147_mlp_norm_logits_shape", "mlp_norm_logits")
_check_visible_tensor_shape("ex147_mlp_counts_shape", "mlp_counts")
_check_private_tensor_shape("ex147_mlp_dnorm_logits_shape", "mlp_capstone", "dnorm_logits")
_check_private_tensor_shape("ex147_mlp_dlogit_maxes_shape", "mlp_capstone", "dlogit_maxes")
_check_private_tensor_shape("ex147_mlp_dlogits_shape", "mlp_capstone", "dlogits")
_check_tensor("mlp_dnorm_logits", "mlp_capstone", "dnorm_logits")
_check_tensor("mlp_dlogit_maxes", "mlp_capstone", "dlogit_maxes")
_check_tensor("mlp_dlogits", "mlp_capstone", "dlogits")


### Exercise 148 — MLP: output affine layer

**Purpose:** Learn to send logit gradients back to hidden activations, output weights, and output bias.

**Inputs:** `mlp_h` is `(4, 5)`, `mlp_W2` is `(5, 6)`, and `mlp_b2` is `(6,)`.

**Forward operation:** `mlp_logits = mlp_h @ mlp_W2 + mlp_b2`.

**Derive/do:** Using `mlp_dlogits`, derive `mlp_dh`, `mlp_dW2`, and `mlp_db2`.

**Ingredients:** Affine-layer backward; sum the bias path over four examples.

**Required outputs:**

- `mlp_dh`: the `(4, 5)` hidden-activation gradient.
- `mlp_dW2`: the `(5, 6)` output-weight gradient.
- `mlp_db2`: the length-6 output-bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex148_mlp_logits_shape`: predict the shape of `mlp_logits` as a literal Python tuple
- `ex148_mlp_h_shape`: predict the shape of `mlp_h` as a literal Python tuple
- `ex148_mlp_W2_shape`: predict the shape of `mlp_W2` as a literal Python tuple
- `ex148_mlp_b2_shape`: predict the shape of `mlp_b2` as a literal Python tuple
- `ex148_mlp_dh_shape`: predict the shape of `mlp_dh` as a literal Python tuple
- `ex148_mlp_dW2_shape`: predict the shape of `mlp_dW2` as a literal Python tuple
- `ex148_mlp_db2_shape`: predict the shape of `mlp_db2` as a literal Python tuple

**Next concept:** MLP: tanh backward.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_logits = mlp_h @ mlp_W2 + mlp_b2
# Visible forward tensors and shapes:
# - `mlp_h` is already defined by the visible forward graph.
# - `mlp_W2` is already defined by the visible forward graph.
# - `mlp_b2` is already defined by the visible forward graph.
# - `mlp_logits` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dlogits from Exercise 147.
print("mlp_h", tuple(mlp_h.shape), "mlp_W2", tuple(mlp_W2.shape), "mlp_b2", tuple(mlp_b2.shape), "mlp_logits", tuple(mlp_logits.shape))


In [ ]:
# Exercise 148: derive manually; do not use autograd in this cell.
# Define `mlp_dh` — the `(4, 5)` hidden-activation gradient.
# Define `mlp_dW2` — the `(5, 6)` output-weight gradient.
# Define `mlp_db2` — the length-6 output-bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex148_mlp_logits_shape` — the shape of `mlp_logits` as a literal Python tuple.
# Define `ex148_mlp_h_shape` — the shape of `mlp_h` as a literal Python tuple.
# Define `ex148_mlp_W2_shape` — the shape of `mlp_W2` as a literal Python tuple.
# Define `ex148_mlp_b2_shape` — the shape of `mlp_b2` as a literal Python tuple.
# Define `ex148_mlp_dh_shape` — the shape of `mlp_dh` as a literal Python tuple.
# Define `ex148_mlp_dW2_shape` — the shape of `mlp_dW2` as a literal Python tuple.
# Define `ex148_mlp_db2_shape` — the shape of `mlp_db2` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex148_mlp_logits_shape", "mlp_logits")
_check_visible_tensor_shape("ex148_mlp_h_shape", "mlp_h")
_check_visible_tensor_shape("ex148_mlp_W2_shape", "mlp_W2")
_check_visible_tensor_shape("ex148_mlp_b2_shape", "mlp_b2")
_check_private_tensor_shape("ex148_mlp_dh_shape", "mlp_capstone", "dh")
_check_private_tensor_shape("ex148_mlp_dW2_shape", "mlp_capstone", "dW2")
_check_private_tensor_shape("ex148_mlp_db2_shape", "mlp_capstone", "db2")
_check_tensor("mlp_dh", "mlp_capstone", "dh")
_check_tensor("mlp_dW2", "mlp_capstone", "dW2")
_check_tensor("mlp_db2", "mlp_capstone", "db2")


### Exercise 149 — MLP: tanh backward

**Purpose:** Learn to send hidden-activation gradients backward through `tanh`.

**Inputs:** `mlp_hpreact` and `mlp_h` are `(4, 5)`.

**Forward operation:** `mlp_h = mlp_hpreact.tanh()`.

**Derive/do:** Using `mlp_dh`, derive `mlp_dhpreact`.

**Ingredients:** Tanh's local derivative expressed with the stored forward activation `mlp_h`.

**Required outputs:**

- `mlp_dhpreact`: the `(4, 5)` gradient entering BatchNorm.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex149_mlp_h_shape`: predict the shape of `mlp_h` as a literal Python tuple
- `ex149_mlp_hpreact_shape`: predict the shape of `mlp_hpreact` as a literal Python tuple
- `ex149_mlp_dhpreact_shape`: predict the shape of `mlp_dhpreact` as a literal Python tuple

**Next concept:** MLP: BatchNorm learned affine.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_h = mlp_hpreact.tanh()
# Visible forward tensors and shapes:
# - `mlp_hpreact` is already defined by the visible forward graph.
# - `mlp_h` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dh from Exercise 148.
print("mlp_hpreact", tuple(mlp_hpreact.shape), "mlp_h", tuple(mlp_h.shape))


In [ ]:
# Exercise 149: derive manually; do not use autograd in this cell.
# Define `mlp_dhpreact` — the `(4, 5)` gradient entering BatchNorm.
# Predict every visible tensor shape before computing values.
# Define `ex149_mlp_h_shape` — the shape of `mlp_h` as a literal Python tuple.
# Define `ex149_mlp_hpreact_shape` — the shape of `mlp_hpreact` as a literal Python tuple.
# Define `ex149_mlp_dhpreact_shape` — the shape of `mlp_dhpreact` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex149_mlp_h_shape", "mlp_h")
_check_visible_tensor_shape("ex149_mlp_hpreact_shape", "mlp_hpreact")
_check_private_tensor_shape("ex149_mlp_dhpreact_shape", "mlp_capstone", "dhpreact")
_check_tensor("mlp_dhpreact", "mlp_capstone", "dhpreact")


### Exercise 150 — MLP: BatchNorm learned affine

**Purpose:** Learn to compute gradients for BatchNorm's learned gain, learned bias, and normalized hidden values.

**Inputs:** `mlp_bnraw` is `(4, 5)`; gain and bias are `(1, 5)`.

**Forward operation:** `mlp_hpreact = mlp_bngain * mlp_bnraw + mlp_bnbias`.

**Derive/do:** Derive `mlp_dbnraw`, `mlp_dbngain`, and `mlp_dbnbias`.

**Ingredients:** Elementwise affine backward and unbroadcasting over four examples.

**Required outputs:**

- `mlp_dbnraw`: the normalized-hidden gradient.
- `mlp_dbngain`: the learned BatchNorm gain gradient.
- `mlp_dbnbias`: the learned BatchNorm bias gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex150_mlp_hpreact_shape`: predict the shape of `mlp_hpreact` as a literal Python tuple
- `ex150_mlp_bngain_shape`: predict the shape of `mlp_bngain` as a literal Python tuple
- `ex150_mlp_bnraw_shape`: predict the shape of `mlp_bnraw` as a literal Python tuple
- `ex150_mlp_bnbias_shape`: predict the shape of `mlp_bnbias` as a literal Python tuple
- `ex150_mlp_dbnraw_shape`: predict the shape of `mlp_dbnraw` as a literal Python tuple
- `ex150_mlp_dbngain_shape`: predict the shape of `mlp_dbngain` as a literal Python tuple
- `ex150_mlp_dbnbias_shape`: predict the shape of `mlp_dbnbias` as a literal Python tuple

**Next concept:** MLP: BatchNorm normalization and inverse std.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_hpreact = mlp_bngain * mlp_bnraw + mlp_bnbias
# Visible forward tensors and shapes:
# - `mlp_bnraw` is already defined by the visible forward graph.
# - `mlp_bngain` is already defined by the visible forward graph.
# - `mlp_bnbias` is already defined by the visible forward graph.
# - `mlp_hpreact` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dhpreact from Exercise 149.
print("mlp_bnraw", tuple(mlp_bnraw.shape), "mlp_bngain", tuple(mlp_bngain.shape), "mlp_bnbias", tuple(mlp_bnbias.shape), "mlp_hpreact", tuple(mlp_hpreact.shape))


In [ ]:
# Exercise 150: derive manually; do not use autograd in this cell.
# Define `mlp_dbnraw` — the normalized-hidden gradient.
# Define `mlp_dbngain` — the learned BatchNorm gain gradient.
# Define `mlp_dbnbias` — the learned BatchNorm bias gradient.
# Predict every visible tensor shape before computing values.
# Define `ex150_mlp_hpreact_shape` — the shape of `mlp_hpreact` as a literal Python tuple.
# Define `ex150_mlp_bngain_shape` — the shape of `mlp_bngain` as a literal Python tuple.
# Define `ex150_mlp_bnraw_shape` — the shape of `mlp_bnraw` as a literal Python tuple.
# Define `ex150_mlp_bnbias_shape` — the shape of `mlp_bnbias` as a literal Python tuple.
# Define `ex150_mlp_dbnraw_shape` — the shape of `mlp_dbnraw` as a literal Python tuple.
# Define `ex150_mlp_dbngain_shape` — the shape of `mlp_dbngain` as a literal Python tuple.
# Define `ex150_mlp_dbnbias_shape` — the shape of `mlp_dbnbias` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex150_mlp_hpreact_shape", "mlp_hpreact")
_check_visible_tensor_shape("ex150_mlp_bngain_shape", "mlp_bngain")
_check_visible_tensor_shape("ex150_mlp_bnraw_shape", "mlp_bnraw")
_check_visible_tensor_shape("ex150_mlp_bnbias_shape", "mlp_bnbias")
_check_private_tensor_shape("ex150_mlp_dbnraw_shape", "mlp_capstone", "dbnraw")
_check_private_tensor_shape("ex150_mlp_dbngain_shape", "mlp_capstone", "dbngain")
_check_private_tensor_shape("ex150_mlp_dbnbias_shape", "mlp_capstone", "dbnbias")
_check_tensor("mlp_dbnraw", "mlp_capstone", "dbnraw")
_check_tensor("mlp_dbngain", "mlp_capstone", "dbngain")
_check_tensor("mlp_dbnbias", "mlp_capstone", "dbnbias")


### Exercise 151 — MLP: BatchNorm normalization and inverse std

**Purpose:** Learn to split BatchNorm gradients between the direct centered-value path and the variance path.

**Inputs:** `mlp_bndiff` is `(4, 5)`, inverse standard deviation is `(1, 5)`.

**Forward operation:** `mlp_bnraw = mlp_bndiff * mlp_bnvar_inv`, then inverse std is a power of stabilized variance.

**Derive/do:** Derive `mlp_dbndiff_direct`, `mlp_dbnvar_inv`, and `mlp_dbnvar`.

**Ingredients:** Product backward with batch accumulation, followed by power `-0.5`.

**Required outputs:**

- `mlp_dbndiff_direct`: the direct centered-value contribution.
- `mlp_dbnvar_inv`: the inverse-standard-deviation gradient.
- `mlp_dbnvar`: the variance gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex151_mlp_bnvar_inv_shape`: predict the shape of `mlp_bnvar_inv` as a literal Python tuple
- `ex151_mlp_bnvar_shape`: predict the shape of `mlp_bnvar` as a literal Python tuple
- `ex151_mlp_bnraw_shape`: predict the shape of `mlp_bnraw` as a literal Python tuple
- `ex151_mlp_bndiff_shape`: predict the shape of `mlp_bndiff` as a literal Python tuple
- `ex151_mlp_dbndiff_direct_shape`: predict the shape of `mlp_dbndiff_direct` as a literal Python tuple
- `ex151_mlp_dbnvar_inv_shape`: predict the shape of `mlp_dbnvar_inv` as a literal Python tuple
- `ex151_mlp_dbnvar_shape`: predict the shape of `mlp_dbnvar` as a literal Python tuple

**Next concept:** MLP: BatchNorm variance and centering.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_bnvar_inv = (mlp_bnvar + mlp_eps) ** -0.5; mlp_bnraw = mlp_bndiff * mlp_bnvar_inv
# Visible forward tensors and shapes:
# - `mlp_bnvar` is already defined by the visible forward graph.
# - `mlp_bnvar_inv` is already defined by the visible forward graph.
# - `mlp_bndiff` is already defined by the visible forward graph.
# - `mlp_bnraw` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dbnraw from Exercise 150.
print("mlp_bnvar", tuple(mlp_bnvar.shape), "mlp_bnvar_inv", tuple(mlp_bnvar_inv.shape), "mlp_bndiff", tuple(mlp_bndiff.shape), "mlp_bnraw", tuple(mlp_bnraw.shape))


In [ ]:
# Exercise 151: derive manually; do not use autograd in this cell.
# Define `mlp_dbndiff_direct` — the direct centered-value contribution.
# Define `mlp_dbnvar_inv` — the inverse-standard-deviation gradient.
# Define `mlp_dbnvar` — the variance gradient.
# Predict every visible tensor shape before computing values.
# Define `ex151_mlp_bnvar_inv_shape` — the shape of `mlp_bnvar_inv` as a literal Python tuple.
# Define `ex151_mlp_bnvar_shape` — the shape of `mlp_bnvar` as a literal Python tuple.
# Define `ex151_mlp_bnraw_shape` — the shape of `mlp_bnraw` as a literal Python tuple.
# Define `ex151_mlp_bndiff_shape` — the shape of `mlp_bndiff` as a literal Python tuple.
# Define `ex151_mlp_dbndiff_direct_shape` — the shape of `mlp_dbndiff_direct` as a literal Python tuple.
# Define `ex151_mlp_dbnvar_inv_shape` — the shape of `mlp_dbnvar_inv` as a literal Python tuple.
# Define `ex151_mlp_dbnvar_shape` — the shape of `mlp_dbnvar` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex151_mlp_bnvar_inv_shape", "mlp_bnvar_inv")
_check_visible_tensor_shape("ex151_mlp_bnvar_shape", "mlp_bnvar")
_check_visible_tensor_shape("ex151_mlp_bnraw_shape", "mlp_bnraw")
_check_visible_tensor_shape("ex151_mlp_bndiff_shape", "mlp_bndiff")
_check_private_tensor_shape("ex151_mlp_dbndiff_direct_shape", "mlp_capstone", "dbndiff_direct")
_check_private_tensor_shape("ex151_mlp_dbnvar_inv_shape", "mlp_capstone", "dbnvar_inv")
_check_private_tensor_shape("ex151_mlp_dbnvar_shape", "mlp_capstone", "dbnvar")
_check_tensor("mlp_dbndiff_direct", "mlp_capstone", "dbndiff_direct")
_check_tensor("mlp_dbnvar_inv", "mlp_capstone", "dbnvar_inv")
_check_tensor("mlp_dbnvar", "mlp_capstone", "dbnvar")


### Exercise 152 — MLP: BatchNorm variance and centering

**Purpose:** Learn to finish the MLP's BatchNorm backward pass by merging variance and direct paths and reversing centering.

**Inputs:** `mlp_bndiff2` and `mlp_bndiff` are `(4, 5)`; means are `(1, 5)`.

**Forward operation:** Variance is an unbiased reduction of squared deviations; deviations subtract column means.

**Derive/do:** Derive `mlp_dbndiff2`, `mlp_dbndiff_variance`, total `mlp_dbndiff`, `mlp_dbnmeani`, and `mlp_dhprebn`.

**Ingredients:** Reverse division by `B - 1`, square, branch merge, broadcast subtraction, and four-example mean.

**Required outputs:**

- `mlp_dbndiff2`: the squared-deviation gradient.
- `mlp_dbndiff_variance`: the variance-path centered-value contribution.
- `mlp_dbndiff`: the total centered-value gradient.
- `mlp_dbnmeani`: the column-mean gradient.
- `mlp_dhprebn`: the pre-BatchNorm hidden gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex152_mlp_bnmeani_shape`: predict the shape of `mlp_bnmeani` as a literal Python tuple
- `ex152_mlp_hprebn_shape`: predict the shape of `mlp_hprebn` as a literal Python tuple
- `ex152_mlp_bndiff_shape`: predict the shape of `mlp_bndiff` as a literal Python tuple
- `ex152_mlp_bndiff2_shape`: predict the shape of `mlp_bndiff2` as a literal Python tuple
- `ex152_mlp_bnvar_shape`: predict the shape of `mlp_bnvar` as a literal Python tuple
- `ex152_mlp_dbndiff2_shape`: predict the shape of `mlp_dbndiff2` as a literal Python tuple
- `ex152_mlp_dbndiff_variance_shape`: predict the shape of `mlp_dbndiff_variance` as a literal Python tuple
- `ex152_mlp_dbndiff_shape`: predict the shape of `mlp_dbndiff` as a literal Python tuple
- `ex152_mlp_dbnmeani_shape`: predict the shape of `mlp_dbnmeani` as a literal Python tuple
- `ex152_mlp_dhprebn_shape`: predict the shape of `mlp_dhprebn` as a literal Python tuple

**Next concept:** MLP: hidden affine and reshape.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_bnmeani = mlp_hprebn.mean(dim=0, keepdim=True); mlp_bndiff = mlp_hprebn - mlp_bnmeani; mlp_bndiff2 = mlp_bndiff**2; mlp_bnvar = mlp_bndiff2.sum(dim=0, keepdim=True) / (mlp_B - 1)
# Visible forward tensors and shapes:
# - `mlp_hprebn` is already defined by the visible forward graph.
# - `mlp_bnmeani` is already defined by the visible forward graph.
# - `mlp_bndiff` is already defined by the visible forward graph.
# - `mlp_bndiff2` is already defined by the visible forward graph.
# - `mlp_bnvar` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dbnvar and mlp_dbndiff_direct from Exercise 151.
print("mlp_hprebn", tuple(mlp_hprebn.shape), "mlp_bnmeani", tuple(mlp_bnmeani.shape), "mlp_bndiff", tuple(mlp_bndiff.shape), "mlp_bndiff2", tuple(mlp_bndiff2.shape), "mlp_bnvar", tuple(mlp_bnvar.shape))


In [ ]:
# Exercise 152: derive manually; do not use autograd in this cell.
# Define `mlp_dbndiff2` — the squared-deviation gradient.
# Define `mlp_dbndiff_variance` — the variance-path centered-value contribution.
# Define `mlp_dbndiff` — the total centered-value gradient.
# Define `mlp_dbnmeani` — the column-mean gradient.
# Define `mlp_dhprebn` — the pre-BatchNorm hidden gradient.
# Predict every visible tensor shape before computing values.
# Define `ex152_mlp_bnmeani_shape` — the shape of `mlp_bnmeani` as a literal Python tuple.
# Define `ex152_mlp_hprebn_shape` — the shape of `mlp_hprebn` as a literal Python tuple.
# Define `ex152_mlp_bndiff_shape` — the shape of `mlp_bndiff` as a literal Python tuple.
# Define `ex152_mlp_bndiff2_shape` — the shape of `mlp_bndiff2` as a literal Python tuple.
# Define `ex152_mlp_bnvar_shape` — the shape of `mlp_bnvar` as a literal Python tuple.
# Define `ex152_mlp_dbndiff2_shape` — the shape of `mlp_dbndiff2` as a literal Python tuple.
# Define `ex152_mlp_dbndiff_variance_shape` — the shape of `mlp_dbndiff_variance` as a literal Python tuple.
# Define `ex152_mlp_dbndiff_shape` — the shape of `mlp_dbndiff` as a literal Python tuple.
# Define `ex152_mlp_dbnmeani_shape` — the shape of `mlp_dbnmeani` as a literal Python tuple.
# Define `ex152_mlp_dhprebn_shape` — the shape of `mlp_dhprebn` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex152_mlp_bnmeani_shape", "mlp_bnmeani")
_check_visible_tensor_shape("ex152_mlp_hprebn_shape", "mlp_hprebn")
_check_visible_tensor_shape("ex152_mlp_bndiff_shape", "mlp_bndiff")
_check_visible_tensor_shape("ex152_mlp_bndiff2_shape", "mlp_bndiff2")
_check_visible_tensor_shape("ex152_mlp_bnvar_shape", "mlp_bnvar")
_check_private_tensor_shape("ex152_mlp_dbndiff2_shape", "mlp_capstone", "dbndiff2")
_check_private_tensor_shape("ex152_mlp_dbndiff_variance_shape", "mlp_capstone", "dbndiff_variance")
_check_private_tensor_shape("ex152_mlp_dbndiff_shape", "mlp_capstone", "dbndiff")
_check_private_tensor_shape("ex152_mlp_dbnmeani_shape", "mlp_capstone", "dbnmeani")
_check_private_tensor_shape("ex152_mlp_dhprebn_shape", "mlp_capstone", "dhprebn")
_check_tensor("mlp_dbndiff2", "mlp_capstone", "dbndiff2")
_check_tensor("mlp_dbndiff_variance", "mlp_capstone", "dbndiff_variance")
_check_tensor("mlp_dbndiff", "mlp_capstone", "dbndiff")
_check_tensor("mlp_dbnmeani", "mlp_capstone", "dbnmeani")
_check_tensor("mlp_dhprebn", "mlp_capstone", "dhprebn")


### Exercise 153 — MLP: hidden affine and reshape

**Purpose:** Learn to send pre-BatchNorm gradients through the first affine layer and restore the embedding-grid shape.

**Inputs:** `mlp_embcat` is `(4, 6)`, `mlp_W1` is `(6, 5)`, and `mlp_b1` is `(5,)`.

**Forward operation:** `mlp_hprebn = mlp_embcat @ mlp_W1 + mlp_b1`; `mlp_embcat` reshapes `mlp_emb`.

**Derive/do:** Derive `mlp_dembcat`, `mlp_dW1`, `mlp_db1`, and reshaped `mlp_demb`.

**Ingredients:** Affine backward, batch-axis bias accumulation, then restore shape `(4, 3, 2)`.

**Required outputs:**

- `mlp_dembcat`: the flattened embedding gradient.
- `mlp_dW1`: the first-layer weight gradient.
- `mlp_db1`: the first-layer bias gradient.
- `mlp_demb`: the `(4, 3, 2)` embedding-output gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex153_mlp_embcat_shape`: predict the shape of `mlp_embcat` as a literal Python tuple
- `ex153_mlp_emb_shape`: predict the shape of `mlp_emb` as a literal Python tuple
- `ex153_mlp_hprebn_shape`: predict the shape of `mlp_hprebn` as a literal Python tuple
- `ex153_mlp_W1_shape`: predict the shape of `mlp_W1` as a literal Python tuple
- `ex153_mlp_b1_shape`: predict the shape of `mlp_b1` as a literal Python tuple
- `ex153_mlp_dembcat_shape`: predict the shape of `mlp_dembcat` as a literal Python tuple
- `ex153_mlp_dW1_shape`: predict the shape of `mlp_dW1` as a literal Python tuple
- `ex153_mlp_db1_shape`: predict the shape of `mlp_db1` as a literal Python tuple
- `ex153_mlp_demb_shape`: predict the shape of `mlp_demb` as a literal Python tuple

**Next concept:** MLP: embedding-table gradient.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_embcat = mlp_emb.reshape(mlp_B, mlp_BLOCK * mlp_E); mlp_hprebn = mlp_embcat @ mlp_W1 + mlp_b1
# Visible forward tensors and shapes:
# - `mlp_emb` is already defined by the visible forward graph.
# - `mlp_embcat` is already defined by the visible forward graph.
# - `mlp_W1` is already defined by the visible forward graph.
# - `mlp_b1` is already defined by the visible forward graph.
# - `mlp_hprebn` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_dhprebn from Exercise 152.
print("mlp_emb", tuple(mlp_emb.shape), "mlp_embcat", tuple(mlp_embcat.shape), "mlp_W1", tuple(mlp_W1.shape), "mlp_b1", tuple(mlp_b1.shape), "mlp_hprebn", tuple(mlp_hprebn.shape))


In [ ]:
# Exercise 153: derive manually; do not use autograd in this cell.
# Define `mlp_dembcat` — the flattened embedding gradient.
# Define `mlp_dW1` — the first-layer weight gradient.
# Define `mlp_db1` — the first-layer bias gradient.
# Define `mlp_demb` — the `(4, 3, 2)` embedding-output gradient.
# Predict every visible tensor shape before computing values.
# Define `ex153_mlp_embcat_shape` — the shape of `mlp_embcat` as a literal Python tuple.
# Define `ex153_mlp_emb_shape` — the shape of `mlp_emb` as a literal Python tuple.
# Define `ex153_mlp_hprebn_shape` — the shape of `mlp_hprebn` as a literal Python tuple.
# Define `ex153_mlp_W1_shape` — the shape of `mlp_W1` as a literal Python tuple.
# Define `ex153_mlp_b1_shape` — the shape of `mlp_b1` as a literal Python tuple.
# Define `ex153_mlp_dembcat_shape` — the shape of `mlp_dembcat` as a literal Python tuple.
# Define `ex153_mlp_dW1_shape` — the shape of `mlp_dW1` as a literal Python tuple.
# Define `ex153_mlp_db1_shape` — the shape of `mlp_db1` as a literal Python tuple.
# Define `ex153_mlp_demb_shape` — the shape of `mlp_demb` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex153_mlp_embcat_shape", "mlp_embcat")
_check_visible_tensor_shape("ex153_mlp_emb_shape", "mlp_emb")
_check_visible_tensor_shape("ex153_mlp_hprebn_shape", "mlp_hprebn")
_check_visible_tensor_shape("ex153_mlp_W1_shape", "mlp_W1")
_check_visible_tensor_shape("ex153_mlp_b1_shape", "mlp_b1")
_check_private_tensor_shape("ex153_mlp_dembcat_shape", "mlp_capstone", "dembcat")
_check_private_tensor_shape("ex153_mlp_dW1_shape", "mlp_capstone", "dW1")
_check_private_tensor_shape("ex153_mlp_db1_shape", "mlp_capstone", "db1")
_check_private_tensor_shape("ex153_mlp_demb_shape", "mlp_capstone", "demb")
_check_tensor("mlp_dembcat", "mlp_capstone", "dembcat")
_check_tensor("mlp_dW1", "mlp_capstone", "dW1")
_check_tensor("mlp_db1", "mlp_capstone", "db1")
_check_tensor("mlp_demb", "mlp_capstone", "demb")


### Exercise 154 — MLP: embedding-table gradient

**Purpose:** Learn to accumulate every context-position gradient into the correct embedding row, including repeated IDs.

**Inputs:** `mlp_C` is `(6, 2)`, `mlp_Xb` is `(4, 3)`, and `mlp_demb` is `(4, 3, 2)`.

**Forward operation:** `mlp_emb = mlp_C[mlp_Xb]`.

**Derive/do:** Derive `mlp_dC`, including repeated token IDs.

**Ingredients:** Initialize zeros shaped like `mlp_C`; flatten positions if useful; use an accumulating indexed write rather than replacement.

**Required outputs:**

- `mlp_dC`: the final `(6, 2)` embedding-table gradient.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex154_mlp_emb_shape`: predict the shape of `mlp_emb` as a literal Python tuple
- `ex154_mlp_C_shape`: predict the shape of `mlp_C` as a literal Python tuple
- `ex154_mlp_Xb_shape`: predict the shape of `mlp_Xb` as a literal Python tuple
- `ex154_mlp_dC_shape`: predict the shape of `mlp_dC` as a literal Python tuple

**Next concept:** MLP: gradient invariants.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# mlp_emb = mlp_C[mlp_Xb]
# Visible forward tensors and shapes:
# - `mlp_C` is already defined by the visible forward graph.
# - `mlp_Xb` is already defined by the visible forward graph.
# - `mlp_emb` is already defined by the visible forward graph.
# Upstream gradient for this stage: mlp_demb from Exercise 153.
print("mlp_C", tuple(mlp_C.shape), "mlp_Xb", tuple(mlp_Xb.shape), "mlp_emb", tuple(mlp_emb.shape))


In [ ]:
# Exercise 154: derive manually; do not use autograd in this cell.
# Define `mlp_dC` — the final `(6, 2)` embedding-table gradient.
# Predict every visible tensor shape before computing values.
# Define `ex154_mlp_emb_shape` — the shape of `mlp_emb` as a literal Python tuple.
# Define `ex154_mlp_C_shape` — the shape of `mlp_C` as a literal Python tuple.
# Define `ex154_mlp_Xb_shape` — the shape of `mlp_Xb` as a literal Python tuple.
# Define `ex154_mlp_dC_shape` — the shape of `mlp_dC` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex154_mlp_emb_shape", "mlp_emb")
_check_visible_tensor_shape("ex154_mlp_C_shape", "mlp_C")
_check_visible_tensor_shape("ex154_mlp_Xb_shape", "mlp_Xb")
_check_private_tensor_shape("ex154_mlp_dC_shape", "mlp_capstone", "dC")
_check_tensor("mlp_dC", "mlp_capstone", "dC")


### Exercise 155 — MLP: gradient invariants

**Purpose:** Learn two quick sum checks that can reveal mistakes in the completed softmax and BatchNorm gradients.

**Inputs:** `mlp_dlogits` is `(4, 6)` and `mlp_dhprebn` is `(4, 5)`.

**Forward operation:** Softmax is unchanged by a constant row-logit shift, and BatchNorm centering removes a constant column shift.

**Derive/do:** Compute one sum of `mlp_dlogits` per example row and one sum of `mlp_dhprebn` per hidden-neuron column.

**Ingredients:** Reduce `mlp_dlogits` over candidates and `mlp_dhprebn` over examples; both results should be numerically near zero for these shift-invariant paths.

**Required outputs:**

- `mlp_logit_row_grad_sums`: the length-4 sum of logit gradients within each example.
- `mlp_hprebn_column_grad_sums`: the length-5 sum of pre-BatchNorm gradients over examples.


**Required tensor-shape predictions:** Write these before computing forward or backward values. Every listed tensor is part of this exercise's visible graph.

- `ex155_mlp_logits_shape`: predict the shape of `mlp_logits` as a literal Python tuple
- `ex155_mlp_hprebn_shape`: predict the shape of `mlp_hprebn` as a literal Python tuple
- `ex155_mlp_logit_row_grad_sums_shape`: predict the shape of `mlp_logit_row_grad_sums` as a literal Python tuple
- `ex155_mlp_hprebn_column_grad_sums_shape`: predict the shape of `mlp_hprebn_column_grad_sums` as a literal Python tuple

**Next concept:** completion and comparison with the lecture notebook.


In [ ]:
# Supplied visible stage: the complete forward graph was defined above.
assert "mlp_capstone" in _REFS

# Forward relation under study:
# sum mlp_dlogits across candidates and mlp_dhprebn across examples
# Visible forward tensors and shapes:
# - `mlp_logits` is the visible softmax input from the forward graph.
# - `mlp_hprebn` is the visible BatchNorm input from the forward graph.
# Values checked in this stage: learner-computed `mlp_dlogits` and `mlp_dhprebn` from Exercises 147 and 152.
print("mlp_logits", tuple(mlp_logits.shape), "mlp_hprebn", tuple(mlp_hprebn.shape))


In [ ]:
# Exercise 155: derive manually; do not use autograd in this cell.
# Define `mlp_logit_row_grad_sums` — the length-4 sum of logit gradients within each example.
# Define `mlp_hprebn_column_grad_sums` — the length-5 sum of pre-BatchNorm gradients over examples.
# Predict every visible tensor shape before computing values.
# Define `ex155_mlp_logits_shape` — the shape of `mlp_logits` as a literal Python tuple.
# Define `ex155_mlp_hprebn_shape` — the shape of `mlp_hprebn` as a literal Python tuple.
# Define `ex155_mlp_logit_row_grad_sums_shape` — the shape of `mlp_logit_row_grad_sums` as a literal Python tuple.
# Define `ex155_mlp_hprebn_column_grad_sums_shape` — the shape of `mlp_hprebn_column_grad_sums` as a literal Python tuple.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks predicted shapes and values without showing private reference answers.
_check_visible_tensor_shape("ex155_mlp_logits_shape", "mlp_logits")
_check_visible_tensor_shape("ex155_mlp_hprebn_shape", "mlp_hprebn")
_check_private_tensor_shape("ex155_mlp_logit_row_grad_sums_shape", "mlp_capstone", "logit_row_grad_sums")
_check_private_tensor_shape("ex155_mlp_hprebn_column_grad_sums_shape", "mlp_capstone", "hprebn_column_grad_sums")
_check_tensor("mlp_logit_row_grad_sums", "mlp_capstone", "logit_row_grad_sums")
_check_tensor("mlp_hprebn_column_grad_sums", "mlp_capstone", "hprebn_column_grad_sums")


## Completion standard

You are ready to return to the lecture notebook when you can restart the kernel, run all completed cells, and pass all 155 tests while explaining aloud:

- why every gradient has its forward variable's shape;
- why reductions broadcast in backward;
- why broadcasts reduce in backward;
- why shared variables and repeated embedding IDs accumulate contributions;
- why only target log-probabilities receive a direct NLL gradient while all logits can receive a softmax gradient;
- how BatchNorm creates direct and variance paths;
- how gradients travel from mean NLL through the entire MLP to every parameter.

## Official references

- [PyTorch autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)
- [Broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)
- [`torch.matmul`](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
- [`torch.nn.functional.cross_entropy`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)
- [`torch.nn.BatchNorm1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html)
